# TAS v7 — CPU preparation / A100 experiments

Upload this notebook to Colab. Use **CPU → Run all** first, then select your
**A100 40 or 80 GB → Run all**. `PHASE="auto"` selects preparation on CPU and the
study on GPU. Models/data/caches/results live in the same persistent Drive
folder across sessions and future notebooks. CPU/GPU environments are
recreated locally; the notebook kernel is left intact.

Bring **only this notebook**. It clones the fixed public baseline, verifies
and restores its packaged experiment modules, then imports them normally.
The editable implementation remains in the experiment branch; the
compressed package is generated from its exact commit. No separate source
upload is required. The code snapshot is also saved to Drive with its hashes.

Automatic flow: blocking P0 audit → 30-image smoke → 240-image screen →
predeclared shortlist → confirmation (3 seeds + greedy) → controls → P0-D.
Final remainder and P3/P4/P5 remain disabled by default. A session
budget saves progress; the next Run all continues completed stages.
Shortlisting requires the predeclared coverage guard and a nondominated
observed hallucination improvement. If none qualifies, the study saves its
comparison and runs baseline perception diagnostics, then stops dependent
stages without forcing a finalist.

Full A100 weights have not been tested by the local CPU suite. P0 rejects
baseline mismatches before bulk generation. No scientific gains are assumed.


In [ ]:
# Edit these settings once. Reuse PERSISTENT_ROOT in future idea notebooks.
REPO_URL = 'https://github.com/RachalHys/Tag-And-Segment-AGLA.git'
BASELINE_COMMIT = '624f74501bdafe9b7d6a05ae739d12b422ce8737'
RESEARCH_COMMIT = '7e287723ac4119b589457ca48282b105b0c3e8a7'
PERSISTENT_ROOT = "/content/drive/MyDrive/TAS-Shared"
LOCAL_ROOT = "/content/tas-v7-work"
PHASE = "auto"             # auto / prepare / run / score / check
STAGE = "flow"             # flow / audit / smoke / screen / confirm / controls / diagnostics / final / external
MAX_SESSION_MINUTES = 180  # planned stop; run again to resume without changing the experiment
SEEDS = [0, 1, 2]
REQUIRE_A100_80GB = False   # supports your A100 40 GB; True enforces at least 80 GB
AUTO_SHUTDOWN = True       # flush Drive, then release runtime after this session's work
RUN_CONFIRMATION = True
RUN_CONTROLS = True
RUN_DIAGNOSTICS = True
RUN_FINAL_REMAINDER = False
RESERVE_POPE_IMAGES = True
DATASET = "coco"           # AMBER/POPE require STAGE=external and explicit verified manifests below
EXTERNAL_MANIFEST = None
EXTERNAL_IMAGE_ROOT = None
METHODS = []               # empty = exact initial 14; external runs should select frozen finalists explicitly
BATCH_SIZES = [1, 2]        # conservative 40 GB candidates; measured, checked, then locked
ATTENTION_BACKENDS = ["eager", "sdpa"]
LAST_POSITION_LOGITS = True
SPLIT_SEED = 20261021
COVERAGE_TOLERANCE_PP = 2.0  # approved absolute percentage-point loss vs harmonized TAG; freeze before outputs

def release_runtime():
    if AUTO_SHUTDOWN:
        from google.colab import drive, runtime
        print("Saving pending Drive writes, then releasing the Colab runtime.")
        drive.flush_and_unmount()
        runtime.unassign()


In [ ]:
# @title Versioned experiment source package — generated, do not edit
SOURCE_PACKAGE = {'schema': 'tas-private-source-v1', 'base_commit': '624f74501bdafe9b7d6a05ae739d12b422ce8737', 'source_commit': '7e287723ac4119b589457ca48282b105b0c3e8a7', 'sha256': '14347b3df80dd276259653c0f81d31785281ae703574422fd476363cc82b0a43', 'files': {'AGENTS.md': 'b796e9c1c6659dd677f2aa0f223c8584854a37dc3f82c219c0e72807bfd7ccd1', 'docs/TAS_V7_COLAB.md': 'f72a6bc2a0d0403bf316e6967cd40a6d97f8a6d57739eb05d93c4bd626f0598b', 'experiments/__init__.py': 'fe7a0225cf031bed0d2204dbb763b431d530abac99b01ea33faa4d0f6b244526', 'experiments/tas_v7/__init__.py': '5ff13022425f78d78413600e40c3941e750ee983755c9b17a5d564609400edf7', 'experiments/tas_v7/audit.py': 'f6ea192ba422b551601e3c558e16429b5b7670b57ef4cf42fc42c7428df42a3f', 'experiments/tas_v7/cli.py': 'b16c5fbd3f0893697b332af31203fa97b686537f83b4663e71ab3b40861455a0', 'experiments/tas_v7/diagnostics.py': '2a1a56e626656d6172bf0c9355a5db5b60519f0ac82350f5003cf3416cd1f77a', 'experiments/tas_v7/evaluation.py': 'f3f8cac6999331c5dc67c0708c628b0cc81a25bf6045224b0c124690cba7c2f0', 'experiments/tas_v7/external.py': '7e9d9eb0f91f1d424c1bf34bcdbd45587d16faa8d2148eace5dec068cd97bc56', 'experiments/tas_v7/methods.py': '2913cc58e41e1a726d7e1ac8c7b2711a5259afa5dff4fa7e6f0d3fbf4d7ad9f5', 'experiments/tas_v7/prepared.py': '81b5d30c97e731f8e05bab40b944835333ba937d041c4cc135e601540d93fecc', 'experiments/tas_v7/report.py': '8bd6aa025a3ef41255b24cfca247f5d9d1779a1a6b32d5b2f246bd39bf7a0994', 'experiments/tas_v7/study.py': 'b48e222fc337c42a8a1e29fef4efb2ec7f40af10cc9985915ab5d8bea563ad5a', 'experiments/tas_v7/study_diagnostics.py': 'fe252e3e95b583423d59157d01364560a557f70e2ca556868ea00bb174f9bca1', 'experiments/tas_v7/views.py': '9bffe95f8996c3462b7e82b3a51587b9629d5dfe6d45c7654a17989ffb160775', 'requirements/colab-amber.txt': '7ee7be8843a02ea16d1333e4271ca0b8174445aab6867efcd46e57739a95389a', 'requirements/colab-base.txt': '992b028d26a85bd15ccd29321a8192bbc81efb6696df38eb834820f06377a9a2', 'requirements/colab-perception.txt': '0e98ee13388aa50ad35045d981a278d347fb0c56f1634fca4b9e5567ca6aa03d', 'tas_runtime/__init__.py': 'ebfdfb5371eb85b8640ab449c246efc862a11ea70e142718bbef4c9f7b5deca5', 'tas_runtime/artifacts.py': '37e4bc9271a13ca334df828af92b58e425706d0164f9f30384c81e8dde7b2c37', 'tas_runtime/assets.py': '675315d0c9b6d195e7941829b0bc93d3101a34c8f5205d9f3aa6fb9e0eeb3538', 'tas_runtime/attention.py': '94585b4980179e09acba88e0b3492c2161d94904faa7783f6f5a8391c51854fc', 'tas_runtime/bootstrap.py': 'd2da9cf6414a509ecb7f3d8e2d50861fb15723a14496423e33f50a07de374f69', 'tas_runtime/config.py': 'a58903c262e490dc90a3b1cb9ded9b6bf1c18c018a1592f4c9c2bd19c6aedcf6', 'tas_runtime/data.py': '39cb6a954ddf713a622630c584087fc3f5e8e5cd4da0d9a2377b7a7aa1be2523', 'tas_runtime/decoding.py': 'a5a370f0345e5ed6fbdbb612f16b9b285ab287bbb6241dfeb39189663ebf88b5', 'tas_runtime/perception.py': 'bf7179deaa1bcc8421d6c6e989554399155c1e391fd16c0be57185e43871e025', 'tas_runtime/persistence.py': '9fef2d9912d19ae2f20964167e121f2874e92ae3e3e695f3c64e96cbf1956a1b', 'tas_runtime/profiling.py': 'a77bb6b706612bcc2ecdf45a12756146f6db0b67f5a9eabc58f17ad7b9d8c4d8', 'tas_runtime/runtime.py': '068acf5e27671c6b1131374ba167cc464c86f74816a2553beb1758bac11d497a', 'tests/conftest.py': '5b20fdc9ab1817a64ddada47c22d695378148741eada4f98b1a745a49f7c73d7', 'tests/test_artifacts.py': 'b21d651d6110626e3e99471230b5d7e9386b91f30ebaf035ef8e9c5427d9f506', 'tests/test_assets.py': '6b51775e20a03a2c9384bc9922285075bd960131ae9368bfde5f73fec98b6753', 'tests/test_audit.py': 'c6fde3cbbc8ad8394a0456390e4b4f350871eda9fc7efa1fd2f30eb50bbb446c', 'tests/test_bootstrap.py': 'df0ba4a7c785e82882635fba952b970f3356c51779c281def179e96bb1f68672', 'tests/test_data.py': 'c55b36a369a9da72cfe17fc8c88b478cde2ae3d466bbbca67b60013ba18e92e3', 'tests/test_diagnostics.py': 'e5cb5776067150a2b1b6aff62102f7e881d0d8d88ee6a53485265478b2e02874', 'tests/test_evaluation.py': '4a8e77ae0c9e4b2723fb1fd1a40f42e3d5c4e15075ea2cc2e2d54dfd350da98d', 'tests/test_notebook.py': '3ebdeb421670b43f6014aea1256ad12a2fa4425ca6bd3e85436580d2e0d7e739', 'tests/test_notebook_delivery.py': '377d9a8761c98b6ba6bcb2752d6f7549bb89f49feb0b109678ffb3ef5fbe85e6', 'tests/test_persistence.py': 'ae9b71a14af81561373fb273e58b773f8ced8fb8e4189a1206742cdbc682ffa5', 'tests/test_prepared.py': '1cb769c861c2f06cff94d17bd5027354a94f0cc8baf631ed3b8cc1019b797a73', 'tests/test_report.py': '65f0088060033ed91f2649124f7c11c7554154eedd3312b16cd7f3992113f60f', 'tests/test_runtime.py': '3db34ecd2dc499a6d56257bf95be6587b5d877fbd5e4add76341b4a51c285321', 'tests/test_science.py': '7303f63cd2f15c550ce45d78c5b6e904fe148a3d58e59f103239d84bb9ef2ea0', 'tests/test_study.py': '6d6783c7f216802eda1b1768353a1f7f780718728f57dc5e3e18e068fb6e7c80', 'tools/build_colab_notebook.py': 'a2fcbea25cc31bb6db0f5dc4448c267b88dd00d80fe969127b40858a58507e38', 'tools/notebook_delivery.py': 'de704ff351f24c0ab3cf43a892e2aeeaef3d5d5bf87cd047f14ebe985886e8ef'}, 'archive_b85': 'P)h>@6aWAK2mk;8ApmejXQzq-004Fd000R9002QpMNU*xE^TCOR7-E;HW0r1R}9cYTOgJa^bnxiE#OVIX_0hmbe&v^!lJ0L&6*+=KAd=eeZQgX*jw}xOA0yjy=K0ldGP<RPPERZQ%>__ve+;!W}j(yP5e4=Fg?3OCQ|2hmS^tBIQk2(vBtsW)AjP^kB8ZZs_*{(?*G1iUrl~Xk>>lf@s3PCSe7~}8N6Dv<0u82q~bDUm(Z|oNV$%h6M$IDzjxPl^KO!W@Wxx!lJkjM@3&Ia2lP<1GOq&g>n7Uwy-5|_GY=%!eK2d|l$|uZQ+A2kb#p=EORFMV<BlMD-?62RSH>AB7Af!$e3!LpEmxDtlpZ6OswM)J=yRa@Lv?jYIJ+{I|4Xp~?i5o{LqjVcaNooUok?ENtrvp07?ZNLbg;ZudJhE8*@Qw+v`c=}<J(AE4vuYwN}oAV^ZWea!|dk%<EPvAv-NuH)GMdf02y-&Y?@<)0WSuKb0WP_4$7=cw_~v>Jt>QPBrfNu`Sh^3U*66Zv-kC&(xy(n9-xW}+||wwG>rVGy*zPDB!41JpNpgb*PKqkn*sU=!|5}fDRibqq~yJgW0|I+MJbmdyF#%Z&Zv|FLl<S~;Xysk4z`Jb2eF_ftungk4eRaTjg*BPck@SKc*l-|Fcji%xe~f(a0`mM&Bk_1Es8IXC#zvk)9IKq`swTV9X|iuh%GW+>qD>Z)M!w83s*pKqtYR-u=8vJ519h5Y;kVXF+d0($~ErE?>H3vgK-Ykhum7D=@{sDB0&_Jgl8VCSDg&DWA3;U5#Huwe))Fm0&soLDeNy0nL*~P$jYy*^Eh21Xr$J9y+w2?x|u%$8zxaqk;PGR45$it;-=UzJ=p;A!KA&IF}u3N#Be}Pds#;UYK|Hlc1)|_zpx|Z5YSSMn2_lp4aSmVRi;pZ%**lXzQnyGnnf&-*!V79U}A<mntvh@84Z)s5h}6h82AOK7@W1t2rv@{L@o-bUjp-&F)VQdNx(~N!$r7A0TxlqXwSz>KbEs<z{u@j4fcu3R`x1@!SUY9l)QLSEtW_|U#OrZK`3_&>x!(8k<cXONFT4JzaGJ&ljgGhu+#wM&CD_epnMPwPg+5F{6bX#C_w@mS;JvZZFZfcOI88e-C=10-=8o*1-&^0YTT)E4`g~@-mrptg%yVtaEcfV>j7C~D0kxm!O?2+FHlPZ1QY-O00;m803iT2R)1ts3jhEF761Sg0001FZ)0;WR6$c;RySWmPfS5VE^TCeSY315xD|cpufS{`Ht|TL>^Pfr-Rz8P%TBtEt@>j-ZDz-WNJv5r0Sp1!R-XDF`Vad{dhP`%$=jK}B(z8X_v74i&&4ZoJH8Q*??tP;l5MpV)&(_l?jg%=mrAHw2RSo%cy3gySIP!axmpSBg|njE{#*p<hmY@vl{0cy=7LuDd(qc2GqSVQQnlhqhoz8Wp{#0UQ1t*?t{Pb2RgK4HE!<p$rSnQm*ZALxm8(0Wd@erYc;l?{S-}2t{fre&H#52tGwGGlRtafqawj<aBj~do+tWrucEPP=(3Lc1oo83ptohT8#a5{rtDTrB(KW`&8Ykh;6LK%lvRALfb!UaP@s}*y&&3^oC(3>kUm&>mkMHl}BHgU*tYq_Gsj32Eu(x+Iz1!OpZD#{4%(H`BOeqq0L;J%#yrm_uc<EaGU5V3HKPvugZuDXqXt9T#b2!XS7}+jVCWNTnlO>Z7xGzYW@q=_Se5qxSyz%A5l)~k@wxYmb4=*3tS=yrfvgY&1mEp}2uj?|;-sa*ay0Eu5-XD#`;Yhevd>D!I<GsBBSz%v#u#npR5fs)3yp1_JpWe}N%0BAWS<2O5o5Z=MoEgX28VGg?@KfUALjE@HgBKG|v6znX>@ep9@xemQ18^tdz4sb;QkG(J0@N8r!N4J%37&TVRw~s5`mB<`E60RvYgGq;=HgeS8n)R>9E#7q_JP3FIu{5L^DcA<qpp?Qc<X_gQ#F@-4E7P__2jR2pRXtP6u<iqqx0jBXVQ2T0a6~1Z%;nm-+cbZ<mTho{ed|6y%b0=pX+8`wMbFchyMI{10Zoh$!}2&0mWQlFO5vY2|N;(@kkDP)kvy6z#e{b-j2AWW{C2GwhP1%kVEwgAG9$EH!(u<R;lls+EP`=!D<|;5Ve$zjd4#x24R%+gph-GWFf7J*cHB!l>#byhg#PymM~@i{eS-T@BI(-H3DBfG)~*#v&M9Oi`SH^ji;P(DpwSDt0Kng7-`tv)oU^9>V*nKLSYFbvL&$YDRIJgv!->xy+0D2oolQ85*t0a_JPk?Dcbru^uWo*SfB=K@!J>d?HaL?>#SAZI*oj{Dlr;`FV-PN8yho2@y@ObO$QEJcJ=|GD#v%XSNAuc?ru-7{&xBCw(XGrIou`K#aKm`DU>PV8?*-zInYWYDrL+d)3!3m8IJD(-xz5v&P^d8n<+w?G3X<A{fJtc&vm7x$-b67kUA^j^1{A;Q~v&X8A9WKE{X-1yPM^p>tf;D0+65$3jQ}ZXM6$t%!(CiE`1Q+75%Qw`c0nQsL(a+*cLsiZ5@(=3UDz1FWmpuQ6N(oda6U_8k9GMJI^Lt;=qfZ8DqEyVjGxk#y3ae1ek~bQuiU)c|_7D7VK<|K$ctfD{3tl)}bmYU+mCTQTBEvGh>sOY<tffWszmaI|VlUMfjC_P~!F5k<cr-P+;i&L$Ole`r5zY4=O;`iq{8+qXBcC2h{nEq;}>*7M@&00cgn6i^>7YEwwiU7F&-75%|0mz^8Ir;O~0P`_$@D8P|}F8J30?d8FT`qv5I8@m9Qk|1MeiCKqRnyH-)IDMTa)nlL1mpS~@ohsE^WK%AWq&j;C3*R`Tv8HlzlnH9rMgp3mbPgy<8$T@01C|;xnOdBPLY>+TYg-8Dj<RnVKF!8Q*phlSjAt(^c-ZsEU5!r*Z#0vQUBiC^65e15O#(56{!e1W2Kw?+UqPNzNL+9dkd2;pJ<a&HQxxc--z{BOq<bFE+c#x0EfsixrOczu(Z)L?H9_6EhH>^z(6_9rumKG<U#-Fb>`eVjV9x-H(7!<Jz2%T6}yA`5J{LEykN%2q!f-^ANRxG>bt^3Z=%n}<=qd`!^x(8rHed^@^4@{e*5pBDH_|^d}b4_q-RfDi{KC+F3+UbG20N99wqn8-trjQUQdI)7qkfils4^e%^#I*Sk>Al2q)c9mnO!m1cl1CSl^YO{wsk(y$dF0|!J#FH_M@>e<PgpVh#A`l{ufJSfqEVh6Wuzm?NIDgbxZeX|1QPGi+=%!oMhmnY?Bq!IanmRLi)?v@Vtkutd~)k%a;6Q%7w#@81!vQ@2S?&b5y$vc1Nd?Or<|%fa)_|exjv>Km4$Ms>=p_Jf(ErfnC==1e-ec)D7fIxO_Z&c@=>EhrmV&e%rUgt|Fm6kt^OH>ooiBoF%?y=ttnp=*@`4}zXH8Lu6>lyNX2!6L8}5eGnX2%B787rq0j-Fg=^QMFCoY<Wfx%yZqX}dRw(zFpsbOaoMWySLj43ETG=E50L7=`2jr_(HykGQjA9z1@z$6iIwEo2BV|;3?!8q2Zlkmr!qBd@++!`gA;M0YhALzwB-B9Ql8zk14L@U(j)XHrSur|(gVBDkL>E+_KGlbS5Rzz`Rz4y!G!f<5=}u)35UHp&sRY=FuIoZ9juq$Aa|+fdagu}r?M$*JKwvwM#)}yRty;#^A?b?_z93$dQ1A@+Zbq1dj69LCFu8_9gs)u9^rCa1GB+a_h*eLsQN6J|@+ctuicSxw9<_Rk9qNR}xg>m}lC4yh4MmB8?mqSHqWqagCuItjVW<`kHBO-33c996pvDm5xdYP1jfr5bla9`5)9ozgX=225M0=Apz*=AB0|F;ELj|~WTawu!Bi=1(j!v8Uo`uGJueYHzurGAx1JS*t2@~Cd>SJdb1R&juv5z*5x}&5iw*cLjkU<K@(jz&~57spNe{_r=NrsXuagY+;$_5Q9pzP!1z*962GJ_Z6M&jRGT!D9=T1~o*r720*f)8MDN21_b9fR`GgSb(o0r_pdKg#!i+Lbpf@NON=^auH2zAp;&5&RV~il(*yC;Uj|dJCj^aLft-5Qq8OclklSk%kYdwaSnQM$I{7C{&t0U7X5@W<VHaYJD-Y<xzHW@p&pZGqJ!#qgjc15`UaGb{}ntKT004#=dk&65@bD%Fc}R%W;n@z9`V~J2HuIO(WUyhGP=>ra>WIX^@;J*AS8Hiwv03AtT5)@?gk4pd2q2C3C+bCU!51Rof*H6aEkqUI@v}4KuAg5*X)q^k=drkl;=<z8!3K$MYM-%`e*u@##9|4)oFL=dA>9)||!MwTl9heuzpTSQHJ_G#Fq=nubchcr(bpc8@(m79C{5cAMIvge1kx<bGLnj}YpXdcLO<uwZ8&2FX|Uo+b!l|6b7Zwxmq9JNIGF;{d?J0@o%MV)43|%WwHItPCcB><m;F8x}}&{N;FZT}-d0lm4DUls6D$laD|2c=(~@%LSeWfutm#8lsJm{pIHB@}gjhN9Bj5-`r9p&6IRC?a-nNr1XXk2}FeAjnfQNwp_`UCidM?a&v7F$&ZGotCOod8vUc+dVxOJqJF!6UURKpkmSh{m%(GQnxyNdW0f4BA!uB+d3X*bH4{Q-r05H}`AA^G&{&ob70c}M;`Z13)A8;2V?o{)!4=82x&T3mm`5>O(G$&2^beH%w`}_+){~Ox5yNj{hw*o4lCHt1TzPrF_Wv@k7!D)fia*EuhM}gL7TSzUH@*FZT!|t-n$}3Z0_6Ukcr@JsK@R=lf)4RhW-dsLkf_9W%XAaCm9czPNPLpkc!Y2z3mTS%U&EN^!3##=QBbq^AVc!(tTSe~Q?XnF1z)>t6&S7JME3}oV^zdO4~8fav(A!SPxn}?6<3{)r$qpwQ$>w)4jM^>G3(4je^-kAkBBli1C4LMfEcC$s-5<}J79o$DkG7|O$hmX5`J_`1(g36P)h>@6aWAK2mk;8AprW+yJ1HF002$^000*N003opaAk67ZDnqBb1z?CX>MtBUtcb8c{Rzw4S+BR1<<=w5C(7omv9AQe=4!G4Uoq1J^8MWnNQax3Bo(!YUoK*u16~3AhkI>wO7hp;}7asaB4$Jkd1;CJ2P(|P)h>@6aWAK2mk;8Apni)N%*b+0080u0015U003opaAk67ZDnqBb1!sZb6<8hFJE72ZfSI1UoLQYMUJ};!Y~j7dw<2sHKe7YpokC!4?#e5=!g$u$@U@J3FPl_XsBm)v|8(Cb;7Vf<ir%)4%WXMdk&ehl~5_wbOphS=^dv(MmN4BPb|$4B%^g?eS$pTnNlCwOmdXrr&qvM3_JB^`|NT81%vhB3}^3=qAh8P3@2pDP=XtAA5r??54ZjF1&&6Jg~6~d^Ga#0m8!3Y`ncKFyXJPOkEhLkhb45TG#?gPeF0EQ0|XQR000O8001EXOu{dX<Prb?rAPn(8vp<RWq5F9a%pX4Zgg`mbYXK}b~i6!b!2IDE^v9}Tw8PFMs|MJujs%J25bnDVrL(y;#Q5bnza+hGwa>*Zo*=rnq-rL1p-)Hq@h*%?>U!lpc@yG(j;4{3Li9Nqc5k=<vW*djOTgZrTO|H$#&x7wOHS)^+P3Owh>b7m8_alZG`%y)=iz{*=3QZ$@(CQB+YA)XF{&)MyBFPCA<5&iagJ|xY(BYUc~XXsbO#&i)3HqWesy?c`b34>f%DT)yZCQ|BbBWI+azWDl?!|MJm^->ztSCdp&(EtGH}393RQDPPRZuJ5bj7K3T`VR(Uq;y;19YgS*w8{R;t~KVYl+P!M?`HBm<uuS%H#S9IeRXi%HDDAhLk6sP$vsr8c*x|f@{Qfd=k1jBWUGT$btwXn(_lQPftDy!qY%#y9D>QGb%SPhz!-;~~Jv{&_gzG-*2Z7O9}7W^z;9pYpY3gA~$B`a8_KERySgPsA%EhGw3PoYO`l|&K2G)3}BrSXG0z*p{T9>yzKBiAoBYHL`MGn;(9wNR|{Y@6&1HKF(_6uS}_ahSiCX$2kj@>86tr?}1^R94M@$}<(rh;2KSsgkrp%=J7`*QPlOgk-LWSZes`P3`^^d1<W>xevp3j#}*t>;wCqTivZ@t8ZS!h0_Zg+~*Jd=MUZUO&(WrU!=}TZ6jG8*oo#}H%#RflMRBd=7#kVHG!^_k=-E-x`Bn<wC(fZF#9ob>%_#}v}IoB>pY#a%p>!=>rvsxd0?gf-l%M?fbi3lYuW>?bDB$vC2mxXerUC$?^^ZfJ8+1<JSOT%e?`00U%%pi91W$G7Ut$&A=eLT<LSAF+6yK=vz4<47M*&U?XGab%wk|mWRFn%q;kL7NWn%vvn}3t4d0S;x1=>gUM9OFlj&>-4Q_w@n9gSMXr*ciKVV~hDLZ3>i{ZCqYbL3h0x_pkWMH>o7Pcng!iqEbB*tj*X*)R%108{udUguo%D(;P>ft)y6E0SnEUNpw)^vg+P$lZNR!Okm1bmrtR1F+bI=_ytztyfn|GeR!ttolmzx(+=KK%5<f8G5-YaU7fdj5;nVP@t0FJ;pyAaFWWncvb8h<}<>OG}FGNGr<(xc?5kY>7L(E6cp}z4zqQF4?IGvq9J~@V*k8WD8SA9t-y=FRL1$8p!9Zl{@71`d(%`^dy=_(||ogacz7ju(?p{8eF7_9Fk1@mJ}WNp%|kZhh}oGu)CRJ=IF^bDan0;DglDd+g=MCwv4uEE^9w9%et-3L}r7qE1$lHg1h;&gt*je1d&`-et<s34+3`%id4XTR5kefND{l_b(&YocjydRGw@HyT7iGuxm*T|ujjx6LrlaE{mUDYI!|gf24H=AzegM>46$-P7v6`TfB2(EK1a)KHb8~GKs5n_enSkhKSTI&1r!PkY>Yt!lheUwhcnuueZ_p5NrPvBF|Qs6PFRzxmEH#ps+S1y4$)>_!SilN$)9_Rz35PzaV+wBP^-PqbKC5pu7G0ZX;W*T5i6w2o3Unhi08SQ1s!^}-1Rg>v6t=!^oX_R20g|ZYda$FN^ULN))x(~ve^KkpocMScNkfg`BT-#reHUl1mbNlvx5c+NIdT3PL*OMD+N)u5<3K8QQxZ!@)UI`>q~iwb9VSr-HJ>-!k8)nyoA77rYXAV=z^wv2MtvXj~`T-DL|?Ud>06F1yNd_r3X|Ez{?Ie*)<D^E=fEk^*w+UtZ!upAY|CO!XSYoJ@#He61zSC9SF8DuYz$}^2r|dRYl^x1S&VKlc^xck_W--V7>&xxYge^o(hF*xdIhU0Bs9d-@~)LOhB0o<5<r*GoYM_G#3*G?4Bko^YA15WZsl{ju?pzekY3dQM7-6Rv%>r9)$fBAaYXGasEI*xi4pvF;O+=Nw0+0&o(0@imAj*71J-yLgyCXV4nzZdQ<t<tWl{jzEz(9<pUI${fr0#2=79Y3)`}2%X8bdtc(q`cr1VOtpLL_su*)p4MP@7>oJM9-?&pb1UTgZ+KsX=u9piB8YQO4p50liO`OUVYypONkglv1HD02Mf`aFQp9;BQ^%Fn-D1Ui}LC3zRJ=@6F32Y_VLgs7$6X$$(0L}aH?vL;O^*?c~3{sazCGM_o0SKQiR|@b;fi2_^=T`t_AHMsy%kQs@QE_6!Lu3iY0Xs_8;_k*Gfflf0Aly(p9Vq>y1e1OL@tc3fh#2;G1y5I!(ez4V(Jfj#*d;)&$eX%oYCF5L7JIR`j=2}OYRy8nxhsw|uK<fo5|Wo87~4h_5XmEA<XiD;z7lBJ<*mU9<0ye(AyWhpPVF|WnLAvsfiRLCtdk)P1{v{h98#VjVl<DgkN}eN5fMo>AhiO0S%$;xIf9nQ7+qjs@*<cA{zgq!!7%lk@~;EP#Qd%}@|di3Z_`#OK<2te-N-ZFBRtW;pvf3O11V<eB|8x|gRxc#t100l4g5m<1whIS2Q5H|Nfp`>*8;#PfaO%eNQMwM67SGY0Kmyw>yn<05CpJ16Hf^E5^%8rT!pBhBDbn>*3^D(Aq%H0dkxwNnkkyBR4D-JM$h;Cdp!hFR|&BkFF@-)<xRQ~$u0wJ2g1Ju%W8H=IF(m`IgG1JUd%E<)^IVa11vjvbqgW$LEo;wC0*BUy0QXi2TnL@l{bbV+E8t*{)67h_V@ttdmZiD9|G8$$bDczWt;MM$h*+^61Sc#?iTcxE-<&72Rm}N@7cybbEOiUD)j&)<^>M(T1_>ke0P0ycXJC7H$XJ#4ZBNLI<0HkM``T0jrt2=f<g*UQ%Wrx^ya9IY~DJUh{V4uRrEYMXQGMm)L--=343!c4J)7tUK$g`$$tfBjbE|HKKYHee=$<37`0yDI~7c50Qf04MN2m9K@dVS;d^)2UMM`k_`uC87+EpNvdWi!H`)!xsW@3V5^p6$2~5vda(@rL(bMH%lmmamrvp3;Th?sqL!sOuFa6}oVgjaEu=`<B#UuiO=p#v+$5W??eX$dc_2|)&$M^V#eY*7>Q$ZnQYYX3#;6lX~*aQdiv>U=o5IcIVj%8K{?bj}*3Fu6sb3JX_kQS%nHjo6V%G7g!irUxG3Wl9<dvmV*_aF7@QkowW`why_V|e{K#Bz(&BZJ&pwOg|u@BZL+4?s51o+eelaMETaAPu)@J66vdJa9HT!bhvVd!|r_?F*FAh>)$qFDFXg=L^zh4kvc=j!98l%m;=dP_&BMQ*m~N{aj)h$pRsN$l&KuWA~VU0F{=ljrY`oVETw|7)Ae14Cp*G^8qt_w<<au!dlP`80X8xXp|yWA4U!C<%krp^aW8LZDt6{r^`8AS?A-m*iu3c2~x+=7YM_^jpG_1xH8`~fK9a>>6-YVgDwauCD##XLy`EiQd|$Ayw@uEOzz1E!)H?_cgnA0;50lF;3$EBe%h#y8ul5+eayfojr^FQA2arI4gRg@9lTWlU0^jTf!JV3+j1s`9L15v<qh~+mm*eDZTr)~NsqEK7@_U2H*hwtPiI>r*CnRnTvJ^|Gi-~4BtbN_T8s&U4+W`YQ7X*;aWLDYH(o>5=Z9E)5qF<-4F>X0Y_kf+B!#Fc5b1nQ{pTJ72}H_NsGV=UXN<ml7#T!yjOFY&egT|;h8M46s&&i3H>I-<H()wmEpW@#T--ynNUh-gObvB_hAV+=3mCLf1rK7knQ+X6>z7Nbb9)<M-o>{wI<o2}3zj;!F&V!y#Umrb3c35FFn_P;IUXov6?XY${tbpSInaHG#N8vrk$Sc+MrrV(2@b0y)r7$wHkbO&184hsT76DMxm{>Z0Zp9%tGoijuuIyduC;EPG`+l6awC3z_gC@phriwZ*jZ*3>cRsqD9jCv$YPJV?lBh&Cp|(%2%M8JE5U-l@4xjl*EofYx!kiD2)QdDomo6u+$^0m4@TnhowVrb(msxS#y^aZOtZd^kz^KiFLY{k(?3x8MW&4!v3Cz^M!JTEB_Xblm+J3L0xT5puS_pt-5tt=7O8GQD{57u=l$hR?^x)%$EpEU2d&1kPdPYP0rSxO<Fh=l)%Anlkv>^9;|oF#-|@I?c<W(h-Fm=_7dKX3F+Re+Ct-Egr^rk9IGXb1m;a%>Ugztr%(aTn2I-)ya2@P*8RA$%=q7{u-T>ZbvQ>2GO{P`Re3p~x;xh%2m5w68C_Ux_*W-o9MI7_1EiNsVZg@#Wk8GnpK@JNE868h&&7Q*~7wv+PqgBx2RE-ukIB43d8O>|u{_RaI-&}+sT=)kT0s80hT3&R8<wP{@NMT`EKAp=<-ME{M117KA^gr6b7bHC~V=0QS#m#bh49jc!Ig>j5{M@KBI3tuoV6G-ZGv6q1r@Z5W?C$B9Gao8i=~EtxfQFu-Ew_%x))+c<1e~)O!Wh^PsS|E&A|z~82RkkCD#G!c<YZnlR)~L1V)DPGNejkDnreAu9(@Ulc!zQ`O#f^Zd~5Q<7b@DkWE_91MT_xE(a&1`p9Rr0lfG&Td!Zv|j?z@Qjp+)ODs`pJM2lVE?B3Gtu1&bECv@F5Pi9>-9zXum8KYT&CzG5Y(nX_+a>Ef~#ihF?XTDYgyOW*%agL@F{cWo#$KnwCOgzborq<`UIw~*alP(n8a)d70<IGz@!FroFKIUt=YEoGq1P1V)bJ(s^hqzrr2k|=Dw8D8j7)rSbrwO|S2~kN{`hdt;!rT+^M(&lE18#HkQAgCF$%{;7Ne9|=4$22o{KP$VHV0egBEP;XuuAGD48bv`ev~N%OS=Zd4uVxA0*ELS%*<IG#;s|dAxnE_+wAw-`!Wy}F55eR+CE|vKxklTc@*mol^cY%ibJW7gDi^{%!IkQyR=R!yOD^IvZ-^Az0{)Lp$nsW;Tpx7c#FBBbKaI{5@lS05smn)PG#B)D-+dOy<?Dw&76tyXG<+!LUAvvj!ca7v9zcztG5uI5vgG+=iIkRxU`8kWjHeE?+<UD+9Sr@IIylxvS##bd7`%1a{l-II>C5l%GMT?xmJ(S&Rpd&3t&Pzj96Z~5zGbtFAJNxQ^XGaz6p;V8FaUoo?p(<8Yg)i3(ej8BsL3z2`<Ax3KR5pCNq+=($vWV#IfjPawMoZcyQw2q@dkA?h&gRiy7@^d9=Stu0kHu<yh?tJAU`(j(!gyLY1rh=^L{z)A1`8xaP0Sv|g2M*>SRF!ST)(f5TJEI+}AGz1x3Y^3_elH<m|h9H)=qhAP3OrM0a6lLN@hoz-h1$6h4M1sG4U<onTQPem0DRKXI<=sw9VIBWFjHBQfScAeooRX0;2Bf34IM`1MzFm0QrYI;ZdwMncinLHW~P2J3Tnl7&KxOlsClO%3EL4Zk$jxu24MV8|Rq~F>&*^|NKbidQDk4=5(l^FX4rIRDfYcuW(Nyo<pZ@>q;iZNDznH^nGpN;iu9J7<Vn+!r+K&YCu7U$GU&aRuBTR%Csj`Aiy@hV@@gGR0ZFVA1-)zy|aIiSxiznq7z3>c(j0D9iZ!``7bbyXhc=$ZBQtUxC?#~>yMJ#84TJ3w}?cRaV)`y!rkY4aXV`=*9IR=%~46p`9C@5d|?NKNpD@nPcB_%tuR5Py~Bp02d-@xCz_Wypt)jN%fLYfzMA-|RW_7prEoQ*|^k9Uh_a<@kwU8IwFb_QW&P_uqEMs286JEB~-E>`7}r3ohOQ6B-_d_pjd!1#hmvj<)Km9pyS?z!d2N+suQn%>(^Hu5Q1W3RzBu8>Tp0;46@uBS5KFn(Ke3JMt?XRa~etzR#O-{>|^Muf<n7NxTv#C(y63VfQ({_qUe?27uuJ)Pi7ol0wM`Fc>h&L%cfFs+t>)_8+`PKBih`j>C|}!N^@js*n3Wvq2H8g*})H#d~~ZHOY1#^CYXOEf^Hir^IQbRw^!QzPhO1LvfwvG#eQuRV*K6k|J*TM*<>ih}swRA*NF6NP26k%KS3b@;K{?a`Mg9t@n}e{n@TZyj(rEM*4E~-Tb*NLExq-X#bL#_2u$qLc6zVQ{7KsLi>g(dJZG~R~~eDKjl`f%l!8&WN+f*^%Y-q`s@P_VQZV;mh$PvQhhSNQtfNgW-Fn1P=~qxuLzJ`<}|x+#bypOvGJn1jMzILfJtYx+ycHWDeQ(X9Ce%8T4-0!!d_`Cuu9ZpMNTYm&PF+Ay*?ODQ`_4Nzu|>@FhF=QP~Dg-@^1X)L3&o!;t4_kkS2O$6Mo#nXu6by=Nt8XnK!$8eANwWnUAte(7w!77`5Fd_-Y^QjhCdGru^PYdhkZxycs*VBGQIOSFcj1i3%V}vK8FYd9j&CU;H;vO9KQH000080000X0M;3spbQ8A05KE*02u%P0A+Y^WpZh4Wo~qHFLYsZUv@VyV{B<IaCwzj-*4l%5q|ey!KyD-&X$wCy_Y@cA(y0^!=Zb@<_fe30)dujn~fyuNGgf3*#EuX@P{SYNsHzomZ{;)aORtDhE9^?>h2$`)1u>=TdCRw`-}x+`)bd4TQPc#w@O>tZkcGU-m^~0*0SD+immtTO4WRwB}p=yZM14wQEYnKYf%(THnb6XT4kMfGn>UVy@g35;`e_W)y8LK;*;suomQnV!^+-xGaa|PTCSswJA4feSZ)g4w^lYHD^<IZ+bDET$GUP$v)QZ?8`jmloqM75(~RM-RM^ULR*U!|YlPjY%A`zbcJa|k86he&m2yj|dmA<s?u~9B_Y?bpMY$bZsim=nGvzjDfrz#2oBKk4iEO2nye@*O<db)7F-Vw3JqfJ;c+SpFUzS>kwg_4v_2J`x5krivDpgGmTdity*zjQ_98TjBSs31QwI~3QVhbJB-g!b>jt_sv<~~|Ku#5DdO>Sa5kg8&=`XC}|BiQ-Wf#_XA?#YI4g;_Aw3YQRMvp3AU^E2DXS1iRn4$M|GodC{(PPXPo)Uu`hvI4H9-7k$V15qU-tCi&2Rv9ZxGQ3H?1;2-NpUCeAe{RlKHy11NY9m`tsa~+_<=xkt>(%x0@0a(#;p3Oz|4Cndy}u8hE>VPdKrO1ZFk=VSeYF*~Xk^=4VG5j}FsKNG%AvwTl)uKsC^*ufh3Ol@3PMO}!wJ2!E1#?tMQ;^qMT=5tq>Cd+TDU6brB=qUU|`~$u~1WNYV|_u*R?92h$!yP7tHQN%Xn`U<+a2qwXsW$P*u$Bn(B#F0X0JzgtR{+X|NJgMwV_ambat*P7pzmP2UuU4K=)`c$LU6eRsq~S-d#+)!<@x5VxwvLtBa?y)e3!Ygq##WLm9_(9a0V)$Qf4UnQ}Edon~b8p;}&b9ieNcnP(wSf!8^2XCTG1f<+SfiEK9R@JhUlYltGttz~2R9XpzvI02C1;d|3pk-&sSk<vgbfP6FpN9AQDAM5K;*b?6dzji(P(m4W(1neLODNUHuHF53vhwj{<)@Pai5C=J`LsrH9KZY9=jG?o+T$3%p3Ujx@SfULB})SFT5J?x%2~uUyQI`QYW$hPA0WF7?<xLBQ6o(>|2;GU1JH|t;#y7n!AVO2o(ym*fRor)wBQxE7Gz4P2Wy*_cR;kA4JxvbuCV1rODhV$=B2b!)OV<n#g=j~Wea+nX5zI&x>1vQp0bq-j7j`<)!VMO)RP__y;b1S0bXb(TV`MgXCpn?wpQzTa<)1<TSZUJ3P9Wny`*fvX4H<8G=zZuo_XC1D1Y~;^I&VB_!?j}bfj<J4{PL<Rxjj`)A*1Cx=9{CP2k`+KS1^&c@q0#$IUKzd?XLl%Rw@76%OqrJHD?Kuad{HA+IWP_6*{N*5iaxCANoQmgj)|Q}Q@?$z_cm5%vCG%Kmb!%wp6L6owF>3_S*+3m-#63{41vAu@4gqzm~M4){5nPs4>dB+4VE>IJT(6N$4UnK)RSh@n+Xb)V{Y{Tw^#^o^>7i=Zd7#JBB>6ActYCyOACyugYX9gzM8hlVGV{IgU&*c|17(}$o^3AhsSfIW!ga2#C8PvS&k89~I%Cn__CYb##uJWVsGo+nX9vH*amLBg@8yTvQY6mXbGxj>AtpPsmv^&K(#)WsN_JU&c$k|5t2t9!yhLDX_6!Tb;1AN8)ti#a7QR7*<fIv~mSDSN>D;thYtWHoix=i$^tn3*m{_4%otF|hOaFnt{ZmdA(b>orIXA>keQD7ZCwAkfJR^jL0>4G$6o>eGF+`c_B+-DbR_raOo5{B2C8>14tW?`Q}KDi$>1ptoR!k`z|c*qEsQ&Y}4^|AM9&7};H&G|(NdQQ4*5_Qc3{?#&!kRchH$!^?}JQe{!3qn#PAs)C0t^JKa7=LXcZ__LSjuMSJ^Zy9>?a@m2K(N0;pQxaFTe4hCGB&|`0q7T<z=>c<3`9u$N^GebqgR>#X|I4a*lHHQ%qaF`BymUtf9aB3;m5j%Xov6D!xdwces;&3z=<17sL0}T}7h9q$x(UPoEU55)!5Qic!9Dju4jPxEG+ZIwAf1ozn~roy#)Lbm7X_0jlZp0OY7mp-ox6Yiar^t1+spgg`<t8VBy}C-JaL8U_PcmH%BA2AZBuiMHSDV&Q*N|YdRnW=J<vJTo}b4N3jNDp@$dHN3y6hmpAm9}>prgcqQ;bqDUdZiI{Mnpo?2CXNu%lU^&m@*>IH{9a@-~V(@`-|R7cnGndi_qHV1SK`~86Bf3yhmRS=rZ1}3B%hB@g7^W>7?_zV5JqJP&8wr=i4JdzXUA6;ABjC{)ZZ>UV)r4Tx`_ca`R!91#SVr6%9f_mW>7x5f<_<TQ1Q*gWyB8Se|sXJ~xW`WK*f8U6&>o+p_#ymZX0UE^r%DyP}BII_50kjqNMd_ylBRC9VkT-ObghEN}aD$s9RRXS4(p}^|2ViPJi^_&t7!A_}@`MXifwKss>Sr-e!|-JuJ`AJB@LQ8R^CYbgW{)a;6=iQZ5Ra5D8%JV$ng;Pi{O@dn{b|V%wI$0xlgFce{8Hs4ceF~Rc+|b$z{xkyp6>ULjvLTa;-`yFW3rsP*{157&V(kiVA9<O^AB)lQ}<>!V)f9Fyd-)Ja$6KFZ|HYB%4|_k1r$X{u4_Z->_1RT0|XQR000O8001EXf=jSlst*7F<uL#NApigXWq5F9a%pX4Zgg`mbYXK}b~i6%X<=t>Z*z2MV{<NWdA(b0bK5o&{;pqv>cfqcqZm8APfFCeq)A%$nm7|T?cHcJ9EgM@HWaB4q%Ae|f4_YJK#-yo=caR>i7k<f-Nj;Go&}KMaQJ<ZS7ni9DiJ$X#%fokMK033Qsu46@rxERPsA$St7Nh(6?}l#tE^UetVEK^bzbNyjdeH}JkRr@l4elpzZJR$_P42qcY(;&ttyd=Dv`^iO1DZ>WnFDVQ3@&6vQpvT`;E-kX}$*eK#STG<>B$N*o#tqs?$<w_)yk)jx<Sem+Puja_i0Gsi-!o2Hi6?ND80_^#shJQZS=xBP+254YC99$>dUHMwr^Cpfh~lO8pVpt%`E1c>qZa2UpetkAJ$MsRC71mrI&FaFx0S{Q?pg7iFp9Dpy+L^j3A2O=Pa`zyw!6goEL5I2f$TVk@F(RRa?hMIznqin0P_oIwpuA}bkZQfruvA##rb&~BH>SPjgZt*kaYtUBy)Lgx7!xx=r4xB)}rECzN&UT=2?A;D5RBQU&8)t!^IOmkTtA~eikF!-kvJp$T4R6hHqtkrlxkHi%$hy>=ZZqo#XOa}sfQu8!Qlc|6eqlaYu<}0jmv_I?*Q&H8sOwCtWA*(?2{8<n^sxt+Z!#t5?DG%aL@wUhnz1x)qxDHAr{$eJE-~mY*qg97Ql7K_R;9QZ0r2;bIY<K{@V})YAMANL8onVK^NL;~c?;tdyG*7E28flfS#%v+7(n^fn7yLm-LYmOCUyA@dD^oe~39bFc;+vWHN0$s)$t)Y8cZ8{4p$2NS9}94Efx#`>Ox$=+Kl~|Ws+IUvLLj^<%c2|&?TUl@C`c$6w~5IfC_^AST-;&9bAJ(t1N_@x!rx0UQcJdPkNu^12IM${1AaVQd|91anSlmYEDTLl36Af{2gW3kLOtaSf$P-f4OHM=Qj>R$+<-AST)IUgKWWpSHxD_I7Ta1wKq>L#{x?q{^8Y5v8ezc+74$~#Am#=GbgDWh&$H_=YK5Ghx`3Z|gL4rGVq`-ODnNi`U@;hr$#-6C@kB`utQI`~LzRUd(cnNNkZ9CwSri%1C`~j_&8-H$_maEW@91?-O%7_XIXX0OAU9^`uIRhxh{slb3HqloRFH=G)k|$Gd{SmB^qjiZD~>(lNbb@4ubTD#+&BMw3KH#CF0IyED05symI{<rZ0TW(d*dv;41dmA9@Em^EGrLz%2$^;k1`HpDn_K`Ryn_kl<|s@NuiW4RdUvUI_VFb&6Jrr%q|co20#*uH>yhvzs&nb{3kA;{^VdAm&9>$!+^b$T{eTN-RIY+`+`>wYVFI(o98$Gdi-+o;`-kLj7RkzJtnOH2(UT{z?CI9TYyv7h+D9B+SQDUI<G=b<ZqCk-fCb2pecvU3I()IAm1fi{|V=S2?f<t$;A%S7$TACK-nN9ZXa?hzbR3Sq<~VUxk^GISo1hjveZsr&M3MNH_zWtV&OWf7_Z<wtBnGOyTd%F^i$&ss5#6kvnlDWsIvqnn6F@t35}ttgw}dCGTnNZ8H&$lLBtUA`~A59ZN}ORNMQyBX-b20xrj5!=LcmFH|pNcAT3@lm$;{6TN5$67<U|tXS22r=O*-q2k~@r2qcU!h?9mxUn(-%c~Zk$WU5EwvD4D(b58tq4hXuKAjZHbo}^8gX`ItQ=EMl!rds2hndp@-CiL8kW~Z|mc}Aht@QvZ#jQ4BMXAMYFB_qRWd~b9r>S|Y4SS}X5xw=V1y}8<V4C9nGxwH`wi3uUyKo{VWWxD`F1V95&EBCMk>p(@rxwo(_dPczz&zSQTyM;fUCOBp`$sIwFX7rf?b)q;@_A`}7f>+4Q*OKIo&zS321lNz_7e4R^df%e38DbhjG@VHWV2Ry<`-%SHEgQ~S-}(nMoxsf4VcLq9NyB4!T4QbiW@zwZqVb73tun2q|0lCr?ChDu;psCA(LbTTz|c^w{x5bm;D!wwh|i8>=<j9>r@}1$@I<)LVhL8<({I>DNG1UQJIqNgPY1qKLrQodVmJr%?c#HKfLwvrhMp(o6Wc8i1a1)(c_3}i2CJzz!do~n?G9BLTKM3yGa!ZR@YJ@5qFDY8ov}6##JZNH1Y^R88XOl3HoycU<lZ65Ei8X@3!T;~#o}pre@!payHn5tgpNHSG%y#|=!PW&$gcy3UNP}%NUxy5*$jJibK_wF+Pz8?#yJcJaA-LS!_i4?fPt>Jqu#;>>-wM;iU7pr1N5w9xmE^=6>(gb0App0Y8H(~CbZ^*ZU#E7z{%45O=||kaq*>S1x1G~-$xH?JPY^$q4&+B?Jge>VohngtW1)|Usn?1A3|yk^1s?={lU|to~Px(5;_c2*9x8rY@2=WjyYPv5tRvmuXbpP+m0ZMcaMaGAS2z4&VkKV-i4)k{ru;fR|ZblHp+K*E4(F)+oIpm<tz^U{`&iCuh2C8t_um@vMq<g=Hu)37tVGkuTjo~<>2;JTx1y-7kZ<#+CUEUHV^xC>xX200q7eKTdg$5Yzqryq<d)0y_z{_;0p~pHJ3O~TWM!vm%kI!kDOToC*?~4N$gv-B}sbXk(q`4f!X!SR$zLe>-i^;96f6Xvc8kMi27?CVI}+O_%+7GU6?QKNf<c_%@ZXmE)w_)>#*Hb9Lum+<W2u$++qjj>qAZHU+L-}@{f6Om$ydJq$X&`Q@P*zpF^QMX6CCJu5qWXN7JAw3N&imAi0xsctDM4w4RuQD)^mg-$l=?7J{tPZ#^E6;mD;UVjrzlu1a>r7|*KIgBclb9{z?og)|ndG3}P;MRVT8jV})8vn;!1B7tgB?8s^)2K@A?H;fh1;>D}i*Y97MBMwX|xSJ%w+g5b=R<(}$!VZ*k>>}C)>g)!ZK<lC|o5q0vcVX8>nW|%=Ozl(XvN<x4pWC2o4iP7WQxk7iAUmcr#K-Uf0P0gAx65>07qvE)Z+6P;Zk1g$v)9q5o!Jw?W(+*EHetc^<G|mK;JrJ4Gs0TIEpC$HX`h(cDTb(isTIWp5r<iEr^=D%`!Otgnt(pI*VQhvpPAo1W(C9AJEq`f1@*MjVo67zQB_2DMVV+8Ii6oG#)ISb$!tP3^2Z1<FWbE;8z!uGNa53->2Hk@ZG+27n62E>7>N0bzquvy6OI?{OcyyffXK01Nc6_<i;Uc;C?n?@W{F^!dE4I=HU7{Du$uWIOw-g$Zzk6w2Z(cm+PUUL9M8nltj;y`TQsJZk1^y#^r`D&CNg&HX#~j!jOyaC*?A;|d{5y=twk5@{(Q(``9Kj)TV(ER(Z|*`tR0*v8f!d-#dWbQFrQWMl+qoa`XfGKI9hem-0wQVt@$V`)@h|jtF+V=)O*OPIi!O}fvD7Whnc=^PJSUs!i%v>TJKajfyQYI6<++Lau71P*`PW(1TBRBTA8Z$_W3U#-ar3&0<v(IxB}Gm`rWrrT(&aj?BM3{=IB9~=i#c(DUsvGoffYLK4NH}=z3by9=G=$k;KK>Ep5KRfW^Eu&X%8Dp~1els<|8Dp<%QiA4_7l$96Q|@%{t7ri0UGUuaQNY)F<hAA^u+-~w7OFLSUAhPW+axvVn@6+-S)yTP9A5P(6lT}QfDRa?2|r44BXN9cv#)DVGfXOl}a_S4Z=GmnQ1-ojvBFkpb#c)uHEJ1Bae#uVxmy%;?)XB~k+Y1uABx-pXRCwlvlror6qHc+!iJMo>k0K_jI3E&D0vn>o_3ObC9vh*X6jmIJGBlwzqq<Q(wJ~6XT#O%{aW}kYPnMBRBaR$ym8lkp-M=@xiCjs<#`Ru|TPLrQRe1|A<cZj&$aqLCQLxlHD9Q1AhZaXqLrelciFa{o68?b&Vc7q=1@vbZoT4mr-bm;|X5uaDkaAd$pLfS7U00Uy)yuqtb24AhxtfH$0=3MX>#A>Ea6EHr|`JaN$vd)FVgBDI;g!kw=RLnQAv}j-@AG^IXm##>Y>VRywFia96stU}3@rD{m9j8cx4HBM0rsn<wZVSpX)d06>>luNBA6b?B2&s`yC)+z!MiWropbFhN2z*FxvL{b?iZ{G|fAs?%5?;UfY4Rg}ctO8k|1kM5y!TU?8}*D~Sld6`nO&Fc*hS7Bow;jVSRd%L)Lbk>A6ON2X&r$NL(|eOdvlD(4+yx;sl)5hVUE8Re8C1k(j6Nv-F5<Fcj-aq2g3rhr)59CSkQ2H1Fv~=IbA$_%jU|vx_6XaYZ&<jhX>fvfss&s*OY~4ogkj`YA;MOzBtOLcH~2HsyiI~eDA<G3JIFah=J>BxYRZ+>!|U!n+W4f?JNB9(tYs&)5!SbO~m_*OlUs0=*Vk{a+%d7)6pkGwP;p_)t@_E7ci3!?C4z6V$%d&o02EB5XsE@+=K9q<i<1$sx=po>f5W)lHVuw{4#aDg2fI!${|YbDr@Pe4aEx)y#Kdo^+Chd?Nbv$#*$CFcPs<3f#|JcFdt=DjBL3~Gf17m@%>Bo=AIKIQ?pDEL>vUsZc|FFaLFQ%uV2EjctSeuoJU|mItwKS4VJvAx5&#=9Dyuk?N`6PdjB86DzFr)P?dH)6<f&EAl7FdhG9?z(*=jP>028m=cpK+HDLvUE+1@$JF@1=2m!9dY@@+Th;y?Hm=k?nw3tt;MToUe2`oUskwnV4s;M4f=`;4UF&xvj1yo2SE0&74RQA<`r|0J=TTYEkJRYQad#WIP6?*PsV_RQNBkFTyr4KQiE0AP01D?H$Ahc8ritU9OKu_P^)3r{1NuYN=d9tu~L);R&YS?(*L(9QiJj<IKujXY82u&}L=X{CKBItdM6TwZW9Oeq31c1gmSJoqX_V2N*@uEIs^YWMPW}W$6a-HM1Y2M7~ti|8?K>-1*p^SNL!O*{r$j(r~TwtVQZ|^Qz+>Y0y6Lp)$YGxGVZ$2mZ9RgZF#Yc}ub5I>ow6~TM@nnpEzO`C`xEv$Em;S;3qm~t>U<M#N&P~mLcJel(A5ci~CtQPDt8z5C3~&zK3k~^7{h>xJY6SlFfyMI8CWzS$qMTSH0D5@qanu3__V_Jmo^UH3Kt9&Z@tpn2FTJFAy;@~y-uf>EJ%F`)wECF&7_ZbOF<#}6-+5~Qi-FB3M>!XA+1o;_OFZPB@PAK$vQv~>ds*gSZ`q;F1jRQ+p>cne`clIB?teKz*k@@Bh<yVL*BkR+6HGp8r!l+OEz^KA`L@AMa?~|1JnPtZJoqnAO9KQH000080000X0I_}`M|>dw0C{Bq03ZMW0A+Y^WpZh4Wo~qHFLYsZUv@VyWp-g~bzyXAZ*DGddDUHebK}O5{~w<MVX6{ROHmJdU-Bt;>-9R>ioLt#wNsUnc`OhZlCVGwjE5+$qN@8E_x0|R+^@T5fWd>(I;py<+aw||kDi{Me$QYs8jW66QCtSgMES4(^6KpmC$HbXet&*?_I&Dpc>m#zAEc3=1#zw-Ki&pwbsQ#DQL5Z8%AkykG7gKG=W$nmrISSO3je#f2wtVBI{8bGBvlxvXsOeas*IE3B-{YnEZZ~WC@6!IVxQ`Czb$6vZF%8s$}%f1PEOWwxv7@3P;XC$;rTP)|8uS@fGP8;-1u?fmmB5(^zQBZcfMYEaS9yMQ2Aj{s<qBzRq&vC`HKomf2&dgQ}}V}1FoN|mCBX0&$tEyyq2zeFVr_G$Nc@J!W{ibCGipkrAqdGRRHqacOQRz_3kxMEY)QopmmzaM1kA8_doitzW@5o58mtde|z%-0SBpD&4XM48%#M%RH-ifz&C^ql0sR830BkT*sIc_$}*joDw_HzMz1W_dIe$xcu9TgR_?jmvcL;Xm<SbwWfdfTw$Y_t=Rvl?LjH*IQeUey{)9CqtiSv5ul`Qw(ZUvg<^QN|%dZo?{A1yN2ud(RdK{@dz5$OEak{ROAV)h7Slq;sl2!SzFE_yU3!gCc89U`;P1ab=#Da0V=OECBAiO4;gMwbJ3SH$`9jrsa)Y^g(u^d>q5cpy6<({GD6?lcZztetDm3j+8g?`;%=UziTKhnE2(Lq%Bn;^?n>hCr&WT5~^-~x)U6j;t4V9r-M{sxQz{-%~EOVH(H8^q~}x(SjB3_`rFZb9s1@2zz1FEQhMk2O+J99lkmTNZwmRs}CekQX5SHpmKJVYYj$YI>ZiH88%R=@uC}DX-N2-`{+k62~ME;&hEJ2}an06t)q;e&rVmFo6n$HpiIg0o-D=qOxsptqQN)V6m5K6WqkW@g^=}QVBcEQ&*)Ap-tB`0<YR?6^Ahp4Z^U>gK&TH?b*y5jYgig%JtS?ELK%n<!Z5j%HZX8c--?$*(NA9kRki}OQBQyoGSqog6b+XLMl~}S_M^766K`Ls!GFB>!fax<kcb!!i|BE1tkq*Yr$w6sVMguO~{nLh6W%dIK@gptoiACFa(-R{g3L`iWEbQds=Ntffat58Qr(|`2MFKApDp9=(82rh<WSbo}a^ABZI=LpT2qf<C|{`z{SVEygGmS3~<MukAI`(>C@Ho)3Z=LQ{m~^`O_$RetNoEEnl3f=;@2)DqNjEd+{t<y$D|}qi5&T_SwsD`SSclq`rK1{(R(3x&!_F-TQZc|NditsAn&pKK<hP@{4CHb$+&d{_-q*@%-%R+4AMH;3Zzv^686n6|GihFIQhg!TITzQ5ZfCp1(YO`t<aCbxuQhzsI70@#7Gjmg>kTZ(}U{oWxNU(p4m)FMDQX252836Do%Q#gMhA%95Du0%=FG3@4EL5<(L|L;PCB`E~IX_%F{b|Lt^@L@0qxiOZ@~02nlKWmbbEN3_$FWNQN{h7wLm2}9(JOo4}>7!_&rnYS?9V<<^nFdAimK_J#~o=p9v-cKQ<c;9LMxsIXeL#QeKQ7bfo2<KBjQ#vEY4I+pLFo5#SO-mRyYPQ@Hlw4W|1+2lE9*dYvIvT`Mqd^<RV}&}KiojDv5=46{+c8c{8UR?r*lY<y8hWMVcU_@DnFlv2p^jTk#6Wc*ziUiBkrA@E47n^zf;}U19rI60!vuLGWkFg0)}@wChAuoN^q-2?X+81QSYRba2NP1M5Jy{bl%z6=ButtzMlYd6Ny-kL3!Oxa8Ej=58Q{cGmR7qUVMq}yhdgMgCMIjR4-?QBqikYcu!^TzDwPJZ0%aBPh+(kJ43HbH00xZdrb>kb8=Y~Bn;0q!ojn1sWclo9^=p;d_idcUQU;E{R$dqY3L8khjH$s5&4OC-L!l!6sctZMfJIuu8WV!cssD>o84<6_2wHF(b7>N9wY+**aGjfRVM{fm?sKWZOP(!i^djYT>#fdI<iS}4W2B14ER#4joI(YtvX5Jx>ZUZzgH_2L++L_yRt1s8v@BG>PaYRDb?BkY$?WVsaBqzHCmMQKljUWG-tK}u6G3Y-!FfeA$>Q+3%BXFg?}=I!v)Kx6Gq>=-vL!8DOpOGcB8U?ymdW!YS@F%D_-?06QIRPX$zuq8s8~Q&;GsQ?915?4eZb3723sg2k|>?RBH{t2yHL{gMuD@TOmY^glBcp0Bo%O$vK<MTAzoRn&G_L4z*sv3`C6F{HU4=-;u1xP87PTY=4nNOxDlQ$Rj`#(W@~v|!s$zt5QSUNBaxld4FIiadhl43D(#!CEFj`RLbW<;x2#{l5A^8KBKVtq1k#|~Jb2uZpm@;W*X%pN<AyB3&TmMw>>yS&H_a{+iu5jFmVtkYhczuf)=QQjbiY=_ZE(~>R{oGES_wr)CN-2@F$m~7UYlwQj$6whTk<~WVHL~p@mf(MD5M-K#^YLa;5U-)O(2X?Bw(QyawP>jt)onZtb)`!C%mAo;7LacDF0ev;Rrxwn$zr8altxxS$+D%i0q(TsH4ob*wG}Y%P++Jt0W-YHEDo+tA(<RDlq`F-dYz4`%XM_87SMRpy0N!7-Den8f7u7vydlB79%C@gHpywu<M%i1DPC|TaYisq6F(KRz%_;)U22Am(+H`Vi?PMWEI#2@NQrb;gM3SG}7EX40haG$S1wECWR}j0`*AJ6(cZ7iRnpfNx3bTdoR*!f}Lfw5?~2JF(U;4I&Z1~|7E#JgX$COa!6_c{T<QF%W|h;o&q?7VZs6~nat$x@|>zQ*@iri#Tx5~27rH>2aG(1UsJnCBLrpBv#rWf%J<AQDbs<VGOAcymGKkCW-TGMFJH$pbunvdztMuFBA<xu4~%@u!<#C&NiaVoNmX&Z_rP@plQm&$neJ%`LuHB)e`7|(5AdLQt{9wDh$i99f<%Ll^pqpHfVd@0_Hgtfb|IFKlhFKVkEcLH*e+>^t$kdxDp5O_Gvc3+bqW~gmR3#Qvdv#L<r_o_r6QKtC}=8X`4yyA3$>>HC$+4|Mo>5c-zD3Sw!j**SCB@5hX#4z!8L;>g6V)5W^5O)Q^ad{@~Z7JSCD59U87~dG9bc~RZJC-RIS2*%m=OFlJLeKA8%e!!5s>ic6jpTF{##TSbD@Qaj{frbR8FL2E5uy@(Kz}5WxjB3Hs}SbqPOM0LpGa*)}ih1P-XzI8zulUm0bd5dX^~M%da%tY~STcTjfH4Wf}EY{ZIR)BNNY#*S$1?Xch&2`-fvRvCr9*WPssq7$P~L-T?kLI!M}qN@mFS_Wh#fpLoecnHJ*kd(!ab@8&w#cQgT>;l8Fw?!Lp1wVSEmq0Y|PYCvm$=!kUfn!S&6fcXR>LAH!#gbF4H~fSJQwiFVD`G&)3+5mXObSy*j-;svchG~!`7PGiVum!J>58x}^eQ0P3XKPVC|R?E0pl#K6_kzQ8IgKiL5L$S12qxQlc!P|lg`gz%E*Jiz)`sbtmX$2bnTToW}d6^m@Oron5`yv-Bu*Q4FxRhY?LM3of~$eV)j@|u>MWV&b4`j6njaKm74v&ZC0`VMpzG4&|v>mmGY3NRNrXUu%>`})t%rPVhUca;PxmFT3%Qwgvd}TO1wGpN9~*Wp$6u$WWlT}K{%+D(v-v>=-1oEWKdvDJk1sbD{-S3dww!MK4C@;K(Byk?iKLefgD6_P3b)%2A(5!eCl7NJ`r<bdR_?s@Y=f}X)eOJD4_Tt7E!eKMQQLeJ$l3!<kkz9!ouW;0WdB?qh($d8&04A9t6gJO%1+N@T+xtAu;u6^u8VBnlz?lizo{esf?nUl+$ap*vT<sA5Kz66n~-y>pkOf2&4&BDduFRJ+@Kw@m4Y)#c`=_OUgMMXC}{gb7ne6YUM8$V83#)kf@*j!N-KMJ(@N-#EUw$;{U^chn2x_?|7lyT;HKq40v);!;e>;>?GPWwe*iu0%=lS+ssZYcj>j-HlQv$&7AREG#?n!Qfj8sS)8sYZ)GuXbE2(|Ezp;(b}m6@E}r#syUAftorc5*U>LNyyRT~~EeUC%#Be7eF)QFka|Bh4C+%rd%0m?Zgq}0<lC$wdnvX|Qe>7=N)fszMWf6FCd`C3DFmSF;=ZprG5gt-{KWSqh))}EoBKs$mLT8R!z@_7O()7$Wc~CH>S1p`IO*lH54%KjQ+O4Xi&Td63aOqaMVa%b964dm2+7PE(<3=`vy$E2pq80A+dRd>NSJ8ME-743^I9SIHM{nM)_t33uyoYX$G2q?e#(?w}voR-q9IUGD4O6S@9tj<b+?%oZaoysM4IEf4#}@S`yj3(>ImB|bmqQeEs^)E(Y1scx9xseUu)LW^fHCdbS^<#+f>WyQ$PR@_y(Q>#z`#OrY|x_sx%WMIde<$oE_sjc11rd$dbiLx?memIdG-*fmM&c{`YO#3tyXEP5hYpuJArpUx-b-k*D&I=Y`?0j<2oQ)V*-Aiw)O9B6vFyM5h(M>Yi6Hm(MG?JXc<l!&6~yjErZrveE~qk-1uQHlD%9m<yWex<(7&aA$CXR$N$K3jldtCr*$Wwk^;)yvp#o>4ck|-rAfS;*=E+f8aJjft%zQ$JoWE7D;z-vRlYFIfgVrX=IcA`$XkVI<8JPBS%x;wRkW~P(?UG4MX?FapFXSm4(`f42FmVT90b<hdFg;lbq>CQ?T-cknSHmJ6tq{G{*Aj^5>Ncm&*^9a$0meNId0qa$!w!;BlyZiIi7ThXvenr$gbA~Tf;c>RS<62?I<@Y^?%XeSK4p%zx&qN(p<5}``x!)(Igj-`JR_=Ebo(^b{D72ZCICe8?X5+!Gg~c5Rya32jpx)eNy1qoD?v=c+pp2a%f;tgy3}-=&=F1?;RI#VB8x?skb!8+u*jqx?Vne`t*xulUi~N8t`8i<UbW%huqX(P>xhwYXA68epw-^zLLMFnEJEXY(Di56vPyM4z_UgB9BgvR?jINQ`My9;<bI-u|aZ{Q0^Q4TVuwy3ITmI^&3gkMJyD<P(bLaX&a;Hw84ir^JD+tC@f_ZijEg+>>pwve>5W=lvY=<e}R5a{PTI6;rlqu_}oA=j@u0pIyAP`)v*^nk(2pZIBwf3`R(8!^L%pYpSOxtQmDs(o5>vlcvctx-tUB)MW%SuKeV31A4&P!lN@3I{y1|ZCu<dm0D=YY;Evd6G=*)Hp}Z;e6ZO1x)=h*yB`k_Q{+6@2nxKC|KysLK1_-bc&ivh_L9x|h?)8aEIC7U}>;;W%=P-YSbErQdOWhYW-u>Qmz-3KB(x=KHiu_B|u#~WjYeK652h*6^wiRLZFdin1ok&OdN*8x9ChYt-0vICZgB^q-PSvEzZmi;4KE(pP8(xb~LRyoKo;bdVG-B_%t@~e@?P=bIFZ{sn%Ocn)P=W&Z#xvN2vDxn@@5!{GNAjGzChEtXrd|aiyRw1P3OofOyWQQ8v<vN(Z$c^99YV*+B?^=}(gHS4<p3addMP9z%nCn+H%`aUYF8CrsDPBPfum>%qWiEKr;*h~;5U(-z;HvuR@~Qz)r5C~n3j`oea$71-Dt2@Ujwrhci5oUiR}y|ag(KP00dE;yQh3IqTvNg@Am3DL0J$dhT$BmM@(<*=hRoXP}P$2tNC*V|8m<);piowh#|FN(kmT5!W2xd&gNYWQTK(uv?KPdk44wopI!qAFw8ihKs}3*1&TuZh*CQ&O6}u)59q`-Ll%XOoe5&tvA<EW=uMvMe7qWooinHJ4B`B3Tx{v^!v$h51L)q#Z4Jj7%=m;Nc^!5MCcS9c<gylxgK4_7b6uT|M@L+3!=RSCXw%0!)SN<oW6vekN#yTz5D`6-IE>3=@23GB7qo|dPw?l<mP=X{=Gb4H+XJI-zo8?)1n->&oq#Z498<1`k7uDRowwxE!k|xDZI>$Nqjm+UL^+@cS9BB5p<){Fgn`lc&S6wYf0C;J^2jae%pJgla%h%N@cr-OqL4FuG!)9{Y$t)Look!OW+p&8j2!csUG)B;cixxJ-)?D_j}zsVWRE7i8*Ve+{eq}cT+{~#$yfIO!G{m$O=~M{pVj!t8HJ|1h$Hl|hZV^?pLY`Mb^FBZBeZrEh8njM?R1p5Jt2^nOWTM)ELJ_)G-8(1jxg!5rKY3XHr9Ie?wf(|%3;X1SJug5n3-Zy_6ZofvmUoPh?czUxGzi=%)chH&HA&0?cagaV_PQyjeR49TlYGcYN=`3i2Vh76%t|<sw57W&w-e8+7&vX>?Xr$V*eTxl`bqgw4+S#wJ?xPprBrU+Cw)BvXE5CfCE$mhEgM3#>CO%Y%(KtA%x4AVl5t<_q3#-^9Ya@OU?%kCp8V2*Sd1r8j({FG7A1oW(eHVG3P#@NG?ZJxjKF^n$Y22+TTwGA}&r&UGj<M867PjOT)I0?IaaiTQQ9_H!2M162PmGV<qPOL5UuKWV>CBs9-c7igLKANOwzYtm}O-Xru=W=1(c<S7SaJjPGcG-{tqCevH-3f!K81U>irH`DAYWkPbpu1aRIXk*NKeKKwdJn_50E{aSd<kVEvTHG`G=j)>d(VLG*iY8FIMqb&|}7*+~bwp)MZ&i4!T^j#hKI1QTChH#66<Y-Ubo!rGgIUgp<VBY~AaWS?0BFN4MK}@_Jk1Y7WbR%$0T-!r(*iRl*x6XV_*1V~k462<6?B@TG)-l9X(L$u9r?th4ImjQM$9*k`_g+sr>hG(Db{huK9Fj`5#=Vw*s2d(|6#oCn(BlkuP=;Dsv=`O2MM@&Cc`wH6lg{{T5-z%vIJ`c+#n_6^?z$be$niYJot&T<_qgk_JlM>_n|gzBVGk41^Icqit9QE^*`UPXFGMC6wa-yp-WB|@-nfUr<m%$=bbepw>-$Y@g=yJ!hGn~eV?cz58yz*ttXyvwj@K5&4)S5*&<Hh(K3mxHZh;9z2ICMu>mjW!{ShKfir_~7e4KK0biE7mbuko-@L1Mqxs&Zo8O|Bmugkw|%D)`lKeC9UpIDL9QL$ftPbrQKkA1SItdJuM8n|4F%C1S)%L{<AZ63Meekb6Qdes+qJiz^oRQIUk|2FOI(m&X|Ci!Lk)J-0^H11?P>7z%<!cJ>Q#JFX92Wbi@aU@@r4x~hfyseh=<LX1#4eiA<1IzYTwYT2qTXz5MEU0^bM<3(?9$7!mewEti&l>Bmvz+x~O*oYg%Q#ysiEzp@Up}HfZBnd;wJz%f+Ci1=^=zB%^Legc{k3v;2cU1iM|7Gu;O%Qpo%b79KME~~00S*}Tdv+8Ks!7RG;`xXD>McuE$h0RKHv*9xqU{$-KjXRI+KAd2>CV|w5+yc9@nrG7G=fx3PFWIHT`*K)~rD}mP+^z_2N97;A{B56Rp;6z(wPE4#u-irqk<ZM=07Q*^#;Z$v>S8wTjV7TAXm!(9D8UH?-xz2KaP#+UYBM)eFl|Cgmudd*_QPHQW$3mWS>hNux5i$510f`+nNfR7`qjbF7co>rq$rldC3Duv_q*Aq>fy{zSG$G?{d5>m=v6RJtxVq6ZAbT+LQ-8U;x*&PP8luYTUmpN#+g;_8q8^I!jcJm=TR(a+0KtwSeWTpe=R$L?%ct*py70gP|3F1rA0jdGyJL8{e?vU^I#>cZY>nfu0i*e{a1hY#fQeAfvuA6C!#q_9=5*J@w;5>`l>vkGk!!-lHA87hYk*ogI1m0;mI>eEAQ9E=7U9Ua{by2*?`A3hzdCSAmzG;tHVHJL{?;2i7(o<yI<?WvJ(k*&Y~aX)<|_>r+2EQ-T>yPd3_X-|y7i_ldrByCAThuhZcoUSZT)@g-Wpik+Fv2^T9f$V3X>)im+TWr|h`xvrF9A;Y%?q@7Jrm4M;LJWO)1~p69IeW{}9cDFYblcuL?1R43maDC!_mv-M)7xC4;Hzy`?&*FDN(9he3KY5VX(Z{LTGXV_aT>@xS{dkCfcY>_2n^T0S^yn2|6u?6;uKd}Eug-dZtrOx=2KAi0U#8F66xx12n1})_j2YlXki|lRZloZO}t4C(A2FknMtu1cYRGXJkY%(A&pz$qHPKfAqpFpc0ss4LvsPns(Bu;d;=as-%fRS0^V{NBz*^<=Tf-yY1e(`w1!=dPNN}rV1+mVu3rXi5*Wt-6%+OZYNj0`M-2=ZLDwz#Q=`sQ27H_c`HB9W47i`JSK_$`y}C$sJQEj}R@G-x^^?l=!mPX1e)J)c0qqT$*1lS~y!1~Wkz;rItrhDXo#a@&K0oQCSL%%iS&y$+^7*DEpKn?+DG96A?e#T?ZZfrKuHk`ORP5+eBsd|g#_?Nko4=t8e43kYR&idG$8?zzY5x=YxGp%68+w2c-=DGf$9z-{jVOA#Q4{|1iq9UYTZ$NEJp0Ma3`YHIa&<bN<@AAHsK%oyWigC?w<jy9ukB(#vN6hNl<JW%%zkXoZ%H@z)Fqi3f(s(a&j>8f95vd5OAV3O7BPj!UbDtS7Eg$4(n2z&mM<qoW(|^HsghPTmzdg{AdiEzw27xylIWi{tbBiyX(v}kQOBhcV>+@%s!B`FhL6e&%yvq-q|`jDY{`mC#N9G{0}<a-h-z<Cx}=73{&bM<8%%lPWQWS!N@`7=`NW=rbrpg1sGUJtEm^?(M~}9=HRv%h*6yWgcMP??TPI6KUv>#z&8p6O2LuW$kw$#@9)`(pJ``ZGrYDw;=EFsjrrtGQL1EAI$bDz<2%#&;s+Zd5+jf}a4!FRLz+-}s=z?A^S#CNh>w>v7qSI`k$RwT+1!sJG2(jUR{#+a(Fht9Lq<X>;p(Pnwz5VmM_dmRO{p#bJiOryUPG~_n@lna*-5yz(LgaTfn)pvTx${vr(23DCozl^OE*5#Xq7`qB1((^>FX=7>{H}<4W4D<Ja74vGh(v-!d_;E4CwV|6jLnCqWd^lB#@{D^%9htW*U1a3edsH)cKg(6pLW}KytAVtLDhB3>iAs@pi^b<HKPN048Kfm!eF3vJs--F*=MI-kVmCFxUS0?O3(cxha^XKu9#+1`oxsB)Iq<x?I5%=KeXc3fVX-1;JgeM+g_HbA8ilwL8qk5ryi+B`+z^?sPpR>_9WGaZ{lE`YPv&`zkHZB_SM{xXO3_ARmvC8Dff%B<{KU77B#2Z7!&&RL!R1R08m}0BK9JMCQ)x*h76s^+Hlp;^a0x^#)~e1x+22QYp>#4I+aE$&S*dUGG0(+7{TmS)73Z;omTx6XU9<5Df}JYV4^^<_9@6dD66;N&izipK|y))sDU&ppN^z%1G<+1I#>M00?V{osl4_z3i^tzK7A@%siMY27nu~U!NgL{{hrI!6qskh-X&)>nxLp-LQEzkPF+k*i44n^!@-tHJ?kW<0+n$KDeCXHK5m_4V~^={NBI`_P^dE;jHrK87>jC4M&3jFsYJ`cqp^2KM+}DcS}b{uuXG_3zA~@#Y0YOBhB&Ul?jA!ks!@hEvap~FUTJ#~9gJjO*xL^c!FmV%9k<2Z{GJXpn5`9@)k3LA{1rN>{XoMuM|5iB?8mf85l$?qiJ6RCR=Y6Q)Gn4<mlz?)1a&&4N?)@t9&>0z@Ka2i?2FHIeI++ki@u4rr-pi&3QFDtCeb-RJvFKsuFHzH5<+F-QF0(>FP};+k<9iv?YY2^{10+5Z>FFK5p>ZVzHiOxF@rr_YgQHYw27WYu+8XhCjagCum1ezqdCg^3O0wvlSB7b`>=HT+{0b~3<z`qrCcHlgbktTI$NHyXrqD`j!(q1!xs4L&GF3?<{DdLk^Ph|8j>~?iBl6Z5b!VL6t3M=2RhBoje^z*VSU1^!D}z4coj)<A^xGdB^>#{NqwZZP9gEa`2Xggou1m`t*3Zx7KZ8`Ox!8HZk6Hu>5{+@0iFJ&=1!+)6Qe+l<J@J$=4@@3?;aTYYdSk?Zf7>3hI?nK^_)rMoM5M6hK5b(4C2(DL!?y~>q>XZ<d8j{M2*O}i~O{P>B#lCHx*~VNpF+C9WKf6ZDlQP@PM+K955Av`)6qU)##g=s|-1Xxfl}b{LW!U0WF@djg!3k{9svANKB9_8)7_Ui`$<2_k4tdPJG=cV}$wkW)cpU>w!e^72j{1pke)i$WqZ=nDyn|sX0{Lb@!WN&X;z7h$d|B<1nPO^SO6Bm3l$9&fF6Ba0z`f$*1n7&EZVN2lr`Ts2*wJ>>81h0DD3c*u`v}`w*?lF^kP#c0U&sb2glqo*qt49T??6a_^x*hjM+*z{9!a$40O7l&nn*ENM6&zJ<mhMP7d$2ek*VnS;R%=STNXAKD3haNGFdEn;We<<Tt0Pt8r|er?z7?C#{Zoylmgl%e0$KLdTl+t2f0U$nj1MuW1}8_<>xi!uQ8PqP`NOpEcDyzYk6r~Wzq7nS==U7`)!_-9{Eyxz88qxzchhYzFjkXLO9yb_6>bF|(UsW!Ux$5{L(l|{8>GH}t1QoO-!TwI=-6Q4Xwa=y9$+^bX5`!VU4t1~CpO9iaL0rDYbN9)P4B39EZ-u;D7RD@zk>1IIZ$PT%LTQO7w7Xy-lI;mfPz_GlXx;3YBd#W*q_rS|Um-p)J%zPBFK>3(-sr{HfDFcTee(!%xxiZ?7+f?Z_e2W_$TXT<6xz)bjDdejF=(9py(Pec7z0`+Y8`BEo!IH=q)WJ8f?gJR}{DrIkz;E(t1+bjK4^-<qPPP9|`Np*t$MFpTW6*C5a5ZqE4-Gu&c9;z0f9P60O7Iw76FlY(Uz`o{MbqCVL&wXj3(RWHXIT6{z$b9tn<Ixz^_jXYi7U~DyF-o9I@WKz#LHIQvZ<TQN=*^Az4*zvSpxbYABr5`Mwvu;urn67s=3o>7>6qh!<KY!<R%uz`n3w!2OrCrQ~zpw28<g|ka9@pz!il<HS<F|i!wa#y3vD^<KU4Nji1HN{=#akdXLKrkDILt<3*x(b(2JQDmU?ZQ`c_dR&9+ponoqAoE9}b7l!#b`z1W?W@tEFL=-o<IHBC6zj<oIIor@-cekyH{hBih)ONr%9W_zVqDo`F!iGV9#s5zrCi*#%c*uT_-^x8HKIcY1O<Ku}-Z0bU!C=(;Ur<W}1QY-O00;m803iVWj!d6d2mk<t761Sq0001GcyMKMX>Db0baO9sVRK)0H!o#)bY*gGVQemNd5u_0bL6%WzUx;Y%!Lzksg)fkj+{Ac6jgbv*1KC?I~Q4DK_obWMG`asN*b&5-~BZnd<;kS>YyQkMx&o!cQZ+n%g4TvrL>G!ythIdRx8a^U1O0qOgv((!^5wG(0kVMPS(O$!+2M*H{bu~@&{&0rDeCvlO#DgskLfZQPhJSv?vND+g@of?>c3<m8vr*Ct<Brodx?w-o%x9=t`?pW8#AU0YyGo*^H|yJ}Kib?76)KRlMyZUL%PXp2Y5ZVIsjN{5Ik}Z-wc3DT3IRn?etrm95CRwzB4>9gn;T3zu3eoiT1zE6d^+qq>ZlTmI$O|0rtNh&88*TOOG-idRK6sXN?c>|SVD?~7>KLbiM-V5XBw)Xun~;O&jjX|L5UqO^rp%4SduTb0RJg+sD)#?9E@Xv3&Mm%?n{t4?grPZ<8-oD1GG%-#z4RMLqZ6ZgCspowM=(%v!&ZQ;tDB!9nu_mLR^p{y)%Ny*DwQROF&ne3!ZH@Jvb2n4ok)D2E>y@RzncSR$1yxcP{NX|YDec$ZqN_)#K1n@EEn03($td@^(5gqsD;x`Q*yL(kS<(!ae!0}2t@oz*|-nLxdA(SeSM?JlW-r&S|?}a9-v8{`Zg(WvR8LYV%>BiB`%o)jNOMgI}w|A9<m$(+4HQOKcATn?3LfyGne`NGeWIrvy6Hd%imblzHrQk^p@1$tQ*}0%N;9}&J?E1mx6j{Lz6<u?s)x%XHtK`~cH{QrmA%d<wVGXBsDd;!sFI)F)gmv#5l8bly)DhAjB@?!wr>v#W8zv1yf`h3gT6#QV|K75Rgx=0xzy7<sf3^%UaD`(WBGs9QOCy#kH(l=B?4f5TI-y~Kdl6BHbwWU81U^G>%ZjYs19b%J#SGRUsMulQ?SJ!RWbCaw%k1e+GI=(>T`gpJBEz1u(Zm@`;-zu91hm3U$@TN%<jfZa(JHEROzJza-!^b*1?;e*J$L^-|INtb+Kc2Dd<?<a%LtNPEfA2%!hY{nHN7Z7w#3$)Rto|z2V+GOk}gdO0y7s}MKoWMh1>kit*VAbtXcYX&y?7w^lvlDkP{jTor_1@>}gZ~<7xiNE(mmxLWqH8?|=aX;+A(iQBh`J>{%~+(MUL($`RpL&!q+&w8%mU51M=?Vcs=8q#UbE-squlz?QcB(a7J#_M5YReYr^|^0|UJ2+O;|>^s%%TT?v1Q-8wl?5nfa{{+L)V=r@>c`^_=Jl!7Fs)-k!D!4HUE+JQKjny+OIi)acJTf^v*h~RzcrDzmW_@Vd%k86`pK68GC<f~zVdIc~D*EdqvxZ<L-59b3!Jj)z`8Stewz7eT?Y?jYa{D^}X4(y94ZI5is%^4MrcJB*;%@uZ>v`(*THH%DL_p5VhbllV$_?<8LR;mZECI~h4`Ivd{RCLbz^8=%^HMQ+imL0>?f}j~O7+Ls3OA(wmKpxdqM+rnOJ@<`xz<#pn_~h*U80b@Go_Y2>EN%)cfyi^Vl_#R){&U%-lzEkY%L0sKb=^HX!Xtq|0j=bgld2<LbWux`0mZyAAlFhg+iZ058pZb!22$lY20#KQc=?)H-hUDRk2R~{{%kI=Rcb-(yO1V{Q8RxPEPk{tZw+uY;orMpWglW_TAg}my^*^&^%MrCTfJ1*8S58<EqSuSTB6Jnfj#)m4&Jt8i{HE>QZZ^1$txy6ptpEm6j>iW+uCWcGrX;079EOVe_3<L!X{)##Uiy$aPQS*_dnv>RE|0iV_o6rWj93HMkwB;-RF>vL_k!D17G%%2^AW25g@l7*lsT@U+m4nz$vCdA8H;W>xuU)dk7g?n~gx)&vJ2b%bJ$Nk_I<_>;ZhtLK>+TsK)Djtv#_bKhO&MO-P0Ic=$Ws{hH@bD$3XCOUT5xqKU&<<OK)ZEfa4><0C%3gSJlI+UZys|D5JpjQZ@7*LqYy08>R3=AscOp+QqRQrrV>Q*6WeA2j?CZAw+?5^E#(nC?-Vgur8xkI=*F1de=-6iSbSMqTJiZf4a8V~o~DCD~I!`|?JgkZ-S!^=3Z8jG``9b8{X{lbxJa(40YC#nb3#h?1g42_2}G@UdE=2~iFS3To6C2x<7c<D=OV|GJ<Lz}$~6yY`aN<9#&##2`bEhN{gVdiC{ZLruw)4^bRJe(qqkDV!9k~#j%+E|1uy=#$rcUK9C>(+x*FYJY$bX_&aZnv1lOw%9<<h=QwLMs|5V_wV<rWqAW%)2RYftn?%FFOs-?~~1er&fjYWtkV6J9-Ds2bFF)OHF(|EsB{%j&Wq$QlHe#1zvs{8QiOZ$m2^y2;d57hrT~!SJ#s|Iq!SXRm3RfHYKGS_dJ<zGL+tr*6j(XP@+0ZpC<?~V${3}c$q9ue(p?5mPoxem<^zYcL_vxt%mz95G+E2j7evuAK0+;F57Nd(mVKE*6dgfC)`=r4u9Vo@zFFqV1%K$Y8=P1Bd4|Go!uTyte7gB942gEDj1kYjw<A!`Hwa7pLte71>eoT{(QFDQw*H4I(hOgdVcjpKEe+<>G_Bqrf*0=6F&fs%P%W$jy75|xIRQ^GxR=gePKi(ef;q8GM{B`2<#Z}2g7&Bp>r0)&!^FO2*Xt#ox8|%Lprgg5#4#=`E$Ml%*~lB%Irw8Y>bi+0=VV?!j@$2^J?L(3)&cS2UI?VoXE~32ipzCBHz?d<1sd`<44=msVPH3kDuYbMU_M~V0NHE+M?b`<AKt<D4%Kfy!hnc0@KkHL87arme&q?Fc!UJx6R4#P)h>@6aWAK2mk;8AprTtf+y?@002}e0012T003opaAk67ZDnqBb1!sZb6<8hFKuOXXm4b5E^v9xTI+M$HWL4?zXD}HRO(TR?6lJvr^?hx<s@xvXA-9~J&%V2k&uKlUm-!;axVY-?E`%4;k!@g4+#Y<b{C7?A9j~?9OpxkDG{p&Gd9Vsip8?Xbprn-nJ{Z5mRa(s;&Ga(rG5}sr=P^#FG4+}D$|jQ&15k6<R_WRg#<R4GDQ}vK)jltU4EPkqaq-bFsNLS7ipSgMl5w`V0FB5MVckP%JU@4MUoA28KohNPlT9De=T&J7DhxxZbYDRKhq0EDt$UVpX1UfHA2RLSg9C=fLPXYGWd`s3%SstHXE@>iZ}rE(lp$NwL(qe#0ZlRuQbVx)dJR+8i+vW2ILh6OfeZ;0IiBaO`&rUC4pXQAl9)+L+L9}_g#cL!-ZzeBt@u1kbqfGmZoVCnfdF<z;T?x0L&T*&s!E~8_z3AWDN8YrRJ6<kVg8U1oiTYC=ax6+<M9d!;>llOA$%4=H+ISqFQ$R>hBM8@AcdH>p$OJym7@j7zJhr1HEidoeTv0HPs#r2ESGEhU^rxtE^C?0S$?BW!6dXDOJ8*9GnQSAwAz|b)TOIQ>3A~hCx?MCX*YQVsEQSt&^O<u_Ymm$S_^Y6R`{vX~aw%Po^}yP^K|Do%~FrD;Y(tVJ7a-{zQXGq*krrUzkUr!Z!Ia4c~&X8pB7HwJ+1=AhQ6EGKrsrv@aNU-kyFuzr1+&&-w2TuS#MAA;<Io78gmZfDHeurn7Q+d)lhF1m{8D^>nO_=M8feE=O#6vi_1B4@W0DfZ(cs(qZ68{p)&qBfbMIFF(F}<J3&>FOxb~;x8E%YMy0DHgrx!L`Mynp)jsuIBE(*Tn)Y{w8<gHGl*Jb?u?pBg(@D_az`kYhfH&LbU=sJEdDuv!8#Gk2!e$00+6DJ^_N1u5CyEY=AQk4I{rY|7JxHqA5fz-w$tL;(wbDWxJdCFi?1zeCx)24Q;ypn&3zg628~>88S>&Ji)5%DRnVOUKvFX6jz;={pd(rQOZjHUY)>H-0`V42S%zorv`f~G>5A(Xf^P?oa_M}IZ{y^?x_7=djQw5q&FLRR8C=y-IPd3gPGA4+xUD!lB-yF>lpe!uFf<x$YWe6j=wc{y=nyQSNCYDMpP&aH>~De>d>}l*2SN~h0L*YYK5hdRfqSC!B}TOxu7H&>8HI*Rvb&+N!=_)6_FJ_vX4f`lJBU#BRQx()3*(mX$5D&Mf=EsFG%B(~o9<A2lTBSjZ2}hNP;LgSe?4`NZnk&^8Cd)Z!b^AqOSgX0S>VVa4TzPC=uRr|x>AKWiDOWS0%?y?8=l-Vb`L^!!^lo3IUKo9h~9${Oi|qWg$8jhU;QzD>y&$`!TZUF3|`tW&^Jh7z7!f0N#?v0fcxOP7a;%dAkY#B8y@zjhsNt7LP^>)@)JP7d2JVy(-Lk;%C17pUG2?GY<>eEge=``s??&X6K57;{IZdY)Z#he-{$w0M|VrjIn1VUYgO7){;3f5(ntudk8idm4==?rlOluKUq*F~WD*Zk<$4eJtEWBZ(@X?;E-pmC)H3uq)yapnIE13$y3TWNm=UD&S5Qc(2k9HYnE)z`0Pcz+GYYCH*y>WRii~r5JVhgOPX|g)A~=R$CRhzHZfYoq;5K)~V(p6M3LK<1>yUFZcjWel%23=h1rM(d5uCjle=bSbmexAg)v9{yLuWpPMXYm$TH;K?X}DGPJ1Z~dhU6xX*V}kdO|gUisHV4gxR?Ccu?FoWSo@aEkDALY?bA0kN2z#%6)af>lpoBXP}#Nc`QC-&Ju825&|&@ukw5QUxJTsKv@%jR5#49ov!im46*})FKQ8rq8hK^G2FUMSP#JE=f*T753<KU`1CcwlKWu?Z$j~i*F;%oxl8*ae(yG&V6sxq^Ow_#FmnVlJ&kkz5|5I(cX|xqqivL=1<Yb!nvzA$u#25?SLc-qR^bcX&Wr_#}jB~h3##QcYi>Lerl!JFFD;p8iTj7`%^Xaqs5lr|1j(LT5J#?`vhb>zp&Wv2BFlX>YKvl<KGf8(3u{o;P935m+p1OCh^lELuNMW+di%31BLws@tPvoO*eG<x$$d>@Y>+KEi@NO3W;1nO&44KJjMBYNPC1Z4&N8J^M&SK|ZimBDKQ9H2E#_iTi3C=l+N{)~YMA!#=R9WJ*Fu%TdnCbQOrd!4H#0Z_XPVDJ6<~CPtGX9+#>EGH_F5JV>Y&t6cx0eoE9nw672G=HMHTW4%67T(!vTMbSB!{_QQG3q0ajoB<G(vJfj@>4#-1-dL1DU)R*9&R<wR@NNa#4gb+uT^e;$8!%!f*!>n`yFYWkxc;o$!%gu(MyCz7aBm6`+Uqv2M@{YEFh5Fm5EZO#&c17GRJCc6I`@Nx>#M_QN8;ejM_PiqL3oQDD{!3m{Pm0!<xB5U1`1gEpn(X`|tZ4n|#%vk%XYC*?-_p4uIl#4+U9v2WvC=HiUE7U?|4ePhOIY+#i+*qFxJM>HzE(k|?}WCOL%Eo*K{VIX|1@bKG-_zGtQ@i(-LMrAwTaSF<`g<f+?(~1ZUi@@(<fP=@WE1DiK!jkZDI*Gvpu?uxv7OpU+bB`K&Akj+`9h)X%_oM1PyGd1S+ojt)io3|U-37%xHm-CD?cMA?fn(45Tg88FcGsP>YLc&Is&H$tBYN(Z=}Z==JMEWr{9^|pdTB|A&rdJsPs5Nm8a7M?Isp`sW`F~%1(n*!+o^qs4P%QxK$|jE?4|LK)7@)0UF_E_N0TW+GSgE_XC!{a9nvWd_h`{67i-paeb+fP%cEIb)ra`K_-JoV&Z7hvBv9bF1V7YUJ;B!-xytUUekSH0Dqg`n8V*jW!ERZA(U&sP;pT@N7O3S?`&z|@P#@z$$v`AaGDVF3sx*0<T!fNed-RO8*HrZ+Wsfj7+*bEi<m0ZH>&oLXX%#>AV7DFx_5XUs`L=@mTl<#kp$8R0H;wvd>#<`=uq}P75~5kV+rKrB+h#PfC>k{B8RdJp@&HLx4o(4SBovU(=tY&9q?;2$d4c{7R{`8}HtDi=3SAp;l{|y1qdriUHM4-opfqgi;qv+!6Oca7kIN{XRX{PCn52EkeQfAQDeSr4>|RSZYqo7hxBHlh@zEg&x~MqO6<sx(n8kO<!s+QceA+>c#IvnPGS~+PlU03}HX-34<TwYqFI%Dt|DNJ#ExIa-_XTdR>k_7L(S|7o@EPWSUY;yXBp({4<g9Oe1P>jOq(`#b7@e}SXOwNp`>@mw1OS?iOSaT)uAl9$!qo5WqGOA;p5Ygu-yt_axjxFZa;T~8iiQ?6GOD&*pXwIGYu2lKu?{Z=yPmt_W$DnF#71{H)ws`tB+@Y^P!(GEAwW;t<IqrJI}JwCQ4!UauJ=bhbg;9IXD|nZvs(5Ryhv4S19pa%9k;jfW35`Xg6v-}8}}-aYOXvtSKKO<;xoU>ewzP$Ip{uI$-sRMN5)yXho!=>>hyu;p)GPwI;Dx*e6{W3w*;zsjU0l9_#0<$MRRG{k16A_l?qlDfr9-9m;u18zfSJq2;z-cWMD(QTkchcT}<T{uxVS=yeD~BQ)fv~_&|k)3*9(a!J?w0#--y+N`WD-RaC3c_YizOmcqIR6fkeA76d?Klu86+aBZqtggB+5lq`YYVn9y9vbnmpWss!y1DFP?D)R!<>4gqpgMvO};V)8hqahE1oyrwRT5M<oa9M_s(e5sd20LHL@s^t}xvH0C`)ULtqO>>Ey;oQusN)m;vZPD!U)Cs{<xeSA8YwuqAVd*32%vxU($)GR;7{F6YQA*sD}f<E%qsEB1NYeAQOPBv0oG4@;Le^8wR7%;`c@4)+iRyU_$A4x>?C;1w1$Qn%39eFzk##~PuKW7yfsN0H4;eb>Es}+YyGZGXLV?Ea@rNxyr>~QljV~03EB-NVJwPsFU(c!Z@jX$BHe@unt*4ueL1rdEQl=%9R^<6<{S3gEBo7qXY{ex7&WHiuKem^T;@f#Pgx3}qF4%lo#gl}8X5suTR3_J1=8&^04%`%dWA#@6xY@3tyDiKoB@dl0*n%cZ|JM7`jaHcpI_j@jX3}O=?YtvigIU=sAH(XvZC_>M_^560DQo<E%d0AneFUoQu}e~H2NFf#pT(%_gC|ePW5rz%)Uag{ALb+Ea!p!Y+T%QQle(=*nFW@#|-yYX8ie8`{t%ayV?*agvMv(r$M5mq!B8Zwp`q$EhcbL!-X^KR6uN#-V01?q~3VXXup5{;PKn59xbUoqksF8<u^dij>VP9D<9qQvbS5U+W0BAJ@^k$O9KQH000080000X0G_sxvXuq^0BsTg03HAU0A+Y^WpZh4Wo~qHFLYsZUv@VyaB^jEVRB_;E^v9JS8Z?GHW2>qU%{y#Cat#G4cIWGFt8-+(roD(1lxeYFa%mU*<2)1C8;ECkpI3Tk9zY<+F=F~*yO$Oxx44cAPBA%)La3Bhq@4{Fsy(zPq(j9Ufprc?gZRx#;c5pvTO`r6~IbSi4`~L26uJC3zh>njRGA9K`@!*N|r21^2V+u2`bi78MIMJ!;O%Yo=p7OhU-lsR^9KaDeEmmw^dDMt}m}z{mYWCp%pYzr5o?caGj{8GNObygH&dsTDM9Hf2%Cf0NK894Jw7&M%0R5cPlrbo5GMzOdxofaKiBO7cQO4Di`ai`<*Jp01_ujE47j-Xf0K;23dkp+k`NsZGt(GtA8LhPLI|x7d$B~x{xIbftMxz)VI?~G?`2?$kDLNSRyPS6pcU8l)1H}=4LZx{qaur4mM>e1<$^jGVjHLG=DvbPMpbmVK#($ENiI3-g0CUv_YK$NYq8Z@7dh1#%TKvlnH~Ii?bja92Z6Q&m9hp48xAyxmnJOV#U+j`EUme&$!`>z}stuf~B)YPCed??6hgXAdcqdbFs9g<I!l%41C%XB?K-y>dPe5VqJ2~C|aA-emq#}l{Kmjt0^3xf>h@NH>6sN2PmxWW5r82=^~|dO5fBYGXFz05V2e;W`%v(Fi}weu`p2TFnSsoUIbTfFHX<@PClIdI(ru^(b0|vN(2S-f_=ETykKRcv8SbBS`<*30$V%%*a&1B&N7kbpl}sKFyaneSreYNfAMHW@zli59*HDUiM6PZaQ2*Iw#c){okPbDF?>XJ3U`#J-HB|B!7}QL7?eUt=4@Am&31KSzCwVoC?4>?g)58(l_NB;Q?0QDuyWp7qkUc3PaeFN85A?mu?{Q3Lo>%X9+LB`%j+Fn!bcb6QxpfC$!L_Pv#Znd-wq{+nz_P;bA8zs16JWDh{mNsP`Vn=#2PS*^Omb)z2V<||3g^SG1puvzIEWoMy|F75R@q1zyoP{P*i1G0Tqa7kfd{t9F*Ytpx=6?4hMuZ9fdJv0qxBctP$JU-M5}!yr-To(c>{H4Sg?H>yg&8%{OG?qdnL*l<lN2piaS%&)olCE@poQgJNs*b#war&mZ0=0ek6mmxqeGk8j~*4rI&o;1?p$N3p|j-Nt9q^<Pp(q0NqD&Sh^p*)L>II3Ay)ywe#uFj*Xl?;|rR8Nh2w#U48|0oe^sd>}m}fy)4H45qwWr(`(OxI4x%Xh#+9JdWTC2KD2O`d`>gh+c~QGaYNR(+b7guR9WB5C-yrD+xQMvH95wuvR_6;KdCY;8m(*{l>Bl@)mMW$9>&oJ?yF5K4D)rpLfjIu#((TJ?deS?$J<QNN&QN(nOoQLV9^tpEA*z_?#Y;wXxvXk+G7pAO&7+kNIhe0>uZSMKKb6qOVWwRn1mVatqrK0oh1w4Vz_80x#IP9dZ(<Hd1`SQ($HTf+z&<+g05dmJ6j#%x(Y|E&p>ymM}l4RmwYD^wxyYOgFW=q~@Y9plAR~w5smyOzM(rMRBMYnabj8-PKauj!^OurxYKX&sng-0B>!58h9{)dlA^P*u8-??zzqzTnwqd{8)O%f$vYEhk=8yaL-NM-ezy0J%qDBgfWiIv+&6daZyZ%7-lc<0Y6)#lm8pQj`M+6zvH}2CO^>~MpW*altNPCodGB6@Co1_UWt2+h21N_Z)3Spl-gbM(@j&|Cbrk-Kl&2H_uAVEUrhM?<M^C|2W+GEQD>mSn=O;*`VGsZ>yZ?tfNg}v*tW9_@L`bID_kMhSW@o!=Vk{^0j%ITLjwkfLwEMaBMspur!Q=rojz)}>TvAV=qdIz;P0jq9~(S#Ayax>+AHwW6#w={GKJM=!tMz?zynS!yuw)#z!xasg&e{b!)RPO?f~e*DlFPjv81Uy5<GTqaw@TV=!VbUng+%0ZP^T9jCA60SUnO861IcVxf+ReAT2_F@wdm1tT3rjb!QgD0<#>FO@F)buY3n^2d*|;xv;Q*@j3e1&RcvfW;z;H+yF%l1&$_H9w@aS{S$BJ<v^#$Q#QtOVL$rYP(tbv)J<>0g$DquYw1G6FE+5&;WCOwsmJYUTjePBNz`F0I`uAhlHa-|7jv(C_!>d4uwi^7Q;W{lv79XX`t0Gg*mji}Eq9&(hYNl*k*JN54!{wEw0z-@-JZ!2IvYtYjx_S}nV$U@P)h>@6aWAK2mk;8Api;4XH75-0068k000~S003opaAk67ZDnqBb1!sZb6<8hFLGsYZ*p`laCyyIZI9!&5&rI9!Kz>EtF3)Wu4s0n0EfFxb3l3*BteUIVF<KD+saC!B2v5SF8057hVOc@Z|_i`-49-g91e%W;mk8bMN#xiR<K$Y+ae>Pt18+SOe~3PX)PMsLKkU@l1n03$ue2+n$S8Y4J}%hlcJ(K1`V`42qxkvie|HI%PW$m+fH^ZOH-g}cq@UZ=8_@<F`JpTOx)YYg4?Hx%DrZi@zTsGESH>ZX;;d;$fTxefVwO;w*My_Y&q3k)f@;Fq;8B@l8UtLV7V2GX$z~SnRL9V<W-TSzX@J1!<W(wj@dmeJGBHmWIiMCm!dX`dY9g@!&0@--L7RjDp{J*1_z`SlWma+-L18kW@RTNYts$qQh>&r<!mvV&0vjoEB?*qEz5YDi^Y{1S!@ZYk(fr^hJP&sZP(-}$~Y`9x+3RG5(&_EC*UQjdCEK4bW%jmGfgi*jU+KEI4QFe!I6R#5#Rg<djmAJ+MtLOwH533eC<eCDHw=VTc#*Z$!f<_i=$cpk$fz3))GPQ!L|$qSV1i*Fo%tR&8$G}pq8BMSj}3s8L0ruIu|kdb<YkYqqX5CWKou&%BF#la^Lc9w+9_8$)=M?hMHucq)l0fJ<Rue2XAK^DziN)K@oL!h-Z2cv=K08LT<gr`9r*8avnL6(y>U<($h`$p2z14uh%Vakv>qR>eAXY5)Fm8jg-Kt>2|!=pW<paa(cL;<`2RAlHV~nh%o(dNw9R?i3?gx%x4)$TEcj+?YYtDA~43dYjWV_4RWloM6p;p3Tsno1E*-uyEc8GZIyiW=ks&&s%K{8%uEj})(^@sKk%9@hZDNxBq*tVTzD$opr$nQC)<)!Ij@_zVzjm+f+5Z8^K??pek8EV@F<mRZfKU|b!ZzPxasJht^N|4Z(e`NOy9iz?3pZot>IMpodi{GItI)4r^E=P6sY3Iq-e%Kv=aL3`(jqO%77K4ci_H&*&*E=Boj$RALq`tN8QJaJ7Ae{Aybj@h9wbNB801qqep96wgq4GC;fg+-v0RhC$eP}LnK67C9E=~B{(hK>Y$oK%#~dO^I)uD_zDpjv|Apc@jjSot*h$KNHFG=-~$<X*z=N^z^z@Kjve9}A`1Z**s(J;@0pj1V8AqU+9RvtjXt?1X6?YiHn=M)b{z(hOZB4r2txQA9Iw<2$9na_O%L^65C7)M^+AN%^E_HFwI^&^E;*nE(=hV86{sS@z(}j6WKm%A>zFnTt8--ATcMxzsW-o!`ikg>Wxv-+az*UpfMO^XFoJ)5qF1j19Kat!Q+w{Y_qh?mPo}snlo|voRVWjhYgNXDWR;kk^{*}FPzW#G769j9UtHW@l2=Qj&-W!h+%q&#@R21v)EP@4h+Udp1_b!a9;QMDgl`0EF_iCVS#}xlwb-czCWssa7$%+%N=lyH0j1?m+OA~rO!0hc+X@5&6{6CdJQQ+IZZ1|o;NsA(VNcZ$Y;5k-5R;onKv{4A8eSHFbam*<0o&E+=*b2k1;Btk$N+#KFE9cTP-ulGL&;Oi7S#zPV41YO17rxgh1JOiuF=7-dk`#V_pIa%E=tLc!Xmz)L=?Lk;WerXS(~7-`n>}+%LCbUaHwm^SZp;IB0WUIaUoz+V9(}b<u8yW5}uQ9lZnm2){6iq(gHmB-vP(6o3`ccJen9%0W2jOa6v>*OSUc5&>wVtd%nC_$I&7b@<PuI7J?6#r>67Pq3P{h5L<91s8qw-Wg|nxGX-pp`JOPS4O|3C5)%FJ=I6hC{P6x?H}4|laTU$*Xlhd9Gy+7lp>Sjh5Ii9@dz5oLqoGiK1$K7puM|VH-UyB;##N_I#}n-5gHZ~t4;~d+0_EHw;aE6IoEaK$=6Uq?hd1wkF3{PjCyZWE2FAU^cxLz9XnJvC!}?C^O$TWaXa;gcyN{6fDCi6sjUW(&@Njyqx)Z66syeiG@;Gw5wHWRIzG%mtaz|V2lw4p!baRPIzxhi2eyx6A<L~zu>R0uCuX?}#I`Uo>jRCw?fBiP5Shb_=qw0G^U%U-o6YKDBd9}QFn?7*mC`*P|!EVY=mw}68kLcl*tkmr(IJ<4pR@zIueE-Bg#TZrm+_37Tq}3*;sHrP5Qr@lA$9j3BOANbKs&A~aNix}X%x9!ilQ_UISd&?qiY|rV^_QFA|JY~(@g@yNY16v=oRyNIHzLQhOjgd}O!gmP+)yC`zQg&xOUP;>?ySfKTU}iCdG@eFYOsTHp{MMNd>hE^8nte`sg0?^ym`?V1aXJtslyIRgq~~0KZPYiIg-9$Jz{h+;Q%olAdU~1+N|%6Z{&zgU+hWIPe`ul+p+K`WbY4|6#r$?FGeRv12{B!e}H>La4rpvK7ZkX9&9sM!>-tr?3jfOY<#5GlSVtDHo{C<*FZq+k985MALb^Hyo-i@J?q;mg}K&Ye&e&6NC4ukD3%K)6)P9uSNE4ex*1tHfX+S!y<Pz5_b&m3J#f%(A4l9#X$YcG7MBqqPR_}_P*CF9W)Vc2Ct8-aI!t405AbQl0iako@A3na4P0wTuX&tl20q0(!+1h$JXw<W??Oe49T+Zo)HpSw15~Z*vUDSO*+D$EHt*b`!`#<cBfNlJM~G*t8ra|PqL#glan)Y8b+Ail2hMa<83&8O)+k)E7cv6AtZl$Pl=ULRFu4&vy{pT^HQp5DjBmF>*|SM*h)r$`GKBy41$$5>29Gx$iW-3SSua7#N?KG#nrD-kGUS$)tlr6e>W}!`*^jq0pN76;B5Mnf25iLKR)`J~QhALyCofP;#I7K$E{zjCfprlX?F20`ljPgjCS?GR1Xj(T!jiwL8C;P8__xA6o70PvzHFph%(I1o-VI@aO$o6ee@N7uoyWT|;DjA|N=N5a6cQ?4F^D4Oic$|U4WF4D+~78?j28ViL2kryO!SN2JJ>3)i(PH6j?SQ|J)8;V>-{6RY0qv5$Uwf?1awlX3Bj0ZxP{*JZDj1{0VmNJm|pB;cYvtIj?!G;7=g`&hTfIn7#X<-WTthBP-!GB6salKJb`St=j(8dhWdFi?Ng86;@o;=Oz>OxQf1S4s{ZR;{pnc)CI45ZRWx{hY=BUOpOH18Gh&b6LLmWC{)6Do>Y{@#by`;iPQK2xtNVl#^bW;&RO?c@T2G>_NPy8)cxPp5Q|eilaj!|vnf5UL>S^dTRh3A&pNeIBS{GvI5+B+El&x9nW0-?YW)eck(paso-Km#X`rnooB^I2MxiKF0*r;VBUdOO+6gHIxP40RU=VMfg<l_ANeAtItPoUNK3viq0DDgga?nGO-uDH5`*15i67Rj$Ix(1b$ru<I5hD9K}^g3DLF%BMFgg&!j#PF?~L#nuZQ4!h>o+?9sZryp=PexCbp;dLaU)5cjl5<eqwZhrL?$xMVL{z~^!2!!n6`_8-`Rkjv|BMVYP$jAGMD|_0o)^biwzC1cABpP1lQQ{0)ZxhSg|ZodYYcCcgT?8b9q8-MOCsCDkivzc<S3RHr7CW}k1t8@wm6cUut&T-3Bo0Qi2#i!gnJ4=TvuzOM!RVl70L<0QJVvcg|AbE>#@euo8W1*G8ZmVcZV@X;$t_Tp1kpR*{z56;K!~z^p%I#NWb@aB4fkTp(D-bR8F9=7wrcYH8P;y2NTD>-56WYi>+vKBwE&Vk6v)1{~Z}@9QDX`Pm?R(-FM|<!o{dXt76jXj$7Z$62aqUc*6cZ&kkQ^kKQh|1x!;se?nU?^9BFcI@RW;DUaISAQAOG>^Swvw|$<5P41TT3K<$AAW!|Y9AbAg4D}H79El9LkgzKlVeu3~d%zHL`yN7b3$fhyS62>o^(q6pS|5^ypRa`Y*8nuV6Y3Ge=hJgQQx;WWzO@6gK4OX9eti2e_!^I>4+k*oEov*qH9QLxlF({X>^j~FR|bg(o2fnRnf0>1cw*33&OZ%yTUOF(<{F+TG#^|lb%0g!V!LJP`@|jNXp@Jtx~mu_9SYvN8{C!)v2vJ4)fdKV2yZ@G*5MZv7+YkgY6KhDhKltrd9R5oosc~%^A&6wQ5&IQKPJ1NQ^<0|iPjdutJ<Icyl}3d@)@24MDsDWL5R|ZuozO8j@*9yqz07urCErGCx=)AbwixRn(78DrhoHcF=}{%taA<8Mp)N(1z-LJtu{~(T6#1Sf!lZXuX;k;4<maxVD#Bmf3B%7$$HB^_6=8$=-B61Lry2YVd{~NKSByB6W}o!tzf?j9{(b-sPz7$soJRgu!XKn+{f=A%lU_DnCl@+v(=b#LBtoj(wIcWuI8=1S<)<G^>5E;4x`5%h*IXp%j#ZdY^F5^38ICjP9<KGYZ)*}!tZGrB%=gW#dDB9i8{Gm{Uz#sf}>Bu>_1RT0|XQR000O8001EXH03a7#wGv&@tgnv8vp<RWq5F9a%pX4Zgg`mbYXK}b~i6`baiBTE^v9pJ!^B@IFjG>E4bXcIx(STCp){dr;M&DaUMIhnK+eXwr)%1V$mXHaZQn0KJ0j0{_och00cl#qB1jg`7mRVK%>!U^c$e>`=8Rf-NsjW>aDY?ZceJQEfVkcd#`TV<lei>nvHjrm#dqsxb{Atd8<vjx~aVw8kI%rz5DpHSG7fxZPS_W`;*DKDz{z~t=p!p(kSw>?XIjEXkU~~++<}@PbT775;yTGkLx<E<)eSqWg-7A)zf;@Hd!uz;aEJ(s?76rML^{^NxF{Pyh*ZE!)<nPv&pk7`TisP6i7C49ZA&Vs>#;zs!?#J6A%8q6p!CjRau4fC~nGawu%r@ez1y*vdC6(9&O@!!_VqX{QS><iPl-3hLfN_+^Q_r*|i+&GgA7T9!(|_qKNl7t!tRqt2Vh#n@?#BWKX7VzOK?8PB)m-FhKB^-Yy5)rioXng{+%$7kWi{n^s=ztz*C<fwAK2)GI5mzR7kwc#wXLx4T@d+63t4)n!|)(kM$(z_PiY?yB-%K;Eb-%SMcde|ig|qWPNM;(Tj({qV>?VYDQxpiNobds*R4eT@iy==tf_U0T84MN^;RtK08=F_7$!%ZFWSAQVtwN(h{GV1wCpUS3W8C$rtXA84HwKFhkPrU+(Mr8#imZ5lNt!3ly(Kvr8TY8EKmi+Gz7+6t(j=kaZPx-FA5KLxs9w?NcL{KQug{C688F5y5YY86+j&7MYE;NT4Y1fEKZ6+Eg=S9!LRKaT5p)n@tVPGU{V{oR0lfwyYf&O;6ACWcjYkGhR;Kie<UTR;G~;IwAO9C(|-*aa?ql%x&eI@B_5uaUJ}XjbXJTVNsPJP;%C)yY*{r?cj3bD$Y8tk{Gj8pqo!AYs=|XKvz6WHFlw1&iY==EW)nSz34xKK1eE+!wYF_kste44TsW;8ASrU6t<QDxI>>nd{;v^qzzsF+sifQnhIaB6R~htzJz1m$*S+;eXEYKd;s90)cT3Qh&EWf_ZP#X0`D;`H2dnY#WbQplU%ngWiaXd++?`*DrZ25;;VJo)`vQ(8=JRE?&QRAASD#=KOQ$CFNa_mvIsS=Vj|GO+q=bh<mvqUI8%yk9a46%1%YTbN4{4q_oAw$2ZZ>pWegxyK)Dj5rrrS$hsSRRj$ei`%HQQk_kQoj;XV{0k&XKIRmK&>a(rWY8t3cv+WHCwJE{`{t>H@eua)vc|$+B?Jg}6AZc`U-;k_5qo>(gETbtsk%vOB-lZ#)V;VhYS(9$-sVy&|3%($M3!RpN<1Wo+ga*pxg(e#He}aR3UKo}#wJ&%Y{>QMZ^fp5s=U>7j9F~W}HvVM~ZqR!Ia|WA&FqsjdU_PEu88Z++%`|}jMRoQ|>iuCMM!;XL`S6^8i{l)oMIexqBgBTa2jTwI^Jm!158M!p#d*5$w((atiSXZ9IQU9OcCgWjHx($`(*@GVH;OW_1gUoxxX0|Kv#!#V1QLqqS(4Q^QCkDov&Nj^>xs4k=8VxQBmh6yr0|}dJ^${>lRuf1ui^}l`ouerjI2Jj*8WGefzY(6(2gn6G=br+P|`e12gVA3ub!p<HJ?3OKYH(8!49cXMz7w*tCZNi*`&O4PHErst|pBUdRG7(exp3}9;lxIZ|40B$j4qDSJz-^b=w>Oqw4go(lw4KB^O{g^Cxbq@2ae5%(<`qpE9rkL3)zOy(^)G2a>F)*J(BP9?VG#%$^fwFFYwES%M%Cw#`7gz?4ArBs1USgmvE58x}hg6F(wPw2g~woz^JaOu0KXce@{S!iah91NaF92t=;d8-A&-@VD4e>DN`>CM?}cQ1o%JLUvqUhSI78=t8hkTa+L*Db9|y*pRivu+_Swy&MwvE1LB)T&#M^K)`4OoztXWVSDMzC2?E&D8~t6B-F93&lyl)ArIzyT7fN!dd0E|XugIVzm2O5%&E|8s~m;1(o1eRE`)PWW%f50zzdbQ#vOoC+DDx5kF`(Q?LiFxI79N}KCR6)OrTy6-%f_*G|gug?~qwF>)Ix}sa%Eq;-q%DUT3QeEmiCynX=Yt|Inia$?kdI(l<N|8dP;<USn%05=y-zA3jh&B<1u6NH_ffQj!d-lLxW>^)AmqT7w*WglDGxCY&AEPE$#xAPAQ8XU{Jm87dQpMAo8jigbu-6X>=z4!Y=}5t=JQJ7uA^WB+B;Ez6?&Gqi@T7vlfji?~f-$W;kaUt*DD@pVyxpjz?sP@81WKvRMiwz#m<uG%7j4Ty6GqD>TWB7OlhGFwb_x_F%f3+iOoHU<y*HHhDB3`RCOuP2&<IJteMoBu-m%`JJk^rb%y%qb;fCry=v)F5?hf~NiOI{7x)y5??za;J_i&|dpxwSn0XeIn^LLu+)UHPSgeY6lQCe;!RIh}Ifaz#LVC;Zb!mvYVU!RCoIn*CL{>9i%k|J(?A$J%MklJjS8@wzw%kV!rT*)?};JAVbN>gr$nF@~qx?Z2^l=coG0jNw9&?Phxh!Mb@@v?h5QMlG1`2Z`CY*A0#jqy&!pY7=QF$dgO{18EpnK5y0*M8GZkPESq(k=l2*HD)U>Aw9pvslT(FaR<m)DCIVo54bwtth!H0d0z{*(j<G)l12eFO7>jx~W-2bs+1y#PAK5hp1kv+vKvq8>q(|u1*(HWE%+Q2~3ljvpFpciY3d58U?{H)mp%@Q1n)E9|MNp6PDjgz7m$|$9&P(ldhCoto3wzVa_i76S6XEblyE63haz*}0H`r)LGjw(#fP<}t{U>BqB+LrU&~>f!MDxvzuF4hZjFm(7e~&?cVYh4*&#)UYISiD7n}36B3++$E0+OWVptfdqOaALX8b+UqaL@LWE~hr<Yv46y`u>WUC0RnSvx3AG>j_~$(NyKMWPs)51S7d8!qOJLMx5h{lEIIpdH3<>-rUjVxk*JlR#P>hL!X3}4Av%?t$Snur~!J~@ORvhrr;P8KK~}ak7t~C4*=m2D6|I8rxv?$gR+YTXax#HrH-tfe}_`BNJ##XDdP0Pghc3hH=Pw~u)SDu-X<tHMP8&zS828bbBEomy7R<3`;s=@*m#eKDkkVuwA&6HXM~_1^I<%f65mVdFiZ}{d?r?$#atnlM88ptCs3k66FV?$KL|m?aumY*p`(o>yl`*>zNZ-oiSFSTC|#Pe^y)2aBlf%G;@zd$^#<5u2inLJYKBz*%sbE17<BFx=vdHPf-p6wZ)%K3tu|$;2MY8dox5|~3fk_j6wf5yWhc^U{w2qmyG8G^G7F<O(A>v}2iR=BWC%7<wN^9cSxs@V5r`hAglkxa&Yd%Y15KG>sMWr>=%&<Xw0_LXVqfH2HH|U-*_B*~O(=ZN&>>dPHlU;&_!O2aL0jg85g=be^C9d-f9zRW_68Jiqym#xmmHt55vwTuMtxm0VZI!&|9QhEDF&3J?o?=~-~qY5C;+;#$p4-U8$r{ajT7p_gni_s19KTnEZfQAHgqe(4$CgkRMxY!xXmhr_~HGRzelfMetF5U8&F-24#LhW^5$kD!dcA?WeaAsD>85ls%Cl?Isn*1&e|PxOgk?A+uP`GKfj7D-oAbR<A*m3U-vA(e0ure^V^G0KfU?%+1=uw7w<2kmp^~G`0>N%FE8J}U-;`7Hm%kI;a~jpW?`VVS)Z&hvFvNc;g^d~=YMmw1@f*o{pK|C`NfB~Kfa4z|M<!5)8^nM<@aDt)V^U+%7j+uk7)}G+^<Jr9S%1Cp}k=#H`ncWdw=L#!FGm`%h5U2I8IAhDhnB!FmGxDn-({s-f^<<3E4%7uB)=$)zQ^G#!a@nCIaDd4U)Pl@5t%EXSEROS1q_?WQ=I#3{!AFn@&kUKNOhQy_1uo;}T#`1;{0|So;5N(+d3U%jL7n;8Mop;Ek~!OTv+$N1?|*y41UXE!9tODQ8x%VCz*=Maqzn(mk&kres@T!Sgmc2PtvBiL>erzbY%<RP1{%2*mR2a>*8|5Ivxt;^KZvGZN3y-sC<OQF^xcKVTfEA#hQo;Hgte*TcC`$X!*Ic-yvaJTT>`+y#83JX^tN=NIP}JaA8i3u^XfI74DYT_sF}2B7JdDI#L-!WY<RU21Jl)94HOZ}(6IQ5U@M@;6Y*BG9GsGG)mLdi!%B<6&K+tN`C2n$~H)4n1e~ST6xtU=&NYN}!u8bXkb*=1aY|tEdExCwdrhd_4+gB9t)=gb`9NpKE3fK~=$jyPTyrrroiLk<p<i`S{xsh>qc#snsq4uQ1OF8ba^)R0gQF@z+QgfYCN9TFlQp@&0mVf{|w6qDzo;lu<<n&etS{op16qxkew!$RJ28fe~ani)wV#Qj_ljDiv#uh)kxRXOqMG?}bSTmviSVVCvaXzGy~cc#HvTAZq<Wg@drT3ARGQ`70@66Y%)b3WES87eA#v$4<=<SviYy@RoY>6GdgH-zHiXv>)-2D<TUC)Dy*Gd%N2Kqb{rYVfV{9&UjDvs)QYYf=P51b_CX%s95rFcgbp^^Lwnq5uTAXWtXHz8TK9=G~jNl@iuKXWun4pg2<70Jh44UqVa6iCh;t*BfufcQ6EpgAvoZZ7{ZB(R4~>dqCx$FA^^($syAhuCnUaQQ~*ohP-6||Vy|~C%76}8;K?WfM;85`s)VI%vb3I_1^WCnX*UyYX%s&@JEQ2SrEp%tb6*5|a5OG00KD+;1|&U_aEq?Y58n4@AUcF|+v%BaiKd*3U<vFdM8Ny&*}GRF!b?F54DdckY#vV^ddQqACmR#s@3L1j*yN}s%^-_pHkgmpmK1}&MI^B$%5W0VV3#%flHx#CL}DBN3t;`={q-Bz<O_z0O4y}a*!qc>F7GBl2K{jmhA+3Uri5^j-lf$G@5^_xzyNS0OEiTkz|P3fYELA9LBxp}FiL=ij3YL!F-zA;-)ErT*4bC8pe5OCtDsJI=zE!Bm?jYvCMpyV_TTF|(R>Yd0oc=JospQ!%j>ME7dorE8rby%?7UZLFE#5^iUmtCq)UAJy;YUqU#CT?`~t0Dcox@W6zj-zZLKO7--RY-iAK-FNNB-2CRT6p%Id&;E}QZ%s|lwAm20D|mvsHz`7#|Lv!CutT`v1sI2%ZvU2nnW#17V^v^Q3t`!X*hY-ROO&#m^9S|EGKHIDVx#t(NQ`d*+cjpe2nL9ey2+hHw1_F&TSp)VH9KunEMVFZ72``quxdVH)5MKcH6zj=1@H&>e#CKA!YYRVk`D7s}A*!CMZprGZr1|JlWu^CihX=ilF&9vEyRWN{9@}D5eBib-r^uP~2%Jmsadm7mM!HL;n2E+GuAI&_0zAj^B+FLlXPuF>Q=U%e7ZOUzosfl>qq*YXJ%Nv`<*aq1Tv`U-m?LsF!AZYfBlr4UF2hu}VS-4;o=8&M-H7<%ru)IwRlQ=<zmfm^07;K9dcrc;@j>^E~kcTB)T=MZx(Z}zikAKGTlB`DSC-DbY@=^WuNAF#RCAHga)|~G0xPdj~qYWxUz1qaZH7nS9mEm4rXDe9=h3V%P8Z?l(H$d+CRGLjWf&ubk`uJ{>7FhR|$M;mbiy9rAU$A6KfDXI^wkqs~nIYVl#+qNwcI9sBUsIh2BJGFdtOSSq$l3LJ&_e6Ub%4;*iXi6td3S6XEa%V9ESE@xcG^OEa?Iaa#0Rq$J~W^Ju{K7}TT@XW8<-s-8V6BfHa1H%fXgx6_29C@Mlk1iN{K<LDmW3Fj|0v|&j;Vg(Am4S77{7+e62<VSL@MbL&WXZo(XZ2-p|o6YtvpKGrS7<6$P<eighq4Cah19CVlX)@(jbsr1$Xu+wzJE{$gZPpU!4rn%IAI>|M*Dk5ov+;XqDdj-gwl2JR3*--{eH0ByeR#ustjaL{ygF){V2Pdg~Cg3<^NA>A*Y{l&GQB2b0{b)Gtgrc}rYRi1ukVqdL3CF`j`G3316y;3$@TBtdsS~06O%`O;B6I8{8q85kIkwuVNE#O^EcD@{56%lkZb>kyqziteAWWvTBnKb4BW)}p*fUZTYk3YVb1o{yNJu1XuveHNO9?uNsmIb1{Wq9Um4PV@t6DvleRjm##NEgr4QpdPO7Z!9X?hFSup0+3?8y4DrQdJDP3o1lNO<ERJlI4c->e$5LST8}H5`-XRiG)Je+RA$Ltshj-m!)<f&<_~)mZI+%JN!b9UM3i#S~w-pfU`?QVZ>Xra))^bs;rG`VSAX!@+m|ao1O7NvS#XkdiP4t2p^1p)%V3}1C|4L^&JCg4|Rvx4C?}=)@(tm<2P&*eY^*Qupm<jEIzE?2XccK4qdPA%|fHPy#kC@X<h5K?MyG_0v3?0rJWdq`Amrf!2zTh0tVP(u9?*MxK4p=NgcHXu~$M#OUK;26FGN>>X_|XV)alC6<1Y!U!yX%#_kIu-u=wZZ!yuRi6v1yqe*cq8N-pooh<FKPGx1LjN&RkFj?|=SL308P1e-nIKbKLEcDKdDgtLt?9e{z$StPCRPC=G30jpi)9Y}wsv(-dz<#S-g3;QI!y>QWt#XKAEA!&$6c-6dCaM89+)GD+alf7)XDgG}rqEpw?Yd_R#?I_6))7norvuYbe8-h_`}m%CVDMZRH|o9RV^FiRJ(DCRd%AEzV1|?lRHR04dq|$OV_oT8)SnD`$AY)b<}`M+;W*W%AB$oO0>dI{L3Tu-`;BTpQmwt6^588<uT%vr)#L+fM7u06n~81RF#qDF_FwwoOVPnLoqedAuxmo^d2p$n-r%jSGY1H%&iFE922Dki%Ktzo`#R3^w4!q5c-6GviG#k7C65I-i<sFV7vqHoR%%VBIx11%3<9c@RnD{h#!<-sdtJqM9R(WqksLrYRAo!QWIMuqIsfibRS+lm%W=Yz%ez<;VgQC}h%lJoEbh|L04%5*ra@6(OrJmh4v78yyC9fhdDOJO8PtSk_@60431(E%>O2QH1$f^LoDl>Q80FD_%9QedUOsvclq~QEI%Iji06#GqcwaYFd6P!m><*Yb-QVyoOPbB%SvcZsXrrGyYK-Gvl9?95Xn~E%_D2Y__nvq}Q)KUFA#E-K6?mtoKp1`<ShZ_Kl7iPp3xt*PMZ3i?bSiq9ODRU5?c!RrfMKTu!ut<30zBhK?olNWtjeDbxaY)Bvt4oR2YtK5+U^dQQR*r~nRv2TMegW<u?f8I^!N|~s+fcRktGnwkg=Zix^kEeGbsYa2+MZ1KyT`9rP{!Z!;(I6TL@>9MHo+%eAF*IimZA>32EoYbadjRqPpvHo7JgnI(o=tA1O}hzp^CHR*c+}qSc%)ViHjx8;Qhm>`Uvc=-U5y22!rZqnzIR5~Z7$eoa>`IRV^h?mcM0U8gB}W}Q|Pph{q-2CAWMSzqf17?3$S7n|eB-Ge9Q0knEyo_fM4Bwjnfmp}NSCWIuv(P5n`!**TY<5T$wsNO{LCAhYLOrN6?_`8-)1VvnP5void|Ngbkj6+F_{8oCvmS)OF>omp$qIRN1Jc+L21_)#V;ldCJ;trIjEEbuV%Z9#2yUWvlDuz8)`h)#cQ8`R$8l6ibEsDD|1BU&Od4{Prwu2Lw5!_Wc%{{B)$<dhW;6zJN6gLFZ0O)L{?%yzu`;IZ@EfxFNGZHjcM!nNY@;d@P*lCnDxtD=f+02z{q0@ZE{^cWgFd$Dn8t6SfwIo_z4kYyG|G!JRmt=`JiCOnjkwQjFBC&9^&1-a_@tzrRp0LlDSm{a!TR1?p^rhe8xUiugj-K{q8rB~?JO<8Kv@<Eh88b?F(4TlY#fgt)XlD9Dt|K2H?#=)hb=+?CC5Pz7EWlaWl%h7Z1AxVf@>QoPSrTeR5d2_jWna2+!ft?2F?rxDag)Z~REl5KGtkeN??{RE3<Kpg8Bd?kbHTRuD?ELJuUVeSqrBc?;Yppv^6nY$0(^&i09i0_*bZDZ+e1uN)4HXIh^E+i3P&XZ;bopeSKhBtHmC()QR@Sn$~Z?THgHHgmG21(oWalEU2HQVxT9I(WQj`}^MUDp2t&yPArihxFFa~`kgZqY!;bayF<!OD$3Gi#@a-2;iw{74)KaWqa71+mCQC$ipSmrG--_Bw4j9_U%Y>?4qOxe(8f|M}sPe|wwG(f>nJCVj=3txDjb&Zxb|#t=@|cn(J@}<|f>hm>VWGL`HZ_*kX+Rz4OS@ssW1f01=LH{PbXSlHdoKMBFL4%{DI{_0_3Ln5`m)gxT`iF=A$2-OuM+CJ!;8hBQZVCSb9XS&C>~zYFL_c%BW2*A79T7axFzZivk&*UBm=h`9m#9f>N)wh*BFH=8@eiZP~Dq!_@noesC1>X@MPnX5CD65+w%D?K5pS5Bc1;Ex}JG|{|Blya5tfk1x*_ASV@>-ilG+OBNR(OwT7pcGCtl+?h4G@OsDbTj-L3ze<Z%<h8sWQ_EUOgCYU=bin1|unPK3xV5l=<g_aoQw?J9cQ(a{C4q+Uya@84=qmIYeP$<hmLt+UGp1?vqO%Q`nH}aM~xjrIH>0bu!d%&m>kd`1U8Twp|@^l}(%YB!EnIS}b4n3=RK6vd{&%79JswkAE30FY~Kh15rux+zEy9|T<Ftx)t2iwt6xw4TPSP9N?Jw=XpuT3%v(709_|C(O%AlUgRxPk|i(dwARpctbY2e|+6V6y7X9X~yg{_2P7(YC3b0eT`!uuoc>VR35T#SHh*OWi17ctxZ*t&>by;b^&kA!M^IRi|~ek4jzc1*PY}B#2>x$V+62qQf|9Hl?@}6cbR`hkv@6gTpa&QUS!%>SnqeI`!A%?8^ZB_TfEfXb(R8Fd4tCl#NO<bARUDxQ?vI0B2c?ipPtw=ywpw2jvb(&p?x<As%9kh*x;?JkH4dt_Q~6I}`n42qTO(j)Hp3+??YPsOXgfj^#P966`GuRtgv^HaWzhJtu_N)p~e9*XhDfTc!Daf{u`4LS<0S?vU!}Z;L^YSR5;CuD!Q<QW{MtJGN5!ViXbH;rn737G|)6gUVY}A>x!K2NbeX&l&|y<JBS^1MaLUIepW5#DaA&@~bSZ4`{-kQSEdIWP(as$m5KJ{nZu2yfjcE;rb_dXH?tF4r}fQsv`B%E@aPdf^lG$qb(^d)iK`ulA-_bs2&G{I)@444D%cub1<s^>ny`>vJTn59M$Z)0utXZlwr=l{c)XDP<vr`Cta+21QLhIAggy|l5$4gYo06R@>ITHpLMSy7&IJ36;@N;qy=506xU@Tt;tc;V3Tv4T`tw~sMaWg$F&r6ph-ld<$>sh-*1R`-Xl7W^guv6I5oq3H23dyF?!-Y{o0TPhLa)V$zhj{f+VkISlBixu^0vJB(LCD!B}c=jTpfmPlNqeNgU*J{|yk~UQLOrN2m^{S6TBy$m&znuBiX(CXExXT!SJ_`GU8Y&Zo93tV5mozp=OzaXDvzY3Sd<p#N~kBa#7wK!~fNUx&lNobO<Fv8MDfhMjqb%)!2}?{~1*({JiLCO0zJ8Mb9puFAaYE}t9$WN+eSuvZs&jP~Yc_Ti2;^qghQ_<l+#V^@|OM`I0+5O)<7nEWF+hG?CTN2a4cj_A&TNAxd>)%~Gr(h-m>RZb|mbo|kPO;ezQYd${$Y`iV222b-x8VB_VFdE(|tsh6Vz6p!5l>5KKXdKCZZ+v!Xd?T)o;BORYl(AnOGfgT0j*P(j{M#_R!eiJQ9Wj35I$*sQ5Y}2oDi=2wCB0p8?kKA1MfD1n_4peh+9)1nyl8;N8ylCZoPQj~k7KPjE|VB<xOMgn_nguHrvG;??a_)I|2=u~a8!G8Qs&<6(VZyH6neK5h+|SY3z0^y&W*g?=Aolk*x5*rN5OpjtzCJ%<us44(%ct$%A?t6Df(aXO?B$$C^$wYo6legu;bA0*L6Hyc+W<5HLEE`b-V)O<Yuxz{D6YNJG`6p+j<VKYUgGhZu=JKhhEV~yaRtkA3C83c+W=#C$tdk{22f8S9;tgryR9uj6=3#{v9u9e7qi`!q3t956%59#V;)cvxq*K6-AG8?*Se>`r+|__Y|o9hQfm4np7RK5L5Xr1&vu9?x<2c2;q<3M{)T&)q&(zjsGc^DR8WB=3QL;<Yn~;pz4dD#@!6v_d;vD_PD{TusrDt&c+l2WOYU#eOwVEg!%<)7j1K~{)y~8+L|M`00mq2rHlKLP~J=^>(T|a`5J=#qyD-D+h1(g4?Ffg_tlRTJrMijKC;C2<LQGe7JHPR;xNv39GUH*4-MF-1rqBbu!gTo7+Z!VFKk?ROS8_)sOe&;L#0}#Iu%^Y0LMW2gA0nP2Gox@IBpYn-34sTt^Wd1L?SL2*M0-JYzz5EzlpS$7jA(pk4|w#9-W$+>#13{2-ELZahLTE!^etb`uvW+YO9Jq!YA(R<nPREGOX>!%0fK5b;9vXc}7<|&V`9*P|#Ybn@sm9w`2e%$^P=RE0aIy)q<kN9iqHLFQ`w0A{PMrn>X7TZ$ZSyPB*n{)jLpazS?)F`cR4Go>JLf#i~J|KWGGk{lc`FPuxN6{WmGHNcv3Y41{39PnF0cb<+oZU|m23FEDtGKh*Y6A4XKAjs(yZ#135L=5qhpb(?f^T5gH?<HWjlK>~@>sdk6#DlhBQd6xR9gJKio@MsSoby<-(=kTqwQO5#sGV?=|w0m{icJE~)Gp0#!;6f5QaL`>WjW=FlGN~$ac}ax4Vh<Ko-G8T1+4p_?xJv?RJTG^6*@hQi<$NPHZGEgwz*r!#R|ks-tiY+S4y$Fe@2{m#UMtK(4<zB1%59QJ!JJI=bQrel9}VwnIo06yrz}L*rweOPQ5YBuHt)x5^ixuFYujodtRc%^vSguB1$bsCdiDP0`QQCej<fKAw5uD!2`{zkZ_4m@K;qNO53etN(y<UXRaEJ<GvxFeU@=fsXJK}J@!`vx|M?PKe0cxQp_A_P;W^ole^FmaLw7eiqrI5=7oUE7=gXpxndlSY+ktSVqLFNis@NZH3r!Xd=;0xvUTQZnypdPmYkTbk$L;tNuURhmBPtYLgT0C00BekEG*tOJ(ufPW>KCRna)SDT?l08$bym~oe~nul@C#3Rlfs*x&At!4XLvcxv)P~Q52NC(jb*DUTdiB^JK-qrXdA^*5^A5E-Aoswc~e5uD)U2qk_+wcL+fYC5oZY&LhHnvw-~4<a-py7W$;$irg@CMJZLyR`oV~*cbBYn#2zn)F1zWURW~gcooP+%!r_u>zVPIgKd=i;(hb4EqwZ6+s9lA<yZxSH@a{!*8a;hRmIzEn(ao1z+I1)309863dfMqd{wM<L(}{yFy+h;<(%2+XwNXLu8acW^ur1sd-dNh+oUx1lOf;;EaxEOWlJ5%Z4_sFrx(?a%4G-OYzmwquNPL>R`-XwT;NqKptQU>AZB3^#K6BuXTfcV~@V<klgXdH_cuSouQ;Z7+Moqi}++}Y_p2Gr9ZZ~NKwDKLNfcV=5qJg^D0qjiqY|{67w70?vb)Njy91p@5)(b4~sX`}G?rCpT<W1Dt#fJSGLe#y^*5OBE{*+3g%aMWo`Re1x`M1eJ(LJ>8bW*bq_wUUSm+u&>#-q{Y`eifjwq|#n&b0>Ad}sj-H26xI`KXnZAE@3B)m)Mlh@`@~CiF%-(jfnOI#%Z~1U$>I7`wuZ&9`ed6Y0I0ZHxqWvx(XuJTF5QPWPZSWai@c#7AG4-gUFDWnUb7YrAs;OowQ%$prJOm-at&YqcZGR#*Mha(7wKKnC&;i*Dq-jRykr93XmGHQ72|H3teG{g)_?e@P2ZEr7V3Jb6JLLQJ^@{uGA{Mh?_3_9<gsxulUK?B;k#B=Pn*Ld@`X7?ub^w7J&y+rw@CYcoEkzDd%37Ksr%?NkVT0~du7f93|?_CA5U$BxYS)ZcPIr|XK`c@kGD_Fu`jmoEAa#8O12tP3}FA2H_7MFGpL+uTcUffRJLJ%1P-U33Sa&HWB<xyLW}Vz^S!Fe|R<gX;J4wtZX*Sc#9%6<V`lo-;IshMxZ@&Xa4O4r=kMaH@ryMcLY+4Wu{L%^vOX#LebrBes~wXh%1z7G(tb>pHv2QzgU)^bg*myYN3C9WlOxfo-5AP;v#tpbzZeM5+?MM^s)>{xbnpndj|F2Ep3`uji$8WJx2%G)yZ`XWnP}T0OqUtrAyRjW<Ku;b!kHrRyUb5AQ8blbL@=w(ob74;Z^s|IHaPz#IC~>)SI+id>E}2;59X+*54_7)rd2-KFj#Y0&?&iuZ%}r+xZhOcgwTbxMpazt>D*p*)Ri{Bjh2$r+r}Bwgini5VYJr|&GYn>SiDjI)g-sSX-P=R?Mk)6pI0&AVJz_k;%5ncecig{Id7o`D&7cZS?2zq<hYwZh9KxU?C7^|{9jpl0NZXwPt7&UsMl`Oy8sBU>N2UoLi}ZFoWyb?bJG`uM0CjuP*Ld#sEup;BdG$ZxxQn1BcOj=H4t|8)<!OgU*gMPwC46?g`TuU{BOm29q$S2_#EoYtcF^e4x*Bk?s$K%6umCUmh-D%hX4<mC|AH5J<uLom;Cp@RTxeSB3Nr0rb~NW~Fz+1a|gs-z@4{?EP^kUmla?_fr8)5s4_$mil)J$5HfW~=X?=^1YORz9xU*J{>#A00h0zP4(A2`ycxRYk{8mafk(d2xcZr?;u|t4!a?<7uZ{0N&a1&gOqc<y942LWRN<TtipGx9?8gVUvWuDF@qG<}7{Z?Pb^!Hpc4a3h&MoUmNVV?p>#-zV;M>3eC1ysR7p2T~?4FVXNfy-z~sv?n(4z*9A65Z`PwOm|DpSrl4_GE~a>W*WF?u12=K4bAlPz@-4q-#7((k+YH^3%Oz9fSS0Oshj)1Ko&6cvbc^S~nA>gt7f?$B1QY-O00;m803iSj>WD!i5dZ+AKL7wH0001GcyMKMX>Db0baO9sVRK)0H!pK^b!2&8WNBe%Zf|pRX=8IPaCyBOYm?iy@w<Kn%YGppioSEEk5Q`3B(a^kb>mFjbULHaa3B(r*rN!BAkU|G{NKB~07!rjoi0i14?YT9>@F6IePQYQ{`(Uv3XhegXjqeraw$_$@yx4uo$?AtJl1A;10b2V&mVZUtZEJqUS95tmX|5_s#@&0j2DZq+D4oJP3olulk)jaJb0<7Yc4BMW(X@iR&%fBFq#1sCUNq8E`bg!h(=z@il+_R75s}ubL3~50qnt>j8`1-G@j%qRwAK$Ug!HfCEh(zd(2B&D*;zq!4kZkXv-}2Ub8~-Ma>Tg1r&Hq@HPLb&4D{WSFxHkyf|xw92u!vwd`DK`u9K@-W++&<Av}0i^aYcCof6%ZPV5~Nxb}2iMp}mDHjVp_8%!q1)qwdfF<HvGa=)9))q~cr%gAZY){o0MO0R5-uG|b8u&LScHo+P!(>voAdM4`87yqi(xyYqU`O&aLE@49qliFIr6@skMRH{F7<ux@Ui{&Y$v!Xm7;y$$mgVKa;N1(*k1SETkQCyOH*!47(jq<Y`)=e1E@8!yJ)&mmWyM*}4yBN=%iU^%g$MuMl(J!<Ot0>9QmTkXNjVKeopP#oYU@OOPDq*6V9ihBo+oC*VU9v;-c|)mc_P_~cdruOkqkAL{GJspvzPeIk>&MU{*b3;!{ZZgjw0)pp?)U2bCPEo0{8|MM72{O`ggMDj8tpKniTfP4CiY97oay@0>>=LMH}q%T7p0&Y*m?sD_X|GP2`<#yfr9Md}0szsXZyU&FaP0z~XbLz;6NkR)goI1^!~ygTu1|2l>?ns{kHw2SDoK)>{J2(0gV^&Ny<uQShvX)L6IhRz2Qqtyya5^ZxCvrnu`d7-kSiKRY<|*yq-JgQeym)Usp$xfd|+Gr+wxL!p=NJvH`a2gbpOcsKElk^70syP!Yn5g3O2aD*NBkY~+NZvbs?bxuVp8X<~a2VgQDZu|>I0zoA|oIp*LGOaBmvhnR#{#NPq9t@LF%)kbSydsy4rjP$XV+ESqwW0><SRZ(xp7-_l=XbmRfIX4}B<^aI;CL}lcH^gDZ5!!tA7$WLqMD}UQ>g5T_O2=tWUzRk!NZuz=3Mb$2YV3e4LsqQf`i`+x;yAnSsFNQXp)ID%ci64Z1_RcIfv&$9reh27I|=H@F=ct;y*0(`oT}U<PER!IH57~M9G(pSi&E#U^`xXHg=P^{^scMhpNa^aK6nEULHk}#prd2Gq5odgs@S{K#1^n&oLSuma&G#2i^oe_}NNGSgiu}Bbch>3#OcW>qF#$c;J76fD=%Huv^d-SZnZeUIi9zfTN`ouF%?PEtoT~qU92!@vS8%HWca(8kDJ<0ABVxkV%h<&<St}Wy`xkWVYW4<k8v}cP=0goB^^x9kdclF4J0k$Uq!7;RqOL^&7ph&!gcQ47k7DdcRwD>u`{Ss}au8ipcDOorFCH^&g!39SB;V_ikCD1)$F8(~g0)W_8WZ!A4<v0Lul~&hY2B4a1>W5XF~kqpV{dNx5QHVYC$l`$DiL0KhOZdSbJP63`?d=F1PI^1@nEKuNk$=LD(4C+QqFK^U1oTA@iPMLR(eM2_1o!HDv#-(cc7^6vO~U9i(GV_x3y)5;5$_!m4wL<8Ug^n-DE)*g*ChKTFnZ*p}DSGTeOpTn7UTsWg6S}sQVqrLe!8X+Uo^OO+N{z^-38r)$MV4dGHbF{`GX4se!%qLg#UCB&{E@oL45RWskmaquc;W2bAKFLg*bqGEaD|QTI(`IK3DImoF3%o`vqlFGfpEM95V*C;K6d?KveFn)v+9+$LU?8BCfCJnP#<&nT_!cGKofMz$;5oo}4ZPy|pLGj%9c-d(5^+aA^{MICnN+Mn6VPwFBKtwqcN-s#c~T3}_*<k5nOY4Ixh<#Xo&QQ8EPZiv`==O#eZ`ofCPfSJHqgKqF&yno215?wnzJlv_ye5CF^?)wK#}wb!w7|%rd<HQ+Cg!kUiPLH-uG%vhm)+zYUn~X>RkyTIa*z^5ip|60WPaE%z#vzO{JC?YGcR@l&n-MpfF`I?N<y&Yh}Pv$ELyPxCI(fKEWK|{<W4Q>K7O)3@U3#CX^G8ypR6N<v+f9|93cEA3sGN{QCOczxyBmc=P@}{rc3g3;K$DqS~$o`U0mRy-Rj&hCZ%8UUoPGGCJUiNWfx7USeywPR9};5&-soz9kBP?<4>)&eA*hLWsK-vxATJrqY(#duO5TjY#hw9uL!|*7sPrwisI~Rp?j`Ol-fhl{Z+aYXp7GGQjDlsrPVV0l>G*9>ahT>n6)L;>RyBjEum@-RJxRguE0JrmRvMP{5k%FRrVM_ybP5e3;yXlN*mc&O}JLu`q1Mu0Z?dk5=%c6krfG%_%`RMrnd<*U1c)r8xy%L0?7Lh>u339EGmKnVPy)1#qTY%!Q*(m3PU$W`rETRkUQw3;;lI49wne?w%FJj-_`nZBobuKd|(ibYPaKKe<}l#S8SSS7}liDi*~8-UsvUq*rYS=P)rnLO4?l!f|6m)w>ur1hfpI=pVlQ+V6?}nq<UMICyQ!EodVhMusJzvDV6_*3rd;G7;4cz1~tlOmW^{Xs>lnqq2R!E$>S4VR(S>U(;3v=95zdaL=O)@z5yc!U=<nYlL(<1+zvvs~FCZ?k%&MZjNjo3XiPSW*k^<!<GHsjU>0@brKvd%|vMvD%|6+cKqX~#U(te#dKbWnJ4+3%`}JGUz~YOu(?4=jIyCE;JQvjM^neTbX5jAo)(NV!3pH!8WtV#*py+^FI9|*COiW~F(s~Y?SJ#`n|GFk{5kc<t-NP-4o(U;vkQaV?S!CYq?9TNVUj0(i~^R^31r3-`uH4RHQ@=cae(V&MP_hf<|AI8U@&@B-1=BgUYe7-w8DVp{fmLn+>8qA%)ATJQU;{Rn8LD03BH2W_ZH)-3aJ&?@B`J+h9&tJOO^={2ndmj-Yv)VnX<M$uG#N~!B<r0U|eD$E9TCjp*eU=dWI9!*xU@`c_#G<|2=y);iqPD;On&MTD>4R_kw|<4os6JukU%*Q^6XO$paow>gg5_)lCP_y-&4vN|eR<3$HB)Fce)vX3cWQ!slpNikq0YHZk{Ui3(YPJ|(@6lnOeXRqI@<by6xcNn2}fe-{HO9WYA~$K_j_U=rVS$PrDM`)J@eC-H@oqgd)3Fus~)$39JE6~mA8E}$5i!bvA^9%9{2)BmYuMc$}n8`Iu-S-7DZNHXcAgY_Q>*3hJEy=N3J^-)Lo1aUm>(I=VT%tj^XbCIpZr6p}9mV*;J{Ey7e%Lb2SNrbi55d!@@nojp6u4ZFePo=_(tNuSJ`7G>|^3MVfE-x;?od5{Lx5i@PjA@zH7vm?oO7|=gbxbe;d^kUxFCP_<D&7jM0W0NgpTBqlOTR)bv_TkFT&C|p+?a7NV?Dk<pBK%rpooP!x~_vnZm&eL*5L&X9OAPmM1s$=D5phJbF7YzOG+^e&$CCtW^}<QnnHyM8rM%RO*s10BT~uKQ_Q(#p=q^!sPl^~kiu~h>I%GsB>^QoiA*&$(87oUy5ei>6Ga%08z|L1k#+Ssxw+Z?0!udCg7MqfFifyCo0EC${LGFa3dA5F%QLyY#>zdeEb~kIjU4?<urUiub&jc!eY2X`(f^4#v<!U43*Jy^1q&P(b`WB;hg`02rWxu|g&w6E`V}}qu%qG?MKF5tt4E`%&1n91RNb1aS6zf4J<<2^ZW-wU-S8qxEQ?{%aY0tI^S86Y8{4&~if`I}UKXH=@6OkVP=n41af15)TZ}z38fP*_={=obk9K7J1{E|5`7G;$s@vka<&n*svypu*ftBarkyGksCW~3a2e#p7UJIEN`5hmY?Z@-x%L{Q^Ws_xFv##!vV6CnQ=I;fZS^QKr=QY-Guyskz!u4OAv~P=IZNV4CBz*XUst{p{`b~&DZJ&)zHG-I^*;o~;s-=omWGJa<USj8m8JR}ZHAt@ACdgohoNhsl7MB9x`74lUPBe1}Rm41gTSFuzPKo3^b7N~+ft{i8OE(2(*+#wkSbA+3vU;_#2D%;<1iKpRMn`(v(Uom4)lA(|Tb;r|-v;p<ykAFbNq0@uU{LsxzK2qjO}{JHWxLlk@L%2dVdM?QZvC)pX*69L)5S}T*IafPVBL&r3c;$qT-vwVeam&PFLvw3^5ydTA6~tLW#DzX;SjYO_D8M*cjs^nJBmW9ufnQpN2>lRo%(i+VI1iB=&F8#vk`Z`_t|u{k+@w9>=AM9-sfCgMKq~#U2-JXYB|y1%ADH%6Ka#kop#&F`^5}y(}@RvrB{!!X#?(da4~w|B|2VJGIzM{-wdYk2cEXhj7k)FI>fBS13+3c<3=RZ3lgfD6b<T_1R;l){3vnN2_UEMkg5E7?G1uDgNSfko3_o-%4xQ_@)mb772;T3BkpCx@NQW*>4;MWp))Q@_a-^kz*17H)2QpdpPXZVMY8NbwH+~bU_dvfz9~39`Wq1&fA`MnLRGWLTUdQgL!_fNCzm)_U2|LADY_qGn(U}Pa^%h!Hd`2OwOaJ&2yxjP>n?228K`z0+%FhyIHRI&<i=?~+mgAfhsz8XbIuru!AW)BtCl(%`n~rfL`c|{Vod8&A3t?jfL{UOM>eV}0F~>jw{JbH{L2^&w?{-W@71lwroIxP?|B*!eWN}AVy30V>yhuyAXU*E0l7NGv8z{o<Fnz+?IBYh!1iz}J?7j)sv$Dd1K(k5gF&VY0!n%9p(C@}%YJb?7Fk-S{Yu9+)QpI3+B(L2cE^)h*ZTT)>{?&HFs_y6+uxE{ag;M2xZ#t<q)onOinbetk6n2ym&OLiRGSYzp?CuS-4AcR1wZS*`}%Lozv9E!^#AK`mp}Spe>0S4<7$b+8i)$?ya~wU{nT=*_suyAYE{9i7!NnMZr%~MrG)AHXC4gF32+UrG~$fk=c#oc>g**Rhjp)zqZn)BH{y<CIs`{`v-)$-hAdenkAmFj)Q44<q=1b<5DU3(B_9R=v`ZsD4;0+7&WWyV7rL>W>o!_U-|1IE=KA9kb<CXvx?ADErYR=4r9f(AvoiQx^`Uiq8lV7MjDnl|{%f2bDE1=HKxPt83nE?{ZXyq2wF)^0RbJo5Hx~l*-hn-_WBb~+Lc6;Sxbr7O6wTagf1Iy1s>7;#)j)visNx4yi%SKP{usZ<dja&FczE9@svw}AcST$K(|A73U6ykuLDu*^Ph!a)77Nm!>qqCn5l6?_@Ot*HS-W5Wsqju)M~G_8!}0C)UF{h!Tw6Czn_Ooe^_mh6VS{1dVqAAL7e%My2>w5j`OjRpD5Ap%9}RZ%?7Aq|j%BG2`UxSem5!=r`qg~aI1?%1l#5~FsPWT^&{j@(7CjP-tu6&%L#deWn^{qu@dE&qtly=M&$^6hC{B=AGBtUk!Rh7l&3FI8^$QXgE4)m6C?y6=Cv{8oMgLuxY(cR1xrC6(>P(5Iu)hB4?fX}Vx5T^;cG2z{w!eGc<G0dF{jKzgmHD2_W(wz<Zxgx_Nt=>OnTR?$!17dkbN6L+hgR1TpR*nQ@v39ZBv--nkr&xgv<;@sc{bHPHZU80*WNPva6NK9C?9-$j@{Dc`*8d|@MJ6~e(7~+Sq=F$NC$qgwBYt9<Q6}g7yUKGwK4hStjR{^hUWhTP)h>@6aWAK2mk;8ApjtKw4J350037h000{R003opaAk67ZDnqBb1!sZb6<8hFLr5VcXKXqd8Ha{bKAE0yM6_xeF&9Ol9P7Vol^DYq)zIco8(-lUC)=}K_nz`p$L^AExTFv+xNTxASg<9*61z~1wL<h@Vvvr;qdHsTjXh8i_2Hf#Z9huBGsj-t0t{;T?$!d;>E@JTd`7lqw4BTEOjNWPJb3R-%bXDm#@a}#G+^_BkHvhFW+5Uh<qbgiV#!1*=m#5ir-esG(|1myuSEN<fX|og~ZgV(oLC71}{RsrMgiSV&s1)6N|LcTlgw<wUGrcff%M)m^!Z;WFSkSHrx76Y^3=x8C<B9Oz*_m6H&?Cc%hKdcw6aI83ULIl$R@!1ER=spi-uFyGKw#UCFeb42HwuV6d$8MkL9yshdhA3G8sIs~T9Ax|S%e84T>&M%HVF&txsrLK*|YIRumCX&tw(7_z?Gk}M2#THbkTWwY7d0h%bc4Epx`!oi)Rng@fyKRmSvK>kqW{HkiyXh2Ki7dTN+>@r`?1_C~yRRaRg1gKif#juuUeDm#ixoXBYPlg2IwBTX9+LBb3H_|Ye$yXbhRJi|{SQc8=0DCey`G$$e#+Y1|37lw|%38HBUr$a5WTC4}Rh~<wmW4{|Cnr{nR2RU}Y-OrE4A|({;D4%2b*8d$lb7}186J-`fuE|{>Vi(iqm@d6CDjZ^R+11j0Oo1CPdqyX|K+JH;M5T~Q%g8_Fyb0W=XH`qMit8u>k>ctQsBzOn+})+7`OP?*IF!w1z9C1Z8!YX(!ft8bECvBvS`#<Rp}}kzAHbJdRO+1aySaPBR>>;lA9%ps7S&|a5^J$mAysroj7?&f)ISum|85rVTlKMcd5$~_EYCK>a#@ZnM5mC@0Iv&{;+&~EGH#Jot|98imk3aroilCZ-o4aff1-ad{jR-TXad4S(`R9(UOhbGd?(y9l4?od6Kek%;GQmhU`90i9N$6l&(=boY4bY>b%9z@Won|E7&2@2Ywq_0!af~`n}03FbG)NO0X4G++l<&)JB!45lSgG>U8DqpH1NMB>NF}f9N>glpuo&^BUn%p2CP9hzAGZ{k+x*hUa|`uM|Z6(#CFn2~igj7I`Tv)_8>f%^*xq%1l<3yo-fd%PmB%x&da_z&{3itKW~r`1=5i*+7y3`ST1-SX#2;$J<|7*^nx0B{QKxOL+le!|v)e*uh#C872;+mo<T!38pKgNW>wg0Viqb2T8~(2X)iZghWJ5laLYp40pb-+2!yj6zF4s{YhjR5}bAUbZxUghdl9-VTOBrGst+v7sOj=T_iT1)VPqFyo{_4uTNs}<o#$Oiz2#DFbJ0DBBTQj0vIeMIJibzKpsk#)062Faf4XKO61O#q>={3iVzqI4!ymbzm$bh?zDZhSvEzH&^Fmk=yAsRctZcjZGyCEk5?Y^*TIw+KadmS0hvy~W#*Jh0!%jGA<k=t_votQJVT=7<EC?(g0c;GEqZXR$%nXx>X(Q?P#GLg0H_O&_xOa_2Y(z1_x<xd2Xg`k5wGka2#ht579p;*#>J%PT~d6)tRg(r$bxebmEJrZ1d2DH^p74lRR0V?C(qwp{&04gT%NtWID2uGV07<69|Y}yxg@X$`1a!b#p%@<gMM<G>gFq#Uj`4?4PI4@<1Aigd4t>B$lC~V6CG2k@*?tQ@t74kVgwAD6<EQXgmpxp!)j|uL6jjykJ{}<Pl29g`jr5N-QNKuG5yz^zV=ps+h64^U4gsKtwyf-`#r;SyqjGH%V!qR?g;h{9y2F*aXy|_3N(<Qjhp+OyTO2y^rHU#m2oEuq+FE;6}tG3c&$qX!1!ey9<$ChI9nXb@ZRSQ-!zFgm*=mhZ=U~i{Nwcbk5g*QAS7bn1)0B4)gx$i3fNzEH!gJg0gjY|^n@MY*SubfpXMhsp%?!KGsZ+oZePgsLo6IgNcXY;OWZc~#3ManaV#Q%7=}Saq-3cOiJZGZ45Qvif!5g*kdO;xqE_m01`f{c;1ag501`e@$t`#h>>;Q+ZgOey6PAR|apE-f=c45>GznMjux4n3)~(Xz3O$665tz6q&Wv=bszA#Au%p0A=3%u?^tSmez;_ckyO!^D;AL2Vv<4YlY);VEv#0Mz!SN*gG8U*#ZoL!F%_4G~jJrSkG_L=P?B1hsbEw#-c-?*9Hv0*cV)FwGKzG!@%Mw<2h_GCB8kg_G@J_kd5(`w-$#F*$clgO$*A74X%pZ#lR3|<C_7Pn)og;-_qtb`x@I}!47$2#X+owHsdS!c^@Jbj8gWQJU0y0pT7=t85ab!5_#m+q&3L<EL{QMAR&p3`@V~jX}zqi{1*q>|=@5g|nq5N1FhZyZc&Wy@O&?{AgN2)N0FH{;|p8o25*Vaoc4bRZ*NC0Z~XLur5pZYe%v9saW{iI=z*g*BlQBzm>3cS7mE?ZUPrLsp0p|FxWj!7coNmwBls^G0dEmzhWx(bb@gy6sm&kKF4%m||IsT-dewZhhjdrnzk<g!r=q}NL>VArx0Sm-ph2^HP7g4Pf^7LNF0w^nWN=6d=|1G1g$@lIVj;3WPTp^>BLWUcbm8e1~@wac@5J@Ns_pZtSWWG76wLE5$tJcn!H_T%M|`s9FnSj$e@nUYa;4R#dyl^G5WV^Km?OzlIAJ*{ktdXCx0Hcysg8~!}{WTol|kv(3gJ;$cZ)Ur%f%Q2?fqy?h{2V`VjS0U5^s%}H=e%zGH4$J<3kL@ag-IROp4tj`!;<uI5DO42Db-yH6ePrzHc??}PbU|QcQ<37xpc>;Wx}D0|?&$RO4|eiQ^nS|GSdF-<z;PiCY_PZZX7cslCQo@+kP7S@gE~u?TcVp9%$k!_f-Bt3V;Usp2{DhLs08Q5`Ope_gDeM+#rcYv>g5tFgFKq%9>$K|O~dulpmXT3%eQ$^=pAlW;#B0B?V$lt*kKcrQU+>BHTC0n0hw>BN}^UkKC8qIE5UZYo@u)y_IcU?tOc70gMn*VT70kG91j8mIRv<15CZHgeG6)W^)C%5uN<$V)!moIlxnpA^j`GfAs;p1^IjC-Du_oCV6-&B8o8iwVFh#9fJ_C5TmvUB2R@S&t^^<p1AFEE!Y$Idqj+T<-4|eJB+Af3*Nj~90>Dk4pTBqq|Jo?}04;AJK#T(AEa>@YGY=3T+}_+d2vi7QV-(LwYg!tiQG6UazfM%PdU(X!pNF2^ZrLg+vy-WReAFTOXeM5NdIkZ@`0=ne>-~YtvB@aXv1fxUuJ^=#-`PI_f6n0it@kq@-3a%d@i`e%EZDX{A~Y@^pDcfZK>--5E5$wmWJn{?h*&>%Z>9{{O)K}43W%@>Q~M751=oW#-nziGbCbYDT3^NOF?1Al*o|WEGjsRQ%G&Yh_J<?Y*~M|XJ>PnLc6xdC^VMDtfKfDo0xtCD&+@w<3jbN>Bi>)=fxL0NY9UQLe9hur&KJ#X!terFeNLHXD;)u>`6fBpb$rSS0&QHd$oO3cV;baQ!tvaFmdCxowB6?E`#kNhbjeG3ek(J>ne`7<>1V=bcf3ccbk?K_$`#1r=5B*gB~K9nZv-X*F(_!ZP>S0+Kxe2us67Z2R7u4q4vH4KuJvYGp(?N+5bka-pv7eeiN>Ocoo<SZ83;HJ960Iiv`|ZMETh{wabGe}r%zDgJEuz^-=;9PyzXkZRJ&vs)7RP#G8sEad6+5P`Z^A1j=EQ8yo<SrBBCnB0SPgkil>m>nb9~TLa(}{Sf2?1#qQab4gKW74i0o9Do!j9x2Ms8aYBG`@euqmxh7jECsafd^2QHBQsKd%Z3H`vL8Lk{?}!#W7F!0`2<oLh|FJ!Puk*kN>^plNU4Q22ckQXh#6Wx{oIxNb`>Pw{_vA)?2o>>7m?J(Kg>Z)<xRiso2mcPEdw<|>>}Lo5h$0wy_zOwux)FN49H&LT9dpYtc69^(jYfdy%fIywkUGU+=fZ)q9>Mqw7`%5|NpA0M@9<K>{tiJ%WF~Q>#_VhdyBA+9&U|JIU`VS0h`-tnU|hRFT1o807htJ#2(L+OTT92fcjL(?HwrKJ@J#J-*)|k7%Y?2k(RQI%(VJtpx>dQN1%{fuQl+Y7t*c}Ax@Wg={$pf7e+>mcK-v|}R%#&ylM#hZX_J_MG6kJg2F8GW4vlc!i2iB~-4NN7rujM~aUAOqF<=l=`lX@d)LqaMIKV(eKd;>&NAi_-4xu?;G=vYVcwROx@(;>hCL>38djL|Wy2?0<Y@rKg&oOGh|MCN><Co741eBF;uoifFUwcM#HJ2~zoo>^pEcV)<y&lai$1O*`;wB_W3s?qpU)h_PrgU{>`?}dguASq*W1|gXVm1jX&4YHXL5Wsxki--moaF~~Ggh!YVBiAf-9aV`yvd<z2FKDnPI~Q?9utOY*j6?)KIQl3_p0XiBm4~fFul?{194O4sLd?DfrgXeV#h6Hy>|0vM1x3+{uLoI196}y){y({yuw5$Q1wDwcO>v0kl`tMA4b@Dgk6BZio8ZXGaKxKT@bmM%Lx`6=Wkf^I_;yhq0T+Qdq^Qphfeog{|8V@0|XQR000O8001EXf8|9wp#T5?(EtDd8~^|Sa%FLKX>w(4Wo~qHFJo_PVPY*|ZDM6|E_8TwZH+++!Y~v>_dP|xbrPhApmd?dr4+nCNs}KN_{|SVO3m#}UAq~E;W6*PhL9b3g^a9q+SPJ<-XAZPG)0IkM6wRy%A7Y4poY&CWd^KTF|&T?2bS_`ifB?$x^!JD&f?U}IfrzS^6m1xPD0Zn;S-QjwaFjJ7cziJMnF&rk~Fk=BxNM{akq~fOe4Peet7&l@=#jt$pcruh;7E^3s6e~1QY-O00;m803iUfkH3^l0RRB;0RR9S0001TWpQ<Ba%F90Zgg`mV{dF>Vl84}b7d}ccyujLyHdk25bX68nxT%c{7AwuDi|nX#sw6VvV4g(ktM}CCz!A2WK-VW?%v)$@HgnK(Fi?n7aW4;m5>Dof3s8YBY1q~pJ6fsw~1SyP&-H51W`|<atl9&iNCgAk7kyo6bvWr6JaAvNvT>?jVKw03wZmxLWPhS8s_ua`twN}*bQ@+@iODu7;uoiSEv?}C4B+}pTZ`uMZ*#W18*Q&FdI=6%+F(7l#-&{i<-68xp1d3D+(4O@o&4hKAD3NUWjAO;$ngc7=k1BGGoj;yeL(PCpwPsgPjX&KzXEf3<(RCjNNX@oN&|_T|j5MO;)m~eXm!;QdBIa8Ld-lFC;~2m+zO4_dHQ<Ho<j<ge$koWW4rc4!A#%bJXUdb5XRev+sgLR_Z7!qWUg3t!ZxZKQ;A&bNv71&<MJ~?gM~Zdm|3}15ir?1QY-O00;m803iShmo}ij000000RR9Y0001TWpQ<Ba%F90Zgg`mV{dF>Vl8lGa${w1bZKvHE_8Twb<9m`#2^%f;k|#QRPR-zaX@I9RiOig?z%I6L?O}8n8p12X1eLByYrsKb1ePaE%#phZtX|~l|YWa@44r;S<5L<<fNB%wHBy(L@c#!`y&-al6tyZt4;vPT>4Y)ecLaLojV^B^=)Nsi`5l&eSiC)=l$X^fJQAu(1_@b2b2l8RLUeX7#umGj6(e9U9)So)cGsBI<s4w$`U_cZ@+fZpj<Fs#z`lRrwzft08uKz#{tKbCN%_3egIHQ0|XQR000O8001EX<x-_@U;qFBZ2$lO7XSbNbYXK}a&>NWX>DaMUtei%X>?y-E^v7b$-xbSFc1X5zpGeX0Ty{jX(A0+1{QO|oRQBY?I-gc5qb0WJXb<(W2Y^|J+$gj`V)Fytq-+=v8&_C%aIFV{IH|8ygJUdvf@iE`|v0tA*A$g*Jdf<1)1U_P7;1lO9KQH000080000X06X^ncNPx-0EaaI02lxO0CZt<UvhPBbZKp6FJW?YX=Y(#baO6ndA(b0kKDEq{=UD0)t5@ip3Vu{B3O5e8m<epX`CQV(+`(}td*!+`m~ZFQg>co@4xpMzUW<hYdffc#4Cv$&J5?}nIY|Y-kU?5=WF)zhksv`MSf%zuceTR7YS!oEe=A7J1$vP)~x1LU8YSUb~$JKJw}QHFO&>dt8Y24nA&p|Cp4brJiXyHD+~Sd=kI<mSzab_&Qc+7L-rG1_7^&UENE8EE*Emo<*MNKSS~5+R5DR8aX2(8re%|TQ`}0TAf%hNk0mQgdg3_)pQ@~rJ?2zpt-_V(d8<`cmj@O_S)-boN0I%=;-V;3tVCJJ)ylq0V-+WPEG5{mudu*A7d!X(PL_rHeTdcG{Vt_hr@}i7cZ>Yc8>0B3%0$lFXK2t&QpZXZH*U_0;<#qt#TC6<tyU?|Sd_#?SqLc6zl-yRZ)i}!F8)G)Z&nO{MAqj%6f$Zb&71%dr_hyRLW)AF7|J(a*2Dl03?2HT-?Ht1U@TLSC`-z%IHg1a_bAmIN-vmH^^-HL#J$NF78gf<#PThNdO$GHc8Mo45<|S3s@IN%<m@Mi_NuPS+V@_TMJ8^VT5mdbPp?N5U`>rwY{!W%-bpVQOYneUoBlosY-h{Dn$cs+NJvl84<47L^;J{l_!~B_f~mr;W>*?4a=lDE%gb1G`%QKVgUCby0{&Q4MomA{TdSwnWpQB!v!Xfd;Hxar-oiu*pBLE?q<cZYYj@Qc4!jd2IJx7D;85|J?aC5?M-<92J5_-mzna^~%=>pCn~I!;rz{!ls&$VCvS=%67)3>VfagDgQR7doawGERu}^_R!;R&H#)N5esHERAIrhk4-F}kbD<Z4f{!~Na?me$!a9P%J>w9Z&&Ag4*o5mp%;!`XWA+~>ybJ?4UFkRk5ilYBvDh$P{r-ZTbER_5B>p%X<pVLGTA}}MM`d*{5i*LLj2>1MbDsH$`zFXDIk0_w|6!bQ<68o<X0BaMfBPl+B6YMf#$sgDk?5k&AyT69K8QK^%;?5Y>y-<5iS1%NVWyK5Mt9KquTw+R*=Daf>aumJ1P9BQaejXooX}qx`!<xsb-!T}#8a;0mI}Af%Qy~O$GdSpL(SDz*CBg<DL=x?eipv?rP)OY{{Z*@g-kPOpEHu!`*}e^7H*mA?aGQ$SHxJgh;DrbvQF*H$&6JeOq!twnZLM=oS;zGelwJM-9ZA3as^*z^zs<bxK`%pI@)*J}vCOu&upz7JW2Ym%C8Z3rRBuLqN$-APW9X0Fg8P~x5$PhB`<&$s40-s87L&&!@!c3mQj~tsjlu|^lsrbPX~kZ%PL&90@T21(zhI6QRI5>T^G4=?VNH>X;<h)UZw^Gm8i3t@?M>6|A30~Y`<Bqud=N2beUyF4yk81$>Z?;e@)<_z@?OsI0mSql+8h=mMmI%@0DFiFk-^3<p<NA}G-Xodnic#GP!HL_Vn5_mj;!;pgfMX)68>70uGO#+)APc2r6|VIf>9@{0ixwSIX3Y>k-ZIz8nmz%HF6agGg;%urXa+nAA(lc7hv5Eqr4?R@PNdu;95YSbyFcjVbvYaOW*>)O9)#6*Ve<lqa}6gzOFvJjFUZY7x|gsE;3?DL<_)PbrGj&%>k@ZO0g9ZYF)>`Ar&m2{qxQDui1gCm>h(JrDUo-ccm$Q+aeaC7GnT518z^MheiY-yDtv$4Yy$myUp(eaUkIjfY$oANg~MM39oeOlw2tSKWlcwODs>`b0+}C?+p70HYy2&5PH?Tni#Nq$Dz8E@~tL_h=JwkC5cNAgyO9Pi}y6CK_-9u*5tEFB)2&?Dedjs&}nOBqZoo|L?|SRe97~SY_?Q=EABZd4G8JGi=gy7n0`JdhPKO-UD~BJ`kQ*oAa{-OFs4jUWXI;Eop-hQ`Wg?J_h6nM!=n1oQ$f`ze;`H&3LH;8n4mXi<Dm4FhFTbL-3@VOH8TiW)v#@=|FRfkDzB|ltl6Uas#g7o>P3-TBZ@q-Udz=CHt&PZ{xtGLcn7cVc-n*Fuy&XtVZJSUcpM~rN?mIQKi!fJh7XqpkE4phOx7g@k-|rML%+RCEkXPE%P;Nokm$TjItmzO^qgLuAt=gVtpfQ(9cSu}c0VC(QK9_O5B(~tUnbGFP&;7=<1<cq4u{_x_qEZ7d>7i>i4<{!1nn?BtQ)e78Ekjtip0AHtC~`WY00yw$l4myzu{`=7vQ}eBc#?n8Q&=T=4oU&T<hO$Xxz4w2jeAe*OvJc`G=FYeGuQi#x_4Z29v22$T~hkg7<zvG+Gtdce67Q@21?)AKF=>81Y}5cwx*?1xC}>))BjCo~{f`<!Nl-R$x)=pO+=xEX`o{?TjaN;#p@}ohB8``I<c8FT*4~%a-?oRb|!Wu`c(tPf^)#YtjUO@P<EkSqhL}4sxOQ1jqf*tRw!wK=(*qNcIRggGrtQ7l+u}Fi*wu!fegO-B;dvI1IGBLEC_cPsiwtfq^qAWHi{>>5%Hkdb>G~)8KhEjL_F6zEB@(*JK$V?ZF6*8Ffs6*V+JpEeJIz_r6(8Rc*_g1vP&{ne!>7ju+aUS}JlTh0f;`JJ!Kve8)dTgewi=vlYymeep%RDO0yPQ3{<`4xLjH&6Y(UR~nrxk0vFOgEf>0W{{O`id|8ov4>sQ@Y&&%^AO2tv$OH3FXy<4Xt<{ZpD+;YbY8}(E!w6CSDQ&(CwtUMQx&#xMmL*?ko4`~vBQR{GzC-;rhHWUjmbhlLs_R@7qtTRyX`n|QRwImoe#uqsG7!9A~_SjHnW80LVsrAlIbx56CH#+#417Is@oHf-mG||HH`>YI`Cfqc1%&eIfEaIqsG~y4F;44oeh{o9wudV^iQW_gJ|_6?dNiKG}$r@rmY<q%w+WAoP|4p<rLT^n(H2#qf&9Gp&3{|9{=jSt1#gJcsExGOtoGZ;E?hBs4GaD_>JnZ1t5eSVEX(7zuV&bM=qOO{Y#%gwFUSaK*XG1=(bZmYEl^Qopg~!YJQpu7Vw-)TYkSn$Od9>k_6#AYjQ^q_*I8wfE0b+l^;<mv!sUAkrA?-$J8M~w1Kc#7wjtA9ihXnjYclovo#P0u&ZNuLIHXjebNb?+Q*7D)TB5FMQ{k^e{Ra9Ly2n16wyCWy<avG6S|T1NR7JgN#3M{xHC*9>Wad)!Y5kSv~*G-lekVD#TR+rI_AZW$F-g=c%JrZnx?990eZWmKKYZXNwIfLQSkaX-hfNJMJtOdbQM-)bW~_r)Telt)N{Bb)013XN^DZXy9UP7T>|8Flr$w`^q}G8xHrjW)H!J9xb{K&B3KysF(^gPY{S~1>~L=v2J4VDZ7ob^xgMQMb)7($%yYv-$}L{cJuBU%c?#T`^V}J5>*!+Ofa`@}4&+WHoWrXKXHKYv)Ql0mWZ4d;U=b<)UiliN!cezuex%^&#QuzeB=dgcStE7PsvCv~6?8yd>)0dvCipPij>l8g2nR|Sa@B7pI|0|N1S6wy@odfnEN9vRa&I^<GQD6riM$#^tu4E{9*w3lOsY)}l}#u0x-({y3o-p^C=_PLY82IL!<iTKO=H9n_OlgcPXQ;9-0%crFwGq+eYuS2@An9pjJ$gW1ZNonmOjJMh+J_L{nNw_n9RFDWUTbYpT#yL8GW6{D?3cSoqU;q(^mfSeh4p#@;vZ>`zN=uIbb|y8R+QVc?(laUwaYybnwTt7G_!vw3KZ!48Tmg@+{%UB{0+qF~xwE*##Up-zhWcv!m+J;DSEDz;ZVSl3sa||Ju+67*yck5Y?OCd1J8L2&&EOVNlg1rg3$?_45f*<3CE%HD&WDKwnnDNbj2S4w<`0Cxex3kv&hUST%Cq4z5jCw_p!mc}s`uoFwobKS5do`{2c_c7~L#kJu<@>Z@Vk(*s5Vuh}y?+WP7fSR_Pxw6`;!;;^!4&?N>N!RI(y6tShzYtm{|>ONe#&AFzgC~d(<NVPap?|R$=u|3pnQX0EQs0Q+_3ERV|y+vA6*=&8{cbu&0Y|Z{^@0$d(bxTh<6i7o<0FrWwzthLEPf33U%IMI>OLG$0(}T<P)*Mh2P~pHHOcHuLp@ptun)|HFi5UQqdC1mWbHURK8g=nNr>P<RFDmG+Q@9fg+8-YK(&RN;PChm>O3rV6BhJZj3Lz<>OByFfEw$0kZ>62ix?SiFO*N>`trvzF|GOIb-nGfL3v*l?S(tUz#-8g_K;k?r?eD;|HR<Qgt5@GfuU`M{q>jy{4nz3=xfd-^G<u!uD-IFrq=9Il(Ihy|bdDx%%^sebxwsJ6^e_mU1oKTpvE|%dI!Nc3r@L~$39n;DbZbLHkx}UG7&Gx@#1(ark0i5wGSf}DZ3xULeE`{QYCR`oKb<VR@2N9V-d^hY?bTuGWwf^hpX**nQ@bNrT{DnvDNL2OqGI~O#0Xmi{j>IkT6}nnaU#_^5`*sZXXn*UnfCK&w%sW+z1MiK?`I;wlZNexkDT(Xyym^=Y~5vW+X~w8ejb0$5;(WMhotX=_i*RZ9vROVw=x{c-i#skXLKn(<!I0S9F4HMyRo}C`s#xEW)4lZ6fKM#&9#{sbBBp~G|U;uX3je18Z$a#v^ClF)FTF&Z>h|V&%Gpg;y&2S-M8QDGT{>meUQlAMvOD~D07d7&86(~lFB$dOB<hEXND36*085eh{SWPok2#SfJKA+LDbTfIV?tJG4swOuTD81#kq8>|Fx=N1+_Yk1c{QeDOA_$0p^@s3m+HCV_ig2(48Km5ja(?^bO=Q^Z+6Ct#Lgma1ohqFWXx|&)A>0h9vzMG4j;?rhgRrP@zTcRVf7>FnAvUDZ1sZjAYlXMs;WXz}3D?dm52|?>ur=(%l8f<#~PbwbQD8>TZO2{Zp1+Mxwip=IDgG)!TyyW*VmYWPn;bb(7OchtzM(KM)#U8`b?X^RJ1v#B|q|@Aw>~Q#06hG@r~)Bn7bu4u5R~VFUVS4f;No;m45c?a<&-tzFDA-Kr`WvEDW`Cx%R4SN{W0O9KQH000080000X0IJDBD^?i*0M=Fj02KfL0CZt<UvhPBbZKp6FJW_YWpr~caCyyK>yq2JmHw}%K$t(0N1}MYDA81T9Z%L(<BZGpOsdk;C4&IiVp>fqB-QPnPXFBp*hkza*>eDpB6Vwd?3CTEvLq5X0M7M0z#$F8c$4M1;A2vXO+}NHAZ#hv^}5LNn(*Q-Ulo*B<Swh1#il0N&AKS-EWaXIo{CcBOprBY*YrwM=3wwz)Rfbjk~><FTTy0dCODy2G|Q`+)Js8%G|gB>SLCMPVg;kFiz=&&G85Ib6e~e1AVmHyva4l1NQ?4@))k?pAQjwXHBb{D*DD}VukOdBDhOTGqReS6$on6rKdk7->6`oaMah<g6>Fd>JLg3`pp0c)<TYKb?%S0KZj$#fEz0WbR*=8G`I(fPyv}Zb4>Tv6O5m(3Tx7khaQ3S#7v{h)jKLr+iyN|7q?>wEip4@RhtM2`qjgr~)nK5nEorq}Wl8(9uGc18WiXxQ(m$&rZ@w3mqK$(>uZHj&_&FwTV6Zp9w2%0+=~0U2TeaNO*{b~o6KTHJ;%1#@tF|SsviZI#S1^z%%CabTuB~ZV?Y1k$zc!+3DgTx!!gA8}{aWgWy7fE<F}(nuqi&7KD-afb)66p|C{qf|uiEzd!b_aVn9#bo$=Cv!Ge(INdB*5!f$SU%{z!h<tWbw;vkJKloaY25qI9_0+*D%}J?PpkPU!~JaJm9jsgPk>eL-zs|MZuaKQ7+AdHLcUnUjYBfj<Lgd`pd){wXh^vMw?X7mZc6S<<4;C=kSklUv*LCt+fYyUo?Su)XI+Hg`gcF~>={&pd7?iO*Bpi37*6tbj2dJFeI5k1MI5^F@0J{~mh3+J-s35jYSl@-(|LkzpNLZBvjRwnF`@X4C0%b9Dur0fq~c71Lv@0?Vw5)vcJGyNu}*)2Noz@dKPmYTFJMQRE2MwnLlRmY-VOO@o-nA`nq%S=6&^FHWu4^HMMIt&qFav%@GBUhEvCp8bZru<gL^-+df^3M;Qf4$6vly!b}Adj5>I3>Kd6T7D>KKqKI(<?zsTg2bhqC%|)uhBRPNN@<dYVMu+)W^w9pKXB79P9lDa!H)Mi{9WJI=%*Jl5=<=kvnNF=zc^-+eN|5Xp1Ey#4tkmLlm;$KUE8zbkjF8nkxgAc3b^mlIHa7}9_YJ8h2^_`>N85602m%Xj}ttP<1h$Z&-bG*%X;S<X1((XtY?-;VoxM)O8qdR9t&JQ2vbKe8YC8FX+S*>EWu`uA5umwo<>oc;uS7JN1R~2>wAt9299rkG1lXH%q-{P<5%}BpE|FOwnrhb8hmyr7<dnGHhvs%D~&k|Z7cEsc)4CIk|?#^ghz=_(<oq0-~|>q8rMF-a+rAVS&Ck*1)|MGU0jR2THMg}S7-DmOliyG(2gV5O;abJECTQ03Kj_-0}Gvq+OE%q1I`6JibYdDPCUzsLeTTXPAwtgBs{@m&;2Y8%Vhwi(v+I>H9336RJI(Ah3^M6<(}t&S_TOa6;2ZQ;6IXt$5HCTu$~*z$Z~9&0$U>?!ZfiX8+6?}$x(39j{8NJ`mG?3vnSN?#HnT5p3ed<{4liwJC5x%jY3;QKDRu}VQvJ&r*RN*Drf`>%RJX+++tzqS?LJ|hhBV?yO-*#zGAcx{4kC$*1+>@3aQw7a+}qY6)msC1oAGnw7rSz2FCzB6E^s_*mfB883(r)(ZF|E7z>CGBC;aJXu`Ptbpid8%WE~^-1VG~?0q}20pn2YmIiLjJZiZixFZfA-?ox4bdyBbJPjk)j#&s1!-qToG?6k0)Yd5jgMjl{F&H$UEFkx*iqEvL{{VwOWpZ0?bH{fhuqL4CIGp)jz#$N@G<JN*pCWK}Zj?lVLG-l~FAf;zF1Wuqj-3E7=_$zR+QAnEJl_VF&zV)nnFW>$swpgh>B8mA;a-|R#6{>1I3IG-1h^_$9tt~h5<6jD;<^qXdONaCVI0VM?HC+hWS{HP65zv8kM@p&lSyH1Vwy;P{fkZ#T8U$hER4d$<$xd{NAzMpbP}*w?l@5d^XERbd;rZ>=zLvAc5GFx;Llyg`Ti@2Vrqfj&^V63TcwP05T_FY!UYray%aE)o7eyfXl$_{Of1U<(FY;uAG6}<l>hSm_;XB3DQ!hlQ3Hf5%Ia$}^sLEj+3LDs6hIJgJuXs<#b7@%zyK#=2_VrR0kj%Bb`04tn49CLHgFT5YycM>04Jw#myIsavi!*VHiE;K<MHo{2^OV6ARG}><_hX2;1Pkv5p#W@DIiQFp0F812*zM4P9nGk&<pr&JCuu4PRsNBZz3)GaxDH`X;~709)r%hK|n)pa}Sa<U@MDLp1MgI#eoHIItpXoaRBhdG3N>OV-W_98w%l`++6d&Oj`D}8T<51x+uo8X<%_*M5*P0Y9>zNiddu}^}qonPMD;=6Yw|<EZ?F@YyrRtLJBG@Akzp=<qWp%#=$p`l>G-dd%jEXB5or*vQiPim(T;pi>$PxII&zy#BPv+YQ`bJrj%RM4r2@abzp;|@Bpu$V6Y!MpTphT0+5+mf2_YIM^8?o<5)w%AcBQH6OiF}kU_K9vOsrJSGaEMg%KAJ*`gHpfx#X4zMVR;&j4&&ZbVO+z2n-x6`hL&?~7Hjiv+jOZnLUOy1K73R!wMCiMpEHMy6|8zYEKCmJdN2tT0HDI0<|&0S<BiO16j@hwLtlgq1oh!u>s;2fiaH7;{1s@OXg2S$cBA&kkbO_MX;(tE^sb5)l8*w6o5sT4#HdJZ%nor)uR><k)RgP40?v#m)6WL3h>iiT01jBy&?FKxI+Uo8$6HLn{EzF;4)z1kA%Mi^pJ$!V6p{^6e;1fxq1OOmPK}n*dqIg?z<z!SXo|Q=7Xy1&;|Cpzqi&kHERckTcQ9wjxkj&OPCVJYdc#ad}`gk>dnUi>st}rV`nv5+(R60E@Mh-iswIa<JxqENOl<eJ%O+?)mHKx!jrF$zp+Ox@Gwpay`}fs8BHhUDN{H!%>-f7Ub1_X#274vY692@<UK)i2jyCBZt{hB5*Oxp`i=xV;)!%?ky%XhKQDacAE7|_I~nT6W2MygB`J`%zC{6zsXGc+ol9$ekIIY)H2&v;J%uE?y9Cw%&HThz8wGd&u`o~1b5^-E$zp_;Pp@c_3Fo$3(&;H+n2w*diUz5zmfST*zDlVPj6oKx`JIn>`7uf6h4J4h&o_=5S5a^iCyNq0ibm!juK-q7(i}B7MN2lz>`s|O&OwqXQZmj5t)21Ye8mmx=jNZqE{LsJB(zLEG5Hsl}&pys}?kY0J*8faMa!|N?9_|wp!D&uDW+=em}&EB+JQzVT_4k%KuE`kw6e8&AmPaHNa@Lod(S+L4KjDjgYk-!_@dW#}$*NCWJI<4aSBae<&Y~c0I<gzZy6S-oRNIv-ang8KV^SrpzUuhtGD`q6LJVK|oAo>w1XAIx}6PBR;NWvC@q2jMWz)v@uq~TuRaPFOh$fzZJP?WGQKZD3O|pRaXm~mLzV<wSEVzZYrx3;8XLh%lqxZvpg*tB3J^{k*_Szn#*z{Q`UA3^`K@cD3{)CC~{VCaLIFHQ>T;27;U@vRH>n7UWs~WG>bMyWIk_M&})nZI@=D?E@fk!FHCfSS59=e!IGs#UiCXQSKE#qE$c`U$;j*CBNM>m;ZN^$k<nH`Iz88bB)W=#Leg??FvX&ggW?FdR%xHgizsz~pCDkV6!Pjb^`ZP-a~ny6Nbp171~$wHR^T1DdBgpq9oJ=X1)#T@wIx><sA-pD+2-<47g!z^->Ko|L?bU`X|btiB!z|I_1IE3<dW|fjIt$6F)4}_-28#ADj>Jxm<+>sE2<6XENxjf)&z(~0And(RpliM^JFNhma<kb&2qYu7HSS8#rHqHBUwdq0Yt!zWW6Fc$$!6p{|2v9hJqEG6gf~rU$g=|zgiYpdx%9}Y4#CI^(s?RdzHe(0E{59>?$wd26+n@M2zL=XzH0BAE}pOy@G^;6Rc$fQy~TPUqk%FB0@><1u{idpjEPTkhX<rt!2xp-7S$ON_i@kS2Ce77*Z7*NM)3)af|y;<im$9+vo4gjrj0^$czV8O;S|_hedHF@Lx?kxrZggRV;dgQ6)KG*)^eQ4GV?{50Lg6r7p^Hv&Nz#C5uwXT2RSG$%q_W0<^>22wRzraJ8D3*mT~j7Bu09{4#_ut;<$TO~ikV*VdQN)?haLgy%M1lme?aVQCe)*?=%HN+p7bn*;!@6wB7D|NQpH%Ge#a0^Y%}f0%3dW61R)nPJE3yG`f1scNb3p*qR=%lOx77{)-`2~7+B+4y5NvyI0mqn^<eJa&q0vAQw?F`!<*)5}0@#!<n5u=AKe*t)8hIEmq~!B1oDSjHsqMq>gFllyQH{vQ+98Fl6XT#<C7IROWxTWHu(3KLaID&Me7jSFiM(@A>y@bI8UeSCa0^h9=j?S904l|#dftN8lZTEtd7+5?e46B~76h<33zFM09L#5&2#x0;gF56}%+Ta3`CZ(m3%FY;Vm$)K)hf({!o6b5|!zP*Y{joHf0HP6bS`cX+M#7Lh(2r91S&ykKVdZ0Pzhr-+N|G0y48!1Wo%#?f@l!_mZ%H3hCk-VdT&={;iep47j`r3N41`m%;c#IL@vM1kzY;37$SkMGrv<ew@4n<Do-ifw&9lv08x~SPy%jsZ@J3+5I_hEg*5l#ud^kxL9g;0UD?FopdGf7NOhDMn{z^4_^%lT?&if9*XxyjL!kqms<a7AwtPG`EcL^VT?JR`Q{G#_9aiDB%hu3ne9Su1`c(a~NDScP0&XBqos8Xg>>#j`7nMsqpbQhZduJN(mp?*px&Vl$2`nOBr8p}E$m(H?dhHj{0-$Z2)P#Lgg=ZBO8hy0_Iq9g|z6tL|@ekv<bnV9!o9cYkf4f_9(WRg}(~${kx;O)wfyZV#Siw19Qi+U%*^R4w~>e%S@F2U@k@b+(6Gk8byk2HO}0EP;#-mow}-!8T)%5=Ot?feOmP_Vbi&hSEd!4Y8xjtrZhVwAZDF8gG@l^;lacx{Wv)?-M+%TB9ci{Udqafrly_H^8t11C<k`D`d^xQsiAkCOx-?f=X{yoZ8F1Qsbdj$VXE*n>?WGOfA6Ss7V`Tl3bY@ENW)!K3bX$uzVuP!)9%0l06o!Y000xp6W{5xsw<w03>g#C3n@Qmo-Wch!lG1VR!J8IpV>r@4RHIGqMaeU9CY)tTbLcj18LQ?S!=NrSx}3Y1(e`RhD0C-_-R>-R16@$%+DP`Hakryp#(=)Z9y|TChfIeBMUwgkVxJ)G+51^NTdCMBS0_&Wa9*QdzjwBZJ1liW|TNwb%lDFo*_Xsw^ZIf;UfTb)o$M4ADdnJ;8%u0soeu9-^$;r)c(^;Lia5C(o|{K%swr|NPxVo^10C%OIE2rxepe-dc8lE=_y#QtKYlkk52w_5Fb^kB6eSTEgL&t*3us$TZNngmt8bOr+)ys^8<p*z2huhT7&p`hxjZeGVo1)3oRMb01kq=(9rseo6?We~+8Ie6BwwEP}jsUF6jf9D*veL5VcYOm*d8V-&e=KPQeA943i>I)7hC-R@G?>r)#xFI20E;wM7L(NT&r-H?<z*~ih=lJQD5WcVxpZZwhqPx1eM|N3kMgY~9(l+f`BAz+nr>KFi6(@<VCugYSx9@^mcz#2{|RX<(nJ>PbEc~axwsr?BR1Myi(K)E>@HyA8Ft=Z@BNqUD@Dpor}w~jBP^-tCjQpc3=*!polwoR8>?<Dtg^pqad=i?E@k+|Vj=30m6B(17(nkt>3iLrGA%6}SnCsK^_(v8*tJ}Keoc^{Rq+?Cn!^yA>#znkN3z*Ai``E<3Zmc!%C0KBBtea?nWd&F18aCEpwV`F+Z;weAyqil9WmLQWIa=U@PUGX30(h~d4N1vv9CwXhU>V(Z2JV=!a2>ShXfW`_nL**yh=}EU_oau5-;Ag3|qurp410I0VP2{i3O%CC_GjI7Qb5&=n6>f@@a!;0<<nLvXU#ZPQ&Bz1vnyr*l+!VL?oOn?$MR|wplrD$d6`K_&*Fvo2M$Wx__P4G`O|zAGC;;zbieu>Q7{)k&;%>64f^qxXFi-oQzcdb~$qA-%WT2+xiN@Z6{yS>hL-}$#W71D~&P{N72>YBRAr5SI`uEn_+<^uD(ma9RSp^wBbW-z%-Lvtd(%O;nbn<lq(~ox_;O;!>ZlYgrVY76%K&iigEy5<4Ot9T$XBF<g(;S!%cgMZw))jr3S?CQvW&%!>+qj+`9I}?5^XnXxV50qvf}dc`_mDH9AHN5$c`Ntv8#pb~GJPhoPwy?r7Vj7MUWAgOsgN2Rsm@8!Hngd=Z&aXin{1!hr>HnErb#!xBR2VSd#JNVRVJ;~3ak|<=0oS%Gu?4aBJJW&`pS;cq+s9(NaS(CgC6m*nMgGw4<MXJ_}I$yQTfCXkq`fGuj_L)dEO1VWeMXhe#_QsSX#0rc-8&0G!D$pa+$asJ!reZ<AD+;)4X^<9)}$~R-+#m(gK>3Fx%0=``I&h+oUC^8(G%EvMhb}1(sf4;2bX76R?+RIUtd%&5D-#a7dyOxB<|vLcdYn-=LJodcOjc_r=>6+AW#QQA=mN%|3PK(rCBMx!cucQQ%6tJ$2ojMhWekH^3u9FxMPV8@lTIiA`NCPCN?VatA0~Tn<`zre7an{v%uefZJPI;?hH?!?6DDB4#t{x`{uL?y$GfglI@M9FgyPQX4!K6pZI;Y&1$z7X|dr@0(?IT%!ySElal7>!OxBJ6kYX36bwLYhpa#a-u<(a*qeWCC-0$_pPn;*r^SXk)zoOGfU8MupVP-?4Y(A=?&HPBM1i1;Z_SU*phtzJ+Xsc+v_Zs%Y!gszFWNd;r06=cHeV6vK<N^kHf6-w!_<(f9?GG%d5AKv%&AiWoauI^q=@T8E~9vGELpKG%Rp7Jx)CCW3avX_d4JrP++rj5yS@HMT0M&(K@SDWq%_5ad8Zja^4qQzfFw~Iqg^`nYK!B2o<U>Tl2*Y<EH(hE{168qjT5RFkDoOuEA&Z-0kV*A<Ao<MlX5ZW3_6uKP7EnPw{ALa^KCj<oC-+^rTeN?>oMJaBjA+Wd{Hl+&81ERdEOF+sJGj0hc~nJdoYA=5DHjf#Fc?E-HETt82Q9>z{x&Hzo~to_1%0<8}nV^}0Q3=G#E|P!%PNGTa|-v|~=AJ-^kg2xZ%@d3Mm^Kr=Bq9neOt#AimY-DEV&H{yU9ec};rdr}9=53aOEB#W@7C~0{QCfU473FuY$eRu$ci!}Q<H%uJ>CkCc?<-9kY!Ew*Dw>R8I%fk*5({?WRH@b6f_cZ`%t~d4JT~!@QWoML)?2zTqULG9ncRG@2t2a*p<X+!nzashPutTMp0@>XfIlLU%_~d3qyHU}!J-SiR$<$gQw#L^#O~xao)d9}qeo<x0#7XbGx0H@>1Db2-zHQlS*dn9GDRd{K)2_CSM#%Ua7--HyB`U)fzwP5+^p)qp-e;d!IJPbU_?}q`)-QByA#s~Qv@T{X-$AEd30M88n{!rV4JttJ)nM}FbTH3bhy##bH6>w7vE3Z~@S!;k`0zodRk8v~gC<2z7D0604kk|wAy3l)AADmjCK6Mm86)Zt6*+7IaY+FvV~G^_8kk5XUx3$C(dr@YmKjsYGUScGO}znA3VbF-=S#qAeCZwJgIHk&jsX}^fNQA>==7oh36Ni2_baV5PM5e}26E8EH7p0~sUXz%F}oB$w3gVo^L{gl+SSb_FiX5H+sW(gp&^23M6K$o+~&qh>St33EF-x_f8E!pHRVc8O>Xs6<*&@cr@X~e>h*w1LlokWWZLRn6v>&+Tp7!5DK4H-J+>u2z1z}O+S_|e*{%fj1R8c0et|762L}+ll$&~T7wcE2isn?(xY#~IaM`I6X_R8FWPbqr;oMYxVy+3PX9g~JiewthwU;>Hor~@B1D8AD?{J~(F6j$d<#f3pD7H(OwsROCD!Os}V6G_XYKsHJ5L-2k5M~`{(s;3X^JTwDuaL6)V|*kSONlLXaoF&1K8E$^%_I1YD3o2bO{RVMTv(S~wh#8lRa&d`36=5LGbAv^fi$4UIbro!J3*6H3oLp2I2=6=x}`N+GN9T+txVf~y2zmMu~@d&uc1+|;%Xitlhuvt2Q*M$9<~3>vsCS3!+aZp<IW-NZ09A8!=}*)TFI$G6Z!eLcQUW%t%sg8gXyOp+xhnbXwyi?PuW$j`B?C5d&~~k={LvxI{|K6eJ87)m8T~a#nXV6u`PR440!S8=l}bk|CO&CmC-=$z5FhBFOMoUaqZV<oxO@yDxI>9er?%F-i}U{ee9^N=KJ>ezkO__8Dqx-HU6)w)R&`vZlMZarB!8HnO-zXcxh&Uu`11r$HD&tP)h>@6aWAK2mk;8Apk-n{!CW|006KK000;O004Ahb6;|GZggpFWiMfLbY*UIX>V>WaCya7OK;pZ5WeeIOx26ll@;4<FE&)beFO;V1Oe>i5(ETS;;z`0NSUPUY=ivw4u^Vr*FNM@9c*ca!+Cx4*f@^gDC@W?#A&SyUc>F5@2`QEymi9BMjLRIfL1CcO1LvzS)?_>SokIT`B%89dBd|Py7PyO{znu}YL$Y~j+;G*W-ZE6sx5F;LM^tu*u&o+!F6pdz>0<|CkqSQ2r$BNiCVJgord3U-n@Txd;Rv+w$^L3iaRHizZztNq78oV@M)pZrQApqgg<yup_%ucK;5-aTUWt()LDTV>KeosX`MG_C-(LVlvbzqzVo_^s#81Lo8Uc`22Z!OETqF=l+wuu0S&jiEQ;eeiZ(_!z}Tj9oe_+AtF*zRO6g#jjiR3B_N~CLs1zGuwrn}W2tIOCCK2FcMZHPst&(La6vJ#dVFPJ&n|a;&JFP@uwAxCd%hJxt+&YHdu?JpvV#d&&Ydgor`)vN2*S2S3fLP%}Hz%H)?9j+98nU`1!r+cJMRh>Pl*%@pqU`&0S$IunC{At$n+&rhYMa#r9QweFF!~&$*tYea1qd<plp%}|Qlb?cy?cm)_RXPFfN=Hk%nGeGa+|f7P=guu8e9uM<ZwP785qGi|KQ=ysK8~8qguWdW|^(wIg+d4S>|lXo2TswXOX!j@4(L`draSwJSA|TtmtjcT~VcQ&#*vJV304J$!x`2K_^2U<uJKUvAJiy(vyYMYCMuXNb#5~m5`X~8deF8j)!mm&Pv(LL<uMuBaxh^a1j{cFGl9jklptiGDiCQO(O9_qaOZOqX<TLE82DDOJoTv-z|Ml0IWdqg!k#}b4dNC(~Hk^SE0QL(%(gf^aY8<m;{g_nOw+6tv>~yRug>EK6S@xI+;$SqqcNWB&(s;4;{fyOU_IM95jV~RB)Fh2Og5HRSFo+{9+2|kjm#p&ry`#mcrOzEY_q)0XY$lczeg946*ix&VW4jvLJQ@0b8TH)-I7YNsaM;QSL{$t1wWu!lqhu&Ie5>zOHdZYE|!Hy~oyqgAQk4eHzAP3Y}_k;NcOeHo^#846tJqJE3qe6DoLTvBT#;vtmfk7AL23CZTBtF}2U(rZ^2}S5u~jgc4E_s+TWy&`BlDvj8#p4JZ2`fHP#%lkv9mmu~BCnGHqnN@j8O5m;v0`SLBhV%Bb_0w=us;o{`B)16IL=^T8|a40d)@K8xV+n*a7rO_=K%G&%aJDbSTvJjR${7Yv|VDu}zJZ%>XQ!+%buli#>Y3O0{UlA^zPe>j5Tu^~k5pvTPM8ED>F$*K>VU*C&PX|<8f^{Z^lL>j3u-1A+SdSXqx^5pOng|?f%{@SG!p_jIuYlYNt-)Q6tF7Pi9);Xij+O%4{&2aa-H)SH7=RMUjLdIyJtBL9sIg$z`x?{yNA$*iYm2|lHU%!Rra$_;{$-3d%iSRUfJ~C5WdmieaGAt)+HXd$Yi_ISahol4)5>~WWxZ;H^(YOlv+=>;E_Q3*z4P97EILMS3ALPxTdKi6zVXP8SD^_7Df%0hGc>Q}d$beX9d6HvVlShE`a!+K%Y6QQUe&ywz3GDmqsY(-7MoPOS{5h@j{+HE1xCm+hHr{Dez(T8Ev8(LVT+d+Oewj;UHeEzhG!FhP_}FRl8N}493z$R^(8C5T@62iLS_T_j^t{Yvx6WW*)08Z9N9fBdPZ>U&iPm_M~+SxFXKQzMY6X73#Ig-N*2FI(&>iP(*zB*f$cR^!{Pv;>m?VTAUOb$jvcfIG`fGAx153WA5cpJ1QY-O00;m803iVOu5|gK5C8yUFaQ7;0001VVRK({b#8QNZDlWFZ*O#SbaG*EE^v9xT5WUNHn#rGUx8&mkZdT%veRs;T;3TsakrkPi8paF)A4FJ5D7`FDS{<P*{W9meb2!Q3F^ho&fdBAx~WZqz`?=6b6x@0bw9GI=2@HOvw{)P&fqO&LJ(H2^O~0{RyMCm$=0kUHLD5=AITeD(3vMl$s0D~{GQyix@1M<IJZmIx3&WM<_+|tO}=L2X45QrNqAXo$eh<iu%@jDEi;nk0!VKna#<8GjbOA+mrlwvM%W`u+Xe}hgpw64>4Ig3)I~HtvpFRXd9#FWRbH0R>y`r<GNb7|E3*ig%LSopo@YXHJc79zS~t)0RmJOuq`aILc?v8`LS~)ml9r3STmUJ-NWKCVBFA-I$C=lBMUrIRHf_z4glIan1R=npc_|#nwAG7>)`FS$-vlqsGZ%_dMVn=j&kUIxcr{(bvTgFBd+Ro;YuO0RP6P|e=M-0Arf&EuPZOLmBq=R<nWwZ!mQ*Z5AWYA{dzsAhf;rC3ueU#ac%OW^`uOSk!+R1F7eyS6U%JlcFLv`;^eQ?(cb$({fB$^_@#-zKZ5;wXZd21z!1k_KE!+ulUqlz*Mdxm)zjD}%{15#NC|lpkc2#Y#c^sX;j9!@Lo4hFagKR&GUPNbI_oBJS){E%-Xx!0fkIk&$Gfn+fbQV=yG&pGII8Mgqmh2J>I({%w6kPXb#;R`b^RmcGMyi50je>wg6X7jv8xU&%zqH`c#o(4a@EYJHB18z1tO4hB2_R1!pNmLO8j@db-u-a-=I5*TZ)5jzv2X*;U{bfGpYj!eJqyXV-`+m}c7*1ys&io0eMVe@|GUv|JTLtft$fkcu)Atl5QJo2v|<_G)-4O<NJS<Bt%FHuTL2Yc#SKpiB(S;fJ4MJ|k&EW&tm%@a_i|Q?b+fQ`?bn<)dMQe^DH@(w<2!m-=IPRR+qJ7^&*xS*H4J~i90|}a9M|(Dr%qsFs<!d(`e7k)M<YGLorVN4qT~_*_j%<99bK4CcZ|Ry<sgK+$@!GTF}XvYLegu&wCBbWQ$MR>zXX7QO^XSxn>;5#Wdz2tfr)9Q;4#ZfT5q6>=?uM~%~ic!BL=W2x@*K9LHfcV_6afJk3`d;1%HO(sqzZe=4u99DKM~$3)aB1ZUsOAJ8``2MX+<H1H!!=Ss}iTje{2a0b3SX{2lG%j#lo8FLE`<KXofSL30G22<IFD4Fj2x#MS8`Mo>q;8{r-OctoIw9gm4PpP$A(Aj?-c37!JJh3>z^);>s>cW*sws?7p@O>=ODAKS9YSL~{;dF{KOOZHf?v;p7Su1SaUPN&?VN$EM&1_D-uekjQ^FhF^;@mCN6gxLHWP)$&WLg|{5iq$Er(8Fpi0Fhv4sU{FAY~8eK5y%Vpv)kwT3oTm1vmCW*1!xx|Fo-~8f`c@xsJ5~6#J<gj`ijg7_<U!yyJo+)dCky%isvc1kP*fVQS;bnT>(D#7SJ;%$wjoPz`^)WknG1w!ULgh`<)a3j1@&s*ibQ2L&3n0)?;gj-Aaa1&}-=XtsV`AQAp+--0xtl{Ibp^zLdlLe%OJgDJxhF0Z0uU<Zcd-HaS3;{9>;e`fM>Sib}d-6FXe-u{n^TBtGG`O9}4m)peo*I(vdw-Aq`prbV97h7Ek65>BW%C*K}`FaD8&r3*>H7YXtZzlTUbBT}T`TM#@L2k1yl{eciK>>r~QYbeZ2L!u%yE<zd3$zY{nmA?WlhnN%TD);Z*RW*Z1b+K%w#CxW{4_&?-=|OK+LudFsfu2Ll`phH$)Qwt014m7Oj6@m4W5VBhN(gd&!kb`kd<;EHP~K#F&o+}jx{1If`p~4d2Oe;Dd{1HQ=JNI@FVrtTefZ@HUf#V;-d+Fj@$%!ZiT?cN^V`eh%k`)0AKqOhZ?C>wzq$GZ&3)A51-o4k+H0mlFck}%1tobo=V;~P*uMzJXF+Ixb&R=+<$?=NGvVWRFzM^N@zh8$V`)Kauxb@Ve;{dN(1ZtbrcFzWSg!wYPt8mRvM@964AYuUY(9j<S52X52^<`cYM_KEwGSj>X(7DHC>mR>I2?t2{SH4u%#7c3lnjsQ`W${&o0O-#;k>}UE81YZi|eR1_~aq<G@ulay=TY=SRT-fwqS9;a`lv`c3<~WsKS?Lp2Q)Ten1Fq;IHZ#dP!pTECC+8ATaPe+E@As-En{c&7c%YNG#ca7I*71LYJQ-Yi8{jy=Pu<cLt#Gq|Nif!=3{YYKA8XDdXscrv?~$mhfTKiOx6t0MPlsYCix{bg}TllN}w54h)77_#U`3uiHRWE69Rr4(=ca7B?~juazG+zy@5Cfr!#JqfssrJrH&lG|Bth&F4Y-t)m?T?+o%IIThsGZP^WrygkAuG{6af)$zF{ZTs7R{U2|}FTuW2n2BV(8L1KmCb|s<4vPQZV?_n@AS?ZG_#%82UYv&+Tjwc@JtXRdS+l8_?l=eiZ^ke8m!6AxUgiz+)oWQs^R`TJ7FtA#i3FHPG!se3g#UYp|AEv}+6vHZ3D6({31P}%t;d1{dH>5N<bK8<nb3>@JprN0-Q8If!oPF;Hx2dM82?SD!*wC~l-CCoqVL1#)nVTwQ0A40%9ZdXAE6=^M&AVy*cS%;|M!_=LqlxJbO|tk)P4_v7l^<O6Ntc}{el#<rp2boQ)zVg3{_gt6<Vbtuh35AtCh4pCydWmTA%+-^J8i**5W$(T5Y!GgM3~^7txC|0RD+C+7DGoDD$51(#k*UaoRO_lLO-ws;!?K>KXU~<>O$(wEwaEfw3~)<dpT<1%YD<ABxMl`vwyfES_vJj@sEK1A{xsIHW!Y6$^kI$YYq~57NkC`iMKFhy}wm${n@teoMO7qN257n<ZaU=h~w@jA$igfVz7m%9klIaZ7H^JMAC4)U0n3O$Iq#Y_0v~`sQlzoz=Dd{nPE+51()Q9w6uVRB=c3QgT+QBNi#7z#}y?CPHpkelQU(+H=gYFwpHYm+q@hmCSo^&;w$}8W@OH_gN0xC0~TD7s<Lt!tX5z42C|`V1^SEm9JQaA!W)lfL83b&3yDq=Ryj2eV;6L<({Y(h*2|v+vB_cDBVEs3_Knp5XNYksPqa;()OxTS>%OP&td$JZBdXeFh{ll-G=#{JCLH9(|szNc7~;bE6dOJlgz=df)n&XbPd@a@VX14t>JR_k8-O7u<Pd?<c?MltR<!EV=9=zLa6YAyAcM)Dro|@f`xAzSx%=HMCI$l?EOd%g-{AfmeWPa1zNrYpdnkGz9@U0F)*y?S&I!xYX+Gbp>hYXG6e|0GzYVtY!H!PY@;l)D&vs6>Fcrw>>2pV3?CAu3Wm1=qb`uC;Kf>oJ%`1B{%c40YXNRqh*(uH;#gHJju{b9K9q}s&)!D`SY0^*VqaBEvFa#tJn#%3xvWR(3dG#4HW%Zwkuu(+?eGA>H{^r6$@qM#p{2|)(CF!@s^)WDZqTqxnM#od-LP7rnU4^Xs<UNcdex9yv4@-hX!spLSvJ5@IRJa%4l2ES<7~?|C}a+~>vT^NbOuE^y`Zi@E8(YBe57dG)SF?A^k@m<pddJ`N$e|=Ro>{U66WDVwbn8Lig>^LM<hmk+c4P&2GJWC?%$|4{kgBMh2+C0`49{js_gT@$_6%03j<NFjNEJBQ70G}OEgQ^m@Y5Za5+*x?M-@kWclqH@YJ14*tPbP+a@d7ckMbP?pht7xTBGVb+}(e>#;Lc)I~d==Z{zwO55@I*zF3Jo!TqcfaOc)uy8(%cz_A#PfFh>O5PUB&Jl3!`5qZ2#<9$WpKa9=_G{I1yBZkBD&0NR8KzHpiN#?6!afVOLH?7P>H*cf%-G|ot&0QhmQ7QM$@AwKe<;z%MX*#E{qlP8{9#!<Kf>zNQt$u!eQ-zh|A?rjL!))968#PNP_gpzn!I^;ed7sIumw#w>ck!dr(LQb)%_wt-`4^O5s@F-ya0cz&fzU$sH^%+UXqZ7CPHU?&1B7h(F_k$X-V3$W(7qrj=x0%Mw!cZi`q=iVtaXC9vgyvSOLWhrlXBc4TH!zeFAlO`n121Pn;(`ah7yyQ%`NI9#is>T-r#jIX^o75CmTdguO$9A&u!?K&Uc5uu@OEr2o_v=~I-Nu|L=oXSu!BNvo?*E7h<)^K0dR_JNl*n{^qYC#)TJFO=<`^6DVRopce_o^C3M!FQcB55Ma8FiT8w4ILH#gV4GkwX^fHp>G~U-gGr%b$3j07}53t^oL57tW;sBHnlp5Gjy%Lx1eqqJwF0NDX_E&^}QePz^K40u+{RQr3;`t=h+9%|9@@eVbp#?QKU$pFquy+?l>#y!TR)_9YDk8H7ScKmX9*`@;z22xJAhWn<d4<3tpT0k?J(899=|zjlT1`E<j73J=}Swk~`I%#@><jm}6Ilc}fC<{Vq$BVo>K>J&4{=P)yQUeDoax;v<=ry#|y(2o2J~TQdUmM8d`xaNA&a2-1SUV+rTry!ELp?BC9_`lG2k_x-!upOd$jx0mh|tqz!sLG;%(n8RdbFOsJ@2>;$o4%;JGG;D!?7WNc$j;EC#Pm0~DE${JKg?dODI_pn{rObQ!-MD8s@PXG^DQ{%B6gC&{!5M1{s{%p`$brF5FEqUsE%Q0L(EjpC@2dN}3O0Y8^A?%2@_#cJBu+C`atQ(%tC|m8jw9pM?XC+a3_Y3q=ofh$sBUZpc;7%u&I0n}s|z9Sk2U{;mE`@qoBh)`bx#iy1whL%KNw(d&(~p5S_U2YT9zReGVCb#cvt7nApGqvi1LQ5204LFOtqX1H9OgqDTPFdP>0t6*4GYCv;Pun7WS?e{fSuiMPqiF%r<y=QcDJ>1bL8EE(GLIX!ZpJ!Ae-l3aE@LMLoA9)k{lmJLX?8hI0N;NFBaFPIuEFIEo25Od~rW{}p#6_Y@29o|yZBzI!Q+4KLM|I{^zL6jEpkF?&i>LT;t^#%1b>yNx!Gk*-YfFt|l!eLrLie;{pv#`a)ul`YV@#c*#lO~97^EbCjHjR*<2Af~IoYNHd-35J{82QP95{uC2)Y1{hUH#6!KT3)4)%<QzR{xeB`Lq0NaRTwlt*wf7hKpS8XWhU<tqa&!Lc7mhe^Erk+(cmV{))`duQDKZW0f*YC9=Qe5-=VQ?g9z{~Ox>i3>P5k4zWZ(T2ZS3MnY+Cz)XBRu;28IA&FIUg`*v(_hsEf^#4yt7&Oe7rV8wmL&P<d<LW@s3a5YvqG~5pYMct;oJH~CsR>>R4H7wQKMvgA)MT=(fMt<=#rVc_deovB&r%4i6BO{t+3DrY<2*@z<ck$LI&ta*tZ(q9tWb2!zv21>l5N?ti5*?3ac7^oF0LP9ARZM3nm?;rIGK?^XbyoDC4<UrS;?di-6RhFxOX5%VlD-?}r}AhxcBNrW8d!!Nhc{udEUSjgKS)6Motc#Pb73|ao?6aj|FF-F*9K!u)FvC{W1yivypX9V9x2|-fhwDmkD4ex4nk*)9D(5^!Rl&~$l_I!Akvb=o#^lbVdwk{P)h>@6aWAK2mk;8ApnM{zg)Wq007Y!000#L004Ahb6;|GZggpFWiMlIZf0p`E^v9BSY4Cbwhn#wuV8f_YIkg{?WRd)lbvasCY?UqW~XWUlIdt@iL%+1NR_0#+2rQG_kh%wZEd`J{jw_n1VP{c1Q<op>km~SQ|ac+3L|+TKk`i>@Zp?PTg!57nA-`K8~srzwtvRNhYFIVP>vU@GTP}>7jYCtlL>@MmLz%YY9o?_$+FVMF|L$$+)1tMWYW+w?s!^oYlUq|e_O4Bii*2kAvf*%YkW>7lkW#>;_m-xu<Yu0Q;XTeli1Hf3F9|>rByDsmlK9x$v-4Y+$K)n2xTvsRE}M-)$^t2I#E`%2$o;6ywKb;<A$lLq#7sR1V#U2ziK7Gqd%UR>BRD~DxhwobpaNc=vkE!skAWfl38cKiSlaoG-`OdtX#+VN!ZlLjX)5jg+usO#6d9CYM+!k6P;{P@I6oVtH3vEn^YRXz3T-0@<l1!PG<`%M7AK|V6Dgzxk_7?NF`m8Osy#LS@0bFWX^~ZA9|6!WGe=UAi!*i4=>`q#a_Zs{c>*+msYSpc~Ohk#%ME*j+B+Pbqr^P8PdtUco7`m(WPS3<$U#59L-Mb>f8;uWsMzuNXeSz8EX!>^o!n+LZksR;-6X?VLQu)aEaL;P}J>z^@bPtro`DA`1r1gSNl6P-NT3>yyQg>qy7k<_42IkQ4nk$<(2rbC?&R>EFSY&heP&Ni@`_x2o=EyfHtdBN;o>ko9ornnEk*qndicwQ1pOj3LdeHGsR1h4a&zE7xiHFrj7#n(Wd{QY+X@>fhE8F`ZEg~yuf*EU5Y;Fez%LT+l8ss!d5s=E>p@Y!ApB${$q;OJ6C6S!#V|AS!o^omKd!=De#O!r@ZKCl+hQc0wj6ixLxd@EpD~B3C36lqx<?&=-WUiTTtCQP741APB<V6qpJbq_u$W29LIk_Y8oO!p6>XldOd$RXRA3|2eEBeelJf-F@%Sn$)zZ=DT)uT3zy_Pb=urr9WQEjm)nvjp5B+@WxRaixp%tI3A~XHdFxQ8ab<Bu@Ecj}1vcEJyTr<mV*Ge?wmEw;XU~Hv+>z5s;fALuUH5n`5{Q9`<}9*V#hU~aA4FQaVe#j+YeE38xFpmV=q~hDIy-WG$U?iLYi#{w!193YSkJrL&j@E^&sYbyDqh(gR*AhZcP;PRhKc&;5{i4bXt{p6TCP{YZ@?lB!K*Qm^^pk51oDf!-hkC|8T^m^jVl0pM6g!w(7;r~FdPUcF7S^^#X$hqjQxk{TOfw<3I%14Y<9Pr@r~6*?F8jhO5MtqE*2=ZArB<qDqMFmJ>2uzcOk-v3Bd_nP%RRkJ7E%A!sVyoKt=GtlcsDye~*MSy0GKi-kN&^a;bRHvHTU&pb3_Nm6Tk`T-YYDPur1B(1T-8MaD$aqNa4Q*tQV&iJ>$KvlqZ?HR2UsZs0P==o;>~_bCZ%$n6gsqNaxlB{<<g%E;jf)2Jdovr7cZL#u1jo>O)|uDS6=(_h1AH9j2_^-x0RhcsSi(j5~$OiqX?dI5DzZD2D(_E<Tl4^VQ<^7uF=wlN>jKcBfJbKHtI3hC5dAfI~2d>(7N!gbR5EZirkcs~C055Xfkw{6twX*(EFGY=r5MuwEcU)cVu_9p9Frw~hGbZ&Tbhw#IHe|-Jh#r50k*Q|!DS9$eytMBU<?A6;>Z(%EyFLX8NGZj@);I`l!QJ@lWkl~vhh#R5OUCGVO*>0yj)z#^#`nxp_lY&3t=#=p2)4q|$x`jbC%)-H%9iS4e$BoEhb{#0`jw{?SZGNJkv!dd#OEirghR9&#7miAh36b(WA);Ieh0Ju*o;sV+`2xM8ox>8f<0t`zz-`K?-qcF|Qw!g|9D%5_VbB4PC8#FTWBQDSopXB+^>{2(E^}l?H#zv|gpPxGxf5JB4mfQxW^92BJ8UY>X4bo|$L$odzH9NyF#JQ+SblZQN}u4qTcO?kdx6hz(w3dKq*vBvIq7O&9(<a6O(QW{(f}7A(074;#5|ckyGHOq6X-8$&kLEcO;P(olL;rnAmeNEwr-Mb)HerRB?0yEBj)cr8f$?JH~^zmI}rJVv&W6=)$#@1Wl|kv+(UEiyWWY%&5w80^6}PRFBd;iWwcPffPw&S%ls=2JsX9(F=)1$VtLNiRUhxe){yAyb3q+CHC()XhiR!al(5A&iOtOOMZ}%<pASOT$g&RhEGP5mRgE`OvtsW@UN#P09%lAI^AE_c4<rEKflw2S`-BD)0adn9nZx1WKy3EV)bhs68Etjv6*oG&N5-+_8foqRzc-vQMW^pfwW5gF9kA*zdD}h$C|vfeS${g_x6LKD=-^X#+*abqwNVEHFSos5SpU6e1t4|`DS?8@w6JrwrxOko`)?YhNg^;iUf&@=!1H2sPt&NUqh^2leBg5Q-U{aiMyI{l{A74P3b|MQeyEtH-u|-_7HHTXvh;Yr={Mi>61~Cnw00xz{BvN`WN^5VMEGZL3Hw&`<RXoZP68J+3cX>MCol^3LWa!rlm4_P&5i?(n8EZWF82o+gGv_SOAp?sxB)Bffk4mWF)guEdJv!bHL(#qOB_Htoy}&Ge*sWS0|XQR000O8001EXcC!O(NEiSB22cP16951JbYXK}a&>NWX>DaMWMOn+E^v9xJZp2?Hn!jOD^Pasl+4PK?IdkisqT&wyIbFB5>JzMW>>4BDH5_+Q>2y;+p1Ro{hkLv0wiU--Muq6A7TkOH~<a~&Kq(Z=f^C(m63S&@!dyJlvT79VYbThvJgQUiELBGSsEllK9n-Y4-1)wt96jyigl32OIeh~XfU|S<tE4j1Qu~2-d+8Cl%>gbBra0{9c3I=Nstdknu(3fLrG}GI*Vk2@Nrs{L0ZN^2}>!n!G_RD5!922EEiFBpC(xliDjI~VkF+_g#u*4+gQkD2HzNQR+XEoEC%u+1_ZdBcpcnG5l0f{Dz`XuF3TWJVPp{ISpm3`MEtXoc`S=@A!P)ca~x+dSmxPU`2Mmgt6cg%VA{akfHciY+IBG*sJ35=EM@pGOT#M9VYU$uEcCd~t6T=rRhA_e4>GLEEawrMpj;*KLW5nwSB*B83{q}4aeAX6&eH8rT*BD|i$o5^`(T5;YP}<*{4xm3dR})R;NO{QzR2?|A5v3LX6rcgaqXRkgj>QO&C)mw5`PsGD{d=R!RhO-eLRriz|*r9g>0t0`gnEW|8()o<)_P!ABOcI`gcI&Wh|qiD1xQ*fx&WElK)%084L!<j{e)TPZ$2rKm91C!daE&rkIS!#miB!4t@ikye~#!wjR@w;>R$Hf--=aM_GR3Fov@aA3lCQ`wSC*QiyECe-h5vI70KY56g+uyC69|KKW(@tLj1d_HUmrJ^|9x*QfrUkH7I>zd9LwDLyWjaTvo@`Sem0a2Sa^!Yd%M#ji3fzbeGV=ip`}&cd=npdZeFg2;+NQe<$-VU|b0kHFBti`!9m*7xVHks`}ii$xHA{p$4i+pi<}YWey!c)5(C;53r2m%(Z149+jUKl}N|&;GlQKYaY@v;XPx@3h*TbLLFM%j2PN-ol5|SNQ22HQ}do_;m96ct2?7^xt2+e|zy04lb)rBAJVpj!C-RZJb7Xe^+dhxZD%bkarvgh1*LVaMOkB5NREl3&`tuH3VW{fG{8dQ6LP<hENs&=?#dR;OJ;0H1aOUV~`yaSdV6sJh&g-#O115R0T+zEG_X0BDb7n@p&HHWa$_YjdvDv?$u(QF-wjyG8O&jZ?cW_cWSDAYiFvNm!5;1$=0}V7=uC&KL!zSdx2oM-#svRB$on30ADZ{@BIYiZ!Z2tm);Xc|1A;_TeBz*%NcYG@v6_~6Ph*8?h7EmY);?qVXHKX(0C(QYwCiS&LZHxsZ*89qi-Eg1cg|^;u6U?^y<@Uy^w%D0WpYMkkA0)q?9g055<YsjKF{K66APE14gj2c;mJJNLm2Z<Ft}iN11QCkgyrp3({ePf}wB`v1ftF2UHp2mmsMm2`$_VH1;41;zHU2&Rsf}c#-2s{LiP4AASVUCU!`v{ba|R-vf0>bOFk~oyLV`LKhYbG7Qedd+yxtwFbj|t}FPk0L7LnaWLsFWzZr{F#@43(+KD0PEcA^o>HG;C7;D$hmVGQ?Wl=(ir6VmOWs6>vHmzKq3|`2QE?A1m{*5ac3<BbrHfM<Idd4qnKu|&O@oTaVIXw_ROHMD#_6~gv07)<AQ-|trA=5|K|kmMv~m}=Sqv<h7hA)$c%hFE95S#PfvL8Cd0x~!QKd0h9=K9@1Jc4^1XQnSu$B!Su*Juwr^t~@Fg=oF9hBk9&7FTlyOY!XKSsZ9Za^++yxrZVrK>w<lcQI2Pkb>oXz+V>zI1+0<--PeU74Av8`Spo9B>Qt$1kiaz=WPPGtX<eum2%WOpLV+TfK<Ee+G<$8ieN?D=jvzSG9LqLzJxngB(;4xTiJ<;Q#>o+v>ZIqjH5y;^XA6^A^xb#y6|7fl}YwU`1AeMeSF4LR;8bCeHrxz3feSqPSW~6i{SfQNztBCbfZyl4Cv;FF>q-Z9EYqNQw}oL?dBHYatx7#+Tv;uoeXxH96X?!CeL@MI=$Afn)(gTflYr>BC>qU`*s1&WKG4WJDC$Ja%NRv;e~r)@F!`qS@1}+)k5Vy@&$i#3!x0VH>k<N0#q8EktXiU$F^7<Q4}w;OMS>?9H(Nh4A#@&mjmT9D?BA12E5E-{u_aTWJII4OAeWI_*UjCy`I4PVLq)Q4AYzqTLHt8;zDx9GPcXX(=CSeW5%B%`j*$AD@1Ga(sHiJ)m@xs^AGMMgvIKYr*)NJi7xn2BAGc#RV`QvQ$wosG_)pT~ufe@>JivR?Z9gihdFQCfoa<ss2T1ENtioZh1doZlB8{%wrJK<q92EJeCn0eWO%sRmyx1UgP(<l)r&E9Fx6^8n^g0!)3i4zZ;*AbGe3QSjHHPlOg%1U>49GL{Tk3x`=Ek&cVAzeAm~;NnBsE)<7c*a6l(b0nLcz`Wg_z=eEkh%TzRKKEQE&=W$@32V3E4%bR^{a;=LsI3l3;P-)ZVH&wpL3hD8cJQE?P`Wq1L#Ofx)42`n|enHyfrG_J1Z%X#Qe`UQ8pewCT8srBG3gjm6@mCp4B?6JkdljF+4$_h-$0m}kC^Hd)HC@(#DDrrDpK~U%0Jh1KV8hcYCkyPDD!om?B^|;z01#mk#B1yU`lkv71^_gF11peugrB_;=rtfXIvK7v0{1>k?xf4wi+4LE-f(+8G0@j0SK`8m2W0{qHhYu}wFCl}F1MgMMn8^xIC9^}bK4_%gLK=-vvRwU?j73A9!`)l=>&N#&?vqWC(4IO$oAlMkLYWmwqhuGR`87-4t5PKRBe}-vH)ElFSi1CBrfB-+M<DrK#VWwf@=Flnb;@dyQz79@hjn+!M5g)E#QnU3=gW!YrvUGV6v@wlk6D{>yK{wkTn>3M9@o3UGJu6`erJ1cX%xG**~i|2Q}a|LJZk%sudU>XmdrpdVP3=baIX8ZXcgvkbWnQ*`l^=R6X0HtG}oxuW3h8LmydfrK&A7mz(q^+w?VlHsy?%kZTBo8TNu9Nj@hUvxh<fdnK^bWqf0ye?t3$K3#!G)h5}Iienz#omfD9YS|%H!)E+zv>O}=y~Z~KhK?J0nTw3EMlgwKf+*SqGzRDs^R?2)73sz{TuT{h$X2s4s#Jo|SAr1fq=era75=3N4G~#H88I8!0icd!FyJ_(s*wQwbiCIjE`>=9{2(8bI0U2(lM7J2NepLT`Yn2sbsNb*2UkEem(d%&A`;LPR1_<jL`PXwsueKb)WO|p+s0##Vjhd&T83qoBN&PGv2bd`$FV_1>s$EjVn7<i*7S2;$swyLKf9%`+AF_-wtVZO>r9eg&oM@hBxW=sNAc1gF(Q<ODQ$}2j--VV8?lE_1gVq{W!FddO!^X{(G5r&N7KhKTsoi@2Iv|fCRL>lm>{|7X?L$<npzo#I-GCKpe=|uqUzE0j&0jo!7C;!+HlK^v={DSrm5}BN5DJlq8*>=IUq#9Znfc28`^@%9TzOsNK2Svj)EmbPeP+^`~0~fB1IuywYUeX1a2Dz$H(iqz!)+($lPl>$I2P1Ep}fs2U#aWCI*Aky@f67IFTurl?(>Ijz5tv>#5jka%d3B>p9D9V?Zs!Q0GoXt=}SRAt>4quW~UFirT@LwlJ`pIc2+&yrgO05_`}=>vNFVi-)5I5817k=z(TcHprPX$1}%nH6e9Y>xSPMl9XIa$C(z|0gI{?EV@jKin0iBJ3$u4H!=7`I*P`2xWtI%Z!%REg%z0E2a$ur{PkvL6JsxPmX+CZ$>(%DurE~4y3M#;F=H_F&=fgl3fDv<{gawbK{l64_fKsAz4Dn;`$bL{|HObJ%|I|ltb)wNzN5Z6J~}^QW6<$t6GN5H4F|CmqpZb?R>&0mi;AvT;UEkqNa&{3w!N5w?|A00LW8{w%)l;&@GTLsd+~x1nGvsH+r-e?1R9i3u#hbK!<D%F8m>8x9$C0_w6Gy{l{5m_ObU|F8`3@+k))_-@JB0HM-q0z;uLR4E0(1#Exc%ixgG1$c2r0D9tistI(+6^F(Rgz)~~xpD^6;aNMttE;6y2{n*K$70L8wYGqn}9C5xXawpAFeA8Mkb`v^N}1t1Q_B-G5!kpR0P0>gqMg-byXc5%%vwE&~rHk&$jjybvko3waBc@2k9nwr_`Fgo$jlj^@YcWo*%btwnEZD_0%9l_{EGol55>IhXUfAWkFSR}zlgRNqM0l1C~$wvc7?^ZIrrILhPE+v?sp^S{kdGSUpgE*nO0u0n>Yg@-MfEi0Lokxe`7k0=(i6kQ}scGP$Kr<X>-98j9t(sDn9t$~)|26{$I)7nER*6c^$Y0_)Vsb3zl-8s#?*SbuN91&5mtq?YZ5qC`*E-FC23)X!Emx&7;%d{brUb&|G6^tp;g=Z`>N#1COF9=?k}SetgH9AD2^3ct0eH(euXhRiCra6~brPwaB<evEm0IA)O4wGx!6ej2$Ir7%9c6otIl95b_u4ZnrtV;*eCpNZkGc$m$TR4GQ6!T3IM>LQbf6-vlir{>2VeQ_y54^VWjlA)ldVzwc@PXkuR2Hb99Z549SV33E^)X{1Tnwsgpb#Y$D15P-_M~{lZT7qwM^L`-!<!KC!dKMXPQ4rn^UAjD{z~&hL5S|#zaxMeD~ugE<ZBjQA2ylUT>D4rH9Um@0#SDS|%)mK$Mz-@!r0+?|Ja!*0)Up=UiQ4mb+9(z)ma5ULd8rLt}*zdwZst2e+MDW|NcS`M$BBEE8ZyYOPR<FR<99u|rzX3&uONO(bsA+RSJ3);Ih#m(59SE<_Sz>(o_S5HcZ_z>pZ{?MNeGB)Zl^S9umyAqJ(_8z2*!)R@Z~M{#hIW`Hs@qy8eQDAV`u>SwWO>H=#uR5Ky56DKcw8Hf9D&!#n3hfaCjo=#zgGTxwiGxj(_QQ(s+_24=q#-~h^K`p-;a2<i6>!~pIgk@k;cX3t~?Z^znKQ|_$v8b?2x~af1LGfJE2UlT3DKZh5NhynEjU64#(-Z$SsUSs}|4_9D=+Y`Fdm_V>RTy;5w>tTNXgf9=sy9@R9Q+nLT`DrPLafeyU?#M21x-;2+8WAkI$U`|HcnG!LiVI$vblxToON#ba7~Z3OS9}l&uSqt_u6#Kn(VK&nnB~inI4C6?nAX!ug5JRE*&BniRC6Wza5g_240k;ENfJSE~kufNACZR6tFV=4K%PALetHbdH)w#V<W)N#@xZAdEe-|I+mHT<ttba_gQ``L6@l|f6LuKn;SU$LQ`gwV~xh6UdELww$60js+GY_%-CeXX=*I(uDMZ#rBs^w?H1(EGQj))@>mB$IBta_zy9+v1r>D@OuLk2xI#2(6yL>Bg$dm^7~mp&C*S_{twIyz;fm6Uh4ODSDZs@MpJ+AW`nrxDU0>G@5v&%jp>jpAh@Iq&I0M90QR-q_E5*)-hk15BJ>f1vMq+JAA&M$oflyRwd7Ux$>&WEwwFfYp4aJjys#IB-ONwm+4Je;3N{oVoUZS$_e<ROKY4T@gC@ks1JJea{Mzm`(PrIesww~#$upYLG7ulcJh&V*>Xpyyz059IOzwFuuTt}~`eTC9Cl)fBO{b4xhrWI5nItuG1H)O2&$TTA?SCm{_I<>XLT28c#<$9u#a3d)M^%0o5TV%cO#uQsIG<8D#Mw9An$HTZf?CB$OfRcTH7A>14-eeitao-2v`P&ISO~fAi=|F1B81pO}%9!6sT`|#=iNpk?%9<qUI(SyvjqhfIh)r`)lzWdFD1y*7yvVod`o6VgL*alpW%lS;Ha6oaHy*dtrj`9ZLZ9(aaKMncshD<o$)3uz13GJNs6TveF`OdxMA7*`1)FwpcHQa#_ROE4*1o6OZP-IX3_eY|_mO99N}7A7Ys<uXW+T(ag&e&X{CC`F%K0i(M{1j>QrzAMlToPAWijVHM`ujqQW+dO<su!J%G>v4RFxyvRCWB(;~HrH0A0~K-`j{%)pex1?38$W;3Mvr-*!R@w=??pI%9;>6eNC66$fQ{EqeWtwD|WbeaF7Ed5lVI{qCf=4$uQl4%c(ieO4)74&xLvr)$IL+-?tyPMXe*Ye<5qQCxNQ&5Zge3x8_V1a`X(<9_;hL5pFwPkRe<OyhlOe$l0`cG{2P*6OQOh-EsTn#40BM`5RXHHJs3uCOAW>TYz7kWFa{%ItJPNku5sqkx{Qq%oQOpht(&n=pRws&$U8C3<+mnHg`NbNs!&qAefAyhl{D)^<_arE7J)UmFnHUY<T73Y~NP3{;QOm1_%G-b=VrC#1?>o&pJ*fCc!h<Y{1>wKrjga=kA~(qje_ag4J2`A|zYlWfN9UyAQRfYzAU@d1ZN6~UKp#End)Du$s_c<tkWb&SS-G#Xgp?e6<gcO5)vI!?PM5EPnRlWwvY*6=VCzWw@3fseoea*1+7#T#{TsMmcOR*O@%ujwFM!Y}?yygsJvgV0mrH;iI3TJJzSTjPRj^NTtFt|ijij&SYAI|G#&$mUZE17*=4Ncpw4#s}JZhvGLuMy<?v^pQD7+cD6ZrIiFzl(G;Xu$4O~42@@xe@xxY1_w+tt9|}NhSl8ZOd&W>Wo5?h&`Pj(VFow3N!8A%Dyn{F=ud1YRmzpC6x>^E!N1lTRN0;;Q-C|r9LS3o_Cki@#f#p3?gvkr($;y{X6kNm&^E4@)3tRUNeR@Wyq;lZn&1%NzR&+{l!cBys`=;o_S9r)x|A;E^nj(T+RuLZrIb-OHZi2+WlHs00!81}4n2XJ6WmhesWO<5*S8{AmLMgA4O$36BJNkPO#>mlHe%?Fq+b`<1UZ?_*^-K=k{}0mQE@8$(%3M@Mut(7=(Gh^nDx~m9<&fT`C8Yi4kEYGWlCX-QRCLV26`Q%Y7?QHZYHeD9<*wo8Htm*z2GO`T2DGP8nE%DV?XnD4kaE85vk70G^8CaGNMb_5|{X|FOS7WC`kw_jnyQuyjIqncyXo^mzWAP1K9(c)!QL`HM|AYbqNS~*Hp0l4j0Klt9Ibm2S_WR&*}}`6@12MHkKT&gM7<VWTjkW*)0b9!2(e9Ap0Q9QZSfftQb+Qa4r|%8bu1<V%v@23sqOB(SB7Z#1bp5H~I-)ujp$#UX5UB#jctfeXQT7I(-~#7wNa<o*|NIiPI?@(-m1g_n}=x)~0t|#?=Be4|>ugzngP6o=kU&N1hWOQvB8%A=>x9_w0RMV0onnO5!?XRT)UT+fWsI(_zwY&(&C!4kBYy|3-}RItDrpV?RiLG1S~b?dA)0d}o))n;_gey5<tNbQwR?z0RYVj1O794iYS=s9%wtVFZSL>_I)tVXcpsB&*8WBIjRJG4dGa6{ZgGMOpJu8;y3j23%FeQJ*R);|*L(z+KQ$yb)17k|rc<dbED;HWD8)P;y{w!fnd&n$fw@>$oIX$V9!7(~g1~)yfi>zX8z{d={31BKsbLRJKP*ta|9FJ%H=C0(cJfSG`G0NE%$tbe-CO`PLeWHmMbIA);DywaSaoHa3b_)!R5!wYCvty^u;xIy~c1{)PTkqgq$%_?&jUd2`JJsOS9dfbpEH{jk3Cw5$)n{9OX1N(+_uN1{!go$8IMvM930>z8b_#4#P(p<p%NPA<SB>1W<o59>Cm;H%^6=qfe44$m2loTgHlVDL9{ihpaXab1z8Vg#HU6k0;#vr$xdn5)AlNov4jZ%z#M8U<109%jZisSaH+o#9sk&l~jp^2bwt#FOvCt#{<D^&En(LFqVx!p8@l4{o2cao`DD&#xt@PdzY?{9FQ;D!V&}WFxK%=r>c}=6hr=^@Vr{#T}KR?)Ghm=Rh5nS^s=bMg;sZp9eltrPQiEf5czg_0XZcZ)*kS@*jJ@XFb8oGt?G7K90+%zJro;cHJ|*)2n?0Hm>cG30<Vn_md8wfi5^(1bP<#C&wFg92I@mj#6DvK<c*<v8a>k*lt}G#;;av&1IR-PenOfI*n_%%aAIB&0j(37QZ^~UnZ3CWT*6%@BPK)UrO=xPo=12yb;|SS+&Ntm{1arr|)FgpZc{FjtvvhjF1KxHUWmaa1mco(^vB^I=>f$k5f=E;V-z*^rM?Rt2XY5=XpK9pn(sW2LB6CO9KQH000080000X023CLJXj0>0G%ZO02crN0CZt<UvhPBbZKp6FJxt7Z)9n1XD)DgwOU(~+qM;c*RMeNkdg^S?>f^=MzJzY6Wj4Lu|2jYomQjaKqRzcmLeDe+=~<ExA$BC5|p&-O*`3#U5WtC{qh}XqbPd2lbtHXrjt#v6Y544y{&YU!ed)kMP-GNdw6OtgltMtHKl45{A#RdRX6Wcp}SJNdHFn>&0cCerMhYl%d!^R-T*1F)!n|YrKpV9^i^$vh}FXG;PYOWeXTO_Q>Ctr07*nEi%W%kW>@fTzL!7iP5=wlrEo+wqShCc<-I^3wgN5PC{b3%cGZTYN6|#5K!j>9FH|;rw^PEXS{0TKTml{7zU9;EN_Dpa*$KS`CerdTAd4(Tqi(<oR<+qIilW(U+v&Z?^KEbYPUSglZgpqDJdL&z*_+wSy(^`aMJ<g{#=qK0v#YC(|M;`fjeoXaIHqoITeP@)|E;WRxv5nuev~aP4D_w;ik*`vYdgKI>foHY1)p}hsotyEY*wnRV0Yz4sWQf23-m?G>gH9MioLQsUBcr6GzOFZ(t~a2Su8~qC1U<H9$Cx;{Iv46*0O{Z)FLbUeQRP`!Jkz`WamtBj_l@F=TVC1iFuMD-Pkiv0`89SIvuE~#-O#ODXMDuOx8w?Yeu!QlVA(8jH47dM2jfl<%E8B3S8A7;|xueZ*HwJvGYV`cJlGxzKESRnQ97MsyIoqvbs>l#>r~&<r;`?stYi{Ct_4FJ{M1(h%Y_|vxsxYZ#MgT;Pq^Hj)C)g)hyq2y-H^EN_?k~)vK0*hHbi+H+iG3a|%*J{sH@*e|bi$R_!6QB?jI?Y(ZRF%-o`G^Y--XOD6Q2c!{a9q(8J+>YVZ$$ZfQ)VF~4<C8Z5Z0sa6%Y!ACu6fRkotzl>UI89s}l*3LhG4qCyw7Sss5JBI6^XA7_FJJuU>30!rFen|LH41#*6l&mgz;WaYMkd)Ip!2-d#^zO1**u4+s<+8va8I?xq~B&!n~Epm94d-{Y!zqlfwqwC#1pXEc$^|fRz``RWZkQ$U8lP^deL0Ty7E<|uJ%wOff^CWqU4DAU}37C%xv@`vON13%eTG)H;^?@`UL1)+3HG#@o<W_H$UlGHR9Lkho{fK`S#zKB7rA9d_;iBD*i;@h(_+AdNMiGe<X*}S>OzwG$`~S1XSD(d|W}l$)X9(yR*h6$}`<v%dUJ2^;NO3+y)k00ke1DSKYZ_G<G5M=4UAOyi+w7c1LmfF*cl!Z<Oh4SK@*MsG-5E=oe60ugCC(Mq@C4c?Sgg+Qq(ta$G}P7rH?Ll$b7Zk<Y1>R>{kJulBmT<toK;;}W;%ywE+=-{I}hu_?t|ncr581jbP7q0F(RQgBjrf$yQ}_AT}Yc;zb3v=F%rf?oIZMi#cdrT!X(+E>DMeN#}cgMwv?oifxZp*J-U72iJ-;4Ro~GR{;NCSVbxx+~b+NDSY334KmEnD7XUWmWfxPjbu8g0Lb&(C{G`UVr|WRTQiuDAL=ux1o%ImINEXQS-}3_#4_dl6u|gjod)Lw6~&=bpZ&4v<J(Ys>H5;JpdR`me&w<xNf7sE+|RnjYWUCdxNJ8q8_xsg=$bFIG(X5a+e#vOONUGPPhO3f%g21jF5&aua8$;9b_XwVPyJQQG>tY1gn5^@K^|zP=j3!5DJ>8KSL21C;qH!MdlKyIy=Tbeu%Jpi8+n6h1v?y8#>F!xMMXW4gra=4zW%dZct@Tc@+PSpHeaBl&_j?a>!^}YyI1>-2R37;vpZ@k{=$OI(4r;K^^^bb(-|GK}u@wlVUokbQn(Y<WLf$bgA2(+(o>_cA1^s&EoZ)EZS2n92DbMu3Y8CPHUrLpUp010MDg7eJEFwKsGe1GnUz6+xN$kOg6W1S?!neb9k=nI6wm2(w<?ELrZ$Cp?aQZZPBx^5<%nGBC2=8z;y-K6l#nV%od79Q&!MH6gP>Wr6r-Tht8WAAd}q%pc5~)$4ss;Rb*q<*66Lpo!&;_iPL78(D`320uFTQ4rAbi?pM?TX;|4PD0yI~uChEWLM3IWR7i@7BMUiGI0c~<Uk1~|L=N5pQzVeq`%2bvr>|3S9>mIWcuH1h>&a=5OVIvd3xc8dRXep&7<gkotVdiwB&q5UsOEc`6DWjTN9P``+!VCIS+*%t@78^59hMG23$Nnz2@7t%ThZ(l?sAh5H>AZl5po0IVm!WfK%ZAnH+gpw0i^>jh-&!lr|Ai3f#{uw^C+Fs9C(bDpbI`{6Dq<C&ha#kk69!X{DL=t1;CRC9^KGSBly78VTXUP5%Fe0WA}HkDjx0W=hqD)k!1dL2o|v<6B$`vfooLEGr($*qJdzhXuC>xxxu{$M>FxlihXY^CD*>kcT5UCqx{h35o7{X`>O#}!S9)P)c`C2kSpss<IYs{4a`R+sK*jBh>%60S_?pUf*v4|2lYqH&UQk~?(F~JFpa}uVI)AZZ`6^U{iFcgMk8iW%=XwGAN$#eLJ~hjuim_P9(j2<QtuJq=}pV`EGu%0NgWfm{F<6145&4rW=p6M$rD*G7|s@iXCN$N;+MF50tdtUKx&xXTjlc|6au5u-GWI_7d`+gU~9NS8HWSoGFAs#8??*lqluH`K0V*^FovP&E|3hkor9Q6zmr#-22gv(6!1!x>Z&T#R2zOr%VQ~mEn_j{n(*8&#(ZMQe-gkJO^?VQ?{|i4mV{KqN7Rt4GaD;gwRNIo1jZ546n!b9#Q{%znIV9;c{rqC<`^diD;C^3P2XGDVOT8bo+|@`Y<ZXg*pM8D|8lH10feHQUi!+K`C>3B_klq>1+v;0qk@7efu520%39#c5Mra#xw)Pt2i<DmZ^8={1ig#n_0^$-VYwoHD->wWzQITI4avAZ(sp2WmuU}1&C@0pbMJyMz}O*Vk8K}XGN$YpX2gR_lw_Pu22>o}0*$LQ;w_m0%oZ2HN*KD~Z5;v*vkL@!uR3kKJRw26vP!3fl3?aHAE!Vfouc4|dURzpRXtjZPq}pXYqEijrCvRN;K>cz1NqX8!f+LKBKI8+*z!P%9s$7#j4?0VshhXy1=24;mGCzf7(n-AF!8afMMeOHE>wI+_)LbGk0d>n{@a`i>r(uBbQ^_Pv*2WLBlZfvrnvEL5(^RYi_akY{oC~soh{))*q06oshWOIS>npa<jjZha(Yq(4&-I=07CDbbVJIg^wwih;*h!$l6*KGQ75-Lj09uH2WpmkT5-2tAK&sMK_JmR=<?pd>uCNwah@aZ8o-c>eb%O@3xYc|5*g|bU@?lBH4(&3!ycc;pDjiFyJwJ4sK;SZ)-7HOPaH4;Ju07T3@ymV6;Gs#^<-R}oWy|CfQP{Ibv#|78;+o&<aKp9YJRF=xWuFMqV79M@5Lcwzv8*$^)ThrXNMQlgW|s(YJR7Ikq9<{LmrD^m8|5~kBIUNd1x|7>iRYhYBovri5UD9X8Lq9xgc50P9`Ds!|C=$H)!@)LRrP{2Kc)a0CSnu)X0-#8@W2p6#+vm!m3P=U$^5xI>`SUD-a4jii;z(@Ffnd3FbOo;0~$*NOAcGE8Zd0nGP_=$R-&=-PMrfp7L~zQ;y~%PYX*FqZD7P`0)B(M51chn!CUr1Yy`uf{S)Q$|%(kx>?coaWc4<SpqM=(G`eO+uY?sIHU%*3vhn5;M42zet=*2f;@_pj!cu_J4qK?<BJ0v7JPfj=15OrY786)rXj#|fqHve-igk+6rH7Esq_$c<T`wt|DZbzFhilroRql9xmQguGQ<&zO$BIbE0{X?qF}s95TB-_081a^v+>J?Z*`4d$ont!Ar~tAw}Ps`{~RDu#15$Gj;@&+9I~LNZH%pKkHL#x+jdHc;D+4~(yo{KfiV~))BC2!hfE(+Ocq$@^f2<L3?$PYv;I>D5z|2E?zAOh>)BC_W|)Txg^`dD*7~C;Z{U^57XQoS*p$DXY;^$wG}sP23Y;LoDf)w1+sa~xUwL5dI7ycDd>|N{0>Y_-PSoH3Us0#zL-k<1$xY=X;C!$);7WG>6LIAA#lfxZ1Sd}C%juj6ehHG|&T!$fcg|UJL@E@;HT>gEcMqfogXz(|OnQew9@0>KPj(ZM^Fr83{r3VsGG_k;P)h>@6aWAK2mk;8ApnbqjRPSG005>O000>P004Ahb6;|GZggpFWiN1La${w1bZKvHE^v9xS#6ISw-Nrnzk=9&=qlB1&gV8CR0xQj)<FQfx!P%qLO2lAijtVyU2e$TvlPSsz0Z*Q{$kf?KU9FvCvs*uGn{#5h8Kq6&u7WBQ74s)l)ZfY<2_UPlgvwPu1w?y_OG8_|Mblpy;rGx_u>sJR3TH9OZHxsXJ%w&#XglRlchL_Qp7><w@TVl>714FQm$E&N|9H^njK^*6NvqW#2n;NRB3755$*9uk*0f*JTfaYk(Vm5fx&MoXYx{{Dp6(3er&Z%#a^bC={&uHA%$@0bHd5?Ppmx4E6e4RGz`@E`TDi^$BW<DAQ6RAt}r?)Ansn8MEa#fBxebYjD!mgvXJ<ZmrNf?8ar0j@oDIBje{@@gWw1ujPqkvRz`BpR90wH!lqo8f@Ip7hFp`R!dl3$iQ1(gs8{kTE8ug>@}g#og(=lhBxTEG&*JWn-|{1FBM5@;Iw9_7&ziT=!oQn<;gbpbxPjC-KL}&Q)h(*rmLgB2o$T!OWoFeT+RQ9B7r$W#m6U;7!{stZVI$njapkq3kCE_H&OWpY^a)A7a6@-^jOq7S*z#J)Xs>m;S`TSkwHt<Z;Zt1Bq=v(NX{ce&G1a1cemCav1~*Lp+b6j`dh5S)R{_7A^`UhKT_<+|7A2`&vnP>Ovga7@u6659w)G@q&w)7_sOcGZ-iHtC*}cwZ3k2Rh3!AkcM!Suo-RcSfIf=pwS)9pD>j%zNu1d~bV60go$}_C~n$Y1%C2|`k)j@1-L*Rzaj|vHh#oPF<=c+mr00GH)yqKh_XxCf{l}6<Ds7)qKn?x@`b#F8rVzpVjudlReaO*{G-%F$N(@+qy$%ynypw!4AX#A%(>0wBjIk4nbWNp@pjA!~FQ%(S7OLJYbcOtttl9EQ7A-}}U^HSKj1!CS{XKMj)x&^{N*PB7-jUxjBc^Adx(O6k7o`gyXtfQVYU1$kqC9ALMFUDvy3^A7tJ)AUEPN>U%xc@s7r2|KKfmLuBU{xl{dJO&^`}kf6<FL>J^+CVSkvRuAk<cHASlT!ooVRMfxy<vCvG!lOs6d7^JKbW-`4xMM(j=X7U3t-!VZYYb>`M>TsGGV(k#kd-!KqH)c&J8K6}R*5k2%lfdxZAv?rK3=ZC-255?Rs2D;?jD-`GJ5{EYBKT|Nx&&LtQU9304Cmg)qtMb*P{x{mG7riU?$I=a}!!9&?gdGl$FX0&o~$`6FBTOYFyT%UdWIfaKx(<d%<<Q+L$r=*HjOQv3Rs$pN7&=GPps<ouO@L{!@6a5vV_Nh=g%w&%9jdNAqN|a}1ksB!B?0er?c_3yYF&aSW;j$_Kq^K=Dsqd<yNUxMo6u{023=)&aLMyZ+;z=uHkYHzZK-{yVGPb-aJe*LfHCYx<&GeTU6H@Z#Ug=9r^uw#_Il+-rvcMB9SU36iiz>ITL~47Hw7Y#6yPWWFI|GHg@Cg+Gf}_sDZSUv~PMcd8C^^t&>3;p$eaIbIDzZt0&^@%!K|SlM`{#FS0IS0ZYDU>vq(?#@w3&o?UH1cstSg+##7Vje*P*!vy#Zit|EhEBx#~MyTq<{6J8;^=AhaREx;h=Xb7HSc8HI0ucp0ubW6n~Y&S=Kps{^|BEwRUFB@4OR+&y$h^&oEKpCKNcWXaoa{^()4p#V+IWk?1xZa|3+9s!O(h^7p4>Dp_MK8!oVnjbA&`}=x<hFjEvM1nLVxDNjt?Zd9$=LaQDxdt94;pk{Tt#CJu@Old2ehPsWE`7!v@+6WgZ-$};s;>Aot!lS2BE=AeW{A#p6&dum@om3<?Kxl=l#TJOW>Ra>`e!rS6~-RZqLB#>z1QuS2M7>W=i5-i3D+*7!eKDgt?ziYnU)o#!*3G68byP6Ur~xpW`ZHZ?VA1PJWgH5^9Rr^P7nj?+s_bA`{`laFJ8&;0-7On+e_x(vt6k7zn~2XAEsIDY67}2Ub9B^|KmF#mlJ-sJaxH=YJ{Vx+m7RpLD{{w(w7IS3InOmB7!3)y&lXi*UL3diYi*gZX%9`n)q>K%tU8?g2~y$sb-G(Qafa8COj0O)of_*LL_Ygn-1Kkboc<FyV`3zdO5f}pw%?)5#fTL-RTbt63L$&cR($)m-k^3Adm98VE|rHqBKr!0ezq_CyY`cTf4YeO(c*`*d~ppx#Qt6JK5jD^}?pEmbNl+p%G<iyMaWWXLgB(O6k&`q0n8DU7e_7pV=NdpaVC;?9FT#_R8rn*U<5}=Cqf=F~W!nXms<8U|>E(L64iTfWh2L9Kywy#8<z_Kol0gW}VnROmwSkyJffk6M)V&_iqIM&&cdEy15sR&J!YkH(!pwO&jT;(m`H?CaUN7=8c6etIP9WE0L;FjhEfysULMzew2p5(zV6o*_?6~c2UmyXSu23!agGJk)3PC8-ON78+@M48wlIt-_CcS8w_C?E>4j!uB)=B#vT29Rt+qVML}<-k<*9<fc}Hrl&^J7!{^c>f+67T(K~TInQ&0@7_|pZyTV;a^F?hrGKesX(x*LZhQrx>G{Q|zj@m2Y&W%NWjl@OS?d>Ygq{ySyYDtDG^iBYENMv<S#xmQnN>j%*;%H7qF7~;Xa&E%7ZbHKa&bo>G_V`sdt|Te&X8cxL4<4uFGo#4lb)-L0_olBw^L$EBvQv7q?^WvFdzJ49{e!8W#L#@=+E4OkuPb_hzQPkJ;0fXD-}wt^{InGW0U1@a9^%xElnPy#dv9vLE<7^UX8Qk9wT~N*1(PSbmUi+|=lwHz?+z_2$^^|FV87{fym@-ot6`xVk?HYyq;aq~%5eXu0U83jzY#=J#bWMuIA8UJ4PL(HMa}UIhL5?q<8w2Idjs-MKjf48-KXyfO6cMK^Sl1O>z!JHW?<*_ADH+I#}`dvVFAcNP|;lso3dPFax)Vpas3()5y5JL=CnHBf`u;rXF8d%|A;-e-+P;$Wmj1?x2^%Z$<s}7Ezdx{rONlwZ+x7l6`D!k{d0P%K@F;-w9ca@_(M-KAD=Bd@G9(N(+por`xyKiP)h>@6aWAK2mk;8Apo^MNEaFo007-J000^Q004Ahb6;|GZggpFWiN1La&u{ObY*U1WiD`e&01}5<HiyGu3xcXKSa7n^j+Ep1rG{vO`A4oni@^6Kw}sLMJ_GI6sdAa`+SD~duMj`m0Zg5wFT-AcOp4EJ3BAW%<OIyMc=o1R<rxIyRDiVR#r6^?7r&v?2dO;S@B}ccJ0s<ykPfT)x+1mWk=qLO7y(xndsXtyWw#ZMT<q*wFj1_<<JiuPg7PMj&0YotZCXl>#MdAi-mgki)b7DyA?F@nDzU*+Ua4J@Tf_|e(0-OKlNFMlA1?;IJ(atD|#fZan|)!ndQCEjF$@rfA7@mKXzT)t>vp9jvv3t@;#@gzjHCv{r9}M;p%(Vw}&cEQFeNfXHDBwpuu#Xi9Nj&`|Qi#ewCsG>&0rZSQNZuJ5XWG(<19L!TY6Xhc4%7*S7uI(WZQ{N+cUl=w-)`S;tx1a8{$8Hbo_FS=QzI>W(wfGSW2r)8%{CaWn<kC%bE_g7sDN!2Zx7UB|ORNCQdnPE5_VQt7UEt2MnE3n_UO{K&yIn*0GqKFU}4*JKBtxO#9IvJ3frWvI)xlW$nnuw|q)v^iu=Rl<^DRCcsxQFQ<}6B}~NTzV6IBTi59-#Xjz8f5Y9vb1s%amQ<50IE^zHM5eLM6?1a%P4R2cJnBCKE>d3q(Jy_!%<aZ<y{!QYLc8+M<k>XF@{wthO(?aEF)L;s?;R>127cJ6@*No8JL%eQCZcLE$~GaaovJ-mzoxBo%Z}gzg$@j1OKIY(Bos6Zy8n1ap-Zq+pS|wj~GR<14Q+a6S)#uKx8jh@L!6b%t%LCQU*9`Bl*bp1T<TAHT4Y7A*@W-3+*gL1*-*Le;|+1o+8joMBa7)WT^u=)_EdFC=%cckie4@A;^1y6XHubmoKc5V-<Hdb-P<eucDQ&=TeT1D?wqfnkZI&CZ7OH-t!J@4>LVR#_#2intmN`)c>4*9GjcyX~Gj$PJB^`F{n?V=Yh*<NOye?|46kG^)-~nKXJs$tb6!_UL|^^kOcC*Ri(nCRH{iulp(`N(&1%e$j(Pd9=BlTrOODGcOS_a)!SrZ7P1DaO8~0#VkvDpay&=CI4g?f<fFyJb?_SuMm=st-zodYI?U6)^`LSU<^cK|04eVH_S>u$foZHIiVEBmA;SevS1pz_Z;ubDw&+N86q;+rWlLNY)(~`9u#Rw!2;dslrx)fdBqAO_KwR7}2n9(P2OKg5F^Ln)YX`EOoT^c5Ev3{@ZGnSQsY-M@m!^c-;{>&)jH!MXFs5oG9w_EXFcILVE7Hid!DdZ_VVvzmTMs>iG6N5c;y6P8GZc`ndtn{0%PPVBLSq@dZ$wsdro=bdTt=)4%1}hv<xohq_l<=9$jLi0;ullz2xf3++?x6Rqx^=np&Z~^r+lMHEt&IbLK7cu0VbDN#x%Xy{?rXHnM`PDdn+GTi?hI>n*jp^f&)M7c!!42kq!<?FTJ#>0bUDEB(YzPcq!gp?Zq@XOcJZCegJ+I=Sl=%AOBQR@jWU;r9DhiOYaI%;gl*2Av)Un4!9w27pPpL7$Sep0Cc{@gVmy=kS=5LfOFl!4AyC6R7Ii5mg<Ep6}|vX5imgZA4wp6OM3>CMjfm~+Jyv`ex)*iosgU`y5EuCwL=dmHq{UHB{7B@vTS$1EHx_{vSz<wuh^T{U+VvS*;)Z&AL}gVHf3u3UX1{+sgHi+;0B&)8omKhSjI|z4>isQXxo1N?h-nm1|mNzSX;6^;AO`PfxkO`XnSZbvgT%h3TcCmA(N_1rKV?4eUt!en<fffAXVB-q2pRiyNdu;1Uiced`zp`QQ;M7F@>sRre!a}f<-r#Rx8RQ7Sdv9Xb&DlsO4{&KciRJpFoHSosYxD%jWa5f)!P1=rZ`xbDHQlG`D>WN{`lo4{4w6LLW1b1B_gxkO^{r?7OxB(i`Y*Q{LQ+Mm#0c7*c(8{Bw2u7UM^`>1>e}SPS$Gj`T^m8sjxooZz|R;@m@7o#{e<ZU><QAD?RgngGr^vKobxloy~F7zEH;<+o52td;Z^m;e0r_}f#E@Ac#@NHL=mW4QlMy<mw435c2rBd}J(MKsSqcO|2SWNt`EXM+ESN_l4Sa*>}Z;us5;I>5c;w^kWaY_-s<Nd@e~$r$KP7sg?%@k-xU26KLB8?JCKK<UHvEK-l!6)Z)jyFR9lb}JBXqr?5+w-Fc{DpdRri3gU>oUV*=Q$vav+>CHYCuQr)$`!Xvak<!9#eHiE7BrYGS4O48a#E5QMY`*C8tu=fs|}TJC6x&7Mj9(K;8i5-(UJ2>$rUb9CcY3*s$ag;awApsF`!m&_fbAq&>Z$v|DgPGmeOyD;6-4Q65skhI@%L7lc}1c;2-=wJ)8(aki_0d0*XRy280VMg9_K7?uFX2S@~X09HyIjqREb8FE?Lh8B;Ymlw}|R@YUaza<E?^s83gsUWr^+bRF`8*;4*HJDC-v8yaH^cBsT5>+`)^*DG7Q8|1MzGxYFn2Q`~xJ<V(F2(8F;TN&UO;$TbSlZv9PlEhlp>!ckL&-PBiX-TG~rPuV4QE6v1-}6Hj$vBLlGukN6$)VuB6`Q*^cIyjpa_o)B$C?t<f^0@_m4O$dUTwy(cUfJb*I5ITK+J5dD~opRz1)hmJFHS<Dwm}Iw1nt7P$xT@BAA3YkmPzBB~aBW!p}mZ!zl!56<1xmpW)o(O=>j+aUSfsE{(ml?*nGDFTA6!#lf10g06;!5p^lIhVTJ-x}xXp(DW~tT=0Jlya8<oLlGv#rWQ&Tnos+vmBu=f3A$&T$DJN)d#suA_He9u&trRNAst7aRKSm`&6{iV6rSIT;XoQu)7BXzO#9^ple2v#<W7IlazQ&Qr~uaQ`P7XGLY$pV6HN)5#MgRCM&XsrM4dB0t1*!a*PEGyOvhn|)v(ESbb1(W)h~&bL9%#Y4K(bE@mrP;he7R3(it9<mL<3?s|2g7hG%lOi*^`noBluAoAVHKRG`jfM+K=CD9;2}<yj7iPM8a~7Z^9h3VV|o;2kx>j$aUIY%)}sP<zIR+Bx(M_{Qte7X_jT0IkzzFYEz;UUx8~ho*i}6z_Q5HaB>B519wzr^7U_V@DzrGqcblQu%fynt>4ZHH|tggs*~2dv8FMN)lYMz1o`UA>8SjLHAbIjUaf5|K%+#ycwWgj%^~H%gBzL3%3dN)+eBo09aj4R<>bpR#&e*G&5XgQSF?p{H!+i8N_-9<y%>DnGO~Hy=8--djO;zbv}Hqf)HD_=PiEaR<|%NSxk;_b!eU}uVJW~TW)c%QctmYq$xjF&!1l>uYXrOZ63*+=r?h_O5S{R{p7%kbmcG-nSBqpo3`|8W&|45G;wysV<~H!5wxh|G2}lAXmc{=7A$tk)tFpX3dzJl$1ov_ghb|Vlz9d##y-1cPP7Ic6i9&II89G<X6?IXPS-KD{$i2C41!rV*4JlDg8RKZnq!d5`U;8%RvivQpY7n+5t9ZU%7fp)2zjfBF0MdY*B4jl!cj~+%kLOgAP`C_xg4P2hiru}Bnx)SA0(R}|NL&FcR_Fi2F(LHirHUjZprP?7QDXr%RgAV`-SHUhZ+v!pxo*zD4jp@B4&TX%q$N(;^z2KT^c}A6?;Xjm%32dwPYzYCRLxNOTp{XKZcx~QNCKUvL2vZz?mPmZ@zN-LKq&)(O@~PTK5>xr*!UQZBJa;3DCdfP(}{*3WD+E7kJ|9R|Eb)uPhjq1|#<2`am}mBic!!V2g!OeE_E1kW<LwfJGp8S8>eEk55mF>9M>w+f29TGd-imf8U7Vc$7C6+zWN&%rsLpu~}09zOU&?`yd|YKjAxf4_8t+zJ(BMpRBjyV#rY;ejM+E?s4GSywd69x{=6L3z1b+o8z8haHmaDO)oi&QijJHP4b}USAeF?bS*^*oSM)ekY-jA{9tv2=7KroFxgpl#i(JG6u36LW#TH`=WdUUCea{SJb!9DX^lsmLdU){jk3F(RJY5YbkOTmDC*1zlPxLbnSMHasIT`lXOHtg^;qAU*}u>+d&Gaz7%w@>ckB_qC~|5<_yWmUbI_$R2|t*dqTrwzEj@R#KY^puNBomRQ3o{U4p9GVR|HG|^^hzMF!_m-{_yNGcMQx;&AEF8o}1J3FTX%=>LP(+NkhYPpT0jZIw-T~C9lHCICgiyY(6L3;hPEKo7;tEb9;nDvi5aw2w$cz*tNT~NJGx2aNrrhQ2-yxS0>qk8f~(}OHW<)An;hO&+syfdnh)ekwLPY4qd%0yWwdV(Yd*0oaU0L0aXQXMWeB%_bw%=eLt6U#lGH7s(}D!`E@zbCzbkIfkcG#t7PmH?xdmIJGy!cMT<J62zU=&^Og4v&Fx#4zrMI(i?x}X-*hF^&-kfnNo`V_Uh_WdN-QANb_8~Fw`9FCVR}c_N_K7vy+0#`26ZQ^vDpmz>}6RU8SDAue&@=pD&w6Cvm<q_=J`B(an`TWt%2d+(H&a%DEjo>oS6zvn#P=-n`tb^U$GxSxwYA$0Tym8|5oA`#){s46I=)ha{H{g0br~8cyUI1<68|B)KSa#iCvUt@R?g`pI06R%@MtuS)Sb6bDJCX-rv$fg^7MB9^&Z44QUq{bY?*hrweP7`&@qO>0qmappTR3R4b6Urk^27-CZDh43N4xFfnrbgci%(?U<1**UoCf&0(C^JnI0+{|!(}0|XQR000O8001EXNj`7UJp%v$p9lZ|7ytkObYXK}a&>NWX>DaMaB^>EX>4h3XD)DgrBzFh+cprs>sJswNr0_Gb2H#Wklmt(V6#A*_OMt86g9S4i;t3&vud#ay+e_fDceB{)E8@>ocU(>ky82x>wDsb1j1BDK7D=oNF*>fSdgH+Bj1ES5O_1a;KCC(Dt@q9iNBc_T9H-{s$*7wt0|>Lv9nr|rr8Z16EqDG(rD{3wbFiya7B@H@vj$cEy1&nd3Ls#p81`&@+`ZdHCA(Q7+iF)%c_8~gOlK}ebNf5O+oONqP({JP(;lxN;MxkRtwj#6B9jadnl`^?UaQGM?xZ^Epgr^ZB;wZtaky4l7<wtN=JoXmie}>Dpd^XB+w8bZ6s4-2hNv2tk*DpWP5OAOTNEP8ieDlB|?#MKAlcsbakaT35Q@W)Rrk1PzCFj?nej(FSU$w=)p1%xmFF*jJe`j^u7L;Yla>V>PIj5@)RNEB2({=3p>2|w%2X0Gb7hrd#(FLDs4dV)7p&wFn5@b!;Tjr2lFL;ydtW%yH}qO0fekfUe%lf-Z+?h>smiY*C*|N4%gt(T5bRR?66?PO{Es%$k5@k?lnIEKQ?;sX7J@A^9$VzPgjIu;My?jAPB!aete*>D-z(`{$>YQ;U0Hj?KVsL@c8@F*Uz8lbYZ`(FlB5xwOfAqb?Rr8m-7j^0=nlYP3G!u2lrd-tU!>_yzJyZwHsT;OVkOrc2b09*XzFqu;Y4fhHcE<q#;IYSa#;O+?`fRAAlhah|Nvf3syI#A7Zz3trpS(&5IHV6t_o#%PjvB@fpyU*GU&C$6XYGHH7NmEpP_j|2>o$TSfPV<L)9C<#{6+ZgXnegKts*)@HzZ<ot3<UR?#cni;&K-ExB8Z_uTRNuZk)RR`&I1XP05c5K?wgS$9;d`|i%+U^8#W>_l_@exB_QjBDO26WGoP}*Lt!~4}_EZ{SyveIc{oz474)lqZayt2v(Ecq<l)5p>=(L``epVx%mOz+MnjU=5mjV5Q+>3%eEF9KN8N9KT6<pN@U29gObf|ST9oB1Z^HS9k8gb$aOLq8YMVyQT07Z)TNYZe}aCb(zO=%7^7B5xlgv&Xv`l&WWk82VoP15ir?1QY-O00;m803iU!vVbaK4FCYDEC2u&0001VVRK({b#8QNZDlWVb#8QNZDlTSdBqxQZ``=?`~C`^`k_*yD7LTdhb;u0>p0io^1{VQ3v3qzEpc~M>vf{EwiD#PcZQ@SQlf0P1zN+{U6I4#aNaYN=XoDWPV$PSJSA`6vM=W(D@Du7golLRWunc>Jt?Y`=OpF3tjyZ1Dg|L>3e$C-C0PqAcD&)Rgj}=ahL@>`JkMLLc1=|fN_Sn`HJnnC6?N6Lu(qsPCh@FRdMv5Rmfy8`cHNJtFBpirHp>;RnzaWQ=@ET|#{olIHOWEm8L_tICGOCJ3f|ITkZLkDaLSV^#bavN*Hv@Nn)DOU$XBaXk~1O5+ngQQU(~wwiohQn@jPX9%NxR)sso0!2X64J%i9bX$8u6Pyk?C&v1A<JDwyQebyf0cCDEPd2v`+$D}bQ{=iB5!4(td>f9n8E&dYs!h{!u8Za@MJCoIoj99f>Fe<o!`4q2M=QX;EaiYIzmS3(MmG&}<aH$aKvJjn#Eu5Ni6kzc+Lfrlm6Jg>^V0A9+oBmlvis6VZu6SM$iii$v~FL=HSm3oBaSxA6YE$ug@d9}}45nnYO4;0Vv57$IWqhhtxo^s&;NY}7I5vry*Xo9M)t-IFPQ?UlHs2R0~8l>{*17x;areefr;Z#31>yW%Sg^%@i02_!O&m83iJ#d!#YE__?S*vQ;dL$K}=ncQ8An1;Z0SYA9HKnDjzS=L87}V$HWDEb1IIx;;UYu<M#j>BIwM2vAQl!lf26lp;?6N%9>{Mz=vlLKg1>19Buy!=HSlM6%IxT3P-EiMrn6~#dj}^pqRpo)na)mW2)2twIOn$UE{X6oZ#vQ;0vZCl(cAax1>YqGmdHQ0<89Er?NGCYC&Dw*sp8_;O+E6q>_9jJwLAlAER|RjI`<GHQFC~d!77fVfUK@jL3%?fxL}4S8P9#V75P=s%--#BiT*;ev=RaT4s}H|izNc^Azq<VU%32C1-=jgcjc;fOVA*B+ka%rZf@BI><TYi=)p#L!&2nMd)1VM6gX4ceExBx(s_|zK(vB}!axj>tB~8+b3veRfDZ5(vQ@~eBPV23K6X5(nl?#UK10O%zAh1T7>K1i>v$fe(X-uFKnh5FezGk)mprVaNm8jqWxP65Qb!NuuAS$c!D{m^F3EFmb&Xp$udzj$_K@4c4vMYG*2jqvCoUWZrd>X4IyI)(%*?=d{@WA|zC2f9Bz*f{rHFs;^v*Ln-y>k{zrwLIQ{RKF9e>emjat_IhYl!leHv;vOwh!lx<TO<roog2eGh%fO;m>_qW!bR(GBbR+gO=H5EeYQSqZ)+70#FBgPENr&20^%p7Mw<W5<Zbz!PEdDs`}sHblOLrY^8?604gP(`umb5Jkj7@Pb^LPBv65(12;d0_ZKR>ldi<z4RX@Sz@ug+c_s$$3l1o`@saS!QBxfl0P;PkT_`^-5VtG$-FhT6!`5fpxpRjRe#{acYf$yHavy?&KU>df9jNK!nZgwkiiSTogHfqfzQrL)fdC4jh=D5?6vRr$q|7ZW(*YU#kNoQW(jzh%pHviIG&-9MNigmnBL(v*K@|@)TSK?>XaNagJd$rgP11>2vetqCKSdJ^64G7Tb1LbeX_mB;*<hNO$n*ATnNG-_mVB`Ei1mj~KKfW8)3itP4x69alMJS<u|(R`o)c^SbUSe}Gp)8~m4<5+>dD2AK8YMn6l#HT&uvmo!X6_tbnZQF9d)Jnt>gSF_fP(7P1OAVPgJeXo)OK!g@@Hq9sGZ+l&hb{@-xyuA;|$dZ&s*~Ip4G7o(fh#64CfN^Q28xwX*8rGEYj)LI=vF&k!6ibF7+dpJDE**F(&Qz>h(O5s_3ixO#{Tx`4(sGbANHVpZ7}!LphjQcXzAN&(9Y-Utw2$yH@3%j6;P{COCEm)&KhGQ5G3csd^`+EzLy>H`@yTh%o{T`Ws(iva<7NjwhHqb(C2J}6XCeLn^+QS$^d6sx3<aFpVaKp;7%eePl_qK9}9>MNFNii33E7o@|AG>6DRue+Sjw^ie6V<f@HkHDfek`0k2;&(uyg7?+e2<PUYDro`G<S0}aBP;JSo~KHhD2v>fT143k&hAG1+JVs(I&@lPxm8eyCP%p8#H3|skRgvlGPqHYZ^DJBj0sEUhtX%_&&A4Lfjn6MN2ou3U_%VlNw=wg+aMEarKqhD8{9mDQ<ZIS-6R4?v-jicW{2~<uE7ab<*NZ{2~Nh>`3eLCs6)Y?{UDfnbG1rVrVlMb0DFzq^71{7?#$~$uv8;heyGwxeNgHBB$sJodk`U+IXWG7yrfIn`ow!$zh~yOdLMP(wbj)i+0|3(Y1=ovx_o{9`Rx^b^NwD=f2C@_a0%km)%mYi55r&q`SJYf;;&P%z(t&H=Rhd$Q0ep|PSIR8_?$FLUAj;e{(M4q247dr1?vRM-@Y?RF#9D~sOy3YL=k0z!s0APukBAtW>uwVU$Ic9cvldonhwg(0|VfaV?n{l#phS&IE*VJg7Hkw$2z}$3DFDc8pyydK7O8*DZ$FF3sD(&pkVEfXOJ_rkcWen@+1>dHxJqV;7ykdy<|9ISw`33?+*oQZomvs!ybSJ*~l#-+2N$^?#Hvo7r_?G;Ho9ly8<P?NJbFssA8uC%2sJL8&9f$2T2+Rv|PqEK{4hl!<gWVih>>`8=&&OmEKOYjjx=OLa3&_Z4g0?T;)d)aWHq{u+TE78F`bu%cd3cs~s3~EW1k!w7Y|>^y5zL{kKT4<`i$@#9=2=a@0)CrCy!>$p%A7CUr*(UQnQB?@An%I|F^J<8Pj~*3aMPnWtbo@U$gF6rmZkns%gS%yM}Tqu)0)`3j^DYE@tqHU?Dx1zhXIi(0adSlTyg{LP~O!g2?anGAC?2|H21hp7R_+q4v{W1;2fBxdGWHVYou>AT$oFZdPZZ^0sf1_iE)W4!keK6Z5m3Jr5nwlpLjSpS&T=Lb_St7g%|s!KAP33?W%`Z{W|7349#4x73}Bo5u3A4~NK>AnHXzT*G1n1N7jGklWYE#!CxZHwvNa>}=u>$wJJ;S19YJ1<d1zIMdZEOB;YC#zS-P2QxI9x}`Ub2#+Xrk>2W^T6JZCJ4&s<~~F}bd-XPm7k)=Q*WBu{NGX)2f(Ze{zwjk=Yx!lnOCKs)MaJWd}{A=u6fXMC`F(&7tTxwoQ7F^4oXLrg2`tqe;N2HZ7eG%o47I<qtCkYK9@ITb*q*Q^!Llx<lBJ%yOCBH#DE?-csaWSH1A^~hija4Ef?^c@+8kKu;)(2FjIshpt6~>84kAv(&~HPN-2)INe_WfVA%8JSW`i4k_|9>ckdsi2VYX#pNs|@P}mDKk`O3MG@a6=YEznIDcA=xi#by}Y->S$;iL=<9m?Iee)`z#)vf1Hc+pmMPoCg)uYmX)WC==+4SrkgqpQDtkd5g9^2|b~&tjA4<4i~}9!}zza^TU>RkJh=srzAyE8>;VVuH$c*b?A2b6@9J1<(WB_7Qt>*8}oh50(@8f&nOA+Lr?T6`>%#Pn4Ig<nqHONoWlbI3gd~1Kt_-TH9ZJ5mt^bj`Yxkb!pZvmJoPBT(-V%Xp!^2Pjm&O#*W*X*@@)=6DDt!asU2gkKNBj-Yg0$T^&dZ<)MtIlHGwC>8K(x$x1q5<&pJzF%nzVeW&=FmsV6|H;|d?tQ^M@jxTc1q^atD;=9yniBak7x9^5&mZ~-4>X^uz67}F~+8h=I1V7V+1-9~e@^rd)J4$bBUa<~rLWEDd0O;=nO|DUSZDz2_?C9z)QklTF@+y<e$UtEvWJW`#&&tMbOW?{8<f}s9vak&^y`HFQgF5XIC)TN!`@@=>tfsCMJ%SQ2tL<b|x(R%H2vJ8^wW87alYNc7@oTdtc(16SwllPb3O`LUL*sR+#b{ihjY_S%m&VAo%q))?zm4UsnyaHII3#Pvy>7woXvuGTMxFh^=wyPI!OkftO?(=im@Uql*7KnoV^@QDp69)XRN~0V<q3M-p<jM^s+;$i;pDsk6I3aO^m_x?Eo<=pzG|>{-1qDs$(4TL_qVV>eA2G)qRGI#Gq!SGbh)$QC-wd8qF26{Vdy`IJmq(I=#%Y`{^%*ydyI)}`+3U47%z2L-EI7nOqmB2rCJTBa8*RRs=1*}x!0yBba5mc*Zh;)msu-!yniv&3g2(T)KcatHcrPYC5p6yl^qx&>Ya}tf$KF9P3QWdcOX;;YCuMre3QzB5bY90v7`m=Tk6eH(BICJrqLi*kG~vo0~)+XdhjmIy|?nN@!Y1Z#!98d<!HLNY}WX_04(iTel7#tNPrtRz{GplkM1Q8-8q^9M1es(RuYRhQl{JR3G4=ST2NWHdMWWlU5)9O3fO`smfk~Vf%;Y-SbPRv51+1i%i@#hkMsTKAyi){+HdYXd!{y?1poeTZ7SJtM@{BO7+uEf=4-)u4ZnibzW`860|XQR000O8001EX{`zot6#@VNJOls$5dZ)HbY*jNb1!3WZf104b963nd2Lh6Zlo{}yz><nC$nNuGMQwhm5?@yloPo{xmrEAnepNWvduj9>(|CiAkq4OaCKF=yWP(7{FyifQWA?GNz=f`Gf{@4B%QHDqdyzlZ#s)eg!Xu_;DynoJ!0FtD$nyQ6KXKlli}*o`P<GL+a9yd8kK&@jBA}GR7yBOH62w+(1#HY$mVSfvG7%JlQj`~F}f3nGHsd|el_+5MhCKa%d)JM;2e1u`uZVU$dYPZy%|1AEPiDqERJ0)>kO7thq5clATB7~9FA)+Mj|Km9H|t7rd+6+ZthF6*%rkowKJpj$4H$t3pEgX#3BB^+YuMV0KScasUuYjp3<Z(XS1Rtlffdf=trw(h4aw%K@)At+3g3ps~#6!mi<l_QFY=Hq?pB`r@qp;%;cp&k06;t<n-r5I<TD@91>i$n|`KpGA-<x6MwPZ-It&6QE-kr;SbvdkK<w$aseJw^l7(5^{CjalyF7s>h0Gcc#k^H8apCzSAWy?cx5M$BhKNKTET@4#)(OcYLA=?eF&7tr-igPj(O9E;{~r*)!-UmnGV1g{4U|^rd3L@(IFFxg<G<op^j7Z3ew@4=0_dITrnjF1{{Xaqc1dzZj}CZ1>c99CYPeZP&+MFiuom-roj)Ho+sbcTlfz-q%m0Y2iq_afgD~-BQ%1gyOM;PBzpm_o>IGsr&&xM**{QA0|XQR000O8001EXdi3mO{sjO4QWO9H7XSbNbY*jNb1!sdb97%}a&&2CVPkZ2E^v9hSX*z~HWYsMuMoJ0N`<2}LDvEW@=){w6xh0<==Nd+1SK8WtSM3zDJO2=|GsmmizV4k(zKhOEb3gpbEWdi6hSH6O35UUMQJJ*{DqIR_P>Lo%%y_vxz^G-ux>WXiPH(ot6Do*KrAXJbCEjR$q8ox{$97o@2bjFbAR-){PsqqD`=iRfvuJM9kL~~?^B_TmZ?yDCG2W?_Qe`KpGC9TELFnVUcqNH%Z8nSF+IK!7NS?4KZ885;QlT{3o369=|_(-xoIj<ZG+3+Wt7k!zLgcE&Q#kl8nac#IOHI}!m*rKivrNbPPGG3y*1bRZtZcZK<K&*N1WF2z4tN$gxhU-1z*r8%SDmMWo>Hwr!$UZxkLznZ-lDB9@U4aLFL|kt%VBvH;2H7w=7|=g2nkfxVV}JOyGlemEEvt7tQZxC2Ax@J`xg=kP*`!%4_LXL&H7^Y0>h3$N`@<1N#qC0ac|2@?548kwI0tGQx#S3r$wur{(jue2|iJ^rnZJ?WO@zK7BDLuA<P`x&R-K)D)!x2N{<|Q9F@Hg~6Lfr3)(N(k&}Y28zI|U?Z(G`b-dt8wuBzjyEqosKTWydPHBOP;A!=TSFs97GeoV@%#eG3verw5&iq;#BKrT>E)iS@c=o8^%(YPKJSf;CM+=)hpDm<_0CZ(5SXNBUsHen`3Kw01M7sV?PW_59m~o0LRkn~xjl9yI=V+~DoTe~k>b}uW|lD<#2y2J<qO$&c{ecYb5%pw2{~!kL2dOMN>Gw`@&yx)ka6saD&jP=0=*Z|_-ex_dIVFW{gRv@{_{gz9)M*K{1SY~D5MU73d%ZBSRF(b|CkRa>T)zTCbjtxv(dIu3Oqv1sZ_ar`n1cIVO<)Poo!wmwg#ismX|agv6(m_v4`W(M9VjTK5#P3)Yw6cm3m1Hqo4_vaXc(*chJG_t<ymV5kmyVWy6dHHbHF-n3hw_TDC8yuC!3GDKVx@BrFPq4OZCW6pUeO&f@C|r%;$Yxy4@dwFaGHFFK#IQf#q5WA7PgW>PzB3U%q8jB*p3X!Zz4j}$IR8q}?KKDwull*E$l)x}F?l5lS+WM>r1v!`()anKwVmRH$<?XJxfYz;T=gw=Z+9_aeCwkFGi@fHtKt0Ct(I$*if7Pk}3x18IxEKAJ7|1sxGC6`ju06XC7QqE@*cZk<`TbYnm<b<apJW3@}2>0S?2AWLf66ap@@FnC?{b2&eI2v8^YJ*^DQIi?AXpS3ky2iX|Hn|cBq1piV<I>)_GFXHl1;2SUS{rZ2YXp1ntPg$-m@=tH?Y^o1kUlcu=!&7=6q*+pm^+q)6C*rs&pR~788Y%n!}Zyw7`QYa8)m~}d2xEE_gsf^zH5oNqZu{p&@mpuF#2iPdpC*)NM2V)FWD&3jaxxQ@jD6W7}mBY`FNgW>4_H4t`_~|xbi7KNVgOx&XTg{KZ9fBeQVVyk$Ov~)vn8LczMIh<{EI`^BV2orA*hVxgV?nO3QIO7uW*Cw8-%?F}AY3S?jtew={os*}sOH6w5C7;3>R^uMTU08a!Q$@p|M%NECry(FhPqjas9(8lmnk>thfKVb^v*J-T#oyAl__z35v}-+}NTmp7;e50_vHx-GR=ntkv@Rf(-7X(Rg?-t4^yUB;dyTr8g9(8V>HZm7?$@K5IOHg}mAqb9dg<~_t6#5MxHL1k<hBL4*q%oCiuF&HtfW6iGFm`DYnL;{M0E-)kgPS0{gG7;$-4}JaaqMsyYyn^r7ZmJ5%=0TT52wkIS3C*3Or$o;wD{;cNg~7cE9GZp0%UQHQUv>8N2X{2ZPW7Vmgrd^8a+K?X{fJ#J+?$CzwTEPs7PwL482O<tH$nV%m_=v-=D=YB?2bib_aomc4s5U@hfyluGXK2>&(3#!BAbLh@kECSL?2-&u<dG;Y%!`~!ZJbCH!R{uBi5(RY$o%7bNZacd2o3dFwP4hHRr71>p!^BDK_ta0Z>Z=1QY-O00;m803iV5z+|Ay2mk<<9{>Op0001VWpi|MFLY&dbYEd}b7gdME^v9(S>1EnxDkKXU%|ojflMVz%H*7>=QPu6;`^AnUNec?hpH+DB0&i;N$>#3vg7gp-d%vC_$fOX-$Rv$Smf?vKYzOmiL8*yczdSzsn|{uB{OetOc83XZn^S=d;d44`OnwaA3pNm4_up{SstfcO}*>A;w%>V4qtw<ER&MuvzED&S1^OZD_I}lYei&uRwN?jj?h%nDpDuDB0l887&zS#SKq=+MmH4$92M2VzX&J06wWkNhunyahfEofu*m3&?}zfil~NMSVn${nq9+qt%v$pkr$6!F^Y?$ezxtJa`f&B`)6_n&L=-L8?m;Y%c`8{<!9^kwJa&%@#S5l5bwb@U%@R)eol&3=1tX@DU^1CRDbRR7Al)^Xt^L{o`1KtGD7dujf3=huH+vboqj};1*FTGQ_`;7;T~gu)zJ-y@4bP3Y^0p-AhO12ELK_jyqCJmp3Mq0kOT?WyDE^<Ec(1(WV$qVITwZ7pvEH2C)N&iA)NdO+O%aoZV6?c*HGen-8<te2#*+SYb?s}OCY`s9Q+Z4k%Xb_ON5XIqdB&9vcii~oBfcTQ7_SalQSdzW?HM5kdD@j5^tok{KosCWzxN71;A6idB-BMJK%W%}*2_)xZ3f>Kn_yT-(p6-TjHjA=i{7igOp$oCjoxbIk-6CAQt_FUO)dwe*M{~_u);L>jw-GTnQQL9UR>ZYC6m?c^<puoilZ4m^d#AZa@fiEA@5uw&Rm$Mql5|B-a~<T5ANK@*+a^6zf9x6n|aH}dF#>Rh_GPtq_(wEzw&%%_GId5qx{Z!PLE(j+)0}DM@Wu{hv88P!z;P#(y5S8y-G^uj%HkIw&OO{r`{{3c6y+y3aHdc3lYlxu^MU-zE75~--ZkLw**TU!L+LKpe4iggIb1f;B;<QC-Lq-nm|V0Vmpc%&t$&3RtG*LDbygr`0ZC)z!f+mA=vgV=kr>#%ZKhkiX!AclF2yE8qU;vfci10t;FpQ20KDWm-b;%Dxp3G4fza^?3WR%;k^raHssi0rDEzHY^=I1bd2)hd!cw_q`Ge$W7(9HSi#t<{kEJiXBqEM>pcVFa)`ihufMx+Y*qFGz~04T2f)-+yYkj~&BG|=EI$-&P7Uo<<T>D6-C83C{VJwM0U<dU%5rJ;;K!Vsqf|GvM8@u64%-@L&pndKn5W@Z<fJC+HSjjVSazFBe+HDa{2er~eWJ}{9kr@l*m)_w@D&0;%a3gu#f=hNua>9BjdYO*+k0ynRr?D5e{HgSI4dc0_4|RPJ}MAmNl=T`K0rxJ_BdzXEmP;RGdwLtaXJjqd)jG%X-Ba*{8Rdbm19GsCV^Acz;NdW!J0T{k<GO4_LlFui4V>5kipOhh+krbZOZ_%uIbY)WIl!2(&1wy6_$DFqE7k)czVz}?2rdTi09Xl?soeQ?xVe>u7@qopnfH0RsAv8DSFjbQLmSmn~~yca)&rD=P>u*#6k^py76EDqk$OM{?~9ul`QMa<zhK7ST(kR;i=;MiDl@AM}vp)G3B6Aa@nfE%2DGZOO;WV-f#i7XtVe`ApA|{{G322S)h8NXsgfT^ch3t6R0zUKIbj(qtLU^=xb2v3_3kA<$V)cjY?=C)_S(wz+wP}YQ?P}ePKE1V8u~MfCanA0cb!ov~$d1n+m<>nkFJ=DaEPaIepOzYnSE;2ArNBnb2TbYoqRNX-_A>)2(Mb@MX}+0you*Hv<IE4ERW}2OGqX3^o@RV16fXKX1(GBR={UOAq{c4yvY8e#f!`Cb&pu^CD$X63|Nks`21%<iTr}xtfbG$W&<P9>5vF1D3$9U!D;$BV_(!v(aGuT)cWkdGd2GX+vrQsJEmthXjA%xi36mK#>{0_FlM4<+j-#M1IRs5zk(cAOOM3-hd112HQf^EDOz8E)skmv7Bx>1QIQcd((5lBM4f%wUF}u2C%%}a|on_q#}lw1olA@36{=LgMpd@+FPL^SCE)tiM?42P+_uZAX$HH+e1cdqbMGnoQR!kpwqAEl;e=`O`Y;9n_%y0XTUoVJhmaiUSEc>h>YL7QM3_-=@ix=<(AtvTvT}hL2utD`mMrEgwhr(l-$8+s8`tfgU;v)ro~UlaYW;SdX-S=w<aP6w+V9(BX4<D^I6ci8?>1XiZm`#N3^J<pHZL`x2}tzal92xHjQ_Kf_{_c7l^)5CoQ02?G6whvO+hEbUp()-Esy_7{dX0zlg%3v9=?;Q4H`)LmO87GY|~mFjLW9-106I<4ti0!aCi!y!a0PEs7tpv6tL1Cmcxt9`&?5sED_d$USqIi)Ip*!~Sf$f}S54Bz(EVfS8BJ&h6+~X83ECa~_Wgu%1BEI``h~sh~EZ<pJ39ivuQg#<MLrakU=Rd-I%q5=-3b<KTF-(fb2h%+DDejEw~^6Ra?ta~-jQqj$Hg*(67XkH+*dZU6oebl{~Im$RXGw$n;|n~v#}%YBxL{Ki&`U3CG4VYAJ)6p_i*O27M!lnP3EsZtSnB0+wG_ZN8LlTzVMZ6Gz+2%{EG+RPZd6<x^X>&4r*Z@uro_m&rDV~hN_i*pmr@t*bgjNqP|zP~k`FPN<<UgSciZrzN&Y~J_(;TB=XxTDp^G%z&7q=gMv<NYFq&PI=1?Dht)nKNc=Dpu>%;B5*{FK`G7!V8r~ClA2^z|&Q~+hr{4`ZQ=kw|XEM4EtEz8PkwT_*cs8i1ye{H`D+_gh!Wx+$^@aJhxtc<U4+6p|5c*<&poc7qh>xS+aS!c=Pxta&|-g3C3hltrCJW(*|$%&+*_iQ!?Nbw{chW8k4~7JQ|iB()r1wY8Y$BSWk?bGsbVmT<)N$Kft;v9a9TNJuQ_cDAXMdDd3Uf-blzZdfJC-g3h`)nTW)r_Hu|)`!gD)7*0ydU(im!WixxP|36Sm0|XQR000O8001EXWo%a-`v(93q80!E6951JbY*jNb1!sdb97%}b!2IDE^v9ZSX+<Vwi15#uVC31Igk~5v)wkq(LvL8cblRo4bt?X$1nt1qHSg+Q6;I}b%Xr(GsBB6wl_%^I0p72YWU4?F5e7IDSb01x2zNeGh7q1<Amj=WhIe~5taCx7o=fkN7$B$S{tI+-K-J!yd-O;c`0hH6H4i1vQe@kfWq>UX$@GSYNUb<yeV1EC;suhmbH^b*ix*6+#7gwFU-E_xBm`*Oa7WwTsH_4aGE{XYr>O}D&Kh<_<h5bsCaF3VwlcuzfD+M2on(PL&oYNYeXqcOmbN@Oz}+DtkFAZv^N9xs<s9%@FbV@Mr;GZHLtm1M#%a{yB?F@O<U~U!|`VSH|+kkWVh_CTboQK)?MVCsQ2&W4X?rD>cxb>XDxMRa48c7Ny)-4^6}!CU2Iq}T%3<3VTJzdss1;_2-gd~Az7BQvdppwBDjf(;rC|hl;THm)2jN=^O2~V)_?|`E^%8jG-WmI!Pm^>yR6{3EVx$yn_}|h$;};8TYao)NeNBfOHoG^Ya(q_Oq9G!j4XvV(R4Z`8>w)Kh#Ik0;I{8w0wf!84;Xxcnr%GcXqrbAC@<k;Rg+<!d7U8==_m+MXOG`C3Pv=I$!l5j*k)*yWgAiQESn~ZYgyj%XqsU5K(;LxD~mK8>Pi|&hsJ=)l2z-1k-u85_OVs<g7m80a`$KxIx)P~Qbo%}OumcB>})=d$$3mJR?}Y6#Wo&j-n#Szm!%%(I;?=P6|14VGtGGsEzZ2H@(u#jK&iq>Fol+@7=G+KbiJqxevi&qtlskIY!E`r1-jD&0!0)(vu2)y=Q>-`?!Rdteo&jRrs1&X0~Hn9a>yn6!s&|%wN8O&`V-f=5^D|wC6F3GGNB)Pq+ohL>t5jJ!|2KS%vlY{$M5X>-`>8uhPO^Een94jUuHjdLgWdVCl?Qh{(5=+^2a~36FZ(gu;c1__QzQA`LX23<0V>g!;ppXY}6MAdaw|i4aYP^27j8f`j%-T*YA05$gSXanjWBi?4F#PJq4yiuIv2_l&hgw0Y6hxniPsS>WZ%i-geBH9+GHeWRu;b^MSOhXB%yuo*k}R?$PPMgN}JLRIN^<vv`2S44MYIvNg+L8<1E;Ip8Wf6yx_iZ>@E-LAhd7SDvkWVaMJh@j>+rxT#|52FU>^)F`kjqM9M6QYfx>ss)HER%#C9`S1>JFK!726RL}bQP_s0hx@XwS!WH9X1>q5mY&t%<6Ca~-HxeB)=muKqFwlU*6jH*CeG5Bc>R+xgFbY4NRJ+FoHl+A=u^fcd@@G<zxZ?t9YOZ6NiP<YLT2tbX*iYPRfDggIMPKjCt~9dOrYF2RNK7I{&5P}7M>ms5z%M-(tDOXCDd-F4iRT1Hzfh^l4=7+X}TgQbnOOT7IY{FZ9B?$Jil2|A6VCdD@=XZhHP!$TekL~VN%QN8;zNBLbb<J-10=uv)+YtwWOUBK?`s&=I<j=299@!TzFR>z;u<^BjI)^i=8aIf0p!ej@59vz~5KrbTvdOc^w5b8zVTjRp`>vdcvjUP-Hppv~=F;+hBnSx)B^ZB8d*w54xha!67XBeE_~y(0TVEU|+IBw+-6pBc1QC2YVOvY1(HnB<0Z3uTpE=i&2q$N}EMFvJdOxB*u6Hsii^Z93zjS>}Mli-U*Dr2AE3=8}U+Zh0$G-O?sO1$+h7&fwEjG)i!nyG&Inh?Lc9+<~tZ5vdy`#%}=l#(BTc=H)#5!8BQv(6*Y`h-YlC{{(xc03YL7!fSS9BX(^x@mSGkF*N?^oSIa1Z{tW6@{haMj4`d#+M6Oj1@^E0%k(og{$x9e-qkfp`CcfBBTBx7bZH0Bqk9GL$WA#3AGa&y9223Xx+uK+UM!Fe3JR#)B9S3;$`I=mvlbYW`4!7FgO(7%PrkSkEJ+T*(guH@Wk#}`C>kQehIDys?3AvMPS-^9}@IFx+uD~w<2od)~47n3P)p#40Ai0HeCr60Ek4x*TRW||y)4B`O$;ocEnBx@heg}v|?^wf^vjvRZ1zvyC?u~ssLYf|4p}ItyxO+^O)`^CJ(7-g-<uwhmG(M3-4!IBt;w04`Pj%F}tcdlHA6pa?5TVH5r!JL}aD}skkkZ$(AFGI)PKa-8L&E&0{j{_2pfTh33rQ(ip`)ljMKC?V*A(w=cx8rbFWfHwU+onrsjxHH$u2Y0+DRVtf5qS1YM~DZAM`2P4^MZPz}3*JEIU7lF_cCsAN#Ni<je&!xRc*Q*=V4U0auq22k0=50*KX+VdZ8}SmC|2{2iU5=f@KA7bNdSDE+80?jY@e&tS7P;!*SM!AO7lD#BVL=>xZ%MNG~{6RPH*xA@f2g3X^joKFLX2&u_Sn~grCE=R2+PrN&v!~)285*wsT`0_0a(++Ze)-^3(jg}1~pBN#C%MancKK|BZxqvx7CeM>+^Ye@EzWrv^v8$8Y08N<1#q{H9?77e4ojv~oxUb0%62CW9aHqJ>%NDbw1-~nD_2$_b@yCP<R%?iDXV`7L4bbSq9*^aAqKJ~M%{b?g+0~oH1*`*e@32*Y*9x-U`#wo77m2@x9el+ZS?9%GAtr$|GM8mae}4J)Kfk?x_20{%{0-#<(Z95Th2$JlJrqEz#afiY?85<vB>xYraH9}d@IO#X0|XQR000O8001EX$<VY#)CK?ml@b5|7XSbNbY*jNb1!sdb97%~Z*O#SbaG*EE^v9pSW9!;HW0q+SD-jisZ80<<KP(`9NU?A>N=W4@k~bbU?38bP?G=)fO3>f{(E-;k|JfwZF;E=76tACyWh7jSmuQ`j%Yhr8l97ZyLBcPM26)l{2Wxzc3a7~Z6Rze@d>^&am#aI3!aEUEnRG#u&z?zmYGsHnTvR#wX@FfqTZk|dMPu(WGX<yZAWCoGnsNHm{1#Ow94VdU@%C<lHi{0<wlF8(mJVDrL=2dSyGfNDO1k4N}1KANd&77id<eWbaQw%An-9@5Se&SBgVWbY#1A1b+!@FFfO=(BkcU--iyP|AiZyNL}8td{AMHaj%QX3z4+>Y_^Ks7Gi4>strY+Xp7F~Bb@x{16kuIzMeoUELg>r*<TyTgL)%g-lycf6>q_V}eiff|L^7cwFJ{DFr8EM8wDf+6;-_BlQMz@-U~IUw!bW#IE5(&D+ElpV=c!-rDGV*<c;ZT)d3G9A9|bn5c9lzvvv?znm0B^WmU@16dXI2KFGu9$xP5vs&_2i_qO-LisYo(lD<u~?;}FBj2~!vWe~GkY>Y?+P(Ezs*J_0Ab5f`By;;TqX#}}Cx5l?Z4r=Sm{@Z^E71Yiiz8?Nk9n;bl-7<d)Gd>x<CKFx}#*cO>|5&e}8=f@q|THq}?;FYpuEwhvfWP(Y{HquHk&j)&es97h{CdNl3*XluR;W>dH2`|8_8tN*o%LQ<e2x|v^S(m^ef)X`A(>b8H%;akA?rSw~IUcUnAByfoX9lXk0stXVt}S|+0GM!cmXVBGH$3ZzU;EvBT%D1n{OU?0h+TUYp$-{tw#Z#Bt%dBaRfoY;ga%QG5}oG&z=*tf@$ksa%C@4U&YmCwY3$)WD(SvN(4152QotZ^w*x1hUQe%{Q2|1psC=4Te)v3VlpqpIqxLIi6qXa6ipj}{gwrR1h04zshp`ooJ7*x%`ig4YIvrj@Dk451XfwS6R_n+hV?t9SH)5PhzEVJyOzgN7|Ko=97&Pgjy3ly{G1%g%gs?+nC-ff~t@s2$i4W-=Z@oE5$3P;aS~^}xHLi*wq-H3mV5j$O0uP-;1#BW*gOcE%{Za7-OIk+`WT5i%lsWO$MZO>gBHBV#g1}ZGoxBDt!luwJ;Gvzgq107CvNc3t2JK0zwJ_3QHe;n?m@h&dNZbFTGD?+6C2G4MY-q02G81QGAix!wc0eTyQgJ0!{iJ;%3miuD_vy{s^NZiEZZCnW?g_(#O4Bd1_aAPj=d<_7GX`i)`(d4%OLlYp_T%}-FRW24tr>&OZ(bciI6=riqki%E@|@jWf4Y8qbHy&N?yfJcKH(n6^iU%8q{@HjEqwvpRO7wS>_m5GGYrh9L3^--)1Q<NCI2V1cJUGyTYUOAx&<z#2PFI2t}63lTlL$M`1DQus<+>^g13r1Xug?#6eeUl<*QXkQ3JGrw|P5${XyLjZuitbtyZCQSKV4|fI?h_!+2_=;8bY)xCzWZ(3ysF+6oeS-<kgEjppr?2oATi;(*S6N>qwZvf_YxZ%TwcjyppNaEJqQ%2Zb1cw$5eL^8cxdaQ6tNg>S=hp(!^9NN&*g+|g_00-TH1+FUX@nu~)nYC9`<{td-CwC8SoLs^T+FV|^)45C-5;MXcb^7{^zX9~C_gr1Pv*>HTMm><JMASOqHRMsy@k#-N`UbL#w|FI>-`(EKeq)#CvvXg>lvd)a-_X(F;R4oH!gd#qs1%@&D7f)di&(v!hWM4=z}9h5ss|T;lLZ~dz@EP^9qKDn?>%==4vzGrg64wf)<c{nfD9=W-@pKZb*U<z4J58_7zyzgwS+qMfmQiKDos>f$|k$3)0*SO5BB{2BXFS&Q7~g+35lrp3nBCm3<@P}xn)EKZfuZ*WgCs}xWNw(M<W~JElzZ?_2@J()|-q-$NGAC*z6E2o!2iQi`t<VJ)@1Yv@8&~@4E@wCTo#%dPZ;}Jc6(~+MLkQ?meahyv47Wv3h)9Er~i2Er`wgpt^4f2U9^ZxDnn2$XnU8$6#|pj4$zm7(A)_DclbBqG6Md!Iip|HYwBEcNFy6R{?DnGKH&X^a987-rInmTFJRDaDaoKR(CV3_wWuzv!#S^ZmcGG2{#$cn(D?(lyHyQJP@AXs7<tjH#l!MJ3TYGBNG5`3OR$#0B7{t@ZqycF^mx$i2;m2t67S1VSaYs846mpDGESgwXSdvSr%~bDF8@%RRkV!Ol1qq`)ENE7+_)eu2Q+@Hqd(scxUuCp`bcU8%SyAp9uSw4*mvEO9KQH000080000X08PUhVPFUV0Jj|g01^NI0CZ(@baO9sWpi|2WMOn+E^v9hS>2D@HWGjLUm@_}AQg&s*Ucrlphl7GwnczL8x*}KpFz+PWiu;_DoJJI8vfsJhSV2j*}Jz*>xW&5<P3-N@tYY+>?)~D@JY*ZmMggnDrUBYSO>yQe1z9o^KAI8%0<E3_qr6uaBXIe(lAX`T^h0D2~$SoEH%32^R5~AK`E)?fEl?HDaCEQm*SLF#tdt@gtGvC>sk~UO=T+Sj+G(@5p-;43du6czZkAcRyc-kqLLM-V#hY{H)|GGidRf=#|U?E!-@ls0qJM1M85BKa$JB#G@H%Rf@vMx0qt)nMA&L1!Pi@+dGyxu$ao%TZvIn+niqL9h(CNyP_I%kwGS4Zj)6lncqbH3ja2(E^2wBx1}M-%q0c8N*^ZA^v$B+i86iszM2BNHn@bhwst_g)KsA9VgD@mUF;UL1Z(fr)P^{eW@Fos!ZlWj-Lh_jv{?s*2&S7?NGW<DJ!FEe;7=MS85&RL6k+&cQ2h-<*t9W`9Dv074VR23l{rb1pH}mY6e5y9&4C#Z&%ofProSENpvDq3}@alBI@ni&b*YA$H0~LC!R<rT?dr@Xq%Oqu$wFfAYh*BF?rd*Se4z2rO2!PRuQz)v)IblEsW`*J9>QqLNyDb-spx^QB<mqvrrN?qLX}r#ejlEF*CbpW_W+Qkb?}C(>gdCl+rgSu|EBx_7^Gl)vXTFh2@J7-`AaF<0xCbsF$JN(4?%A1v@5YZT8tUot(&lh`+MMa~>h)TFK@Y%CjDv6EV7_{U0<e!wJGw8iPL+H>5w6d#Jf59-hSf0zfgyU3y9$VwwZiCtXNY)o=p#K?*n)5euqG#523aLUj<L0Obh2)Yj>|EB>lMhs2UIQou-JAW??5+maJl#J9~4h+;r~nce~0hKS(gC}XX;Kql$Vo0k`|nmbp>$WQQC}W$>QLfZ|)zM+UNlT#Z9fs$-EfOE_<*;dJyzP4INm}J{e576lR0V<ZZeb5g+|m-`b|m_9F;S<rYc+YSq5zE#NO=btVj`0BVQD+@EUo){jO*(?Dh~WO~n0fw+;Tnb4o4C=GoOW=kDsU9TaJQ&wV`kruVa6OQevxULJ+da$#dCrVGcSi0nS4bj>U(@M$Dyu^I4I5e)0SjNAAL#D#KKyk@&aF7n{GwYHFerOXDiZYkW4s`(GA{&vo0P;OdVrV*n!DQ~_WR(bTM;%7TQ4OIrSLR0ssO%pII4{BdSHVBr<1IYe)Yv>X!*_$>_ulZtQ3gByrfMGArPc~S`Q4%;ouv5T_nH-9!AnP`_zqlHlp9zM+I|zf0F_S?{w&Rh(p5B|z61(7+3)~?<L*Rpw|(<8+@WU-zzOE)E6T9dbQhmwwFLZFMa%2e`Evl9CHcMPYLEMoI}VU6);us<p<OAEkn@VO)OHmZ7OD)eFnV;@R1;TScwdodk_#w=)a88%(1kT$=PYF|Hnp<xj>oWkKm!zfR~4+fXd2IbT9&SqGW#)9oQtE=Sej!j-TawYvQ9WH$`~&+2efb;sZ@Q3kU_%$eFC&F{1cS!JbMa2AhX5mUSnvYq1t|BLUSE@eblH_9CLfqbF)ptr1ODWN6CYNfTspJr;r?u|1HT^mE-kTaWc;fQF7`!FzY~QrmThxPxX!!1s$Xki?-PQu{D(aif@EA&<&mF)Qw}V9yfO3`ApNS&AxK7>BXnmtTa3Zhw_(a5L&i(bs@R15bG}i{cOO1dhxR?c|47m*4eCEnXuKz0M#wh_{pH7w!sUe2vZ4)GO8RJ2WokGMf0-nUWT<olvv=iUKL#ic!voGD;dTfAiS*`@f{PC<+^O!pVcw$PJ4-64}7#^iHO`Y^o}vfwe`CTstG3@>IxkX4Dq6kHdm(pA4-|Xv#i{Q))H8bMhQR+TWeX=hKDY%+tL}%*NQzlVgj{T4bdA#XF;vIJ;8hfjTOD;yrMe*4pBk)!Kijo39hZ7if}GFj>f4H9`9hR5{kZ9A71_VN3|upH}d35gjU9jjZ>2mArsvGa{EhDq`sur?eAv(?WKDJG$0ax(2#|>;sL`p_xp3p3%2H1CX+q*@5IeQ{JgWx|E{xTa`X7vO8NNFR#VW`<->{lyFu>;(X_B$1XYM_%@zbQ2_$h=yikxN<c>U247M2%P(a&KaDIWnzcUP%C<xLlL4B(W5!-G8h>hu=Zh4W(+6>Y5Pr7}7%h4H{IfM4bbLHZcV@*)6pF}-ryJyq~Pp~rEt+^s&Y|54VgM;gL*=ATJr1piTTfCE`YrcgZ-XT=eiFO5A=-JL4Y9OMlx?9qje<3p{3fh);qAVbTy5KZM$+$F_Xr4zDqv`+%Ch7yy76>rymk51rds-NssfD0fZnQA)*-pL|*u8dtvILloZZHQav?m^B#AtTmat?Z=rFSca7c=vzpc-%=U;8}6U!SEm-*X*|*iQ)}ZH&)EVYBE?Xl>gjMWL}{6+UA)oZKrsL-gs77vb{I$9}WDMS$Zf?sGP1j;Au?t{RR00x?doX((5cTmfooXv7{9&F|eF*L=F=={<l)Y5?YGCU(5kU8AvOddp8#4@IwkBKw)@<A;B@U#33bHnTxf$}1eiIjfL~4oTmO3cnMD<fAAGtV?wOUK`j*DhhC@Ugws5Qu2q8{QA>-@U-wv9K3>LM9}e~M6=J__eysyfSQ}gvrn<d&=x(jc=!or__@Th&r_=X{vH8w5~y#Un7kcowvCLH3SRV<&HwqzCiPb@|8{4cRM5YkprFQOXCe=%{iH>y{Zd0I-X~Goe5vU^EaE6M4*vsCO9KQH000080000X0GzOH{`LX@06GT%02u%P0CZ(@baO9sWpi|2WNBe%Zf|pRX=8IPaCwbYO^@0z5WVMDSUEXT6qRMQP^3Lo>ZymSSA;AR4_S8;JGC7a7V+OZwi7Vj4>dwbJehfK{AR|;uGhv9>A*OxTbpGvTGY_>N5U;peYP{YBmJ?~wGK{X^y#>L_AdSd^NTFH!l4HvJ5bJ6j$5{0RIummz@5};3jX%QrTGB|S!?xSn013VG9h`Rv~{w+PzSmaGjR122*#DtF2FOXtmCQ%8$msg8I(iZ3gK($(<FXgEWf}Gt;@1Zz>YYuF2xKqQZ-CkrWG)!!-Ffq;83@N0LyBlUxW>|K|L$|qFBqf&{|roGJ;ocK*&0<KfWQR!Np&yuU^4uOX;#C%c3M(l)Nb4`98<*AGvU2_>r#Hvy$NN9)FvX{Ihx}$wP$L;1Yv&t>$iicZ-A@wRE1?PI+&1uPw?RH<+);I<L9QOENw5_BYfn-<)Gu5$;z(Eh?~10Hr{T`xp+5Hb)jLvjv*ATYc~hF?Zbad_qlPA3l%0`z&_x-@Is*Hq_ZWy+!)^7HOFvMZGbXyD5uJxW^{;ntayI*ABThQkiO6lGALlno2NR;;Yb2elBlDjTUJ|ti`z}Yl`#Vt@A4|(h4}_xa0UGO-SUMg6y;jpNLdb{u}a`R+WCiHcQ%uqKXeiy6=&_1w~Kq(!9rVE<BBo84X1`v*9G_b$e`35L|V*C2NDCek>D%X@NCzkF)#)%<>Mxi89TqKZ<Gx0&8-ew?!N_Gz?O?WpU%Vo~^Kf@rE*xSseJ7hwt_4<c;$eUp7&QYg*`gIZP|Sd<9TH%@&n#hO0Q%6K7G3;ds8?j+ED@mDbV5=%FuSy;w=Y(yMtn@T*|jUO_9v_s~*p1enMCdp?rHOj8Mw@RVEm3ck0!Hhw;0E@+7_=Md$|Ej0dQAO8Xrp`$uwvitG|e=LQ+1b8JpW2_SLCUD4>F0tW5r7qE9pN$}uaK|2e6Zum!J?VT0;UQl|e-^Z6@MqvH9b?N&xr^*CP)h>@6aWAK2mk;8Apmez%TV(N003zj000;O004Alb98erbY*jNUuAY-Y;|FDX>V>WaCxm+ZExc?68`RA!ScSyIIU{u+Rg3)2AZY;dbn#+>}?Ai3`1Zf+GZn*DoG{lUF5&t3|}P5mh*yM5I7b!GvvHIGs93yFK;ey$Xa|ht>T)nx+HSF7KLCH`Ss@4EAnS6)SXU=s7u~(_^1s~kPWZ7Vn)cC6td(QrF1e8yGANQ{?M}S9-6)3+Ds;EC3l2>ZnzRVkfAffbbf!H@q1RaPLStFCIo&iw@j#i^UtD?b?R1RA?@mwW0A_Ov&}~F4KqA1Sc8J{9XCo8+P|*Erp_DL@SN59fh)fr)O4>J#Y<5jw|m>bKD^9}s?~<8{8mb%jbe>^s|%^PWlSg2$z<|_(=zjROyGUP7j!4fw&HX;DfyaIPM2i*!HQ5EDC>cm$%rV+%7em**)@5?%r*_;yfjg3!|DP&xFBEc3jR>BV;ipN2eSN1MG1euJ9|ut|9(DuT%|p20T$dyB^-UXjz6AZo)TKK9fz+J{0LsAkLf7kyNI%sn-gSj&myt{4G|N@XJd6OjSZ5~!=aYB*Dvif*}J9r@#as{zk7<|e0Id}_U!o7kM{F#<O2M>=8A)B-eA~+2D~J<qSzM|Cqjc8H^P8?!^v7Ia{2ScKR=S&){tL*`_CtGee=l+67{-5BH1dUdBYSw1h2p8?L)%ZAT949LpPj2%F3OnF@7>D&8Nv<h?Up+!|sP(z924eEYLt?kXx=r$q5TK$><BLABnSJL(mMR273^*<wQxj%jhYmbU8%61*JpY<d9-#v-x?3;6<m|gAxEJhJQ8*u4HAqYjo0)Pp8MSf>~cCDo}e#0S=BZE<@nVdxPhtP)Vm*j|ag@da*>KpU+kuMlAea%Y_(&n1UdaTkGYJ^c#o|K*FtJ8ey``wWv33#gzEsb`2FG8RCv#hDh@30k=RjFYXej9MhTuu=`g_+TcP-S7fmu??x#KR?AuxaHa^B9Q`b22)hR4H!^>as?-U++={A%#3K*P6kALS9fD40bh?~dX}u)_q74x-encQkPuZ~55JO$A=koRsUKnjn38e+jdJpeC0_3~4GNP%V{Fr^xfx9=Utu0{CZauRriA|*oJbwy`NC<T0A38ii{TQm6(G*aEmy6lJ0annGy2FLjc)NZ#LTFUW56J#uH}UV+tkQ=B^AR}~)x!H&fPEp{i&=K|{RnS|wE@op^(ZmYVA(zhn!%<%mqy6WVv*Eal_IPJ`PSDPGvFL&f;}R}031iWQ6TL^p`;Hdkqe!Jh@3g4bBNBay+Q0h3Y$vSo9Ch?2HyyAhq0o!*n}NRve<hu8eO3Q1Fn<JShB<urcV575SAU!(*bcvrbi=TOxQP&=(u~d^mliVeq{;}eiplf?;5l3f;(q`Y3L9z;Tp52S9Hs&sx7d<qPqMZ>KS<Yb$R`13%vmRf>$_ejaau8FhGwDQ1u!cTal~jsT>TJea>KTl56NxPLO@!5T_nQ;+Px(fQai<B3tqY0X5WjzKRI|)8uzB6Teal+I+G@3|~;|66)>wWb2<SR|Al4>$_S$)VZsYh{`*mvBpK19o5)16IJiPv@5fI>ygv`i|_4Oqr)9K`u+R&As2&JU#9jgfPL5Nk&Ek}ViO#949PO?O4=GqQ|WpUXbGHM!jQ3J>Mp}((wglHcjb_}IVr_ytpJ?#Rj+UJ5_&CDY_$}3L2FgQW?s^18j*bpB#4Hm$^OfA1_5UqLe?~#DQ)TY0FeX2>zmQ#wgwzyYV_UcVlBTpSYP2F*M-Z$YzXH8_`T!2(S5U2gjT76?Ts5@b2lwJGkPnMV8)5}noo^W^Jn7trkETZYZS_sT-4xgQ9|_U2G9w5i&9#6Dus3y%x&M;ZDK=E_rqs_T@qkP@6@lfVz<1)wi@k{hA_Yda8T;}ht9kgTHGGA>Jc}1cfgFXQcLIjmvQ{&{CSG}=y(`i{!oEsHNf0{UQfL0DY=K{mepoKSqXTg@u{K%<V->v)L`a>hdO##$8Xj(RuOC}VNmwJLwTJtp%W>JR<UC5kkBteBov?vrfakci%@0;OY8Zt0FI&l9=p;Gn!9Gi2T%7Rh<F(uRlMGq?Er_5?pOeTQ=B9RPJ3Lv2KE4<?YCz(8cd6@<DV^v#a!&}Ec!rkhe(gfum&F-wUaw2B>rBgyAOg4Gz_pg$A&0e7rDOQUXS^Lbkq&p0(w-U<|#B(bty_$*{#vs&UhPuGO>wV&TKTDt$ITQ{tU)SUY7?}7(9*DBHtFq%^ZGA(`Pw-8|b0@6lZL`@U2tyubugsou4?fZLME(YCuew)W9#NZUd7MH>SV_0BrPP{{77FH9sFDr<1#+C9e$2J5K0HpjhWYrc2E?^D)js<duAI6w%sNY_?Cag$5|ZsULgnK+w~8Z9P&tnzzT|aQXjbn3o|!Db}`GGchE*<_OCVZ|1APIbO5<Pc6>u2%4|i9!8C2_`aO2R;Mg7fEiea=_Kzj$D>;WoG*-k*~Eg3b5h*AYT2p+G&qEuE}HxF@ZWf!ipyuN??MX)u(p**La%1{jNY`Dt*hf&6e6n&C@{k^NU+g`LZ0WadmPUgffxibuqp#;8u#*|!b!WI!F@z8qpQ#HTfnH12;)6=8W+F(c=eH7-u!;`(cTS&{q0zTHQL`ppcU;u0$|MlA5cpJ1QY-O00;m803iT`+YQ^$1^@t~6951f0001VWpi|MFLY&dbYE_7bY)_1Z)+}adBs?5Z`(Ey{_bBva33lUvX&I<Py`S#CsCWAiIdq*hs`qxTB2>W5-E_B)2!%!-|<DVoWyC`epo&v67Pk_yXPLBl9!c`8d0TtxqIY`rI00U->W;#K)RHoL|TYKMT<HwGLnjdE(jNzErj@pytb8fl^X?;u_E6vdDmzj4<`vZxw^bOpZ0paj4hGbhgb(p63LV(HcXLKuHj`#v>?m;6Eho^Wv&BVR-~eO9ime3kL(Vf({=x-hv3I=@j(f0-)%fbxyTod@ETsb%Bo&eQlw0&t;yHVThg5Otbr6lqu9C)2!vWNLi&+EM$nHcDjC;mb~v}{KcS&a*N_pUCFROG%azRm%aCHAPBkm;BJ_^23h6Zn(=|&!qNPY{1v-Kw$!Oudz0LIi(uC}Wwax_(QO?t%&T_tjC-A_GPzwpR3#y<ToH>`G){%A7$z-IMrdrFu3d|4QpA9%%`jM^}F<J8k!eMJ!&~lMco1qXn=@Eir>RK}5O#`>ycUFD-JA-g@PR(H;vrWO#F9^3A=hYow;AaG{?o>qOYJ)>G!yaVtj0b}zT!hb(;i+4Lj8wH*Q{bc%Mt|N3&(uC}j~iOl3}Q5hiwZ$A)wKc?(2QvINe6wW5AlI;rND%n<u=Zg6~zoIQOlI!In-L(B~w7V#|)cS1)_ZEmCbLW?<%7o#|Wn)W4IG0rzKO6WfR}$FLsH2VrkIC5N<X4URtr#pfK(<MP!jeTw=EI&hX0RPO{AaFsoRb$T`&dHj7==&EeyzteFceg1qLHPaai@8RBW8nBj3UNrvN-GtUq1U`<$_E%xBY5rAYM<LGmz{t$mj@e#n7szD|54Yb$5C3>(q?4T8!=4e2C)a9v;jx*e9xrdH|Apiw}&tJM$Yx?rlZzhQDe$){GS-gJL5q;^hV#}iL7=%K}Ef7oumEju7GteR>6dy$Jn`CuW<f#Q<;M3B;)Z#zD6r-QfuLpo~cN(9{_<m{|MGxMV1LO9xh-~9$d8z}uhgWQ*u$E_#@1QaDU0}C0O)ztPW#FFkjC~4Q)H7a}Om=M}dvJduOW>{(qP7vXZLn2xK+}=MXkhi-_8~H*GT@7s*wWg-NC%&7AYX8Q$Au3$qE*Fs*7f7`J_uk{(>eGbjwACHc#|Eh<FtS$bWzX^<Q7h5j_Pc8)GW|}?MWe^&(ssDFMJ<1gzk7O10GD*_{AQZ>%XRFSEIAx<ZO~8r!hX*r->O?1`&*Jaw&KTHyPfe>F*e_i>Y98bu&Im$o25#-SBO)&-i+L{(d-3$nnkS^ulOQ&z?H7tU0OH(5`_yH}1dqx+XmC>UH{T#kd7qMjCb`)HFf?R8L?MS_@8YbNEwhe+9ZZU{8ZBp4(uHodz>^f5|kH|LgtWnI+gp1CZLj7=Rf)+<$O^Jd@CyQ}3}c#Cvf*N_K3zQ{&@i*8u0lZ$N7;@E*O%)ZozREemnGR}VU4;`C-v$-@x6obNm3&4|1?9}O?aIJq32k4}@ZDTG3AXrb61qv6f;icHRKrl(hbj+}aXPd=8&KW@&)2^k(99FW%sZ;!ve!SQekiJzSRoixXI>I^4W?~^fPhD@(6;9+EX_S)IKj1Hhzes>=Pk;ky@9ks4?CT@9qPX&8ta+t$9^EIGSHIhq_)dicvQivw7R<KbMR`343<NGUpZhxga{guM?*|~LWBg5-9fB>(ASQRX?B7X(g=O~kAW{>Yj7%Yogt?eyb^HS7Y$8PF9a=SGb2DW;*-*pLur1SDIr!|KH<tx6&mt+M4N^88`8-R1o%D{}hr9GGF-knp0Zu`9?@O>=Q6zMyN=m%NH&wNb)9eL>T)+PEDv9=!YnQ7}v7HnzmE|*xKf8uKsJV3Zd_5R(=^nBl_*Cfwm^nRjdgTwhYsSh(^)Iu{uI;VC@`8o;vSODxEdUjTKxS4OeAxdFM2KT`5ke%WV)jSzpmMgj_SoSRu&O7u{QfWRbZQW)5FSI1gXx`z>5M<~gnfJFvT~df7BNcoH7x`+feI22?zKpg%K@9tve=Qh&pb3ZT2gM4J-?6vLV8&7JM~nYNJGIMCwpDlf#dfkGN>gb++w64ph|_-0e^$d#2}cjCS!a;mVHYQCWs>HKp|Q(jBBhW4ZWG;nj3HM6>^Ew&64jL*|66Z6u!?=2A>qp<eBzl#w7bW4Rtr%(yv4?UXe$ugGHl7iOlAImRgsNV1>Imn_*w^EjGu^kn}^4M?X(81`M0)TF!Lj-@%5k)O%;6>5Cc@BMjPP$A5cpJ1QY-O00;m803iTuMdViU1ONc%4gdfj0001VWpi|MFLY&dbYE_7bY)_1Z);y<Wo&77Wpa5gaCxOy-*4MC5PsKR!SPAT6nQDyZU|st?UrGE2s&)t9%dK>Ezvd`Nfb!RNgCvT-<?RxqT?jZ>KB{j-FH9u-5o8PPH9V4+=#cY6X{l4Znm{tg{4&C*Jq>JWUW<0I&M*f_=7)S8H7x~>a;3_F=6M6?AEdtNmzHcRv4Q&B&$?yidI>%QtFCTqLw?MZv%3g5cs@*_typc<>Te$hfn#i!8%@E@r_^zY~0?s1+c7CO(U&4&|pnz!Hnv4DS~RAWJ!|z?3^jq^2YXBB$Zf`ST4=Z9g0$x%!g1@HrQn>D*#*7I}j>5u0?Cj;%w=SmO^%@F(}RmEi5;u4!gii(m_tHgJ>acq_HO0!;g~&43sZw#VeBrX)Sogthljh27v?(^uOU(Vv2&FQneP{1FXB2H{zV6-Y;Bb4sC_GNHellde|ni^)4*vkZ8KRk5ILMq_qdPkp^4&sO&T9E`=^9{(1!Gaz5lLA6AL;8Wq6jJ*kRvE6OXTdfW9j9dQ<P1IRg{TI^2YDvZFz?-%cWrOTX@*VWuXW5VX2bno!9!dNZ4)H`%6dEUVz<Vqc)TX=-W!$}5eA=kXR71aSFnKZzED|91UV1g_eZ!6~6YuL54TL_!=og+fBXUNEDVw;X(aV9>5@nG)?rZdtS1i3K+YtwC!`P>KL?g4WSwsX=6ZNN9UUVsV4kljIq=FxVgp`7VqMI0^2boBL@IUtkOL&~#k>aa(Bz%h)#13oyL>ZidNdMj?gplp!5GpJU!`*CJ$U)>hnEzJtJ9wc}g*@w;*W1ya_1TUHjbV5x>CinII9Id~O+WUOP^o}VWb|5j6e(LJrW$uk0RdT(?@6%Ik7pJPNZ=difa)Hu^mTS?bHRLNzCp1HDsY*aDH?>-&^rV1!LbEJGl(RwIa=j5qtiyZIeQV)5XV8(>a)Y-xkuR>bw8FJeDxqa*-FHxx1iD|j<?RN3)~e*SFBj^3RW;URpz@?KM=W^I?6bIGA0UnWbrEq;tzf!9M?lYk(S&wdf_F$S7s;dSooZ%Lq~U}B){et>6y0+{Opu1qtG?+>I*H`4I=`*b@s2(4lSTJ<=ua1*f9e!w%U`{Dn|fkQhhbK1#Z4tQ5J+4@V{;jA?l|L?p6Ag37^rsG{?}aAhONXJGgj8B6-;T?0QqZXV6b%mtL3vt`n@i<(3;645WmG%KfUo@hccLxl%7$1t9Y5k4@>eL@DVmeZ3LmOXhyytPtVSsK8`H^+$%Jf1`_%YuX}N!wbD3^Y`LAg(P8?AMq~FVaiYd}cXBeu8aaO+9KX*ah2!(;Q4u%0;`5Q3`|XG0;HaGm75C51hU+VQIYRe@F8##^I8X{ZsyDu%MKKem<lR7Oh;{~u2HJpM{t3h2@&YGL9ydPFi&U5R>K%D_z&|ZMi_#iqHHRm-;q~zT!2brdY{d}KGp~-E1kYbaMmJaBOse8NX6D03Uu+e(9QId!gcJWxr+a$&e;38Svil9Ev%`WsTmvP!37p(h%Wt?=hW8nK&IfDGu4r-SAoCqw0exYI)E!?$p)&49;5{A3E#Dl@$z3pVe%Nyco6{yUAuV+>^nsFhFVdT+#l_(M-4(EKYJ2TU3~R4?|ALzRyES|*URtp}Yx&o|94Euz8w&0}P)h>@6aWAK2mk;8ApqG=_yJo6003nc000>P004Alb98erbY*jNUvOn|b7^yQWo~0-E^v9RSW9o?HW0r1R}k!B<%J`6z1toF_z?5~6fG7x<kTP_Xo<ENOQcFtPP~EtdxtMsl4ZwkHy1}D=fQ90n;B}UWQi(j)&*Z6URF{WbPtcC@PD;4Oq=*5>ayA)s*$KhODk!pCaM;Omn@~q@FmTRj(YCGi{F%z>Ue*}l;+wnk+CR|j1c_Y#|87X`t}RW)@<V5EAT`rMhcd%nDXzcdQot_?z~fs=7ft(u@ZF9f@nc2y_RM&8jnV!oGp<(6wzy{a+Z_KnjpGXTfSwQEWys2Y)e8#P854)a3qpRrRvIXDH2mw#3DAn^&T|dqB)9dXo_&IKpl2_mRxDOe0;K`EwL>>;Ymy31TBjid|_LrcJtZzpewCvle}SG(iOwE=+l0J-f&6v1|CF}vK>plOi*ri6`L=lEXHobk`~2+W_Hu>w9ssSTH~yzQ3|FLtMkivg1kmAOtoeZxq^uV0%4Ma_vxyTiv)j64OMtN9;5jjeR49IWm4pq;4$mq5iBZ2cY1<KW@t_g1wCG2x?){WIv&RDj|Ui@5Z4|Fw-JtM^MZ?2>Sc&CMc6+zEfR0`(PQ#78HU+gbcqFS3HEvhki*vWMRCK(!g4KH;DwSKCa~XN)?8f#)#m}IUO23J3a5&44eC3j3@O~$=^;4VclZ+d7db&m&X-H3z?>yDkt))JWb0aN`b^<D?Br$#r+LFV<Cgg@PHxpR7}mNpA0Wsv$tI2NaZ6XKAT3mW#FLCVBEvJjLm4$G-k%rqWMCFCE9BQ%j^VX7)YN(&7RLiMDG8-%!0mCntmW?=eRk2&L$*hyX(r36V20(VnbGX^2h)k8Y6lE>*On0S9SgJl)d_g!>@P}+)rr%$Gbvhi*%$ej4B#G^7|wR{F@4sM595|}kCGv;@%KDc)cIaiTMRt#&AA7BqXJ?<Bs2)`aL7IAYHMyS@2(?ZY*c65c3Un=#iq=k(dPGDq|UdgKu!z|37;i%kA+V9GzmioIjcRFt48}3DbRDoTN?JH73%i<;xxe7z!Q>77*TA@74s37g`HA#T^QYu_5UlWz~+E|t7QDg${O?P%F^hls6oz+7FRR5HJf-%tr<irzTyJFbrhc*o=ZENg2|)_pqG98@hNqarLPI&$-uhp2FVzHycQy~-*r`c;R2X#E5qpyb6(TQq`eaT0N;B+SWLTUHUcm~pGBu5>Rz_sic`DG-xCwPr}a^^<MLxn4E*HW{QO-S?jSKP?7I$<i&|&vIL8gTb(&sK<KB0}+Dz=%(8}4Jby^aP)SB5*Ia`Bh91=CL6;jrQz)VV(ys#^!H1>z)N0sW5%i*FHIj9K+Cr#|42Y!9zb4S)HW4#NRe~DWg(sGk?6=s_x2w4=o%A1l3zO*IYA3qEQ#*P&q-r_EN3qA4G)QaK#ha*)ktojBTG+%}qKCI1Rr19<B{-hERIpMEgpVIWXS~ZH}?BW)0qKju`Q|~5Z+MtVkC2VTpTcc_g-HaFk*xd*P8y5|}UE)ox2Hd&xcY~Y^(Xy)pHwgk%h(31fT4Vnh!G3+~qu3vYPwF~Cl)XY)_r~eVakIX}!?al_imJ~*vSF;!mTm=MV&!Utu@_7JjShXs2rQGj{Ui#rxtO5QFgP<TbM%FX8d>s;SZF5pDm?q?XICqFB(@)<5~))rwj8bk?v{$b{`K%D`StGM4kC@G0w%UsfAx*spD%rA4yhiF51vmeXiA0{x=6qAsuh;j*ye{jz2Q}Y?-i@4a=6{d7Xfj$=F#7O+@p{Z_ESPF=$7(=E{d4qKcELo4Z5OSz^_>|1m~}5f!s%@95pT>(7uKrv!()3q;~=y4bM;^<)*GuKM%=@AMxdI>BsHR>2+;*;gL*&U<ZSyoWRLISkE7T)nQPus^@4(s)t<oh+$1ZH>-9wdY2$RaW(=A4q4EeH5%KHR!*Px6gtdZaDQ<r15NYow58j2VkOS%e%~I;T4@__>erGfEcIrh;|<dx52k^6(@YkH^yOctu+Xmrm10{iYdh;&&ZV{%q}IUW?UVlwq;`kgJlX<ckJ_kY$nU4aI2H<)m41Q79_s;1{J4%ZH#+rI<`Fw$!CU)d7wm6}pTay36I=0)NAXOfe*sWS0|XQR000O8001EX*slW(oCW{@x)lHb761SMbY*jNb1!sdb97&Da%FH~a%E&LaCxm*>u=jO5dZGKg3vHP>S!&`bxmPl?Vj1PHpm`djDVme%I2ciMB1@i^nc&+B~iBJxNC#N3CX+T-S3`E5Cng7_J{<_St5B=qPk&%H7h2NTrf1FaKcN5=Bz51Y&M7{O(g_^#iB}4QkAl)auEkXFdAh|RiKngn&ean5W$POY9wk{ozsMk>~cZHBIh$hE;n^Y_y_#X*&A9gQ7Z<=Q7(%bt`?}Q4cG1UjpMm4=)4tOZzL1atz^|C3wupNOOaJg!J78A?{BVeHTP{(B}|B_u_mx}%^F^?Qi@noK~~RWV{gkEju3p!mOM|1!vO2O1#OC|<nLHY9C-G`h#<2K;VB+PgI9qEYTB^0f0yOX7?sP$iL|6{7jBfvqJPAdfrqYJf&W{(fP<1kNdf%<va~8-DIPG&*_<XDQqBH>Kp~3B0t-@_Whp^uoU)`!c{%U=o>War1Y_wgF({I-TF0CvzioH!m{{g=m&g8g3^zGl(L1v@8jW<+p<BLYx!t_dP(W<{C76z9n#d*1wdXZL#_^!?OHfu)4)L$pg^D#5Nyy=C0$c-Y2qwMTW3y{0QkEe?c*!LpA%s8{89mgOTE^NZq;^pUvUWo;Y*{oL1k~hxAQoUYs6|*hXTcRZKl^|YBxFr2!GiJmLh2KYf=8apg&}LDMV*e$HmxH@S}Wn{3WzM5((cDuPNigJcoxMvnd(Zg@D!snkU09zBwn2HwGuf};*FIFBta;Pn$%P-tndv+YcRU3V_v2l><z<<7Z^Q1jUs)nIe|bU600bnO+z;wB)}=4$~=VO8a+ej=u32V`3*)JSX{aV9DRj0ky^s`+2v(~(fBlqsZb>+1QAPK%5Tk$=2NK%*E(&{*C^1c#C16jI?ZB1S1i=_nAC&>G3LBuP#jxhFq&0W-qB27T%BJ6V3$`HFYbZi&t}O+9?(jnbpzH60IUMirLu;yLlvWE7_p+3o5@c!SL=-C`3zujYn6DD6ASu|bkLv^W~|GMye<T`Od~meV4>QyfOR=M*-789Ck6Lb(>GPg?y>JseRCL|U~~b<!A6!>RGyzjd#FwZ2;K*_OgSVV_}!l$u;*!4NG82<5k`LLOyRUwMLUBmdv~!0w_CMHbbDK^EfjoSP$;oLH93h-E<7u2-PPVxA3eoQs5k{$UAN=LB_w3dLktvDb#JH<VbDT@(bUF?3dH+J$C|D#4J&~@C7=h%crIB(sx0gJ8F51babrO*EZq$@@b>6R;M+r22X<?|FUo|s6X->_`@@6AxE{p%jW!H^JPjOl+A1Kh^GIko0J?M%*we8y(lV8m0RXuA-QF@s)06wHz7M4PhJY}do^F$pVeuVNdxfjl9&)|Le;_`m4tp=>rp|n8I&O`{o{D&%y7zuamJq`b3T86s;BUK2JF*}LS@8Dtuj5|=W0jf|-K-X~f1y2xJ&E{$K&yEtG5TbY&ctn*2`yJJi5y|2@42TeZJp5*kB^d0c2)Jf@&&r1kGdMDfo#TL_JwhtCrkJN+|;~)Ql4dM9@cXfRiaZ<@u8lpX~`({(#oSQ!*o)PdAWiyvVtdp%S8m$qypIm%u&J$2NEM*3!g#SJ;Xj3Wp&U`Jy;^!56uL$u5+Hih^1xfQf2F=OQEhxfi1k*SF3D=QbiVAdis2*UiG`?U_Wjv_HhMCYd%np^x4?h4V7Bi#MW|R9gW??%MucM9{3kqJcjC>J{+4ZU!X;?VRPtTR-hMjJ3`wK9xQsj3ynvIFZ4qA<LjMY=eqi)A;NuB@UBgM_UuX3;gr<!yClS|;?v;u&8zQ!b0x%tO3LTzVKH&f;8-ju46JG(rNYd)nh?h9YNnca7)9}dtqq?YQ5yQ_KhHqg(93OT3WMtm6};Lq3ga4S%6b3M;XU6%J%{uxfcJ$#bZk9q95`ON@YZPJT~N1lkHxYGBQ%+yZl%f9TQ!lUVcS<$3;wp}?fWa-eWivrceiehA~S;iU!V7a$@jg}I(o9?A64UbE@;`iAiI0@=7+bxlO9EMsK_3n!jOHRLwlRu@bI<&MS^E{cw~}#&(lNL9@fyDD7EgcfA)<!29rH(y~bpN%o?J5rH(klh3!?39Bx=jVtbuypSykO$jNzkU{(XYH!Q2=ku&lUEkj@NRL6L-b{{U;wjzH@5)Fs$PBn2FOaoI!g8Q~|n<^1~&ZPRdY#K_T{{T=+0|XQR000O8001EX5csr-@dW??j1vF=6aWAKbY*jNb1!sdb97&FWpHnDbS`jtrC4om+cpsX?q5Ohlbxxa?9Odcz{83zTMKL*&;kQC!w_hRwz*2AMpALJBL98IH;J-kw`ubsk;FUR@t)_7cflCH&l%;KZCQ>KEorWq;E3=7X;v}5Lz3<!RSLeU1TkLFn!+!xk=UY26tqO5(RHKM6k|LZv8onQqxxP`rAMQ!6cwT$Ybsd<N@}W!%I=;|CADH++8NHajFXZn4Te-xRYbD?Xo3!sX_`3&x1ZTVHsEwcr@7!;whJ4**G+N%%6=M+Mg`rXoYdw-<BICNC=zs)pj_NhNp^IZPHz&lC8eUvS(+f@pgAVIz#FSTBxA*LmY|aIo!*=8lNTcdKa%Q3a`ef*m>(R(@rzb7=~n-3sJuVBswg!pY4pjt3m?7T-AXNm7P%;44ZnW*?)N|5zWMw1XB<iFD2MZzqhtmkBdADKmoyd<n}1Bm!<nrm_ZSJ;*u*mSLnNSOLx=W2PfkQ%y?*)TosnT&kL@D*5v7x32^OrN?}t3fiKaUNu?!%BczBG(%qLo8UYt>5Sc--h;~<T$(4uFaa4s~+A89ROSR4_ugOG(;bFXgw!4uB#K2v*OqoFeJ*2G!0{=!39k{ZGaIhGXya?IQ01uNij+)^wzR?rfyTKj>__;&W>HIAI`7UsA|@N^y+o(_sXJtsaJ6944~9M<Q4eeIo9O{u}x<4hj=V{}$BElPOcxwSX3>O?6D+3R*$VYdn7#2PK(_{QvdNbYn?rvdv~@y&ha(gz&-B*kGB(rRtUa6nrj?O(>bEdvY+uODKYz>RLdm5K8@a-asOzK^l&QzKgwC>o+e9K;6+(<rJkvQeUJpzJ~6%pnV`4FQ>>ENv>H^F5Fz)Pe!d98={8#0c~2n-0_75gv`@33A`G-E4A!2_@Z;{NCVtbrl`e9JE&<f4lwl<*UE2gYv-i-rr19Nf-or{3j_JdMl-nQ2$L-mp))GTK1p#y>iBr5bo7%y*_`47-dp}Z+8HkJ_v^=++!u&20SmD0&w4$5=EfA8I<Wq|FtA?M`czDr2^6$0&X!*Ap(7*d(b}XrB`%0j&Z|*Xa$~3PyO~OG`|SmDtK^#S&Qq4S(8<K;dxkH2^r4v@dX6F#Q5`@p7n@J?ZzEPdoXd4K|f)TGB;2rWOkgq0A9)Rd-wqGKxxRJl;lv-TOR{Xn$hC3>G+wnGkcuNt=6MQ-V?v1`0FbDIVW33hCCgOe)4GtP2GnnM3X>qH0|9O-<b>qW2+}7T@p02zqh3zdW<>YCRqNB@h!w^(gDRdXtScbTx~YFD;Gf8P=JL(9?`t9$FjCOGGA5<+BPF92t#=^nE%e)H(EO-|N4@#h{hX#OOyZ$2eLAiDQaR5_loJGGg85&10X;>+Cy`++d~_Fsr~5MU7N*r|8{o;3fy)@*8pVF7ws%X&1t7GJJs-V#y9bSawioDJ+x2e7SHS*PWX<FuO|^(-56o+iFTj}*G{iAdO5hI7il(pF3aWX&{~}6-7LON5A1cp@jB+#&+MGwv}fnEeY!e@y=Y{1AhL2wPoJIB8i1g4GUAUYi^w<EAt@+%uHcc4WTsJu1(<F?+#fs;XejinPxw6k=P-X*KOQQk&|AUjqoKN}_r*{}F?3^V1|cG4=BI~TRFH*CK?rKXq;mSMvooV0RQ#%jCA~kr>{rLPeLJSIc1)e86PpywB*hqtY3S?P=63k9ur+^R?473<XNbPqyj5EqYlTt4PU_3vT;HBJ=??o{=P!H}LMtsvZ4@ZcYPs;ULpyr7omTivtO4FtIygoc5G=uMU2r(g?Y}^RMCiABBD0~rg7>E5_w%eDNe0E+6Q0VaL-N6q5N;J3<5|CV!q7J6k^w?@6vy)ow47Ad(9GBr?9U**z<J{C!NJn@IBw`*e3yl)b4=4=WLv_t5Vv1W$lMY5xUmbInyfqZAx<Q|o3jjUvHu!&bZckuPtQqr90>Kqk|`F6aBpkQ|4>T<1QY-O00;m803iV1bT1H*2LJ#e7XSbi0001VWpi|MFLY&dbYF6HZggpFWiD`ewODPB<F*n0?q5MDU!<#6wSBvHITr$&w71t5hg{Jn1qwqTXo<E7%aTG;`xe}P?+i&<r0w-?g49^pE0XhYULOt>MbQsj-?wsiwU(R{(P+MrOpCT5_d;)p-f|*2t4SxChF9eCXZD}({`k3Odq%{*tNETc+N{r_D4I;xvfY!8>1{1mxS*Aq{1YB06Zg<PYOeI~q+3~TofLoQxD<$|GR+jdd!OyO-nNw*fY!a@v~D*-tCZL|x_YFdN=a9<UWt{cg?^-argjz+ab(|UfX$fJfQ^<pIK?$R4AK=Vcf6@i<tkpbm1s7DSUe%{`74g{FCEIN)C`%0oUSBm$}M+;Yp{@4w3B=-9tP#jjD+Q~qKfk>og}CGy5C`2@cAA4-YS`rnr~S7NEO2v$jM|<@ij!R>$QqkOqbhymMqLR#>Ls5H9f1%X7RU)+h6yxar58>)yjCeoTcR3lw8fH)0AAN<XurDndBX3+Tx~UmZVMy46Up;@Z+Z68o6sToH)fW<kkgsP1k_cORgwL)0*K-tAvTc9wBHe#YQxsE%@%9$tvE1TWe>Is37*8c;K}n`C#$fS~Lj4gb^8aGg4yDHeBUaI3;7~F<)|PQrm!Hwb=0(M8HtYxROCQ^B-l;tx=hc;&K6gniUgkyNL^I?1#xIZdTY`se)qRu*?Z$&MC2HmR*1F(nSfgLB<=Cc-Fqyp(I}P(f^Ra8B>Z&P0PAfJoYuphN>D|_@Z>E-!t-7f`mKqW&^~M=NObsZCS^cvx5AA%n130fd64vFci`gY8pz1tR4w46p{*7mBp>@pvHLRxy22|N9lZ1q-6OXk~j^t2-F+UJk`!=3>rRbR0nCioEGrsEV`N(&)`5w{MnKq6s6pOsARxpQatfE5*4Q8T5uVG-qFV?{6rsTQIU*wwBZf66ivZLvaR|Oi>cID;2|&A1cb~#z+S2%>lDkYSVA7b#vX!Mbtm{eivALP`Pbh+MGFV^)lBZ7YBy%qK8v-d0lEcES{#bgU@fCB-~Z$4ZxOg{Q0xtvX4i*R7BuU-3Va#cT&HD~hter2E6VC_%kp_PHBT$9Sw78ftm4Rku!TpyF)W+S-v(YXOU(f|a>&Se?LGr)+ELkT;v_vRIPyIndoiU@tejFLQPtLWJWep;c*sH4r-V0bRr4x8^^XHi7Do#BOpag4VIGKq19Hg+{>xmd@X?4N_~Af6jKNgZI(rQNg>$;?=q{h04tDu0J09ly4pjllIGx<H2iov^Y6@TFvw1ija&S2J=8esZaWq*@(~w#gMW)+YC>;+6pXzH3PC7cUM=za{IGnh_NoDyizh$-JUJ3-RHJRz92ivbIMxGXa9}_<fos5SPk4sPb8Wg}}POz%1WTyJgv<MJAPJRFi9if6YB|ddD0Rqu&+17dV`QxYW{_@|*(_@MGRgBO4ftNiZ<q=SzV0Gkj)aOO(-OOwO%%L!!mxt6W{Eona&<-74QPEV;XEay{bPLgfqr^1{3Zhf>P)1mAuWbgNVJP5JOBD;BW^Q2k_S4l*;pm627Z@?Ry^fMs9{vpG&NYJZYUt;TB{pH;yfQu52c(A+kGm~@2SDp~2MAzfEu_*ZFl5<+HKHL;qKjEdRvri8x`ts_+F10(QT0FHFd+I8Ws6~t;fX<c=5u}pYD)L7$JQ9crO($NTq}n$gBTc{%m6AC8!ErPd6(GEthK2R7?(?7>ihuJ1~-KK#cG`#THZ56ayAMUI<*Bl=#^R%oweY!8rgeU9D;&X@n8*u!uKK}+xtrt<|Sr&%>GLYx^Upo*c9QKUv>*}cV?B*Xa^P@!k-CXc4!^Gh5*jiU@RmiFE-(zAswe;airIC1T>P+=*CPmn0<J<gR4e9f*vj4tHJ5Hjpn#8V5+KPRQ=q;$!(9K$`DfnC&4hf?VFvuz;MV#`D&tinLZ5OYc36DA#|I(d$}`yi=FYc>5Pq-k6?^>1w=~T8x-W0BcQXF>)|Yo(4WxS_D#pC3fdn$*qt2AyvEdDZ6pF)(cfy_h;WuI<Jg$*)}<sKqmdeQ$1$TpY86RNI=tuU^Ki|wyB;1d5$L5b7;T~#?|PSlanQkFxu@{n4+a~0ro6T1xh=7kY*|$@Y|E?%N@%eZarr+w1%W7Z2SQT{9dEr(!ZQblh8QYKo`LM3HLgqjN<6}O@x{-reI4;{yv$7k(Qn8thR2;kpgq2_+Ve`l%KGt7Lz0*Yw$@P0c+Z6~@O5wBo&|XB)r`Mk$n1W=T)Iy2x?{fP!@rol<M3h5d!=&%iXZ3-XD1U$dNOoM==iw_hmtTDL#^3h*uxjA1MlR&TR4J@$XJJ6Wo-A#Afp?&geCtMe~*s6w(rU8S^w^wM@HHo82`1mlaph8e03Gi%?r3nKX%-`(E@unUhod>u$_J4vK^%GwO!)%$Iq<zdhSZ|w6wV1fqm&BGF~gq4Jtjp0^azy5NLen#cOXG%Q<HKTNEk$#YbeX%&VH8d(NNmN<Q2V4TQB7zl+KL08mQ<1QY-O00;m803iTp6a9w%4*&pKH~;_@0001VWpi|MFLY&dbYF90X=QF>WiD`ewHy0y+s65K{}lv*0c1*ZdRTVsO~aZjNi%e9(LI1o5QsB*l5U|$hNK+3S^Br{d)$#nijt*vGhkah-hKDIUw3j{_ph;%S*=zoc2xEtN+q(|S*gc*SIX*lPL?~{JdvprZ)E(T$Q8WJOXuAWe|J>nl=9<Fl`;uk*S)w{mHF05L@nY}RF$k8wJq|phFL`_VtJt-x1!!$n1^h)EuI`vIa$H$e|Z0`S@(V`*3GKosg_l3p6a}eH+o0;SjbXsfr=`uMHM~VgozUCEU#)6H%NFF4*a~&s#;{ReDk0Z3I78c6-g330F0HANwgAaC8JWVRi5!{;0BBrrGyWW%2w&FUHb1TLL{OzU#q&}u~KgH2iZbYdQ<b3xDi!iE1Or5>>yNq5=E4Xdzn^Io~2JbT1x>^D5Dh+j|go_0hq$hfLneq?p3O4J$07>_6>Bv%Rk7gt*keB(&N+pPNhi{=UH9msU~Z+<J}rMH=^9;nfeKExd%*Al}Y_ZWvW&pjWlc?<cH|~DN+fK03&vlV!WfRs6HIS^Fhf^JuF2j1+1%qD@jy}ECE3<7<?PU+Zx1x>v&`h+f|_8(Q22bkwW$27pYu}_$kWo|1IO1R%isv9Jp8IWwe$#;9ovPu>dO}hF)AGa^)nds!MghtC8W%9nZhhk_IEnoX_-tKkoNGzJKT5vG#_w%mA{tSd7A(z!`-T{4-uI{Sd%-eqb$o_2<iPhSrU7$rS%w53PIq?(*+L%V?)7`qNwXMN@^g7Imc<)#5&t5pWsRt<K{dSeT&isFiGJHrh1r>wxF8$iOjx{1!YKs>A_p)~jHNS-*7Vb7#>~@@C{(biM9AjXO^h`{_;R-J9;aY3J#x^E9)c-gfqT+kyGE1M_VM=34`EY13}ofrpg&VYMqrt-+<fMfY=L`WU7g{v(szZ{NK8`t?8E!%F}QpRH$cIVQ%kVw@z8AshF677ySKY8fDx;eUrlk*1yo*+KT3uYdf@_usw$*PB1NCvD{0`e=E%XsaVsv=QKH(OtwfEd3<adQ(?Xme*{I5xg$J&W(Pug@Sqx+O16jTMZBrr+Fp4Myi$@^t3nZGkoIueh4_(#<*vMS8Vh%`uh^>u$?RFJe`l_<uwoeuh1Hugw{17MBZUKI*yhl8;}@CK!Gd@nTeGAIr`-ayhtKp3_V3e(H>lA8t+npoZl163C{8*`AOp2Xm*ynv^YMz@{q$Ek)s_EQ*LK_K9fSlC~%4#eq*4b6^kfK@e~r>p1&9^TX})@(@RwT1=GJ^nwMHgPBTgb?F|#QdUR=Nik1N2krm+<{P1?SZ85?z@knFv5$Pym-?<zbAHinkbME6LjvX*O4rkVmACYF@@634tOmMy;Hsfc2`~pEBj+YuQUSdoZRzmc`xg%P41s^{;zjZ!>Q`{PhjO(2M2Xcy%Uo}eJ=Q^N)s;aVe$hH(UR>T_w4N^g?>mB3cd5g)?`Tco*U;)}O>3B%Jr}GDk{!0gTyz$YxJ+!Vra?%TODL;gTC<Ue<;37R&Z=@1%zYB^_9Em9AEdAClWi6tgWSK__GRab{6^MXhIEQX9MNW{SBqygCZ$!41k?kG5X0y|MC~kQ`UONBvI=pJP03p<3J_5#7`Kktu^e)H893bvwSqj^awUpO3)S`<)g711u$R6<pnIjJd#+3xxGc4KE^Icu+>K=K`4m~YsDLa)S27ipEj(S>7_=6}4E$&jrP=l0}36v!BRFrFja0I2nQ?$Y??H30}sQyHHJOm9&o2)c&2wfC)aBgt5<Dz`5q2Q9I#E$U@enK$C%jOgOSuO)7sh<it=jp377xno1gox6DITPN&IAf!xKO6_g8d=48DSHa3`7-DrZBf$NkaoZn&Dvx-*%!I0w93#j<EjoTGec*8j1HZ&m*?b|;f=c~o-V{{v5r1SSwJEuIH>@ARuyU^>TwRmA*B!C(h^yopF~W<=@iPQO9(f&w~$YaXAt6lg&siYgP*{ehT|(p6)tbW*?4@4;EYRd-4X*0fqNO)W#g6t{Q}(+(;0#kwqTZD`r)+a*uLswJ3-YNLJLArFW)#f_F`=^5H1IGYkYmko%Ca|->)m|$r@wi5OCXYlb`*LPv@C_uW@mJuHj;2F<y3%X=j~mK8$s{aL>!rN|5_k&SYO8=55zx069@cYq8yG-;XAe?1~hkQ7x&71HQmS1hOraMXO>uIm7QC)33=P)j7Xn84ee3C|nSQ6fPQUtT50J3>bF1R|CLMls1aZ;?Qu5)CMb7qS67DT9ddH1#i7xqHxR;4w2h{^lg2&gu5?BO;uA~)f-hpwFMqA#tKWfurQTdjn0to$?T)}+pzDLb7+ZU|G%I1-`I*{RX$V*Ob7J8g)_|KQ}cLW&!MGf)zKYU83|#-w&7l+o55Nxrp-$>iV-eKWDIE;IGPG_$8$0<bWAW|U%43oc?EcHEUGa(T-Sy)mCRj@_$9A#JJ}d)J9o8<(etbcWC-)t3Mj9!0;XOu=NER?Hk7d#cJp@6W))CTg|j^kBHgivIdf-dIvKXu!g~cBKImC)&@jLx(S`%Ryd%Sa_S!I`@b;K3jQ8lV0WdaPxg}cDFrj$y!fe}1Trf&h5I8<auV-O8n;3{$HZ|R?8i55YRV`rC5VDx9sqb>TnP~C5?Iv|OZg~$IpCL-Sy!`TP8hnUW1GquykgJxDGG;RsyF_8UMdnnIhzz?r#~t_iE=xrDWGw#Bu?GP&#~yLMEx=hsRVIpRlh@USKA<Wql=$NF!KN7`eR%Fhy&SL)R;v*vz5bV_78vk4ITOldAIc|HZ}eGRSPE4^-t;vw7#Q=WEc22e#GAQmbHmXMa91;feMgU0kH`!(qTTKzFOOuhhLDK9Ro4v<6(u@*+;049kHjk^ky7MKC<!2?T#Ts{<H`(+v5wO09bh%;G8!n24$mGp#Zd_H3S2(eJ$Bf~P1nc}u#fVX#Bv)>M_1P`Z)S(8j^l_J@L0gkz>2sIbE$9DO_eZoKEJ-ra62$?u9ZWE{2Z+mu<GX{%mh^f{&V*0s%+J>T?S)!NNCSH@RA?BjwA-u^B*GJNm5Nq`3~a(wcgn7&FQ0Z(#3LpAL(bKXt7LF)HCd<sw`14?C4zqc2_Tb7W&9{enTnrWHtj-E0i71-l?p9sT;cLQ4$mwu4)l~@D>D@y0egFq@I2ReG!kUnvc10OUIn#=FR)j$X}u+`}P4PowJANfT$tu@pb_n{LkGV-@S4>#<p7Tba8$UsH{sAJ$OGzpvdyTcUIN`Yh(^xJ&QZ$un-u_R8d4~7!!cH=c(V!0Uf`aqC_Wm)9WRy_*=Q-Gcuitcg}k4jRIQl_iJs~QB+K<jTKv3x+_akQfn3;pLKExB(Xbb;CQ)<OQqIW!Gt4ha$k+K5)bsH5nRNvNTaj(840m#Tpl<Fgoc3FEv&_sVIwhz7dBfDEZ$?4Nhma)c(%Y!ZFGFyD_XR-#?n8W7|Cq`mPw4JnwMBh`Uf!-$_~}^l2%~-sHg+_M?eBgl7nhUrbNxEA^kj2tDaX>LS|4R!;B3@_T-%+{*LapQaocR%A~*Wd4iCwXKbSB>hE)u7q5N|Bc3(aad>@_iMA=rQC?{KkEnGyANl(tv9kFScig|YGN4qjy|#flI!=FZ!(l$33!iQ;__j!nM%*SDya#N`r~jZ?Ns)EZz=J&10nV~Y%oC2En^o`^kIcO3P2kL?u*REB17~`JpQe+*xxT_r*APf&x4wOl)?NkT8$eA1h8qV_oT?(C$QeB#eU*oM*~xufimY1Y<yMx>J*L;+zW;&3?+*t2Mfc(nSlIrYR{prh{Id)BamhspB*2ikPbX0D#ARMYdcub8MbI0_&OWK6-taU_1NZ_J`!W6;0`+2-LM@Z6y{m~nN<&m&RzGWKaMEibeUWP*)x(T>b`4xbTo&VHKv*v@yG8+Uvf52}{NePV4`4r0nBL*VfZ5$-)JYTcr7cSY311Y$s!-sSMVNi?vR6wQUm0_FX*1t8TYycmpn2)hnrg0>a>^BP_Fm21tbhdBwe?0fPYWoT1?M5fzT7oXLsx<aek5u#KpMbypag9NEij`4n=|f%7#w#&Oli>I*)67fd~W;WC@<=!?oFvSf~SxjPA7ivMiYr?;qt!fCqDlX1q6F2GM4StDJxUZR{~KJ%LhE3g%vtc#hZheEdcgqp6p_s>o?Un-OiALPJ>63Sl^%;2hPh|%tJ@Z)<f;7>6lX0j&hoLP>zXuafhHg3QDBHz@ZzB9;@82W#0JhCinjfAm*@zuVA1|5Tt#B<cMMjpu1a~cP_d%jZuB!0Po&H^Z;SNlRng=XwzOP?DH3-a6M2X+)c6On|AmL+6grmcZ?SO%fFsmjInE(<ZJlnjY-T?_DWPX=+ax(pg+ek;5mT972<H!!J(zBcQ(1M$KZtbvBL2_S^yU#6YpamNv`Y30U&f8qyP_`X9x!mrwJ4mhd*}!eX%blCmpAF{I+uIkQzn@HobxK8F_riw0_x7H&O|8q28AiZA{L=41tdi=^8X0#UMjY@JzkC&z?nB<7Zk{>h8+p|C8>3npsdnWHqEZ`J=nf2*Lo#C2a?1_8m+oUbL=H-?pz1DR5lnE_0V><XDwlV`q^tv@@(ezYH^4W;J)x3&SPgQ2&m)XR*RiFH>1;L{__B{9horc1K%oZ9#9vhK1W)0p&=xmYKOD!0p!qAp(%D_(0~K|J%U=Pj86jolA^-uRreGUo?t=dC_q%!)6E(Zekaf`k5TcIeD}Bj)5)DLtj7e76!{DItE0lnZs<TBb8M1#nQLF#sKV($R2(rP<+?z8EE*_=sZV{wWMs44}O88HB|gr0FKQUEgU0pv(RKowtO)}$+`Rb``6z)TLtA}wsuJ8-3GAVy)v)A9fJ4ms;4a#9x=C{0X5Dz_*@`StCcK~Km5&-_71&i=kMQMG8cr}Ms(A)zob=1&ZWBFR58tT`qx~^;qW&0{j=fdrhL#tx;ZVFKrwajyyY2?z1KW$Q15ml^d%bR;4@eHUhQ4%GlFOVEjaYP?A{Ng-M;8=x@dvpk<{S#gt+YxIA%Pl@C*tE=VreSW158v_pcFl;M^r$^|8BWW}#^zICF#P?Kzgz+*!13=~qx;`&IsO5*-L~OBj>9YIpxBFB3kZfh>NV!B+Oyn!vuEJEGASz$bV(qrU-j+uX?RoGqX9eW|9g&&L!4SL0gdYlcEpD)UFrIthE}yg)R3b23k<_n8=kka9>-V<^x`+5koxv{fuk0-x(C%bxTsDzU92*%RF>i^W!na~TOZ{})h80|XQR000O8001EX?dAqoDGmStlq~=N6951JbY*jNb1!sdb97&GbaiBTE^v9ZT5WIRxDo#DUm>U{Af<|8C;N6mU3716uU&L|x7a-t1q_3rCEDRdmQ+zr;tTTM`wU4*lq@H??RGzGY;uMi4(H{Wq2;zLs@nTi75QXc6<e?5^;ITUo-`Am@YPJz`%<W8_zV7K;wQcps^p27Oze2R+m?HeD=#mbp>kgfRhu<FefZd{@L|h0!meELT&;_0E2?&bcOO4|(j=d%A`waz)x;2syHZqgEAm>!HCOEBb-WezRgt!=<||w6nM{xJsCt*~+tXkJS>)gE(v7G;3!YqwH1fU>iIosc)h@5GMw}G+T5cNFzlmH_yax9>J#N{|mqTXAk%*GWTLrfxpM~0G^(c>$sup=7940eV|5X0=j^sULvws_wj~Tz=pUv20GD$M7l=n+<Epqvrs4gcS{)s}dnuen8Ba|3<tKPSgr|may?)iT{Q-ARz?>zEmkvE;_rBbo>7)yATF%~G1ts}1%cXj0PG-aw3iR77CM9qJxb|N$q&_B7>q~3h)5o6qn(vfH~-1yvEoG;r^kts1sbK(8T%S0aB7F8O0fAlV<9?Z?7ub#~HZm}veSqEXaS|#dTl@BdFka@YQad0&c7PBP@dw3!(iMLpW*cX&xr;=18MJgJFXyV9I*Rp&<Csr8$t(WjR_HluKHGUPZ<sGKPgjUx^9n{;Bkrzka7JFrO6>GM0*FJ-GeN|U2pLgc@X(exjA4V-fR^TpKRTTAnNPDw&2{)}n%!j@aZ229_#cgxO7Y>`sd8w|7nu!}x?dLORMhH^&R9n}%!?J=fkcF4561Y4lMyUC0IyDsG#ORHDsyTIFlwhV(UUN+JPK_a7Px3r3YArfTWd$-#4HA_UmCQG+BT!=;@%9@32PLm4=$P9y0u{H~B#oVs70MJOaxFzgwiwLLFCy>7%U7;LP#gr4j)yThI<G3e4|=AamodU<T5Mxj0>`0Qm2ZM>@i3NoEj9=oeLngJ-JZXE8F|5XP%^f^2qQ10ct6LgJMg{fv4oSuPg;eon%4e(xAQ|><{Q6%npp805fD?i1gR$GWhwHsFXH`yz`jiVOGk6j2nB7!o=FEHk|oaJ<|8!!*}*13<NqL3QpuI@>MN=2N&FE89ZZPfhG*x~+3Nw#!$5(Z?LdXyX^m_O>n!fgiJ_teeOf+f)4^A=8B*M`+rcMK*uYGCth3jV=hOK}EKv9lE{ki<;CY_<AJ=*->j;^ZaL*di4oOz4;*LbQ(1``YzZ_dgi4CQ7EjdwJ^4YZg15S*+UM`PaLF<Q6qfY@8LxD>Kq}k3u?GlDH7=#GK2AQjxBTEBdm}`#-R{3FkTgh55N;3hC#3@jcaux{99tqbtKpioHj(*V$KKlxr8#{7CbR(tJVUuJ29~x>-_&fzQ)S(;8TU;~mL7FaN=%m-_rG+?wlR6I3Wk2F5FwF0LB&3RxEEB1Y5pNzMFR2P7Ycqc(<mQU}{0%gCE7ST)6LsF*?MOH}U~DZj!B`mMEJcP07{)e5081R^tr!U9x^E2>V09YCPW0I^fG$O2ZjhiIvKCI(W3(pmyVqz%F+sH6lQ_Osib|x3s|jl+$(egxR;C>gr5$m%-%-5lG~bu`4AYL>2SU=ka)`pweUB&NF>#)xE`T$P3RXOUVUZj80)|up0Q=%jB)i6qN+L*Uc*X1FinV@x-qTP*=p8E_Liz#7I@6R`?@$wF#(|k#nL!TfK#46^WKc-!@Q6YMCgX>M@QW<$kuA<IDRy4Yrb}>;Du!sZ?=!<f<`8Dks?q4^;-?}PBk>(_8mQw2c8cXRv(5@$U$}_ykZl^jd5ZWjy%}#38=mah8Wjv(HSX<44pbkFQ_K7YH&<wsu+Zssd4#e={3w)2S;ki=6P~13e9Ov8q%x_&B#Rw!=wltwKlFzNXT5Ksj*0tC9#{_8=%ynFpbWgl-Z7(uFn0`X;uwaZ*4aX#R3VGh5u*}R;K+Q>wwRZQ)cBqhStb&_Ch=_ruyx9ka!0Tp@mB@9tf6WqzZUzFoXJ}BvEn|-WK;GMUKfneW@=BOW067OdSN}<v=D&I-S$C1fGO0;k3MOD?!TXsQ@@|l|3AL)ZOI6{wUIf_gA3l4RB})}X=`{rs0qzSMwk)^&C~*&<Lb#k1I~zQwi`zI%<<qpdxMuXLpXHMflP<XnvEUB#!4V#qrh{7kF?Q(9|txcnwSXPrivoFR<${Ccb$Q+wn)CGwnFud+M`C-QCHd71kaTc)G!(-2pSp(-b<H@>U%3uNIb9HNh2W~t)OC_9I4pVsN;d4h_;y~Q%VkIdn+Lv1df_%zlHzq?{7bRw$xMLw6vY!Nxg&qdl^8Joljh3Wym#_H%D+X+G87=bu&)Qp*>R=%~R!fJq&DQ+XS$as<;bUxwbekKupTIdcEWKPEcq+OTgA}L9~)7U_fxfs|_yq<mCF6KDkW{E|u`!>hoZ~_f-Yl<-cq0q{F}4ovJ-xM-LZH`)zR<NTts+*VtxeZMFh{D}&{=;&thKOFcgJ&8*fgY0Rlp(bw6u7AZrT2wHh5j#0PG321?lylZK6qG)Qecj&|z+^kusrYY9?tj60&j;RVXM(vnw`1O~I>5>pDb^QpTADr7=@mWsf8uEY}YUa8Pvs?gvbDBa?g-n$cs_t9%?_qgdWOs!oL_loZiiSN-2J6xiyB4BUOy&)PPFrgZfZraS-AE-@S%>7lE^oCgZhM!*ht>!$ZQI+Ru8&lzRO2iojpTB-%A`UFF>DjlH0_XnS2mCr*mgo!LC#PR^#;y+T?OkiP8WibXOBdClZG^=zt>|&iZTERAP739guAK0u=jg~rV$?Ex+)m9D5LgVp~Dm#Ukyf1B`7?R$c6;YN0&<>0n;$BdZ*@=$gJ4cWnlh9#|jr4U(R&2IcwUV${+MCU3Ygow?`VbZK-_RyRq)`efu$&eq%G;qp5W|w%&1n(-`|Ots6G0f2<suh}Lz%x4gQhQW_wwu8`LPe@(y#){cwU)rA3T%dqq1RPs%Z5P|x&l%>dIF5ugAIgCIzxwG+cs!$NRv0M%0YGr5my2*uYRUSfA<Qyx(<O#3WypA-jM?(~M_)+{io)aVMw~?3e?JDJfe$>*?C4g$0PFXfYp>%w$i9mY=k~zGmsc=|v1FMHKQPFrORQuh@^m@?-o3_<hpPX=Lxi>X-c)UP0dC_!QBAX?3t7W`son5{`6uF$ujuowsm#Ml$bxTrjC^RI)gbHRM1&Jfo*>|($!$Ap!WbHiv%@1CdKrnAL5A9I?pPPHT^hlb}DOJGVD^wa|3+Y9w`C{Ioh_#WxwI^dBW|eU0bNU9gn`sYh&7}zix(D*Z>8H={e`pI*4+H;v`{f@rV6}&l?|=Sxs}w`>;ma2qu__{XdBZEobKI}-GJ;6dWwd+6vKCtvIQFIEA(;uqEUQ?zjkC#$Rj~G*!=B#TJswU^&KjVp+mD$ah67-&{lN*H6dFtwNS{bFEr1(0&HEuuEB6ucNO98Y6i7R`IJ~l_(0Bipvk6Jwye$lzLcI&NJtiwzGrJfuWgWoV8|go+uA4g}n)Xr;S`r7l4BH=ikHu1_l8#D8Tmi(4O#f|bjLsyW&b6pC7}E8It~GY*+twN1z~e^jTPOT(S;7(`J=Id&o#I&8j*it>x7%HJ9*K&o+LcsRIRN=q0!`SIsx_&CzA7m;LLc)dH0rxI@{|tTV#k5nEB$K>CKcnRF~otSf$WRYG=%IHdO6lhhO}S`*VNA1K#-B6js5JSFL;ch<l2hT7w@(z@(m+%g>A5+VhYF~*JCTa<cjAT!V|yh5H_p;Mx^-{$F*rCJ2D)DeDpwTwx0WCQ3^l&>Sg;6AN@TZd(1Zno%<g&Fz~oXMb$p`-Pd79U+K=i(KCYl8ePF)6G=C4psnAjr4iZe%5#6Up1t}XwBw&a9cQ~-2L`A3_oiX*cm~9wfMaFGbF$i8)bxDA)~Hy%q5#}-nVUFdpVnQpZ;F2Nde+h$c||*FLSc+eDclEPIH)>k^I+kho|=v5;B^&Pqrh-^1=FK)j1Ehnde8`=5<?d@CceKopN>)_Pz`jIBfs#>kX~H88mA@Ps`31i(!-tH?zSlK0n4atL-bHg=e%53{$(Nay6v<*Rwh8(6;`3UU6=;HMvZQrz;|_lsE0CPofWsNSc$!IAHQ1B*h;LaT1y4tUJ3@f5222i2+i)j62Z30wG~)nuz+RqnDMlogJC)Gb#;Ti6)35#9xiE-@CiP?OJ(h_IKcIr)+kQLPHXv~Pd5S%dS!{`>VnlDyr1)I@6}hr(=!16yFBI9p2|`vY_$U5^M1j!>FIi%1<=pbAWBg=iy;6BROAXUDU+#JqEev_C-vi$20Mr($`go4qC)Hh!14<H0mkQT;C}HKCVk>}87C*!P=e97B%_*GrsGddYcl`&`RJTnTdfe<Gk)vqV>A}r064ROCPTSBx|T+`v+1D_Bad^07wZCdje^0h-i$o;dk$MC0W@2a=zgUztO?tfjQfYl6Q<gkogX<ic-+FBpV$rjk_^9(jXEDVVV8Ytb@84}XM>n+p{eZ{<HDM&xg1s_ZA{foEE6T$J`7vSisBDhzdz)(jySYL8|$ONpfQudsLm+Wb~ua*ucREltHkLve*M_=ES^3-eKE3UT@|P{2ID=|56q+gW75YomlsTAazl+?)^?qaYpzjNr=Y$d%gKKMP)h>@6aWAK2mk;8ApnLpW}luH006>00012T004AvZ)|ffVs&Y3WM5-%Y++(wZf|sDVsCG2E^v9xTkCS$Mwb4sr|5xyEa_07WP9Q)9d0cxF}5nYElFi2qj+HuXp)FP06_zinQW@&*FM1RK4G3@zjL}9Y*38tsY=zZDzOCg?ew|d;CbE~naMJ&q^MR>l(NWGQsw35a2`f?G8S20$$6gNy%H*~%ZR3|!z5E8$;9=hTIHEo=W(4%<$IoYaIh%zwFrVmUDc%w0+FnXysSi+VWY50@=P5Z*mvc!2umgHGj=UxRN2QLRi4?uRXs1tJd#S;H<kQQJ(OXg4=loJl_qn0+BF_=%mRQExJI}HB!dHiAD->oO9in^lY3ci?7|LDqMz5J={PvKy0|!-^_rJ<GukDH^7T5Ywu7ZqK%5LTp6tPTyn7&1B^?m{4<I-=h~+}e>m-f!A?|V_d{CF^&|6hip$7f_GO1Se+{dK;B#c&J`gWuGvv7Gh%HqSRT&`tS9gg0dk9^GZ^e#{Juif*z`r@F+_IB+#x1M>{FnRC9pN0m$W<_2;S~L$;dx?WHee1y%Ibout_%2LqIWEh*?0A=k24K$EA~v>cp2WCcQYBJ}I9V)Y>3dy`HJPSL48`rc1AZo?uEX*!&L6UlUhJI8v;D9rWEO8PPd~r3p6JJ;$w9R#q&E<r-P7wmp177(7>8AedB6O+XIz7UQ7a{WEa^TLK3`R3QgpiF3*r5+d%~~HQ+IpAD37IC-2<7$Dv7b-@S6~9-2?Zel1Pzffh64O_YN8iQiq=jP_qs!v%B}81#Gi%=P?$lax-u@vp-*Kp-3_Zik|RzU3bq4PkxAGQHj?8Wz2s<mHp1{yo*&+Wf)0Pr5~Bv%<+pd$*KTHb-Ev!Z!uEV;Lx@F_*~-p-SO@pz@Eeq`&d(ckWo#@fm{MUOufuA`3WB8t-nj;1MR%X%eD6_kzO;IAl12_&-1z}>PlhN?YobOfuZlm*}M^AvSak(iI|P1;{K)h?eG61POfi60WBGpY{f)hjE;|vg#1v*G9g{4^e#6=nujrrX`&h>FREOe<Y_qf#SO0V^eJuj+kgI-nADjF)AZ?6u}Df)^~eTiLV><fB{l$BYt^TG)QclhfnOboH?P}^eewSKEfn9-3+pQP-kS|n3ts$3r;@6}AzDpWb-ba0H`h15xBx*@)hCklBPe{S`miT;ihamJk-aAmL4|86aA%dM3bz)gB?h%!%hOnvB8<vhDZro<S$jTFdjkzVluxkqajcNpeNyHbBM=UOG{z_@C0vqNq<MrZZeH=x4La{+naNZD!>L@5-%y27<!k)Eh6pWzvOG&SZ7L9=&MFb5;77$9E|L#8qp0U;5(%;;X#!RiFvrOPil6r*@Qq!C&qGMHhGI*4>j4@1Y0>8+$k3N-x`NNU&2XxWlPa92lIEq7)@uqmjuiptY(Fo<ELy#42nUM?(kdkzvqiXA8b>bH9tOcGKL9btj`|QpP=y&n8j0ZDYR$3Op_D(>NhxEW;UozcDhms>$}1X%PR9=6Jx%~aGvUPpyaYaK6)LcdA;JiJxW=s_u}JfWftZ7icSM)#BN5iPKQr+2qr+rPz^iqBC;8R07tK4c0C0(yyd&-#%P0khz(TKp-W2@DOQJl3@T@fsJI`@N87rv2CDzBAuANhrIm|XaTpt~tYHECwWMOLX&a4YsUqA0(zvy58*}R7FS46B>%r|5RLkNT~MwY#Kn|U2CWyP2wnah_%8^f|NG*(a<iX^Ksi3pfVB9*FeS*5|5+ItbE3@tRHy0?#V@F!ttS%+oJFo)o>EY8=s4))}|^0|^FQ3{S*T1PZO_yNn6SWjF9VOY>QDSoQMlo`IKC7jiQz6~hba9mg>@W{15)ZkMqFr*EPK?JtKWtQUxkrf6=P5~E6l7+#kM)A<4nRlSUp*YHjt{g%jK@cFou+cX)u*SOYK_<%;bJ;!X0Wl}$s34b`G6Xm43RZHePAcgWvz78k(uA5!7T$HDz%HcWuni8EpkR7u<p<Z$q7o(K0WQGAOO{JuUxZ54YX}H$2s>e1LUN84TeC4H4hKlk^-5MmP6ag^0i=nn@$z~+nVwB&<I7nvxw@K>#2XC>uQPPiB=0vFUta|`lXEyZzgRc@Po*|r<nNxJU0R=)rZ_IJoi!Ox$D_%~TW6Nm33kFRKttj6iCckyeoS)Szu2(6_hAtarz>JaecAJ?lhOI(MOCN{?_VB1<mH{7#|Da~u!(A!=_i8T!Juk$CGi*d3;(Jp=O<7qy3-Oe8@=IONUiMdLhmqJ>zBS!Dt)U+`c_fszqC5&y9|pb=0E&~)WXDsDIa<<`fD&9Pp4;Bm%+u^<;`q71qjD5#5lA_5!tdF=fx|AJ0xx)1xeO(EgLRJ2$d~~&D+Xd-)lTRrA=>-dg8bzp1m_<`?s63$v7Yh2);Uk*&hmatogdCF0`2;KS5Xz;mzw;Vpi5TAtSK@i-(mUcd^o*gI3Qsv#VhG_GWf^^_NR*$_ut=tkp_ucO$~a-xS8*6yXBc(YDp7!GD4<gVY-8n@jN7<u_-Oi_r`?GyA(^v&q%@boc$~+33yX)pT}t@@VLrvrA&K@x|!u@^n10XlQtQ^4&PNzPcU<XBXg)*7DO4ICjj;8Rb!n^PY&&#q05;Pb;mu)f_go^k-!UTERa6PZ_@;hNqY%3+NuwA7DB@X#CgNctT)aj4sc<8Bgg*@_gHa44@U4X82+}dwZoB;N6Z7`$WifQEf1!ZF!O*CIw)}FJ3iVs?{;9Vx7j;d4vn_Ge~C@s%mbLZj9oXK_5)d{>JFKWv1Jn2>~JV0c8kv%UmTQpdx{DC}G*wU<YWcxTi&v{$NHSD{t)2XhzzEMCSGA<lFJ3zMUtb_)8CgmWqqe(`<e|n$AGJ=^0Nyzj|{vYmsnzeSS6zNJQ}V+0nBvkB^=m^T^57cjE~mbar))KQ2$e2G=ynXW2wpL&gsFajx>Tu8gJ1$od=>Iow7-3|IA@>`J-Lv*c$?1_QhT`lS3>TJvDSjBPahC;}U}5o=4l+TkjDX3IQZrqYinBCv5{tmVyY*jmFuVaJ<>_oO|@j?!=AewDCvswJmZ#c;r3H}rRcT0YKaQ4c`apw8A5t##V7?BY6u-Ak648E)IXW`*}e{A&dUlj1uH5Av+-Bb!LwI;<2^H6CkE#5tRF@{QeS6;bT;>SjWO1sfUV?~KA?l$~}~sOv8NA#3kuQj9+QHNC2{KWEn6#)>A>LF{NVY5SwaCqOxD0fdisq&qsXgVFlh9gn4EB~3$EKRABL&)P#<ha2)CA@uAr5X&}nAh#qcIdK9Vpgj`y$8~xw(<W|N+mNrcoa>9JP2MtW+80!8c(tESptj<kc@D>(fDX%O#bA;u<V@uvNgL4sLMwQf#%-eTiZi`Lm^6%NmDV@BhPj^uFRh(+A}<|waDb@;P&O)bFB>h|pwj;+9~y!f$;u%fYJ-0$pWJgY$vvAj#>-V*B`L(n${s^{rt{lALI7N*$<Ug|p13e9&=YTG7v~3`mhXeD=xK4=@v_WyoalOJVe~1b-P!TecBlL6yErL3<U}Ey)+`IA!(fo#@uR^PX>IaJcD8@yzuAcfFiPOI#c?ME+Y`IPJ89i6C22c5UTbgHLH%gfb|RN<#lp?8u!|)*9Mz#chp_ir8!4*;4M?SoWKs|g_N#!MoAw&4^&E`%8t+{14f#eeWv+{@+h;vOhYr=0AFYUP_J9W2s&ZbkeuON5mwlp|lUy}#{DiD_)qxY)TG=&W$q3=-Lp#J^byIRs!v5TM7doQXDR1Jy$5{^FWP3{UBFIQ$0U0?O9RkKoY+FTl@(@}aEN1PeC)%6&&?)e236hRG4-{;h(^#g$Iq=rm<DIJm^e^za^OCqe$?oAEkyjIFjbkjFF}c@m{WO5%vctNq>EkV`Ol3@yo7S}$1OCr{Y`sZRjf&=^(iRZ548wzvo5u3G)t;TC`{$UbGk8MU;uZ-9E=~eY8zlq!bN~3rKmL-{zK)h-49yTztl(e0@Q)EDRrGY9kaP)5{%5?T@EgQDxj7y2ZpY8?@?j+<%oRmP#K4UP1~0M7PvtU<Ht_huOi}h1;w4`A&;RT{qf|iz6QFDoNM|+J5?UTUH0eMRW3UZt`%12bhSeA%riu#~InNnGPPvUFZERvDVPJ6r*_$=%55pol{ig}Z`>^u-EK~GTO?GqWT-^sUI=3E>j|lWotrK#J4-a|b;j*ZQ#KXO{T<7J+uX5N}J%P{@R>Mf$_p*E?!x-)~)NT|OTyNF(-e`WRa%ip3%7ks4|FB9TNR@MNS0~t;Xk-O}0|Wq9gRN7jJkH#}Z~ao@z>F0vr1^+PK#jDchgB5=7F*WQxRKB)5rq^%v7&eqmcRoVJWlnSLm3p$WCA<}+@ubg+yPJFmzv8N1mR&d@AT*tjPWUY>j=|ap?w>c@dM5x0o(`rW8lEzURzDjIkbM~XcluvzGw<T+vxY-1(lF#52e2!X9DPL$D?P@8U`17Vs(OT-;I2@FL2YL(4g*akKXC$j@R?Lw};2?+S7POS9~pA9^vPaVV-EkY)tcvOi0@YZPJ*Wpdip+QLx1**A|siCk3VPv`6V6tWVfVDTh`UzV~@(BV-uVeH_@S4wOR{bYoz8gtywV*U+$GUnin;1?3-sqvLpTeaqF_RVHWD>-vcbNW*tNfmq5Qw<}5<8-VB`Yh0U|;VKg8)DRvgG(RQwRo_6NnE6s&k)bk=#W<4&PJA9|*vt*Jx9(V999)4p!or$WC{R*54#iym+I_FZMO&8Po^>4pt@k*Ah$+f*T(3tpzTF=upBKQV8n!9mjP{+=w<K-SMWk)kb12s>VLlrlys%m3zU*ieeEOnBVP^NhR-8m|tM;O`)9s1t?`LnXF2S#FDMd2$j<C(_>f7<<+20T?z<r&J&d=eqOfQDsg3@M>tG1Wtvu^{s`jGs_Hp;TJ@NmlaSLatct@kE4J)7u>ZtSp41b1m!odPlGhA{1Qt!Z=>!_cWPd)p&}5=iw1JAn=m#@T%{x;=|BNtDj<2AsIt-pu1N5MEPewa3-wRZcHVt_`o|<kbW!`56w@z{!U>n}ij?M8$0GhRSGz$C)jHE<}vQty45P&6f#PrT1mFoZZ*qhrmR%!8*z6iU8S@ZN~majRqkp{J|r0;A{%!4BV0!X740qd(9+Hf;SU5E4#CkBQScNlchELaEeh6?&N9Ry|m*76jArUo}4WalI%@DbOYI%oS(Cat$YPlo=Pf-Vypcrz`g)x{3B;A={wjFZ(t<6N1X~feHz$wt~Yp;#C4}}UD9yio`kN;Ht<sdTkQ)tT^?{b!W(GM%xm0y?LB*gc6_kiYFm`@2EQ=JAcg}1z>J{C@mEJZ558_ebD(AL#-}%V5fskfciv!s6NVK(gc}A9Z&4h730EM_0~Jz{(tr%)XZHbVY0OE}@IDOgk1<cDs_^bqs;ErnCXQrW`JS#W{pyH+KuY7&c?X_rjO6ZWpOc;edAA*!;m7s5P@R@~=?UH0Gkj*04`*$`t!$|~{IA=dwBwhvc4JlDC9iw!F&{LY9zI!t`<H%{8n<1a?e#e9-a^{#VT{l6di;w#^@n)am@}yIA^z|23nxMI2}|@j;O;e48d_%I-?u@S0i|ze{jNp8G!578rG&ke*uB=403OxNygw?ay*7ozoKvOSBDSd_lT)PY!dlr5bY<WJn6#4VMjXHV?eG76{1sI>@`VPPs{=|DX|3GWRny+co+CU~=5p;%U7lZ40Lo&mUg@M8W#ZdWK4)w>-C|0}SJb8Q{D@k>lC0SelNORg?jHM*)I4x<b#kSrJv%ynX$s_SPIMqrs*&qTIfcp#e&&`3T^bJ&aaXMrFpW`I$BrTI$gLps+9o~v_lbOPnl|7HLt9h`9#p?tm3h5fA>tH>pmV1mV@c&9r`GR_YgwAkj>+X4Q1MPuzX?JO(`Upfq^3=P2=QY+2SpL!`eJ0-f_~I^&kL}%O_+1vN<<UmSKAu@B23bn;tp;-5ouU1bt2ss)lF*?Is3_aUF(jVXjNx-nxIZogGolyR7J5174voO{<DIA=J<ef6M%ilNoF70FLZ|-yuy!06gC##ivJTarIsS*Ii{(C$T9rH`T1hO7;yR*7RiT-k)h**|4(}4^OeVr0^wnA?@*IH*f+?roHJyy`JIbKujuwXw|CCA>-bNfLYr7QcJl6G`j9|gDF;m?;DD;BmQ(z0I)J!0z_$)DIQ92~$iY8&X<GA%ZS~a$pTf%TuGgo360mKj`~C#}O?H<7(ONFrnUty4WqJsj4U83YY7}}Vi#vNHM;x~0t0#{CxLm0&7MhkfYP~6K5%_%kW_0pB)AsFXa&bk$)~T+?*$z8MR+UF^uwU8YW|A$a$;_rxb-^HA=5S|LYi=^K&6CtQ=!=uLqqB)_@M=sDlu;$IgtGS38fo^{h*zY=61C!VAWi1Zu4+SuvB{khF0?iCn;XKALuKc+aomiZGrOo>qfL8F_Zeram(_}EX58RJaMtk2%L*64nN_|-%mwXj^sR1INwO%dV+ntZ+{J<$47SZ>ipvT`m~f+;IbK&JxOQrq7bM+_ii1?)smUP|uL-AuQrlUy(KuV<K(6Q;KI&`p;}S{3#;ySb0N3biDg#Gb$D0eK_rL(FSO&(l+iI!#M~ab@w-u{uoqn>)c&33>;#9xqXX`~N4nxR|ZfR5+@tW$4j^9d7x#uJsoo1!KO!GN39W>GH;q$?}!AI2L6G#H-ulF9++x4D4S8=`&-g~Rdv|ps9ci0@=r#PHnYG2;12iXC3l~iZ9vhxUz_wV1+9zXn$5g>MTDZ^NY0{Yv5A2Q5yfm%XHbn@K+ihLU77um7}k(v4lh+KMf$g3k<a6ZJW?>eUw9pKwQyKLHYU@RIc^vw((+yT(r$Cmj6eQm<EHH`Cxq4?|)#M4>FJ&`{sU`~a`{+j>#7ebil`9H#`?ciOx`y%ES#Fjq9GAv~0p7b?i^y@wk(?$3tT>~)LvC@p;l^IcXNFV(Psy=<n)HJuzmFtGPo<8+<Ze};2f_sNg=(bbwTX}#vNEXl;Y(KVg(waWmRz6MlI^Bh|NZI(+{YOPpsGC!*(cv}2EwJqj+qN)5h~zNUY4v62^AC%8CZ;KmgB{b1%wvBc1@*qwgTx!Y@0p%=>KLQ%Ep(?JoEf*3$n>o6-n!_<p9Qu${~Xr{{u*&b({B<s4Ux7fy2=Y*u;n-X^wdP2WcSpZ$%v0|R#q1_84a^pz)hmy$#1NN>BVll-BWB&eYUx2isX*}2Wo`6ld`aFP)%x_D<W*}NN?=X*%rBvn!DO_b>Yj+Uh+pY-avm$<L39AFI-w&?CFXn0rO}~XR9=ub0M2PiLo`=eIMKa-^-d)pokTC-o9COCp}|#qn&SSnmq%!EIEIm(_T;By=kkUm(RW34Y>EEwQx3?2KO%m{TWPBY_d6R!VP+vPUVT)cg{A>pI$cCWlzj`vF9}<PxEp+_^bKUk$A>xasJS>gif>{o%*A4SyMLWnnyZ~Ly~91Ac*rQ2)fQnKaArbG)p?1^5GVakIvE6SlrWmI56}Y71G~>57R<YE)^%pcuW38o7&a{WB5657I5(hfclhrNbkX`Mun=|<9E(Cp6${zJL{%Yy!BLWJ^qf9a<%t|d@LS1bh2+YfdqMP$D>n^HT=Ju9`av3=hG{S<p{#<o2CeD6q+qB@>5gJ8k+A?ZGS|&6=%5TtJe-6WA0~-fX*#4iu#O5U*EDBwHD?x8>M#Bjm@o`yraEClX5)nH8mf^;EDMb(tc`*F=!iTr=eF@i^H!xN6%}k#F@hb^Z7D~PY`hU6$Hf4L0}><&D{t83s6e~1QY-O00;m803iSxBlf|D2><{j8~^|s0001VZ*OdKFK%yiWnyn{YhPq#Y-x67a(OOrd8HZKa@#icU0;FniOfo@>^gShtuo^@PV1Q_@zib~>eX@}60%TJ1Pg$^)a!rmIk=D@B{|+*k4F}OgLA*aQ52oi{DxjLl8g14Yt0J6?rSFbnpK*tMbVT@5pl~T(JMxl{2n0cW>NB-EU027uUHyI(P*@kVokDa+2}^HEF*kf3#kdMDxs<7qEe%g+rx{82Y$7pYE|-uUvlBUsb*12ku#<Ij;3<SOXin<@!Bj6>6&Up<LREmax{8>a&emd@a5B|vr94~(c%8V^6238bZ=46C40L#Dh}!NHDyQNy)E|k7YF<MIeT;T-BC2U`10#7Kb&UgCzn56z|D9>;3qoy@$~cMMY=AcvE3*{uD-oIxyXJy0y00JybsPaRheum4XX58oobcelI+ihOV-=dOa%)eN?atP(Wqcckkc3}L!mQkwO9(F$IgVwvAq{XADCux%`2`nSW8P<DQgm$ulOw+lZv5TDN!sJ6&SwYnl4I)Dl830#%{*Eh!UxQE>N?EVq$lh66Gy;#Vjybny*-XlZi&xjke0epR4Ga>u8Kp-%e^O6==q>P5wcme?QvqPthE<?~0j~OS5}DyOa$xc#>3U$?Mosce4Cf#N8eII+-kmTvI&*&w{m=WlUa$4iIo-SumfVGmTaQ+K*3W=Z@=@_t8|;tcvlQSo|^LB7Oe=W<C27JL*Y7s3J-<GH1svkH8ICv-JYZ!Yg;0UNdd?RGf66_`{bDI_0Y1a(L{y<dxH1a1WxgyNhgYyDTYJjQmE+hMh_&WE`D#$-#0*){WAJ3I&cOyKYJ<2_l5FQI!vAl#GV_pw_M&@rth59L;BC1yf6!Yr~L*NK(mY5qoEY>{5}e4FaSSdz_sCeX#5VwB)j0$XKRZYg+$-bzr=tv$IP+>F>P$h}^bP+I!$!ch<RUU`jz8&DtAh=h=tTk3XMWo_^>cN(jbs_SQNW`bGg+fLA62`woIC8<T=+YMn$v)^0^hxn^r{l%c0oO97$;OrOC{+@fL`=BZh{_x8<nO!mz0)cz*P2By?Ao#}^~g?IzeHgAD(nXG7;QLQE1nu@*Y^zh(-eE&V!JM6&+!l!qVV@8xQBC&#RsIAYVvE-F^EmvStwW9m44`aNaq$_q`@M}QpY--dwbH>AvyG5_>yWL|;BvO!L)-*acX?{{mehWIBSg)Gg?z!Y`K(X36M+5kvA_Xu#cLb>3I)pPK0KnH|3ho$Xz!md1*msF2TfBJ<r?w`iuwYd#3g%E2%q$|Ra?T@%_Q^#T=wAyc_KsT@Yq9$aHo`nCFcu(bC@4eVpctdI$QoI8h3d1=Y$3!Afe#Iaru9FG(gjv?Od(^kVBq&qmnx-c2`$Gp0d@cxUqWGX<WxE`_%2smk!4fm=sN_y=c49No)!<LgQ+~NYgREyfqzgoY3fqYf{0}g##We8R1EQ`W(nkwY)Z}`?bAlD1h^(RDHI@=4WW`l9#nt^Zk#^=MCH1u)L=Wk$7&1$_~yxFo!s<EASkM*q<P1%L+qqXz0S<vem@oL_?{}*g1#zP{h>q^u7wI$l7%Z!y6&yJzAV`F&6tV5oei0*K3*D#v)IQ{LcW>p3MT?)qSZr}WF<?NO<6+e&Y>BJuIOYroxGht9!#Gg>xSrW7&uTALA?cwd<I+gaw(i*5{-LNIvxh3%>qG6-7Jh}6(kH^kX!V0pi&GC7xnT?7j7=X9N1MaEa!<=*w^A6b=RVBSz`VizZaG}-#?j9Rn3~LxiU)i{BDb3-vs)zWNg<RguRPGl7O#J$8+lom`21Ms4GKvG4&`^Gb?R$IfCZ6J%v;t?o8uZLaXIrQ|K$TO2|912QE{P{{Gc;j`n`+m5VK{dV93lvbRTDTK5iP(870{u9f#2PaC5AXJ)FqVKJOGc1#*ifu>A_ihBed*8U!-1T#&m7Opb7fUc?0zS@PLQJO{u-ZXciCO1G^cOc)kpxC^bjx0`)kl)U<XrViZj(RfM-Dd^en27?@D>*mrJjXT6;2Z8J#pa>pZM}DKg`VSJO$_0>^;cUt=Lu$?zQ<sbmEy6n7A6whK_xk4osAKzf;_Z=wcnuaj8Ft8Yg+LoxYR~?Ix!$}<CdpP@xQh*g+cL#ydrzk{exGp_Fu?=W1O)2oUuXyQmm;3n_z7whE~ZTq(hcAXCt3*2P_?Ln@fU(&{MQA1XiQ;B0Kx|=`!9JSPXc-&3yN6*jpyvp&y~WZPER5`s33}CZJy{MVD<^c2msuAt&%YGs7Tv$xz@yW_MS2CwJ#QS3S+lU&8rC$nx%tzcH!xDS=S#WE7Yz4E`v^K|29EK)Cg0)?h}5blW#yJD%s*aM|I!5)+R%LQ~1OZ658itw3|vZFcVj-Jre{O*MF)=uz(|Q$0aB4RQQ51#H{HL{}AaTAK$iJ2(W-6ka=qdTR4ChRsfGv}`OHSOdR6cX`b#$Hyk+R$ecrY>S7tT}Q0#H)k)j!Y*ALNAPJpp<ms4%=J$;RWu%$$myzmQS8)3W-upyB~xwq{xEg9?i!yl%`i#m@D0<XS(xUWZ2f#Odd^ybgM!BvOjBdL7nzLax}oL&{Bjokbb9h((<PsKF9RhzqTkpi^tm4Ke4wK?#)l~bCh=P?8cg@SJb=x-d)UE&BWl}!2frmnh!R5;p;H{mRNTU13e8Zquu}+~9;+-|w?l<ZlVd19&>>9BnGcFV<$Ymf{D0B$e1v>8O8Swqy;Qap@?}x<Y;_aZ5i%W2pL(M{gel+9PMZqI-o)S?3P#*acvo*beMN6^($TB2&rpFa3=ME?9I)3n_?&4M%Ibm`U+IwgYfzkEzNGTOOd5a!yT%L|GL*~%BU&cyUJs%zx9uxaw!qf1EyA5wqWhV-JZ^7{4LK)W_b%sae~^X!n_Q6kQ4s_EbvkixPC4T>_7IOSq#hIV9_44&v0DNWKVLc;=omQE;w&AGl20L0D<=M5!vtE<tZNm!voRnofQ#Aw*t{)ffX$j9K{fuX!pd%+h+JYE$g&ys<33$?4WEvR_J1qJ1KRnq-2!w-Dhf7YH=Iu})Vi6sH?Yr<)d7GTim#6M4&7Ycro@f~j48{R^&|7>b00@3j3Gfw(>aD5HgM0}HW|!%ShD#7i9g$>_~F}w(f4jD_JK4o%~nujmt?D9+$iZzMt=iPO928D0~7!N00;m803iTyMrWsr0{{Sa1^@sF00000000000001_fdBvi06|AZPE=DaZDdeO0Rj{Q6aWAK2mk;8Apka3e`HY$000CQ000yK0000000000005+cu>$}AWN%}0FH}KOUsg9?Lr+XWLN0A&P)h*<6ay3h000O8001EX`qaB&M*si-P5=M^7XSbN0000000000q=7UK003opaAk67ZDnqBb1z?CX>MtBUtcb8c~DCM0u%!j000080000X0FCNN_^to|0O9}u03HAU00000000000HlGi4*&pVcyMKMX>Db0baO9sVRK)0H!ojbX>MtBUtcb8c~DCM0u%!j000080000X08GL!jpPyl0HsI(02=@R00000000000HlGM5C8yWcyMKMX>Db0baO9sVRK)0H!opzWNCCRaCuNm0Rj{Q6aWAK2mk;8Apq7Ho1hE`001!*000>P0000000000005+cw<7=mWq5F9a%pX4Zgg`mbYXK}b~i6$Y-uiVc~DCM0u%!j000080000X0D?=fTdEHL0Oc_N03iSX00000000000HlHXD*ymxcyMKMX>Db0baO9sVRK)0H!oyqVP|e{b98BAb1rasP)h*<6ay3h000O8001EXv3?*&d?5e;d1U|qAOHXW0000000000q=Dl*003opaAk67ZDnqBb1!sZb6<8hFJ*RNY;|FDX>V>WaCuNm0Rj{Q6aWAK2mk;8Apri4OrKW>004v*0015U0000000000005+com~I`Wq5F9a%pX4Zgg`mbYXK}b~i6&cywiQZeeULaCuNm0Rj{Q6aWAK2mk;8AprTtf+y?@002}e0012T0000000000005+cFlGP%Wq5F9a%pX4Zgg`mbYXK}b~i6=Wprq7WOFWXc~DCM0u%!j000080000X0G_sxvXuq^0BsTg03HAU00000000000HlFea{vHkcyMKMX>Db0baO9sVRK)0H!pB<WpH6~Wn?aJc~DCM0u%!j000080000X014S=O)w1r0IVzk02}}S00000000000HlE^dH?`rcyMKMX>Db0baO9sVRK)0H!pH!aBp&SE^v8JO928D0~7!N00;m803iT0<uGT)CIA5OoB#kD00000000000001_fs%&+0A+Y^WpZh4Wo~qHFLYsZUv@Vyb98lNc`k5yP)h*<6ay3h000O8001EX4C;tMA`t)pqdx!uCjbBd0000000000q=AvH003opaAk67ZDnqBb1!sZb6<8hFLQKtWO-j?X<=t>Z*z2MV{<NWc~DCM0u%!j000080000X03dy|ouv%`09Pph02=@R00000000000HlHVzW@MbcyMKMX>Db0baO9sVRK)0H!pT+Wp{HfaCuNm0Rj{Q6aWAK2mk;8Apn2nMLMAX007Yd000~S0000000000005+c*3AF_a%FLKX>w(4Wo~qHFJo_PVPY*|ZDM6|E_8TwP)h*<6ay3h000O8001EXvX8%%O921?@c{q;8vp<R0000000000q=B){0044jadl~OWo>0{baO9bZ){;=En;DFWiE7hbWlqH0u%!j000080000X01B5jpuPYA0003103iSX00000000000HlF6&;S5(WpQ<Ba%F90Zgg`mV{dF>Vl8lGa${w1bZKvHE_8TwP)h*<6ay3h000O8001EX<x-_@U;qFBZ2$lO7XSbN0000000000q=7Qg004Ahb6;|GZggpFWiMY}X>MtBUtcb8c~DCM0u%!j000080000X06X^ncNPx-0EaaI02lxO00000000000HlG((EtE+VRK({b#8QNZDlWEa&&2CVPkZ2E^v8JO928D0~7!N00;m803iUX$w4bt82|v*RsaAM00000000000001_ffL~X0CZt<UvhPBbZKp6FJW_YWpr~caCuNm0Rj{Q6aWAK2mk;8Apk-n{!CW|006KK000;O0000000000005+cocaI&bYXK}a&>NWX>DaMVRUq5ZggpHZZ2?nP)h*<6ay3h000O8001EX_O5jKp%4H7V=w>!7ytkO0000000000q=6{_0RVJib6;|GZggpFWiMiHZ*+5Xa$#^TaCuNm0Rj{Q6aWAK2mk;8ApnM{zg)Wq007Y!000#L0000000000005+c01*KIbYXK}a&>NWX>DaMV{dL|X=g5Qc~DCM0u%!j000080000X0CuwjYe*OX00vM1022TJ00000000000HlHL7y$ruVRK({b#8QNZDlWHVRT_GaCuNm0Rj{Q6aWAK2mk;8ApjE=mONMt005mO000*N0000000000005+cXE6Z)bYXK}a&>NWX>DaMWMyM-WNB_^E^v8JO928D0~7!N00;m803iU2hm8Xv2><}58~^|r00000000000001_f%H8A0CZt<UvhPBbZKp6FK}gYV`Xr3X>V>WaCuNm0Rj{Q6aWAK2mk;8Apo^MNEaFo007-J000^Q0000000000005+cOh*9#bYXK}a&>NWX>DaMaAk6HX>)XCZewLGaCuNm0Rj{Q6aWAK2mk;8Apl7}Z_zyi005r|000;O0000000000005+co>c(=bYXK}a&>NWX>DaMaB^>EX>4h3XD)DgP)h*<6ay3h000O8001EX$FhJbVGRHPsw@Bi761SM0000000000q=6Ax0RVJib6;|GZggpFWiN7dZggpFWiD`eP)h*<6ay3h000O8001EX{`zot6#@VNJOls$5dZ)H0000000000q=BYp0RVJmb98erV{dL|bY*jNE^v8JO928D0~7!N00;m803iT+^z3H-1pok26aWAh00000000000001_f$C@h0CZ(@baO9sWpi|2VRCe7W?^G=b1rasP)h*<6ay3h000O8001EX<G^H~%Lo7fmLC8B6aWAK0000000000q=6l80RVJmb98erbY*jNUtx1|Wpr~caCuNm0Rj{Q6aWAK2mk;8Apm7;S04KZ005#E000vJ0000000000005+c8h8N!bY*jNb1!sdb97%}b!2IDE^v8JO928D0~7!N00;m803iU$(6mL=1^@t+5&!@f00000000000001_fkuD<0CZ(@baO9sWpi|2VsCGBb98cHa4v9pP)h*<6ay3h000O8001EXO~V>tU<d#Jw;ccg5&!@I0000000000q=8R|0RVJmb98erbY*jNUu0o)VJ>iaP)h*<6ay3h000O8001EXoUm^G_5uI^ItKs%82|tP0000000000q=Da$0RVJmb98erbY*jNUu0=vXKrtEbZKLAE^v8JO928D0~7!N00;m803iTyR?ATH2LJ$R82|tn00000000000001_fee!Y0CZ(@baO9sWpi|2Wp-g~bzyXAZ*DGdc~DCM0u%!j000080000X0EF8O+t3C80HYHC02TlM00000000000HlF6n*jiHWpi|MFLY&dbYE_7bY)_1Z)+}ac~DCM0u%!j000080000X0Bl9%R`LV@0Ot+>03QGV00000000000HlFAq5%MOWpi|MFLY&dbYE_7bY)_1Z);y<Wo&77Wpa5gaCuNm0Rj{Q6aWAK2mk;8ApqG=_yJo6003nc000>P0000000000005+cXr}=HbY*jNb1!sdb97&DWpZ<Ab97~HV`VOIc~DCM0u%!j000080000X0NAes4V(r50J;?b02TlM00000000000HlHXtpNaZWpi|MFLY&dbYF0CWpH6~Wn?aJc~DCM0u%!j000080000X01)`JiSY#h0E`m=02BZK00000000000HlG+v;hEgWpi|MFLY&dbYF61aBp&SE^v8JO928D0~7!N00;m803iV1bT1H*2LJ#e7XSbi00000000000001_f$h2h0CZ(@baO9sWpi|2a&>NWX>DaLaCuNm0Rj{Q6aWAK2mk;8ApmC+{f7Pz003Gz000#L0000000000005+cvBCiWbY*jNb1!sdb97&GV`*h>V`VOIc~DCM0u%!j000080000X0PW@mS1Aqv0F*5N022TJ00000000000HlH9(g6T;Wpi|MFLY&dbYF9Hb!2%iaCuNm0Rj{Q6aWAK2mk;8ApnLpW}luH006>00012T0000000000005+cJmCQVbZ>8Lb1!0bX>4R)V{dF>Vqb1=bY)_1Z)+}ac~DCM0u%!j000080000X02(9q!G;L{03;j$02%-Q00000000000HlEx`2hfQZ*OdKFK%yiWnyn{YhPq#Y-x67a(OOrc~DCQ1^@s605AYB06Gl-0Mh~j0000'}

def restore_source_package(package, checkout, persistent, repo_url):
    """Notebook bootstrap; stdlib only, before installing scientific packages.

    This function is copied by the notebook generator so upload of the notebook
    alone is sufficient. Its authoritative source remains this Python module.
    Existing baseline files cannot be replaced by the source package.
    """
    import base64
    import hashlib
    import io
    import json
    import os
    import re
    import stat
    import subprocess
    import zipfile
    from pathlib import Path

    if package.get("schema") != "tas-private-source-v1" or any(
        re.fullmatch(r"[a-f0-9]{40}", package.get(key, "")) is None
        for key in ("base_commit", "source_commit")
    ):
        raise ValueError("Invalid versioned source package.")
    data = base64.b85decode(package["archive_b85"])
    if hashlib.sha256(data).hexdigest() != package["sha256"]:
        raise ValueError("Source archive checksum mismatch.")
    checkout, persistent = Path(checkout), Path(persistent)
    base = package["base_commit"]

    def owned_path(name):
        relative = Path(name)
        allowed = len(relative.parts) > 1 and relative.parts[0] in {
            "tas_runtime",
            "experiments",
            "requirements",
            "tests",
            "tools",
        }
        allowed |= name in {"AGENTS.md", "docs/TAS_V7_COLAB.md"}
        return (
            allowed
            and not relative.is_absolute()
            and ".." not in relative.parts
            and (checkout / relative).resolve().is_relative_to(checkout.resolve())
        )

    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        members = archive.infolist()
        if len(members) != len(package["files"]) or {
            item.filename for item in members
        } != set(package["files"]):
            raise ValueError("Source file manifest mismatch.")
        if sum(item.file_size for item in members) > 8 * 1024**2:
            raise ValueError("Source package exceeds its code-only size limit.")
        contents = {}
        for item in members:
            if not owned_path(item.filename) or stat.S_IFMT(
                item.external_attr >> 16
            ) not in {0, stat.S_IFREG}:
                raise ValueError("Unsafe source member.")
            content = archive.read(item)
            if hashlib.sha256(content).hexdigest() != package["files"][item.filename]:
                raise ValueError("Source member checksum mismatch.")
            contents[item.filename] = content
    if not (checkout / ".git").exists():
        subprocess.run(
            ["git", "clone", "--no-checkout", str(repo_url), str(checkout)], check=True
        )
    found = subprocess.run(
        ["git", "cat-file", "-e", base + "^{commit}"], cwd=checkout, capture_output=True
    )
    if found.returncode:
        subprocess.run(["git", "fetch", "origin", base], cwd=checkout, check=True)
    for name in contents:
        baseline = subprocess.run(
            ["git", "cat-file", "-e", base + ":" + name],
            cwd=checkout,
            capture_output=True,
        )
        if baseline.returncode == 0:
            raise ValueError(
                "Source package cannot overwrite a public baseline file: " + name
            )
    subprocess.run(["git", "checkout", "--detach", base], cwd=checkout, check=True)
    actual = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=checkout, text=True
    ).strip()
    if actual != base:
        raise ValueError("Public baseline checkout mismatch.")
    previous = checkout / ".tas-source.json"
    if previous.exists():
        old = json.loads(previous.read_text())
        for name in set(old.get("files", {})) - set(contents):
            baseline = subprocess.run(
                ["git", "cat-file", "-e", base + ":" + name],
                cwd=checkout,
                capture_output=True,
            )
            if not owned_path(name) or baseline.returncode == 0:
                raise ValueError("Unsafe obsolete source path.")
            (checkout / name).unlink(missing_ok=True)

    def save(path, content):
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".installing")
        temporary.write_bytes(content)
        os.replace(temporary, path)

    for name, content in contents.items():
        save(checkout / name, content)
    receipt = {key: value for key, value in package.items() if key != "archive_b85"}
    receipt_bytes = (json.dumps(receipt, indent=2, sort_keys=True) + "\n").encode()
    save(checkout / ".tas-source.json", receipt_bytes)
    stored = (
        persistent
        / "code"
        / ("source-" + package["source_commit"] + "-" + package["sha256"][:16])
    )
    if (
        not stored.with_suffix(".zip").exists()
        or hashlib.sha256(stored.with_suffix(".zip").read_bytes()).hexdigest()
        != package["sha256"]
    ):
        save(stored.with_suffix(".zip"), data)
    save(stored.with_suffix(".json"), receipt_bytes)
    return receipt


## Connect storage and restore the exact code

Drive authorization may prompt once per runtime. No credentials are saved
in configuration or outputs. Source restoration is automatic and verifies
both the archive and every file before installation. It cannot replace any
file from the frozen public baseline. The same caches work in CPU/GPU sessions.


In [ ]:
try:
    import os, sys, json, subprocess, pathlib, shutil, time
    from IPython.display import display, Markdown, HTML
    from google.colab import drive
    drive.mount("/content/drive")
    persistent = pathlib.Path(PERSISTENT_ROOT)
    persistent.mkdir(parents=True, exist_ok=True)
    checkout = pathlib.Path("/content/tas-v7-checkout")
    if (SOURCE_PACKAGE["base_commit"], SOURCE_PACKAGE["source_commit"]) != (BASELINE_COMMIT, RESEARCH_COMMIT):
        raise ValueError("Regenerate the notebook to change its fixed code versions.")
    source_receipt = restore_source_package(SOURCE_PACKAGE, checkout, persistent, REPO_URL)
    actual_commit = source_receipt["source_commit"]
    os.chdir(checkout)
    if str(checkout) not in sys.path:
        sys.path.insert(0, str(checkout))
    # A new idea notebook may reuse this VM with different experiment modules.
    for module_name in list(sys.modules):
        if module_name == "tas_runtime" or module_name.startswith("tas_runtime."):
            del sys.modules[module_name]
    import importlib
    importlib.invalidate_caches()
    print("Public baseline:", BASELINE_COMMIT)
    print("Experiment Python source:", actual_commit, "| verified files:", len(source_receipt["files"]))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Isolated, pinned environment

Python 3.10.16 and matched PyTorch 2.4.1 / torchvision 0.19.1 CPU or CUDA
12.1 wheels run in subprocesses. The legacy Transformers 4.35.2 stack is
preserved. No notebook-kernel restart, system Torch replacement or CUDA
extension compilation is needed. GPU mode validates its actual CUDA path.


In [ ]:
try:
    query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
    has_gpu = query is not None and query.returncode == 0 and bool(query.stdout.strip())
    effective_phase = ("run" if has_gpu else "prepare") if PHASE == "auto" else PHASE
    mode = "gpu" if effective_phase == "run" else "cpu"
    print("Hardware:", query.stdout.strip() if has_gpu else "CPU")
    print("Phase:", effective_phase, "| environment:", mode)
    if effective_phase == "run" and has_gpu and REQUIRE_A100_80GB and ("A100" not in query.stdout or int(query.stdout.strip().splitlines()[0].split(",")[-1].strip().split()[0]) < 70000):
        raise RuntimeError("This configuration requires A100 80GB; hardware check stopped before environment/model setup.")
    if effective_phase == "run" and not has_gpu:
        raise RuntimeError("Select a GPU runtime, or use PHASE=prepare/score on CPU.")
    if effective_phase == "run" and not (persistent / "preparation.json").is_file():
        raise RuntimeError("Run this notebook once in a CPU runtime first. That downloads/persists models and data before A100 allocation.")
    from tas_runtime.bootstrap import ensure_environment, subprocess_environment
    python = ensure_environment(mode, "/content/tas-v7-envs", checkout,
                                diagnostics_dir=persistent / "diagnostics/environment")
    if DATASET == "amber":
        from tas_runtime.bootstrap import ensure_amber
        ensure_amber(python, "/content/tas-v7-envs", checkout)
    env = dict(subprocess_environment(), PYTHONHASHSEED="0", TOKENIZERS_PARALLELISM="false",
               NLTK_DATA=str(persistent / "nlp"), YOLO_CONFIG_DIR=str(pathlib.Path(LOCAL_ROOT) / "ultralytics"))
    configuration = {
        "persistent_root":PERSISTENT_ROOT, "local_root":LOCAL_ROOT, "stage":STAGE,
        "seeds":SEEDS, "methods":METHODS, "batch_sizes":BATCH_SIZES,
        "attention_backends":ATTENTION_BACKENDS, "last_position_logits":LAST_POSITION_LOGITS,
        "max_session_minutes":MAX_SESSION_MINUTES, "require_a100":REQUIRE_A100_80GB,
        "run_confirmation":RUN_CONFIRMATION, "run_controls":RUN_CONTROLS,
        "run_diagnostics":RUN_DIAGNOSTICS, "run_final":RUN_FINAL_REMAINDER,
        "reserve_pope":RESERVE_POPE_IMAGES, "split_seed":SPLIT_SEED,
        "coverage_tolerance":COVERAGE_TOLERANCE_PP / 100.0,
        "dataset":DATASET, "external_manifest":EXTERNAL_MANIFEST, "external_image_root":EXTERNAL_IMAGE_ROOT,
        "auto_continue_after_smoke":True, "source_revision":actual_commit,
        "generation":{"max_new_tokens":180,"temperature":1.0,"top_p":1.0,"top_k":None,
                      "precision":"fp16","do_sample":True,"conv_mode":"llava_v1","prompt":"Describe this image."}
    }
    config_path = pathlib.Path("/content/tas-v7-config.json")
    config_path.write_text(json.dumps(configuration, indent=2))
    def run_phase(phase):
        subprocess.run([str(python), "-m", "experiments.tas_v7.cli", "--config",str(config_path),"--phase",phase], cwd=checkout,env=env,check=True)
    if DATASET == "amber":
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp"), "--amber"], cwd=checkout, env=env, check=True)
    run_phase("plan")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Prepare or run the study

First CPU session: approximately 17–18 GB of checkpoints plus annotations,
selected images and NLP resources; existing verified files are reused.
Only 30 train images are fetched, not all COCO train2017.

GPU sessions: lightweight preflight → local asset restoration → one model
load → baseline/view audit → correctness/throughput pilot → configured
stages. Per-image RNG makes retries/resume independent of job order. A
rejected optimization is recorded; baseline failures block larger runs.
Results use immutable chunks and perception caches use shards on Drive.


In [ ]:
try:
    phase_started = time.perf_counter()
    if effective_phase == "prepare":
        # Small NLP assets enable the offline scientific fixtures first.
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp")], cwd=checkout, env=env, check=True)
        run_phase("check")
        run_phase("prepare")
        run_phase("score")
        display(Markdown("**CPU preparation complete.** Files are persistent. Switch to the A100 runtime and Run all again."))
    elif effective_phase == "run":
        run_phase("check")
        run_phase("run")
    elif effective_phase == "score":
        run_phase("score")
    elif effective_phase == "check":
        run_phase("check")
    else:
        raise ValueError("Unknown phase")
    print("Session phase seconds:", round(time.perf_counter() - phase_started, 1))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Inspect saved reports

LEGACY and HARMONIZED are separate protocols; do not infer a matched
algorithm gain across them. CHAIR and annotation category recall do not
establish caption quality or preserved relations. Inspect ambiguous images,
captions, failures, coverage, length, timings and uncertainty together.
Reports include descriptive comparisons for paired captions with exactly
equal word counts. Batch latency and amortized seconds/image are separate.
CPU scoring/report regeneration is available with PHASE="score".


In [ ]:
try:
    import csv, html
    from IPython.display import Image as DisplayImage
    runs_root = persistent / "runs"
    for run_dir in sorted(runs_root.glob("*"))[-3:]:
        display(Markdown("### Run `" + run_dir.name + "`"))
        session = run_dir / "session.json"
        if session.exists():
            display(Markdown("```json\n" + session.read_text() + "\n```"))
        gallery = run_dir / "views.png"
        if gallery.exists():
            display(DisplayImage(filename=str(gallery)))
        for summary in sorted(run_dir.glob("*/summary.csv")):
            rows = list(csv.reader(summary.open()))
            table = "<table>" + "".join("<tr>" + "".join("<td>" + html.escape(value) + "</td>" for value in row) + "</tr>" for row in rows) + "</table>"
            display(Markdown("**" + summary.parent.name + "**"))
            display(HTML(table))
    print("Persistent artifacts:", persistent)
    print("Reuse this folder for later sessions/new idea notebooks. Do not delete frozen manifests or rewrite completed runs.")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Release the session

With `AUTO_SHUTDOWN=True`, the final cell flushes Drive writes and releases
the runtime after preparation, evaluation or a planned budget stop. Errors
also invoke this cleanup before propagating. Reports remain on Drive.
Set it to False only when keeping the runtime for inspection.


In [ ]:
release_runtime()
